In [1]:
from pathlib import Path
from collections import Counter
import numpy as np
from IPython.display import display, Markdown
from ai_edge_litert.interpreter import Interpreter, OpResolverType
from transformers import AutoModelForCausalLM

import hashlib
import torch

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_DIR = Path(
    "/home/randy/Documents/ACompArc/model/project"
)

BLOCK_DIR = PROJECT_DIR / (
    "artifacts/models/tinystories_3m_blocks_w8a8/"
    "20261011T134954617091Z"
)

BLOCK0_PATH = BLOCK_DIR / "block0_w8a8.tflite"
FP32_DIR = PROJECT_DIR / "artifacts/models/tinystories_3m_fp32"

BLOCK_EXPORT_DIR = Path(
    "/home/randy/Documents/ACompArc/model/project/artifacts/models/"
    "tinystories_3m_blocks_w8a8/20261011T134954617091Z"
)


In [3]:

block0_interpreter = Interpreter(
    model_path=str(BLOCK_EXPORT_DIR / "block0_w8a8.tflite"),
    experimental_preserve_all_tensors=True,
    experimental_op_resolver_type=OpResolverType.BUILTIN_REF,
)
block0_interpreter.allocate_tensors()

block0_tensors = {
    int(t["index"]): t
    for t in block0_interpreter.get_tensor_details()
}
block0_ops = block0_interpreter._get_ops_details()

shape_ops = {
    "RESHAPE", "TRANSPOSE", "SQUEEZE", "EXPAND_DIMS",
    "SLICE", "STRIDED_SLICE", "GATHER", "GATHER_ND",
    "SPLIT", "SPLIT_V", "PACK", "UNPACK",
    "CONCATENATION", "PAD", "PADV2", "TILE", "SHAPE",
}

def valid_tensors(indices):
    # Negative indices indicate absent optional inputs, such as bias.
    return [
        block0_tensors[int(index)]
        for index in indices
        if int(index) >= 0
    ]

def tensor_labels(tensors):
    return ", ".join(
        f"T{t['index']} {np.dtype(t['dtype']).name}"
        for t in tensors
    ) or "—"

def classify_operation(op):
    name = op["op_name"]
    tensors = (
        valid_tensors(op["inputs"])
        + valid_tensors(op["outputs"])
    )

    if name == "QUANTIZE":
        return "Quantize / requantize"
    if name == "DEQUANTIZE":
        return "Dequantize"
    if name in shape_ops:
        return "Data movement / shape"
    if any(np.issubdtype(t["dtype"], np.floating) for t in tensors):
        return "Floating-point / mixed"
    if any(t["dtype"] == np.int8 for t in tensors):
        return "INT8 computation"
    return "Other integer / boolean"

block0_op_status = [
    classify_operation(op) for op in block0_ops
]

print("Block 0 operation count:", len(block0_ops))
print("\nOperation summary:")
for status, count in Counter(block0_op_status).items():
    print(f"  {status}: {count}")

rows = [
    "| Op | Operation | Inputs | Outputs | Classification |",
    "|---:|---|---|---|---|",
]

for op, status in zip(block0_ops, block0_op_status):
    rows.append(
        f"| {op['index']} | {op['op_name']} | "
        f"{tensor_labels(valid_tensors(op['inputs']))} | "
        f"{tensor_labels(valid_tensors(op['outputs']))} | "
        f"{status} |"
    )

display(Markdown("\n".join(rows)))

Block 0 operation count: 74

Operation summary:
  Floating-point / mixed: 31
  Data movement / shape: 20
  Quantize / requantize: 5
  INT8 computation: 11
  Dequantize: 7


| Op | Operation | Inputs | Outputs | Classification |
|---:|---|---|---|---|
| 0 | MEAN | T0 float32, T10 int32 | T31 float32 | Floating-point / mixed |
| 1 | RESHAPE | T31 float32, T11 int32 | T32 float32 | Data movement / shape |
| 2 | SQUARED_DIFFERENCE | T0 float32, T32 float32 | T33 float32 | Floating-point / mixed |
| 3 | MEAN | T33 float32, T10 int32 | T34 float32 | Floating-point / mixed |
| 4 | ADD | T34 float32, T2 float32 | T35 float32 | Floating-point / mixed |
| 5 | RSQRT | T35 float32 | T36 float32 | Floating-point / mixed |
| 6 | MUL | T36 float32, T31 float32 | T37 float32 | Floating-point / mixed |
| 7 | SUB | T5 float32, T37 float32 | T38 float32 | Floating-point / mixed |
| 8 | RESHAPE | T36 float32, T11 int32 | T39 float32 | Data movement / shape |
| 9 | MUL | T0 float32, T39 float32 | T40 float32 | Floating-point / mixed |
| 10 | RESHAPE | T38 float32, T11 int32 | T41 float32 | Data movement / shape |
| 11 | ADD | T40 float32, T41 float32 | T42 float32 | Floating-point / mixed |
| 12 | MUL | T42 float32, T6 float32 | T43 float32 | Floating-point / mixed |
| 13 | ADD | T43 float32, T7 float32 | T44 float32 | Floating-point / mixed |
| 14 | QUANTIZE | T44 float32 | T45 int8 | Quantize / requantize |
| 15 | FULLY_CONNECTED | T45 int8, T27 int8 | T46 int8 | INT8 computation |
| 16 | FULLY_CONNECTED | T45 int8, T25 int8 | T47 int8 | INT8 computation |
| 17 | FULLY_CONNECTED | T45 int8, T26 int8 | T48 int8 | INT8 computation |
| 18 | RESHAPE | T46 int8, T13 int32 | T49 int8 | Data movement / shape |
| 19 | TRANSPOSE | T49 int8, T14 int32 | T50 int8 | Data movement / shape |
| 20 | RESHAPE | T47 int8, T13 int32 | T51 int8 | Data movement / shape |
| 21 | RESHAPE | T48 int8, T13 int32 | T52 int8 | Data movement / shape |
| 22 | TRANSPOSE | T52 int8, T14 int32 | T53 int8 | Data movement / shape |
| 23 | TRANSPOSE | T51 int8, T1 int32 | T54 int8 | Data movement / shape |
| 24 | RESHAPE | T50 int8, T15 int32 | T55 int8 | Data movement / shape |
| 25 | DEQUANTIZE | T55 int8 | T56 float32 | Dequantize |
| 26 | RESHAPE | T54 int8, T16 int32 | T57 int8 | Data movement / shape |
| 27 | DEQUANTIZE | T57 int8 | T58 float32 | Dequantize |
| 28 | BATCH_MATMUL | T56 float32, T58 float32 | T59 float32 | Floating-point / mixed |
| 29 | RESHAPE | T59 float32, T17 int32 | T60 float32 | Data movement / shape |
| 30 | SELECT_V2 | T21 bool, T60 float32, T18 float32 | T61 float32 | Floating-point / mixed |
| 31 | SOFTMAX | T61 float32 | T62 float32 | Floating-point / mixed |
| 32 | RESHAPE | T62 float32, T19 int32 | T63 float32 | Data movement / shape |
| 33 | RESHAPE | T53 int8, T15 int32 | T64 int8 | Data movement / shape |
| 34 | DEQUANTIZE | T64 int8 | T65 float32 | Dequantize |
| 35 | BATCH_MATMUL | T63 float32, T65 float32 | T66 float32 | Floating-point / mixed |
| 36 | QUANTIZE | T66 float32 | T67 int8 | Quantize / requantize |
| 37 | RESHAPE | T67 int8, T20 int32 | T68 int8 | Data movement / shape |
| 38 | TRANSPOSE | T68 int8, T14 int32 | T69 int8 | Data movement / shape |
| 39 | RESHAPE | T69 int8, T12 int32 | T70 int8 | Data movement / shape |
| 40 | FULLY_CONNECTED | T70 int8, T28 int8, T24 int32 | T71 int8 | INT8 computation |
| 41 | DEQUANTIZE | T71 int8 | T72 float32 | Dequantize |
| 42 | ADD | T72 float32, T0 float32 | T73 float32 | Floating-point / mixed |
| 43 | MEAN | T73 float32, T10 int32 | T74 float32 | Floating-point / mixed |
| 44 | RESHAPE | T74 float32, T11 int32 | T75 float32 | Data movement / shape |
| 45 | SQUARED_DIFFERENCE | T73 float32, T75 float32 | T76 float32 | Floating-point / mixed |
| 46 | MEAN | T76 float32, T10 int32 | T77 float32 | Floating-point / mixed |
| 47 | ADD | T77 float32, T2 float32 | T78 float32 | Floating-point / mixed |
| 48 | RSQRT | T78 float32 | T79 float32 | Floating-point / mixed |
| 49 | MUL | T79 float32, T74 float32 | T80 float32 | Floating-point / mixed |
| 50 | SUB | T5 float32, T80 float32 | T81 float32 | Floating-point / mixed |
| 51 | RESHAPE | T79 float32, T11 int32 | T82 float32 | Data movement / shape |
| 52 | MUL | T73 float32, T82 float32 | T83 float32 | Floating-point / mixed |
| 53 | RESHAPE | T81 float32, T11 int32 | T84 float32 | Data movement / shape |
| 54 | ADD | T83 float32, T84 float32 | T85 float32 | Floating-point / mixed |
| 55 | MUL | T85 float32, T8 float32 | T86 float32 | Floating-point / mixed |
| 56 | ADD | T86 float32, T9 float32 | T87 float32 | Floating-point / mixed |
| 57 | QUANTIZE | T87 float32 | T88 int8 | Quantize / requantize |
| 58 | FULLY_CONNECTED | T88 int8, T29 int8, T23 int32 | T89 int8 | INT8 computation |
| 59 | DEQUANTIZE | T89 int8 | T90 float32 | Dequantize |
| 60 | MUL | T89 int8, T91 int8 | T92 int8 | INT8 computation |
| 61 | POW | T90 float32, T3 float32 | T93 float32 | Floating-point / mixed |
| 62 | QUANTIZE | T93 float32 | T94 int8 | Quantize / requantize |
| 63 | MUL | T94 int8, T95 int8 | T96 int8 | INT8 computation |
| 64 | ADD | T89 int8, T96 int8 | T97 int8 | INT8 computation |
| 65 | MUL | T97 int8, T98 int8 | T99 int8 | INT8 computation |
| 66 | DEQUANTIZE | T99 int8 | T100 float32 | Dequantize |
| 67 | TANH | T100 float32 | T101 float32 | Floating-point / mixed |
| 68 | ADD | T101 float32, T4 float32 | T102 float32 | Floating-point / mixed |
| 69 | QUANTIZE | T102 float32 | T103 int8 | Quantize / requantize |
| 70 | MUL | T92 int8, T103 int8 | T104 int8 | INT8 computation |
| 71 | FULLY_CONNECTED | T104 int8, T30 int8, T22 int32 | T105 int8 | INT8 computation |
| 72 | DEQUANTIZE | T105 int8 | T106 float32 | Dequantize |
| 73 | ADD | T73 float32, T106 float32 | T107 float32 | Floating-point / mixed |

# PART DIVISION

## Block-0 implementation plan

Divide block 0 into seven parts following the exported graph. Each part
gets its own Python implementation, comparison results, and Markdown summary.

| Part | Operations | What we implement and check |
|---|---|---|
| **1. First LayerNorm** | Op 0–14 | Mean, variance, epsilon, integer reciprocal square root, learned scale/bias, and conversion to **T45 (INT8 projection input)**. |
| **2. Q, K, V projections** | Op 15–17 | Existing INT8 fully connected layers: MAC accumulation and requantization. |
| **3. Attention** | Op 18–39 | Head layouts, integer $QK^{\mathsf{T}}$, causal mask, integer softmax, probability–value multiplication, and head merging. |
| **4. Attention output and residual** | Op 40–42 | Output projection and scale-aligned residual addition, producing **T73 (first residual output)**. |
| **5. Second LayerNorm** | Op 43–57 | Reuse the LayerNorm algorithm with independently checked parameters and ranges, producing **T88 (INT8 MLP input)**. |
| **6. MLP expansion and GELU** | Op 58–70 | INT8 128→512 projection, cubic arithmetic, integer `tanh`, and GELU output. |
| **7. MLP output and residual** | Op 71–73 | INT8 512→128 projection and final residual addition, producing **T107 (block-0 output)**. |

### LayerNorm

In [4]:

# Read the checkpoint that produced our saved benchmark inputs.
inspection_model = AutoModelForCausalLM.from_pretrained(
    FP32_DIR,
    local_files_only=True,
    dtype=torch.float32,
).cpu().eval()

with np.load(BLOCK_DIR / "validation_inputs.npz") as saved:
    benchmark_ids = saved["benchmark_input_ids"].copy()
    benchmark_x = saved["block0_input_fp32"].copy()
    calibration_ids = saved["calibration_input_ids"].copy()

def embedding_input(ids):
    ids = torch.as_tensor(ids, dtype=torch.long).reshape(1, 32)
    positions = torch.arange(32, dtype=torch.long).unsqueeze(0)

    with torch.no_grad():
        x = inspection_model.transformer.drop(
            inspection_model.transformer.wte(ids)
            + inspection_model.transformer.wpe(positions)
        )

    return x.numpy().copy()

# Confirm the saved checkpoint reproduces the benchmark input.
np.testing.assert_allclose(
    embedding_input(benchmark_ids),
    benchmark_x,
    atol=1e-6,
    rtol=1e-6,
)

interpreter = Interpreter(
    model_path=str(BLOCK0_PATH),
    experimental_preserve_all_tensors=True,
    experimental_op_resolver_type=OpResolverType.BUILTIN_REF,
)
interpreter.allocate_tensors()

op0 = interpreter._get_ops_details()[0]
assert op0["op_name"] == "MEAN"

x_index = int(op0["inputs"][0])
axes_index = int(op0["inputs"][1])
mean_index = int(op0["outputs"][0])
axes = tuple(
    int(axis) for axis in interpreter.get_tensor(axes_index).reshape(-1)
)

# Retain these cases in memory for the next integer-reference cell.
op0_cases = []
samples = [("benchmark", benchmark_ids, benchmark_x)]

for i, ids in enumerate(calibration_ids):
    samples.append((f"calibration_{i}", ids, embedding_input(ids)))

for name, ids, x in samples:
    assert x.shape == (1, 32, 128)
    assert np.isfinite(x).all()

    interpreter.set_tensor(x_index, x)
    interpreter.invoke()

    mean_ref = interpreter.get_tensor(mean_index).copy()
    mean_python = np.mean(
        x, axis=axes, dtype=np.float32
    ).reshape(mean_ref.shape)

    op0_cases.append({
        "name": name,
        "input_ids": np.asarray(ids).reshape(1, 32).copy(),
        "x_fp32": x.copy(),
        "mean_ref_fp32": mean_ref,
        "mean_python_fp32": mean_python,
    })

def print_range(label, cases):
    x = np.concatenate([c["x_fp32"].reshape(-1) for c in cases])
    means = np.concatenate([
        c["mean_ref_fp32"].reshape(-1) for c in cases
    ])
    max_error = max(
        float(np.max(np.abs(
            c["mean_ref_fp32"] - c["mean_python_fp32"]
        )))
        for c in cases
    )

    print(f"\n{label}")
    print("  Sequences:", len(cases))
    print("  Input values:", x.size)
    print(f"  T0 minimum: {x.min():.9f}")
    print(f"  T0 maximum: {x.max():.9f}")
    print(f"  T0 maximum absolute value: {np.abs(x).max():.9f}")
    print(f"  Mean minimum: {means.min():.9f}")
    print(f"  Mean maximum: {means.max():.9f}")
    print(f"  Python vs LiteRT mean max error: {max_error:.9e}")

op0_source_sha256 = hashlib.sha256(BLOCK0_PATH.read_bytes()).hexdigest()

print("Source:", BLOCK0_PATH)
print("SHA-256:", op0_source_sha256)
print(f"Op 0: MEAN(T{x_index}, T{axes_index}) -> T{mean_index}")
print("Reduction axes:", axes)
print("Input shape:", benchmark_x.shape)
print("Output shape:", op0_cases[0]["mean_ref_fp32"].shape)

print_range("Benchmark input", op0_cases[:1])
print_range("Calibration inputs", op0_cases[1:])

print("\nFirst token, benchmark:")
print("  First 8 T0 features:", benchmark_x[0, 0, :8])
print("  Reference mean:",
      op0_cases[0]["mean_ref_fp32"].reshape(-1)[0])

W1015 08:46:39.153000 7431 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1015 08:46:39.181000 7431 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 7989.71it/s]

Source: /home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_blocks_w8a8/20261011T134954617091Z/block0_w8a8.tflite
SHA-256: 06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9
Op 0: MEAN(T0, T10) -> T31
Reduction axes: (0, 2)
Input shape: (1, 32, 128)
Output shape: (32,)

Benchmark input
  Sequences: 1
  Input values: 4096
  T0 minimum: -0.328789324
  T0 maximum: 0.307286024
  T0 maximum absolute value: 0.328789324
  Mean minimum: -0.009422798
  Mean maximum: 0.004446551
  Python vs LiteRT mean max error: 3.492459655e-09

Calibration inputs
  Sequences: 7
  Input values: 28672
  T0 minimum: -0.469448209
  T0 maximum: 0.455547750
  T0 maximum absolute value: 0.469448209
  Mean minimum: -0.009570063
  Mean maximum: 0.007038387
  Python vs LiteRT mean max error: 6.053596735e-09

First token, benchmark:
  First 8 T0 features: [-0.08634469 -0.03653849  0.14335527  0.0147367   0.02153916 -0.04537516
 -0.10777003 -0.2002483 ]
  Reference mean: -0.000785323

In [5]:
T0_FRAC_BITS = 8
T0_SCALE = 2.0 ** (-T0_FRAC_BITS)

FEATURE_COUNT = 128
T31_SCALE = T0_SCALE / FEATURE_COUNT  # 1 / 32768

# Keep trial results in memory.
op0_integer_trial = []

for case in op0_cases:
    x = case["x_fp32"]  # T0 (block input)
    mean_ref = case["mean_ref_fp32"].reshape(32)  # T31 (FP32 means)

    # Quantize T0: nearest integer, ties to even.
    rounded = np.rint(x.astype(np.float64) / T0_SCALE)
    clipped_count = int(np.count_nonzero(
        (rounded < -128) | (rounded > 127)
    ))
    t0_q = np.clip(rounded, -128, 127).astype(np.int8)

    # Integer Op 0:
    # Store the sum as T31 with scale T0_SCALE / 128.
    t31_mean_q = np.sum(
        t0_q.astype(np.int32),
        axis=-1,
        dtype=np.int32,
    ).reshape(32)

    # Reconstruct represented values for accuracy comparisons only.
    t0_approx = t0_q.astype(np.float64) * T0_SCALE
    t31_approx = t31_mean_q.astype(np.float64) * T31_SCALE

    op0_integer_trial.append({
        "name": case["name"],
        "t0_q": t0_q,
        "t31_mean_q": t31_mean_q,
        "t31_approx": t31_approx,
        "clipped_count": clipped_count,
        "input_error": np.abs(t0_approx - x),
        "mean_error": np.abs(t31_approx - mean_ref),
    })

def report_trial(label, trials):
    input_errors = np.concatenate([
        t["input_error"].reshape(-1) for t in trials
    ])
    mean_errors = np.concatenate([
        t["mean_error"] for t in trials
    ])
    q_values = np.concatenate([
        t["t0_q"].reshape(-1) for t in trials
    ])
    mean_q_values = np.concatenate([
        t["t31_mean_q"] for t in trials
    ])

    print(f"\n{label}")
    print("  T0 (input) clipped values:",
          sum(t["clipped_count"] for t in trials))
    print("  T0 (input) integer range:",
          int(q_values.min()), "to", int(q_values.max()))
    print(f"  T0 (input) max absolute error: {input_errors.max():.9e}")
    print(f"  T0 (input) mean absolute error: {input_errors.mean():.9e}")
    print("  T31 (means) integer range:",
          int(mean_q_values.min()), "to", int(mean_q_values.max()))
    print(f"  T31 (means) max absolute error: {mean_errors.max():.9e}")
    print(f"  T31 (means) mean absolute error: {mean_errors.mean():.9e}")

print("Candidate format:")
print("  T0 (input): INT8, scale =", T0_SCALE, ", zero point = 0")
print("  T31 (means): INT32, scale =", T31_SCALE, ", zero point = 0")

report_trial("Benchmark", op0_integer_trial[:1])
report_trial("Calibration", op0_integer_trial[1:])

first = op0_integer_trial[0]
first_ref = op0_cases[0]["mean_ref_fp32"].reshape(32)[0]

print("\nFirst benchmark token:")
print("  T0 (input), first 8 integers:", first["t0_q"][0, 0, :8])
print("  T31 (mean), stored integer:", first["t31_mean_q"][0])
print(f"  T31 (mean), represented value: {first['t31_approx'][0]:.9f}")
print(f"  T31 (mean), FP32 reference: {first_ref:.9f}")
print(f"  T31 (mean), absolute error: {first['mean_error'][0]:.9e}")

Candidate format:
  T0 (input): INT8, scale = 0.00390625 , zero point = 0
  T31 (means): INT32, scale = 3.0517578125e-05 , zero point = 0

Benchmark
  T0 (input) clipped values: 0
  T0 (input) integer range: -84 to 79
  T0 (input) max absolute error: 1.952394843e-03
  T0 (input) mean absolute error: 9.877213568e-04
  T31 (means) integer range: -306 to 144
  T31 (means) max absolute error: 2.882166300e-04
  T31 (means) mean absolute error: 4.977694425e-05

Calibration
  T0 (input) clipped values: 0
  T0 (input) integer range: -120 to 117
  T0 (input) max absolute error: 1.953020692e-03
  T0 (input) mean absolute error: 9.742461435e-04
  T31 (means) integer range: -317 to 232
  T31 (means) max absolute error: 2.882166300e-04
  T31 (means) mean absolute error: 7.735267893e-05

First benchmark token:
  T0 (input), first 8 integers: [-22  -9  37   4   6 -12 -28 -51]
  T31 (mean), stored integer: -25
  T31 (mean), represented value: -0.000762939
  T31 (mean), FP32 reference: -0.000785323
  T

In [6]:
# Explicit format definitions.
T32_FRAC_BITS = T0_FRAC_BITS + 7  # 8 + log2(128) = 15
T32_SCALE = 2.0 ** (-T32_FRAC_BITS)

T33_FINE_FRAC_BITS = 2 * T32_FRAC_BITS  # Squaring: 30
T33_FINE_SCALE = 2.0 ** (-T33_FINE_FRAC_BITS)

# Candidate coarser format; not a final choice.
T33_COARSE_FRAC_BITS = 15
T33_COARSE_SCALE = 2.0 ** (-T33_COARSE_FRAC_BITS)

ALIGN_SHIFT = T32_FRAC_BITS - T0_FRAC_BITS       # 7
REQUANT_SHIFT = T33_FINE_FRAC_BITS - T33_COARSE_FRAC_BITS  # 15

np.testing.assert_equal(T32_SCALE, T31_SCALE)

# Identify T33 (squared differences) in the reference graph.
op2 = interpreter._get_ops_details()[2]
assert op2["op_name"] == "SQUARED_DIFFERENCE"
t33_index = int(op2["outputs"][0])

op2_integer_trial = []

for case, mean_trial in zip(op0_cases, op0_integer_trial):
    assert case["name"] == mean_trial["name"]

    # Read T33 (FP32 squared differences) from the saved TFLite.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    t33_ref = interpreter.get_tensor(t33_index).copy()

    # Op 1: T31 (means) -> T32 (broadcast means).
    # Same integers and scale; only the shape changes.
    t32_q = mean_trial["t31_mean_q"].astype(np.int64).reshape(1, 32, 1)

    # Op 2: align T0 (input) to T32's scale.
    # INT64 temporaries ensure multiplication/squaring is performed widely.
    t0_aligned_q = (
        mean_trial["t0_q"].astype(np.int64) * (1 << ALIGN_SHIFT)
    )
    difference_q = t0_aligned_q - t32_q
    t33_fine_q = difference_q * difference_q

    # Requantize nonnegative squared values.
    # Round to nearest; halfway cases round upward.
    rounding_offset = 1 << (REQUANT_SHIFT - 1)
    t33_coarse_q = (
        t33_fine_q + rounding_offset
    ) >> REQUANT_SHIFT

    assert t33_coarse_q.max() <= np.iinfo(np.int32).max
    t33_coarse_q = t33_coarse_q.astype(np.int32)

    # Represented real values: diagnostics only.
    fine_approx = t33_fine_q.astype(np.float64) * T33_FINE_SCALE
    coarse_approx = t33_coarse_q.astype(np.float64) * T33_COARSE_SCALE

    op2_integer_trial.append({
        "name": case["name"],
        "t32_q": t32_q,
        "t0_aligned_q": t0_aligned_q,
        "difference_q": difference_q,
        "t33_ref": t33_ref,
        "t33_fine_q": t33_fine_q,
        "t33_coarse_q": t33_coarse_q,
        "fine_approx": fine_approx,
        "coarse_approx": coarse_approx,
        "fine_error": np.abs(fine_approx - t33_ref),
        "coarse_error": np.abs(coarse_approx - t33_ref),
        "requant_error": np.abs(coarse_approx - fine_approx),
    })

def report_squared_trial(label, trials):
    print(f"\n{label}")

    for key, description in [
        ("fine_error", "T33 at scale 2^-30 vs FP32"),
        ("coarse_error", "T33 at scale 2^-15 vs FP32"),
        ("requant_error", "Additional change from requantization"),
    ]:
        errors = np.concatenate([t[key].reshape(-1) for t in trials])
        print(f"  {description}:")
        print(f"    Max absolute error:  {errors.max():.9e}")
        print(f"    Mean absolute error: {errors.mean():.9e}")

    for key, description in [
        ("t33_fine_q", "T33 (fine squared integers)"),
        ("t33_coarse_q", "T33 (coarse squared integers)"),
    ]:
        largest = max(int(t[key].max()) for t in trials)

        # Range check for a later sum over 128 features.
        largest_sum = max(
            int(np.sum(t[key], axis=-1, dtype=np.int64).max())
            for t in trials
        )
        print(f"  {description}: max = {largest}")
        print(f"    Largest 128-feature sum: {largest_sum}")
        print(f"    Sum fits INT32: {largest_sum <= np.iinfo(np.int32).max}")

print("T32 (broadcast means): scale =", T32_SCALE)
print("T33 (fine squared differences): scale =", T33_FINE_SCALE)
print("T33 (coarse squared differences): scale =", T33_COARSE_SCALE)

report_squared_trial("Benchmark", op2_integer_trial[:1])
report_squared_trial("Calibration", op2_integer_trial[1:])

first = op2_integer_trial[0]
position = (0, 0, 0)

print("\nFirst token, first feature:")
print("  T0 (aligned input) integer:", first["t0_aligned_q"][position])
print("  T32 (broadcast mean) integer:", first["t32_q"][position])
print("  Centered difference integer:", first["difference_q"][position])
print("  T33 (fine squared difference) integer:", first["t33_fine_q"][position])
print("  T33 (coarse squared difference) integer:", first["t33_coarse_q"][position])
print(f"  T33 (fine represented value): {first['fine_approx'][position]:.9f}")
print(f"  T33 (coarse represented value): {first['coarse_approx'][position]:.9f}")
print(f"  T33 (FP32 reference): {first['t33_ref'][position]:.9f}")

T32 (broadcast means): scale = 3.0517578125e-05
T33 (fine squared differences): scale = 9.313225746154785e-10
T33 (coarse squared differences): scale = 3.0517578125e-05

Benchmark
  T33 at scale 2^-30 vs FP32:
    Max absolute error:  1.116245985e-03
    Mean absolute error: 1.534825375e-04
  T33 at scale 2^-15 vs FP32:
    Max absolute error:  1.101002097e-03
    Mean absolute error: 1.537359188e-04
  Additional change from requantization:
    Max absolute error:  1.525506377e-05
    Mean absolute error: 7.570248272e-06
  T33 (fine squared integers): max = 118330884
    Largest 128-feature sum: 2239203200
    Sum fits INT32: False
  T33 (coarse squared integers): max = 3611
    Largest 128-feature sum: 68342
    Sum fits INT32: True

Calibration
  T33 at scale 2^-30 vs FP32:
    Max absolute error:  1.424428076e-03
    Mean absolute error: 1.494486447e-04
  T33 at scale 2^-15 vs FP32:
    Max absolute error:  1.412481070e-03
    Mean absolute error: 1.497123103e-04
  Additional change

In [7]:
# Op 3: MEAN
# T33 (squared differences) -> T34 (per-token variance)

op3 = interpreter._get_ops_details()[3]
assert op3["op_name"] == "MEAN"
assert int(op3["inputs"][0]) == t33_index

t34_index = int(op3["outputs"][0])
variance_axes = tuple(
    int(v)
    for v in interpreter.get_tensor(int(op3["inputs"][1])).reshape(-1)
)
assert variance_axes == (0, 2)

# Division by 128 is represented through the output scale.
T34_FINE_FRAC_BITS = T33_FINE_FRAC_BITS + 7
T34_COARSE_FRAC_BITS = T33_COARSE_FRAC_BITS + 7

T34_FINE_SCALE = T33_FINE_SCALE / FEATURE_COUNT
T34_COARSE_SCALE = T33_COARSE_SCALE / FEATURE_COUNT

int32_max = int(np.iinfo(np.int32).max)
op3_integer_trial = []

assert len(op0_cases) == len(op2_integer_trial)

for case, squared_trial in zip(op0_cases, op2_integer_trial):
    assert case["name"] == squared_trial["name"]

    # Reference: run the original mixed-precision TFLite graph.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    t34_ref = interpreter.get_tensor(t34_index).copy().reshape(-1)

    # Fine path: sum T33 at scale 2^-30 using INT64.
    t34_fine_q = np.sum(
        squared_trial["t33_fine_q"],
        axis=variance_axes,
        dtype=np.int64,
    )

    # Check the coarse sum in INT64 before using INT32.
    coarse_sum_check = np.sum(
        squared_trial["t33_coarse_q"],
        axis=variance_axes,
        dtype=np.int64,
    )
    assert coarse_sum_check.min() >= 0
    assert coarse_sum_check.max() <= int32_max

    # Coarse path: sum T33 at scale 2^-15 using INT32.
    t34_coarse_q = np.sum(
        squared_trial["t33_coarse_q"],
        axis=variance_axes,
        dtype=np.int32,
    )

    # Floating-point reconstruction is only for diagnostics.
    fine_approx = t34_fine_q.astype(np.float64) * T34_FINE_SCALE
    coarse_approx = t34_coarse_q.astype(np.float64) * T34_COARSE_SCALE

    op3_integer_trial.append({
        "name": case["name"],
        "t34_ref": t34_ref,
        "t34_fine_q": t34_fine_q,
        "t34_coarse_q": t34_coarse_q,
        "fine_approx": fine_approx,
        "coarse_approx": coarse_approx,
        "fine_error": np.abs(fine_approx - t34_ref),
        "coarse_error": np.abs(coarse_approx - t34_ref),
        "requant_error": np.abs(coarse_approx - fine_approx),
    })


def report_variance_trial(label, trials):
    print(f"\n{label}")

    reference = np.concatenate([t["t34_ref"] for t in trials])
    print(
        "  T34 (FP32 variance) range:",
        f"{reference.min():.9e} to {reference.max():.9e}",
    )

    for key, description in [
        ("fine_error", "Fine variance vs FP32"),
        ("coarse_error", "Coarse variance vs FP32"),
        ("requant_error", "Variance change from T33 requantization"),
    ]:
        errors = np.concatenate([t[key] for t in trials])
        print(f"  {description}:")
        print(f"    Max absolute error:  {errors.max():.9e}")
        print(f"    Mean absolute error: {errors.mean():.9e}")

    for key, description in [
        ("t34_fine_q", "T34 (fine variance), INT64"),
        ("t34_coarse_q", "T34 (coarse variance), INT32"),
    ]:
        values = np.concatenate([t[key] for t in trials])
        print(f"  {description}: {values.min()} to {values.max()}")


print(
    "T34 (fine variance): INT64,",
    f"fractional bits = {T34_FINE_FRAC_BITS}, scale = {T34_FINE_SCALE}",
)
print(
    "T34 (coarse variance): INT32,",
    f"fractional bits = {T34_COARSE_FRAC_BITS}, scale = {T34_COARSE_SCALE}",
)

report_variance_trial("Benchmark", op3_integer_trial[:1])
report_variance_trial("Calibration", op3_integer_trial[1:])

first = op3_integer_trial[0]
print("\nFirst benchmark token:")
print("  T34 (fine variance), stored integer:", first["t34_fine_q"][0])
print("  T34 (coarse variance), stored integer:", first["t34_coarse_q"][0])
print("  T34 (fine represented value):", f'{first["fine_approx"][0]:.9f}')
print("  T34 (coarse represented value):", f'{first["coarse_approx"][0]:.9f}')
print("  T34 (FP32 reference):", f'{first["t34_ref"][0]:.9f}')

T34 (fine variance): INT64, fractional bits = 37, scale = 7.275957614183426e-12
T34 (coarse variance): INT32, fractional bits = 22, scale = 2.384185791015625e-07

Benchmark
  T34 (FP32 variance) range: 6.802664604e-03 to 1.629877090e-02
  Fine variance vs FP32:
    Max absolute error:  4.531163722e-05
    Mean absolute error: 1.453356526e-05
  Coarse variance vs FP32:
    Max absolute error:  4.650373012e-05
    Mean absolute error: 1.458609768e-05
  Variance change from T33 requantization:
    Max absolute error:  3.814697266e-06
    Mean absolute error: 8.474162314e-07
  T34 (fine variance), INT64: 936912768 to 2239203200
  T34 (coarse variance), INT32: 28593 to 68342

Calibration
  T34 (FP32 variance) range: 6.579582114e-03 to 1.848954707e-02
  Fine variance vs FP32:
    Max absolute error:  7.481500506e-05
    Mean absolute error: 1.634376947e-05
  Coarse variance vs FP32:
    Max absolute error:  7.554609329e-05
    Mean absolute error: 1.619344403e-05
  Variance change from T33 r

### OP4

In [8]:
# Op 4: ADD
# T34 (variance) + T2 (epsilon) -> T35 (variance plus epsilon)
# Continue with the coarse INT32 variance candidate.

op4 = interpreter._get_ops_details()[4]
assert op4["op_name"] == "ADD"
assert int(op4["inputs"][0]) == t34_index

epsilon_index = int(op4["inputs"][1])
t35_index = int(op4["outputs"][0])

# Read epsilon from this specific TFLite model.
epsilon_tensor = interpreter.get_tensor(epsilon_index)
assert epsilon_tensor.size == 1
epsilon_fp32 = float(epsilon_tensor.reshape(-1)[0])
assert np.isfinite(epsilon_fp32) and epsilon_fp32 > 0

T35_FRAC_BITS = T34_COARSE_FRAC_BITS
T35_SCALE = T34_COARSE_SCALE

# Offline constant conversion: nearest rounding, ties to even.
epsilon_q = int(np.rint(epsilon_fp32 / T35_SCALE))
epsilon_approx = epsilon_q * T35_SCALE

int32_limits = np.iinfo(np.int32)
assert int32_limits.min <= epsilon_q <= int32_limits.max

print(f"T{epsilon_index} (epsilon), FP32: {epsilon_fp32:.12e}")
print(f"T{epsilon_index} (epsilon), stored integer: {epsilon_q}")
print(f"T{epsilon_index} (epsilon), represented: {epsilon_approx:.12e}")
print(
    "Epsilon quantization absolute error:",
    f"{abs(epsilon_approx - epsilon_fp32):.12e}",
)
print(
    f"T{t35_index} (variance plus epsilon): INT32, "
    f"fractional bits = {T35_FRAC_BITS}, scale = {T35_SCALE}"
)
print("Output requantization shift: 0")

op4_integer_trial = []
assert len(op0_cases) == len(op3_integer_trial)

for case, variance_trial in zip(op0_cases, op3_integer_trial):
    assert case["name"] == variance_trial["name"]

    # Actual FP32 T35 from the original TFLite graph.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    t35_ref = interpreter.get_tensor(t35_index).copy().reshape(-1)

    t34_q = variance_trial["t34_coarse_q"]

    # Check addition in INT64 before performing the INT32 addition.
    sum_check = t34_q.astype(np.int64) + epsilon_q
    assert sum_check.min() >= int32_limits.min
    assert sum_check.max() <= int32_limits.max

    # Integer operation: same scales, so add directly.
    t35_q = t34_q + np.int32(epsilon_q)
    assert t35_q.dtype == np.int32
    assert np.array_equal(t35_q.astype(np.int64), sum_check)

    # Reconstruction only for numerical diagnostics.
    t35_approx = t35_q.astype(np.float64) * T35_SCALE

    op4_integer_trial.append({
        "name": case["name"],
        "t34_q": t34_q.copy(),
        "t35_q": t35_q,
        "t35_ref": t35_ref,
        "t35_approx": t35_approx,
        "error": np.abs(t35_approx - t35_ref),
    })


def report_epsilon_trial(label, trials):
    reference = np.concatenate([t["t35_ref"] for t in trials])
    integers = np.concatenate([t["t35_q"] for t in trials])
    errors = np.concatenate([t["error"] for t in trials])

    print(f"\n{label}")
    print(
        "  T35 (FP32 variance plus epsilon) range:",
        f"{reference.min():.9e} to {reference.max():.9e}",
    )
    print(
        "  T35 (stored integer) range:",
        f"{integers.min()} to {integers.max()}",
    )
    print(f"  Max absolute error vs FP32:  {errors.max():.9e}")
    print(f"  Mean absolute error vs FP32: {errors.mean():.9e}")
    print("  INT32 overflow: none")


report_epsilon_trial("Benchmark", op4_integer_trial[:1])
report_epsilon_trial("Calibration", op4_integer_trial[1:])

first = op4_integer_trial[0]
print("\nFirst benchmark token:")
print("  T34 (variance), stored integer:", first["t34_q"][0])
print("  T2 (epsilon), stored integer:", epsilon_q)
print("  T35 (variance plus epsilon), stored integer:", first["t35_q"][0])
print("  T35 (represented value):", f'{first["t35_approx"][0]:.9f}')
print("  T35 (FP32 reference):", f'{first["t35_ref"][0]:.9f}')
print("  Absolute error:", f'{first["error"][0]:.9e}')

T2 (epsilon), FP32: 9.999999747379e-06
T2 (epsilon), stored integer: 42
T2 (epsilon), represented: 1.001358032227e-05
Epsilon quantization absolute error: 1.358057488687e-08
T35 (variance plus epsilon): INT32, fractional bits = 22, scale = 2.384185791015625e-07
Output requantization shift: 0

Benchmark
  T35 (FP32 variance plus epsilon) range: 6.812664680e-03 to 1.630877145e-02
  T35 (stored integer) range: 28635 to 68384
  Max absolute error vs FP32:  4.651769996e-05
  Mean absolute error vs FP32: 1.458522456e-05
  INT32 overflow: none

Calibration
  T35 (FP32 variance plus epsilon) range: 6.589582190e-03 to 1.849954762e-02
  T35 (stored integer) range: 27611 to 77610
  Max absolute error vs FP32:  7.556006312e-05
  Mean absolute error vs FP32: 1.619393255e-05
  INT32 overflow: none

First benchmark token:
  T34 (variance), stored integer: 68342
  T2 (epsilon), stored integer: 42
  T35 (variance plus epsilon), stored integer: 68384
  T35 (represented value): 0.016304016
  T35 (FP32 re

In [9]:
# Op 5: RSQRT
# T35 (variance plus epsilon) -> T36 (inverse standard deviation)
# Integer implementation: integer division + integer square root.

from math import isqrt

op5 = interpreter._get_ops_details()[5]
assert op5["op_name"] == "RSQRT"
assert int(op5["inputs"][0]) == t35_index

t36_index = int(op5["outputs"][0])

T36_FRAC_BITS = 16
T36_SCALE = 2.0 ** (-T36_FRAC_BITS)

# Derived from input and output scales:
# q_T36 ≈ sqrt(2^(F_input + 2*F_output) / q_T35)
RSQRT_NUMERATOR_BITS = T35_FRAC_BITS + 2 * T36_FRAC_BITS
RSQRT_NUMERATOR = 1 << RSQRT_NUMERATOR_BITS

int64_max = int(np.iinfo(np.int64).max)
int32_max = int(np.iinfo(np.int32).max)

assert RSQRT_NUMERATOR <= int64_max
assert T35_SCALE == 2.0 ** (-T35_FRAC_BITS)

op5_integer_trial = []
assert len(op0_cases) == len(op4_integer_trial)

for case, epsilon_trial in zip(op0_cases, op4_integer_trial):
    assert case["name"] == epsilon_trial["name"]

    # Actual T36 from the original TFLite graph.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    t36_ref = interpreter.get_tensor(t36_index).copy().reshape(-1)
    assert np.all(np.isfinite(t36_ref)) and np.all(t36_ref > 0)

    t35_q = epsilon_trial["t35_q"].reshape(-1)
    assert np.all(t35_q > 0)

    division_values = []
    output_values = []

    # Integer-only candidate calculation.
    for stored_input in t35_q:
        q = int(stored_input)

        n = RSQRT_NUMERATOR // q
        y = isqrt(n)

        assert 0 <= n <= int64_max
        assert 0 <= y <= int32_max

        # Verify y is exactly floor(sqrt(n)).
        assert y * y <= n < (y + 1) * (y + 1)

        # Verify it is also floor(sqrt(numerator / q)).
        assert y * y * q <= RSQRT_NUMERATOR
        assert RSQRT_NUMERATOR < (y + 1) * (y + 1) * q

        division_values.append(n)
        output_values.append(y)

    division_q = np.array(division_values, dtype=np.int64)
    t36_q = np.array(output_values, dtype=np.int32)

    # Floating-point reconstruction and comparisons: diagnostics only.
    t36_approx = t36_q.astype(np.float64) * T36_SCALE
    represented_t35 = t35_q.astype(np.float64) * T35_SCALE

    # Ideal RSQRT of our represented integer input.
    ideal_from_integer_input = 1.0 / np.sqrt(represented_t35)

    local_error = np.abs(t36_approx - ideal_from_integer_input)
    upstream_error = np.abs(ideal_from_integer_input - t36_ref)
    total_error = np.abs(t36_approx - t36_ref)

    # Allow a tiny tolerance for floating-point diagnostic arithmetic.
    assert np.all(t36_approx <= ideal_from_integer_input + 1e-12)
    assert local_error.max() < T36_SCALE + 1e-12

    op5_integer_trial.append({
        "name": case["name"],
        "t35_q": t35_q.copy(),
        "division_q": division_q,
        "t36_q": t36_q,
        "t36_ref": t36_ref,
        "t36_approx": t36_approx,
        "ideal_from_integer_input": ideal_from_integer_input,
        "local_error": local_error,
        "upstream_error": upstream_error,
        "total_error": total_error,
        "relative_error": total_error / np.abs(t36_ref),
    })


def report_rsqrt_trial(label, trials):
    reference = np.concatenate([t["t36_ref"] for t in trials])
    integers = np.concatenate([t["t36_q"] for t in trials])

    print(f"\n{label}")
    print(
        "  T36 (FP32 inverse standard deviation) range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )
    print(
        "  T36 (stored integer) range:",
        f"{integers.min()} to {integers.max()}",
    )

    for key, description in [
        ("local_error", "Op 5's own approximation error"),
        ("upstream_error", "Effect of earlier input/variance errors"),
        ("total_error", "Total error vs TFLite T36"),
    ]:
        errors = np.concatenate([t[key] for t in trials])
        print(f"  {description}:")
        print(f"    Max absolute error:  {errors.max():.9e}")
        print(f"    Mean absolute error: {errors.mean():.9e}")

    relative = np.concatenate([t["relative_error"] for t in trials])
    print(f"  Maximum total relative error: {100 * relative.max():.6f}%")
    print("  Checked intermediate/output widths: pass")


print(f"Integer numerator: 2^{RSQRT_NUMERATOR_BITS}")
print(
    f"T36 (inverse standard deviation): INT32, "
    f"fractional bits = {T36_FRAC_BITS}, scale = {T36_SCALE}"
)
print("Rounding rule: floor")
print(f"Local absolute-error bound: less than {T36_SCALE:.9e}")

report_rsqrt_trial("Benchmark", op5_integer_trial[:1])
report_rsqrt_trial("Calibration", op5_integer_trial[1:])

first = op5_integer_trial[0]
print("\nFirst benchmark token:")
print("  T35 (variance plus epsilon), stored integer:", first["t35_q"][0])
print("  Integer division result:", first["division_q"][0])
print("  T36 (inverse standard deviation), stored integer:", first["t36_q"][0])
print("  T36 (represented value):", f'{first["t36_approx"][0]:.9f}')
print(
    "  Ideal RSQRT of represented T35:",
    f'{first["ideal_from_integer_input"][0]:.9f}',
)
print("  T36 (TFLite FP32 reference):", f'{first["t36_ref"][0]:.9f}')
print("  Op 5 local error:", f'{first["local_error"][0]:.9e}')
print("  Total error vs TFLite:", f'{first["total_error"][0]:.9e}')

Integer numerator: 2^54
T36 (inverse standard deviation): INT32, fractional bits = 16, scale = 1.52587890625e-05
Rounding rule: floor
Local absolute-error bound: less than 1.525878906e-05

Benchmark
  T36 (FP32 inverse standard deviation) range: 7.830498219 to 12.115504265
  T36 (stored integer) range: 513254 to 793160
  Op 5's own approximation error:
    Max absolute error:  1.451589615e-05
    Mean absolute error: 6.953473999e-06
  Effect of earlier input/variance errors:
    Max absolute error:  2.598329202e-02
    Mean absolute error: 8.710513817e-03
  Total error vs TFLite T36:
    Max absolute error:  2.598571777e-02
    Mean absolute error: 8.710637689e-03
  Maximum total relative error: 0.250124%
  Checked intermediate/output widths: pass

Calibration
  T36 (FP32 inverse standard deviation) range: 7.352235794 to 12.318875313
  T36 (stored integer) range: 481782 to 807734
  Op 5's own approximation error:
    Max absolute error:  1.507452264e-05
    Mean absolute error: 7.47512

In [10]:
# Op 6–11: normalization
# T42 = T0 * T36 - T31 * T36
#
# Preserve fine intermediates, then test an INT32 T42 candidate.

ops = interpreter._get_ops_details()
expected_names = {
    6: "MUL",
    7: "SUB",
    8: "RESHAPE",
    9: "MUL",
    10: "RESHAPE",
    11: "ADD",
}
for index, expected in expected_names.items():
    assert ops[index]["op_name"] == expected

stage_indices = {
    index: int(ops[index]["outputs"][0])
    for index in expected_names
}

# Op 7 subtracts the correction from the stored zero constant.
zero_tensor = interpreter.get_tensor(int(ops[7]["inputs"][0]))
assert np.all(zero_tensor == 0)

MEAN_FRAC_BITS = T0_FRAC_BITS + 7
assert T31_SCALE == 2.0 ** (-MEAN_FRAC_BITS)

T37_FRAC_BITS = MEAN_FRAC_BITS + T36_FRAC_BITS
T40_FRAC_BITS = T0_FRAC_BITS + T36_FRAC_BITS
T42_FINE_FRAC_BITS = T37_FRAC_BITS

T42_FRAC_BITS = 16             # Candidate output format.
T42_SCALE = 2.0 ** (-T42_FRAC_BITS)
T42_FINE_SCALE = 2.0 ** (-T42_FINE_FRAC_BITS)

NORMALIZATION_ALIGN_SHIFT = T42_FINE_FRAC_BITS - T40_FRAC_BITS
NORMALIZATION_REQUANT_SHIFT = T42_FINE_FRAC_BITS - T42_FRAC_BITS

assert NORMALIZATION_ALIGN_SHIFT >= 0
assert NORMALIZATION_REQUANT_SHIFT >= 0

stage_descriptions = {
    6: "mean correction",
    7: "negative mean correction",
    8: "broadcast inverse standard deviation",
    9: "scaled input",
    10: "broadcast negative correction",
    11: "normalized features, fine",
}
stage_frac_bits = {
    6: T37_FRAC_BITS,
    7: T37_FRAC_BITS,
    8: T36_FRAC_BITS,
    9: T40_FRAC_BITS,
    10: T37_FRAC_BITS,
    11: T42_FINE_FRAC_BITS,
}

int32_limits = np.iinfo(np.int32)
normalization_integer_trial = []

assert len(op0_cases) == len(op0_integer_trial) == len(op5_integer_trial)

for case, mean_trial, rsqrt_trial in zip(
    op0_cases, op0_integer_trial, op5_integer_trial
):
    assert case["name"] == mean_trial["name"] == rsqrt_trial["name"]

    # Capture all six original TFLite reference tensors.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    references = {
        index: interpreter.get_tensor(stage_indices[index]).copy()
        for index in expected_names
    }

    # Promote before multiplication.
    input_q = mean_trial["t0_q"].astype(np.int64)
    mean_q = mean_trial["t31_mean_q"].astype(np.int64).reshape(-1)
    inverse_std_q = rsqrt_trial["t36_q"].astype(np.int64).reshape(-1)

    # Op 6: mean * inverse standard deviation; scale 2^-31.
    correction_q = mean_q * inverse_std_q

    # Op 7: 0 - correction; same scale.
    negative_correction_q = -correction_q

    # Op 8: reshape inverse standard deviation; same values/scale.
    broadcast_inverse_std_q = inverse_std_q.reshape(1, -1, 1)

    # Op 9: input * inverse standard deviation; scale 2^-24.
    scaled_input_q = input_q * broadcast_inverse_std_q

    # Op 10: reshape negative correction; scale 2^-31.
    broadcast_negative_q = negative_correction_q.reshape(1, -1, 1)

    # Op 11: align T40 to T41's scale, then add.
    aligned_scaled_input_q = (
        scaled_input_q * (1 << NORMALIZATION_ALIGN_SHIFT)
    )
    t42_fine_q = aligned_scaled_input_q + broadcast_negative_q

    # Verify the equivalent fused expression: (input - mean) * r.
    centered_input_q = (
        input_q * (1 << (MEAN_FRAC_BITS - T0_FRAC_BITS))
        - mean_q.reshape(1, -1, 1)
    )
    fused_q = centered_input_q * broadcast_inverse_std_q
    assert np.array_equal(t42_fine_q, fused_q)

    stage_q = {
        6: correction_q,
        7: negative_correction_q,
        8: broadcast_inverse_std_q,
        9: scaled_input_q,
        10: broadcast_negative_q,
        11: t42_fine_q,
    }

    stage_errors = {}
    for index, integers in stage_q.items():
        reference = references[index]
        assert integers.shape == reference.shape
        scale = 2.0 ** (-stage_frac_bits[index])
        represented = integers.astype(np.float64) * scale
        stage_errors[index] = np.abs(represented - reference)

    # Optional output requantization.
    # NumPy signed >> performs arithmetic right shift.
    # Nearest rounding, ties toward positive infinity.
    shift = NORMALIZATION_REQUANT_SHIFT
    if shift == 0:
        t42_candidate_q = t42_fine_q.copy()
    else:
        offset = 1 << (shift - 1)
        t42_candidate_q = (t42_fine_q + offset) >> shift

    assert t42_candidate_q.min() >= int32_limits.min
    assert t42_candidate_q.max() <= int32_limits.max
    t42_q = t42_candidate_q.astype(np.int32)

    fine_approx = t42_fine_q.astype(np.float64) * T42_FINE_SCALE
    t42_approx = t42_q.astype(np.float64) * T42_SCALE
    requant_error = np.abs(t42_approx - fine_approx)

    if shift > 0:
        assert requant_error.max() <= T42_SCALE / 2 + 1e-12

    normalization_integer_trial.append({
        "name": case["name"],
        "stage_q": stage_q,
        "stage_errors": stage_errors,
        "t42_fine_q": t42_fine_q,
        "t42_q": t42_q,
        "t42_ref": references[11],
        "t42_approx": t42_approx,
        "output_error": np.abs(t42_approx - references[11]),
        "requant_error": requant_error,
        "input_q": input_q,
        "mean_q": mean_q,
        "inverse_std_q": inverse_std_q,
        "centered_input_q": centered_input_q,
    })


def report_normalization(label, trials):
    print(f"\n{label}")
    print("  Intermediate errors vs TFLite:")

    for index in expected_names:
        errors = np.concatenate([
            t["stage_errors"][index].reshape(-1) for t in trials
        ])
        tensor_index = stage_indices[index]
        print(
            f"    Op {index}, T{tensor_index} "
            f"({stage_descriptions[index]}): "
            f"max={errors.max():.9e}, mean={errors.mean():.9e}"
        )

    fine = np.concatenate([t["t42_fine_q"].reshape(-1) for t in trials])
    coarse = np.concatenate([t["t42_q"].reshape(-1) for t in trials])
    reference = np.concatenate([t["t42_ref"].reshape(-1) for t in trials])
    output_error = np.concatenate([
        t["output_error"].reshape(-1) for t in trials
    ])
    requant_error = np.concatenate([
        t["requant_error"].reshape(-1) for t in trials
    ])

    fine_fits_int32 = (
        fine.min() >= int32_limits.min
        and fine.max() <= int32_limits.max
    )

    print(
        "  T42 (FP32 normalized features) range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )
    print(f"  T42 (fine integer) range: {fine.min()} to {fine.max()}")
    print(f"  Fine T42 fits INT32: {fine_fits_int32}")
    print(f"  T42 (candidate INT32) range: {coarse.min()} to {coarse.max()}")
    print(f"  Candidate max error vs FP32:  {output_error.max():.9e}")
    print(f"  Candidate mean error vs FP32: {output_error.mean():.9e}")
    print(f"  Additional requantization max change:  {requant_error.max():.9e}")
    print(f"  Additional requantization mean change: {requant_error.mean():.9e}")


print("Op 11 input alignment multiplier:", 1 << NORMALIZATION_ALIGN_SHIFT)
print("T42 (fine normalized features): INT64, scale =", T42_FINE_SCALE)
print("T42 (candidate normalized features): INT32, scale =", T42_SCALE)
print("Output requantization shift:", NORMALIZATION_REQUANT_SHIFT)
print("Rounding: nearest, ties toward positive infinity")

report_normalization("Benchmark", normalization_integer_trial[:1])
report_normalization("Calibration", normalization_integer_trial[1:])

first = normalization_integer_trial[0]
location = (0, 0, 0)

print("\nFirst benchmark token, first feature:")
print("  T0 (input), integer:", first["input_q"][location])
print("  T31 (mean), integer:", first["mean_q"][0])
print("  T36 (inverse standard deviation), integer:", first["inverse_std_q"][0])
print("  Centered input, integer at scale 2^-15:", first["centered_input_q"][location])
print("  T42 (fine normalized feature), integer:", first["t42_fine_q"][location])
print("  T42 (candidate normalized feature), integer:", first["t42_q"][location])
print("  T42 (candidate represented value):", f'{first["t42_approx"][location]:.9f}')
print("  T42 (TFLite FP32 reference):", f'{first["t42_ref"][location]:.9f}')

Op 11 input alignment multiplier: 128
T42 (fine normalized features): INT64, scale = 4.656612873077393e-10
T42 (candidate normalized features): INT32, scale = 1.52587890625e-05
Output requantization shift: 15
Rounding: nearest, ties toward positive infinity

Benchmark
  Intermediate errors vs TFLite:
    Op 6, T37 (mean correction): max=2.653919160e-03, mean=5.205223279e-04
    Op 7, T38 (negative mean correction): max=2.653919160e-03, mean=5.205223279e-04
    Op 8, T39 (broadcast inverse standard deviation): max=2.598571777e-02, mean=8.710637689e-03
    Op 9, T40 (scaled input): max=2.488905191e-02, mean=1.030809522e-02
    Op 10, T41 (broadcast negative correction): max=2.653919160e-03, mean=5.205223279e-04
    Op 11, T42 (normalized features, fine): max=2.518225089e-02, mean=1.027650931e-02
  T42 (FP32 normalized features) range: -3.677406311 to 3.020779133
  T42 (fine integer) range: -7845208188 to 6513916896
  Fine T42 fits INT32: False
  T42 (candidate INT32) range: -239417 to 19

In [11]:
# Inspect Op 12–14:
# T43 = T42 (normalized features) * T6 (gamma)
# T44 = T43 + T7 (beta)
# T45 = quantize(T44), the INT8 projection input

ops = interpreter._get_ops_details()

assert ops[12]["op_name"] == "MUL"
assert ops[13]["op_name"] == "ADD"
assert ops[14]["op_name"] == "QUANTIZE"

t42_index = stage_indices[11]
t43_index = int(ops[12]["outputs"][0])
t44_index = int(ops[13]["outputs"][0])
t45_index = int(ops[14]["outputs"][0])

assert t42_index in ops[12]["inputs"]
assert t43_index in ops[13]["inputs"]
assert int(ops[14]["inputs"][0]) == t44_index

gamma_index = next(
    int(i) for i in ops[12]["inputs"] if int(i) != t42_index
)
beta_index = next(
    int(i) for i in ops[13]["inputs"] if int(i) != t43_index
)

ln1_gamma_fp32 = interpreter.get_tensor(gamma_index).copy()
ln1_beta_fp32 = interpreter.get_tensor(beta_index).copy()

assert ln1_gamma_fp32.size == FEATURE_COUNT
assert ln1_beta_fp32.size == FEATURE_COUNT
assert np.all(np.isfinite(ln1_gamma_fp32))
assert np.all(np.isfinite(ln1_beta_fp32))

details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}
t45_details = details[t45_index]
assert t45_details["dtype"] == np.int8

quantization = t45_details["quantization_parameters"]
assert len(quantization["scales"]) == 1
assert len(quantization["zero_points"]) == 1

T45_SCALE = float(quantization["scales"][0])
T45_ZERO_POINT = int(quantization["zero_points"][0])
assert T45_SCALE > 0


def report_parameter(index, meaning, values):
    flat = values.reshape(-1)
    print(f"\nT{index} ({meaning})")
    print("  Shape:", values.shape)
    print("  Stored dtype:", values.dtype)
    print(f"  Minimum: {flat.min():.9f}")
    print(f"  Maximum: {flat.max():.9f}")
    print(f"  Maximum absolute value: {np.abs(flat).max():.9f}")
    print("  First 8 values:", flat[:8])


report_parameter(gamma_index, "learned scale gamma", ln1_gamma_fp32)
report_parameter(beta_index, "learned bias beta", ln1_beta_fp32)

print(f"\nT{t45_index} (INT8 projection input)")
print("  Shape:", tuple(t45_details["shape"]))
print("  Scale:", T45_SCALE)
print("  Zero point:", T45_ZERO_POINT)
print(
    "  Representable real range:",
    (-128 - T45_ZERO_POINT) * T45_SCALE,
    "to",
    (127 - T45_ZERO_POINT) * T45_SCALE,
)

# Capture original TFLite outputs for the upcoming integer comparison.
affine_reference_cases = []

for case in op0_cases:
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    affine_reference_cases.append({
        "name": case["name"],
        "t43_ref": interpreter.get_tensor(t43_index).copy(),
        "t44_ref": interpreter.get_tensor(t44_index).copy(),
        "t45_ref": interpreter.get_tensor(t45_index).copy(),
    })


def report_affine_reference(label, cases):
    print(f"\n{label}")
    for key, index, meaning in [
        ("t43_ref", t43_index, "normalized features times gamma"),
        ("t44_ref", t44_index, "LayerNorm output after beta"),
        ("t45_ref", t45_index, "INT8 projection input"),
    ]:
        values = np.concatenate([c[key].reshape(-1) for c in cases])
        print(
            f"  T{index} ({meaning}) range:",
            f"{values.min()} to {values.max()}",
        )


report_affine_reference("Benchmark", affine_reference_cases[:1])
report_affine_reference("Calibration", affine_reference_cases[1:])


T6 (learned scale gamma)
  Shape: (128,)
  Stored dtype: float32
  Minimum: 0.922644913
  Maximum: 1.139880538
  Maximum absolute value: 1.139880538
  First 8 values: [0.96397287 1.0049611  0.9837676  0.9226449  0.96249086 0.98930055
 1.0272598  0.9648354 ]

T7 (learned bias beta)
  Shape: (128,)
  Stored dtype: float32
  Minimum: -0.242233545
  Maximum: 0.219664216
  Maximum absolute value: 0.242233545
  First 8 values: [-0.0593875   0.0696523   0.03833386 -0.03081142  0.04557584 -0.1962216
  0.11669759 -0.07880756]

T45 (INT8 projection input)
  Shape: (np.int32(1), np.int32(32), np.int32(128))
  Scale: 0.03362198546528816
  Zero point: -17
  Representable real range: -3.732040386646986 to 4.841565907001495

Benchmark
  T43 (normalized features times gamma) range: -3.614773750305176 to 2.9820332527160645
  T44 (LayerNorm output after beta) range: -3.7485108375549316 to 3.0682313442230225
  T45 (INT8 projection input) range: -128 to 74

Calibration
  T43 (normalized features times ga

In [12]:
# Op 12–14: learned affine transform and INT8 quantization.

GAMMA_FRAC_BITS = 16
BETA_FRAC_BITS = 16
GAMMA_SCALE = 2.0 ** (-GAMMA_FRAC_BITS)
BETA_SCALE = 2.0 ** (-BETA_FRAC_BITS)

int32_limits = np.iinfo(np.int32)
int64_max = int(np.iinfo(np.int64).max)

# Offline conversion of learned FP32 constants.
def quantize_parameter(values, scale):
    rounded = np.rint(values.astype(np.float64) / scale)
    assert rounded.min() >= int32_limits.min
    assert rounded.max() <= int32_limits.max
    return rounded.astype(np.int32)


ln1_gamma_q = quantize_parameter(ln1_gamma_fp32, GAMMA_SCALE)
ln1_beta_q = quantize_parameter(ln1_beta_fp32, BETA_SCALE)

gamma_approx = ln1_gamma_q.astype(np.float64) * GAMMA_SCALE
beta_approx = ln1_beta_q.astype(np.float64) * BETA_SCALE

T43_FRAC_BITS = T42_FRAC_BITS + GAMMA_FRAC_BITS
T43_SCALE = 2.0 ** (-T43_FRAC_BITS)
T44_SCALE = T43_SCALE

BETA_ALIGN_SHIFT = T43_FRAC_BITS - BETA_FRAC_BITS
assert BETA_ALIGN_SHIFT >= 0

# Offline conversion of the final requantization multiplier.
T45_REQUANT_SHIFT = 47
T45_REAL_MULTIPLIER = T44_SCALE / T45_SCALE
T45_MULTIPLIER_Q = int(np.rint(
    T45_REAL_MULTIPLIER * (1 << T45_REQUANT_SHIFT)
))
assert 0 < T45_MULTIPLIER_Q <= int64_max

T45_ROUNDING_OFFSET = 1 << (T45_REQUANT_SHIFT - 1)
multiplier_approx = T45_MULTIPLIER_Q * 2.0 ** (-T45_REQUANT_SHIFT)

print("T6 (gamma): INT32, scale =", GAMMA_SCALE)
print("T7 (beta): INT32, scale =", BETA_SCALE)
print("Gamma max quantization error:",
      f"{np.abs(gamma_approx - ln1_gamma_fp32).max():.9e}")
print("Beta max quantization error:",
      f"{np.abs(beta_approx - ln1_beta_fp32).max():.9e}")
print("T43/T44 (affine intermediates): INT64, scale =", T44_SCALE)
print("Beta alignment multiplier:", 1 << BETA_ALIGN_SHIFT)
print("T45 multiplier integer:", T45_MULTIPLIER_Q)
print("T45 right shift:", T45_REQUANT_SHIFT)
print("T45 scale:", T45_SCALE)
print("T45 zero point:", T45_ZERO_POINT)
print(
    "Multiplier relative error:",
    f"{abs(multiplier_approx / T45_REAL_MULTIPLIER - 1):.9e}",
)

ln1_affine_integer_trial = []
assert len(normalization_integer_trial) == len(affine_reference_cases)

for normalized, reference in zip(
    normalization_integer_trial, affine_reference_cases
):
    assert normalized["name"] == reference["name"]

    # Op 12: integer multiplication.
    # INT32 operands are promoted before multiplication.
    t43_q = (
        normalized["t42_q"].astype(np.int64)
        * ln1_gamma_q.astype(np.int64)
    )

    # Op 13: align beta to the product scale, then add.
    aligned_beta_q = (
        ln1_beta_q.astype(np.int64) * (1 << BETA_ALIGN_SHIFT)
    )

    # Conservative bound before the addition.
    product_bound = max(abs(int(t43_q.min())), abs(int(t43_q.max())))
    beta_bound = max(
        abs(int(aligned_beta_q.min())),
        abs(int(aligned_beta_q.max())),
    )
    assert product_bound + beta_bound <= int64_max

    t44_q = t43_q + aligned_beta_q

    # Check multiplication plus rounding before calculating it.
    affine_bound = max(abs(int(t44_q.min())), abs(int(t44_q.max())))
    assert (
        affine_bound * T45_MULTIPLIER_Q + T45_ROUNDING_OFFSET
        <= int64_max
    )

    # Op 14: integer-only candidate requantization.
    # Nearest rounding, ties toward positive infinity.
    rounded_q = (
        t44_q * T45_MULTIPLIER_Q + T45_ROUNDING_OFFSET
    ) >> T45_REQUANT_SHIFT

    before_saturation = rounded_q + T45_ZERO_POINT
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))
    t45_q = np.clip(before_saturation, -128, 127).astype(np.int8)

    # Floating-point calculations below are diagnostics only.
    t43_approx = t43_q.astype(np.float64) * T43_SCALE
    t44_approx = t44_q.astype(np.float64) * T44_SCALE

    # Isolate parameter quantization from earlier normalization errors.
    ideal_affine_from_candidate = (
        normalized["t42_approx"] * ln1_gamma_fp32 + ln1_beta_fp32
    )
    parameter_error = np.abs(
        t44_approx - ideal_affine_from_candidate
    )

    # Compare fixed-point requantization against ideal quantization
    # of OUR represented T44, using the SAME rounding rule.
    ideal_t45_q = np.clip(
        np.floor(t44_approx / T45_SCALE + 0.5) + T45_ZERO_POINT,
        -128, 127,
    ).astype(np.int8)

    # Cast before subtraction to avoid INT8 overflow.
    t45_difference = (
        t45_q.astype(np.int32)
        - reference["t45_ref"].astype(np.int32)
    )

    ln1_affine_integer_trial.append({
        "name": normalized["name"],
        "t43_q": t43_q,
        "t44_q": t44_q,
        "t45_q": t45_q,
        "t43_approx": t43_approx,
        "t44_approx": t44_approx,
        "t43_error": np.abs(t43_approx - reference["t43_ref"]),
        "t44_error": np.abs(t44_approx - reference["t44_ref"]),
        "parameter_error": parameter_error,
        "t45_ref": reference["t45_ref"],
        "t45_difference": t45_difference,
        "requant_mismatch_count": int(np.count_nonzero(
            t45_q != ideal_t45_q
        )),
        "saturation_count": saturation_count,
    })


def report_affine_integer(label, trials):
    print(f"\n{label}")

    for key, meaning in [
        ("t43_error", "T43 (normalized features times gamma) vs FP32"),
        ("t44_error", "T44 (affine LayerNorm output) vs FP32"),
        ("parameter_error", "Additional error from gamma/beta conversion"),
    ]:
        errors = np.concatenate([t[key].reshape(-1) for t in trials])
        print(f"  {meaning}:")
        print(f"    Max absolute error:  {errors.max():.9e}")
        print(f"    Mean absolute error: {errors.mean():.9e}")

    differences = np.concatenate([
        t["t45_difference"].reshape(-1) for t in trials
    ])
    values = np.concatenate([t["t45_q"].reshape(-1) for t in trials])
    matched = int(np.count_nonzero(differences == 0))

    print(f"  T45 (candidate INT8) range: {values.min()} to {values.max()}")
    print(f"  T45 exact matches: {matched}/{differences.size}")
    print(f"  T45 match percentage: {100 * matched / differences.size:.3f}%")
    print("  T45 maximum integer difference:",
          int(np.abs(differences).max()))

    unique, counts = np.unique(differences, return_counts=True)
    print("  Signed difference counts:", {
        int(d): int(c) for d, c in zip(unique, counts)
    })

    print("  Fixed multiplier vs ideal quantization mismatches:",
          sum(t["requant_mismatch_count"] for t in trials))
    print("  Candidate values requiring saturation:",
          sum(t["saturation_count"] for t in trials))
    print("  Intermediate width checks: pass")


report_affine_integer("Benchmark", ln1_affine_integer_trial[:1])
report_affine_integer("Calibration", ln1_affine_integer_trial[1:])

first = ln1_affine_integer_trial[0]
location = (0, 0, 0)

print("\nFirst benchmark token, first feature:")
print("  T6 (gamma), stored integer:", ln1_gamma_q[0])
print("  T7 (beta), stored integer:", ln1_beta_q[0])
print("  T43 (scaled normalized feature), integer:", first["t43_q"][location])
print("  T44 (affine output), integer:", first["t44_q"][location])
print("  T44 (represented value):", f'{first["t44_approx"][location]:.9f}')
print("  T45 (candidate INT8):", first["t45_q"][location])
print("  T45 (TFLite reference INT8):", first["t45_ref"][location])

T6 (gamma): INT32, scale = 1.52587890625e-05
T7 (beta): INT32, scale = 1.52587890625e-05
Gamma max quantization error: 7.629394531e-06
Beta max quantization error: 7.601454854e-06
T43/T44 (affine intermediates): INT64, scale = 2.3283064365386963e-10
Beta alignment multiplier: 65536
T45 multiplier integer: 974600
T45 right shift: 47
T45 scale: 0.03362198546528816
T45 zero point: -17
Multiplier relative error: 3.956765795e-07

Benchmark
  T43 (normalized features times gamma) vs FP32:
    Max absolute error:  2.628150489e-02
    Mean absolute error: 1.019115742e-02
  T44 (affine LayerNorm output) vs FP32:
    Max absolute error:  2.627590206e-02
    Mean absolute error: 1.019107212e-02
  Additional error from gamma/beta conversion:
    Max absolute error:  2.371183291e-05
    Mean absolute error: 5.079321407e-06
  T45 (candidate INT8) range: -128 to 74
  T45 exact matches: 2854/4096
  T45 match percentage: 69.678%
  T45 maximum integer difference: 1
  Signed difference counts: {-1: 624, 

### Block-0 first LayerNorm: integer candidate, Op 0–14

Source block-0 SHA-256:
`06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9`

Test inputs: one 32-token benchmark sequence and seven calibration
sequences; hidden size 128.

| Tensor | Meaning | Candidate representation |
|---|---|---|
| T0 | Block input | INT8, scale 2^-8, zero point 0 |
| T31/T32 | Per-token means / broadcast means | INT32, scale 2^-15 |
| T33 | Squared differences | INT64 at 2^-30; requantized to INT32 at 2^-15 |
| T34 | Variance | INT32, scale 2^-22 |
| T35 | Variance plus epsilon | INT32, scale 2^-22; epsilon integer 42 |
| T36 | Inverse standard deviation | INT32, scale 2^-16 |
| T42 | Normalized features | INT64 at 2^-31; requantized to INT32 at 2^-16 |
| T6/T7 | Learned gamma / beta | INT32, scale 2^-16 |
| T43/T44 | Affine intermediates / LayerNorm output | INT64, scale 2^-32 |
| T45 | Projection input | INT8, scale 0.03362198546528816, zero point -17 |

Op 5 uses integer division followed by integer square root:
`q_T36 = isqrt(2^54 // q_T35)`, with floor rounding.

Final T45 requantization uses multiplier 974600, right shift 47,
nearest rounding with ties toward positive infinity, zero-point
addition, and INT8 saturation.

| Result | Benchmark | Calibration |
|---|---:|---:|
| T44 maximum absolute error vs FP32 | 0.0262759 | 0.0281122 |
| T44 mean absolute error vs FP32 | 0.0101911 | 0.0101453 |
| T45 exact matches | 2854/4096 | 20020/28672 |
| Maximum T45 integer difference | 1 | 1 |
| Fixed multiplier vs ideal quantization mismatches | 0 | 0 |
| Candidate saturation count | 0 | 1 |

All implemented width checks passed for the tested inputs.
Earlier input/normalization errors dominate; parameter conversion
and output requantization add comparatively little error.

This is a Python-validated integer candidate. C, Spike, RTL,
complete-block accuracy, and performance have not yet been validated
for this LayerNorm.

### Integer storage, scales, and requantization

In our integer implementation, each tensor has a **stored integer** and
a **scale** describing its represented real value:

$$
x_{\text{represented}}=(q-z)s
$$

Encoding an FP32 input uses:

$$
q=\operatorname{round}(x/s)+z
$$

For **T0 (block input)**, $s=2^{-8}$ and $z=0$, so
$q_{T0}=\operatorname{round}(x\,2^8)$.

Subsequent operations calculate with integers while tracking their scales.
For example, **T44 (affine LayerNorm output)** stores INT64 integers at
scale $2^{-32}$. The scale describes their meaning; it is not automatically
an instruction to multiply or requantize.

**T45 (INT8 projection input)** uses scale
$s_{T45}=0.03362198546528816$ and zero point $-17$.
Converting T44 to T45 combines reconstruction and encoding:

$$
q_{T45}=\operatorname{round}\left(q_{T44}\frac{2^{-32}}{s_{T45}}\right)-17
$$

The scale ratio is approximated by $974600\times2^{-47}$, giving:

    q45 = ((q44 * 974600 + (1 << 46)) >> 47) - 17
    q45 = clip(q45, -128, 127)

Thus, requantization converts directly between integer representations
without creating an FP32 intermediate. These formats describe our
integer candidate; the saved TFLite graph retains its original tensor types.

## Q,K,V Projection

In [13]:
# Part 2, preparation:
# Read Q/K/V parameters and calculate their integer accumulators.

ops = interpreter._get_ops_details()
tensor_details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}

int32_limits = np.iinfo(np.int32)
qkv_specs = {}

for name, op_index in [("Q", 15), ("K", 16), ("V", 17)]:
    op = ops[op_index]
    assert op["op_name"] == "FULLY_CONNECTED"

    input_index = int(op["inputs"][0])
    weight_index = int(op["inputs"][1])
    output_index = int(op["outputs"][0])
    assert input_index == t45_index

    input_detail = tensor_details[input_index]
    weight_detail = tensor_details[weight_index]
    output_detail = tensor_details[output_index]

    assert input_detail["dtype"] == np.int8
    assert weight_detail["dtype"] == np.int8
    assert output_detail["dtype"] == np.int8

    input_params = input_detail["quantization_parameters"]
    weight_params = weight_detail["quantization_parameters"]
    output_params = output_detail["quantization_parameters"]

    assert len(input_params["scales"]) == 1
    assert len(input_params["zero_points"]) == 1
    assert len(output_params["scales"]) == 1
    assert len(output_params["zero_points"]) == 1

    input_scale = float(input_params["scales"][0])
    input_zero_point = int(input_params["zero_points"][0])
    output_scale = float(output_params["scales"][0])
    output_zero_point = int(output_params["zero_points"][0])
    assert input_scale > 0 and output_scale > 0

    weights_q = interpreter.get_tensor(weight_index).copy()
    assert weights_q.shape == (128, 128)

    weight_scales = np.asarray(
        weight_params["scales"], dtype=np.float64
    )
    weight_zero_points = np.asarray(
        weight_params["zero_points"], dtype=np.int64
    )

    assert weight_scales.size in (1, weights_q.shape[0])
    assert weight_zero_points.size in (1, weights_q.shape[0])
    assert np.all(weight_scales > 0)

    if weight_scales.size > 1:
        assert int(weight_params["quantized_dimension"]) == 0

    # Expand metadata to one value per output channel.
    weight_scales = np.broadcast_to(
        weight_scales, (weights_q.shape[0],)
    ).copy()
    weight_zero_points = np.broadcast_to(
        weight_zero_points, (weights_q.shape[0],)
    ).copy()

    bias_index = (
        int(op["inputs"][2]) if len(op["inputs"]) > 2 else -1
    )
    bias_q = None

    if bias_index >= 0:
        bias_detail = tensor_details[bias_index]
        assert bias_detail["dtype"] == np.int32
        bias_q = interpreter.get_tensor(bias_index).copy().reshape(-1)
        assert bias_q.size == weights_q.shape[0]

        bias_params = bias_detail["quantization_parameters"]
        assert np.all(bias_params["zero_points"] == 0)
        assert np.allclose(
            bias_params["scales"],
            input_scale * weight_scales,
            rtol=1e-5, atol=0,
        )

    real_multipliers = input_scale * weight_scales / output_scale

    qkv_specs[name] = {
        "op_index": op_index,
        "input_index": input_index,
        "weight_index": weight_index,
        "output_index": output_index,
        "input_scale": input_scale,
        "input_zero_point": input_zero_point,
        "weight_scales": weight_scales,
        "weight_zero_points": weight_zero_points,
        "output_scale": output_scale,
        "output_zero_point": output_zero_point,
        "weights_q": weights_q,
        "bias_q": bias_q,
        "real_multipliers": real_multipliers,
    }

    print(f"\nOp {op_index}: {name} projection")
    print(f"  Input T{input_index}: scale={input_scale}, "
          f"zero point={input_zero_point}")
    print(f"  Weight T{weight_index}: shape={weights_q.shape}")
    print("  Original weight scale count:",
          len(weight_params["scales"]))
    print("  Weight scale range:",
          weight_scales.min(), "to", weight_scales.max())
    print("  Weight zero points:", np.unique(weight_zero_points))
    print("  Bias:", "none" if bias_q is None else f"T{bias_index}")
    print(f"  Output T{output_index}: scale={output_scale}, "
          f"zero point={output_zero_point}")
    print("  Requantization multiplier range:",
          real_multipliers.min(), "to", real_multipliers.max())


def qkv_accumulate(input_q, spec):
    # Promote BEFORE subtracting zero points or multiplying.
    x = input_q.reshape(-1, 128).astype(np.int64)
    x = x - spec["input_zero_point"]

    w = spec["weights_q"].astype(np.int64)
    w = w - spec["weight_zero_points"][:, None]

    accumulator = x @ w.T
    if spec["bias_q"] is not None:
        accumulator = accumulator + spec["bias_q"].astype(np.int64)

    assert accumulator.min() >= int32_limits.min
    assert accumulator.max() <= int32_limits.max

    return accumulator.astype(np.int32)


qkv_cases = []
assert len(op0_cases) == len(ln1_affine_integer_trial)

for case, layernorm_trial in zip(op0_cases, ln1_affine_integer_trial):
    assert case["name"] == layernorm_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    reference_input = interpreter.get_tensor(t45_index).copy()
    candidate_input = layernorm_trial["t45_q"]

    projection_results = {}
    for name, spec in qkv_specs.items():
        projection_results[name] = {
            "output_ref": interpreter.get_tensor(
                spec["output_index"]
            ).copy(),
            "acc_local": qkv_accumulate(reference_input, spec),
            "acc_chained": qkv_accumulate(candidate_input, spec),
        }

    qkv_cases.append({
        "name": case["name"],
        "input_ref": reference_input,
        "input_candidate": candidate_input.copy(),
        "projections": projection_results,
    })


def report_qkv_accumulators(label, cases):
    print(f"\n{label}")

    for name, spec in qkv_specs.items():
        print(f"  {name} projection:")

        for key, description in [
            ("acc_local", "Reference-input accumulator"),
            ("acc_chained", "Integer-LayerNorm-input accumulator"),
        ]:
            values = np.concatenate([
                c["projections"][name][key].reshape(-1)
                for c in cases
            ])
            print(f"    {description}: "
                  f"{values.min()} to {values.max()}")

        reference = np.concatenate([
            c["projections"][name]["output_ref"].reshape(-1)
            for c in cases
        ])
        print(f"    T{spec['output_index']} (reference INT8 output): "
              f"{reference.min()} to {reference.max()}")

    print("  Accumulator INT32 range checks: pass")


report_qkv_accumulators("Benchmark", qkv_cases[:1])
report_qkv_accumulators("Calibration", qkv_cases[1:])

print("\nBenchmark workload:")
print("  MACs per projection:", 32 * 128 * 128)
print("  MACs for Q/K/V together:", 3 * 32 * 128 * 128)
print("  Output elements per projection:", 32 * 128)


Op 15: Q projection
  Input T45: scale=0.03362198546528816, zero point=-17
  Weight T27: shape=(128, 128)
  Original weight scale count: 1
  Weight scale range: 0.0017367515247315168 to 0.0017367515247315168
  Weight zero points: [0]
  Bias: none
  Output T46: scale=0.025061121210455894, zero point=-12
  Requantization multiplier range: 0.0023300248233497876 to 0.0023300248233497876

Op 16: K projection
  Input T45: scale=0.03362198546528816, zero point=-17
  Weight T25: shape=(128, 128)
  Original weight scale count: 1
  Weight scale range: 0.0017351132119074464 to 0.0017351132119074464
  Weight zero points: [0]
  Bias: none
  Output T47: scale=0.018255505710840225, zero point=-30
  Requantization multiplier range: 0.003195635996911343 to 0.003195635996911343

Op 17: V projection
  Input T45: scale=0.03362198546528816, zero point=-17
  Weight T26: shape=(128, 128)
  Original weight scale count: 1
  Weight scale range: 0.0006367104360833764 to 0.0006367104360833764
  Weight zero point


Benchmark
  Q projection:
    Reference-input accumulator: -37136 to 61267
    Integer-LayerNorm-input accumulator: -37350 to 61266
    T46 (reference INT8 output): -99 to 127
  K projection:
    Reference-input accumulator: -29880 to 47705
    Integer-LayerNorm-input accumulator: -29896 to 47718
    T47 (reference INT8 output): -125 to 122
  V projection:
    Reference-input accumulator: -33514 to 33939
    Integer-LayerNorm-input accumulator: -33357 to 33587
    T48 (reference INT8 output): -99 to 109
  Accumulator INT32 range checks: pass

Calibration
  Q projection:
    Reference-input accumulator: -49709 to 60970
    Integer-LayerNorm-input accumulator: -50003 to 60812
    T46 (reference INT8 output): -128 to 127
  K projection:
    Reference-input accumulator: -30751 to 49734
    Integer-LayerNorm-input accumulator: -30862 to 49670
    T47 (reference INT8 output): -128 to 127
  V projection:
    Reference-input accumulator: -42909 to 39839
    Integer-LayerNorm-input accumulator

In [14]:
# Q/K/V requantization and output comparison.
# Rounding: nearest, ties toward positive infinity.

QKV_REQUANT_SHIFT = 39
QKV_ROUNDING_OFFSET = 1 << (QKV_REQUANT_SHIFT - 1)

int32_max = int(np.iinfo(np.int32).max)
int64_max = int(np.iinfo(np.int64).max)

for name, spec in qkv_specs.items():
    rounded = np.rint(
        spec["real_multipliers"] * (1 << QKV_REQUANT_SHIFT)
    )
    assert np.all(rounded > 0)
    assert rounded.max() <= int32_max

    spec["multiplier_q"] = rounded.astype(np.int32)

    # Bound covers the full signed INT32 accumulator range.
    largest_multiplier = int(spec["multiplier_q"].max())
    assert (
        (1 << 31) * largest_multiplier + QKV_ROUNDING_OFFSET
        <= int64_max
    )

    represented_multiplier = (
        spec["multiplier_q"].astype(np.float64)
        * 2.0 ** (-QKV_REQUANT_SHIFT)
    )
    relative_error = np.abs(
        represented_multiplier / spec["real_multipliers"] - 1
    )

    print(f"\n{name} projection:")
    print("  Multiplier integers:", np.unique(spec["multiplier_q"]))
    print("  Right shift:", QKV_REQUANT_SHIFT)
    print(f"  Maximum multiplier relative error: {relative_error.max():.9e}")


def qkv_requantize(accumulator, spec, output_shape):
    # Promote before multiplying.
    product = (
        accumulator.astype(np.int64)
        * spec["multiplier_q"].astype(np.int64)[None, :]
    )
    rounded = (
        product + QKV_ROUNDING_OFFSET
    ) >> QKV_REQUANT_SHIFT

    before_saturation = rounded + spec["output_zero_point"]
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))

    output = np.clip(before_saturation, -128, 127).astype(np.int8)
    return output.reshape(output_shape), saturation_count


qkv_integer_trial = []

for case in qkv_cases:
    results = {}

    for name, spec in qkv_specs.items():
        prepared = case["projections"][name]
        reference = prepared["output_ref"]

        output_local, saturation_local = qkv_requantize(
            prepared["acc_local"], spec, reference.shape
        )
        output_chained, saturation_chained = qkv_requantize(
            prepared["acc_chained"], spec, reference.shape
        )

        # Cast before subtraction to avoid INT8 overflow.
        results[name] = {
            "output_ref": reference,
            "output_local": output_local,
            "output_chained": output_chained,
            "local_difference": (
                output_local.astype(np.int32)
                - reference.astype(np.int32)
            ),
            "chained_difference": (
                output_chained.astype(np.int32)
                - reference.astype(np.int32)
            ),
            "propagation_difference": (
                output_chained.astype(np.int32)
                - output_local.astype(np.int32)
            ),
            "saturation_local": saturation_local,
            "saturation_chained": saturation_chained,
        }

    qkv_integer_trial.append({
        "name": case["name"],
        "projections": results,
    })


def report_qkv_outputs(label, trials):
    print(f"\n{label}")

    for name, spec in qkv_specs.items():
        print(f"  {name} → T{spec['output_index']}:")

        for key, description in [
            ("local_difference", "Local vs TFLite"),
            ("chained_difference", "Chained vs TFLite"),
            ("propagation_difference", "Chained vs local"),
        ]:
            differences = np.concatenate([
                t["projections"][name][key].reshape(-1)
                for t in trials
            ])
            matched = int(np.count_nonzero(differences == 0))
            print(
                f"    {description}: "
                f"matches={matched}/{differences.size} "
                f"({100 * matched / differences.size:.3f}%), "
                f"max INT8-step difference={np.abs(differences).max()}, "
                f"mean absolute difference={np.abs(differences).mean():.6f}"
            )

        print("    Local saturation count:", sum(
            t["projections"][name]["saturation_local"] for t in trials
        ))
        print("    Chained saturation count:", sum(
            t["projections"][name]["saturation_chained"] for t in trials
        ))


report_qkv_outputs("Benchmark", qkv_integer_trial[:1])
report_qkv_outputs("Calibration", qkv_integer_trial[1:])

print("\nFirst benchmark token, first 8 output features:")
for name in qkv_specs:
    result = qkv_integer_trial[0]["projections"][name]
    print(f"  {name} reference:", result["output_ref"][0, 0, :8])
    print(f"  {name} local:    ", result["output_local"][0, 0, :8])
    print(f"  {name} chained:  ", result["output_chained"][0, 0, :8])


Q projection:
  Multiplier integers: [1280944693]
  Right shift: 39
  Maximum multiplier relative error: 1.092208546e-10

K projection:
  Multiplier integers: [1756819468]
  Right shift: 39
  Maximum multiplier relative error: 2.116242737e-10

V projection:
  Multiplier integers: [1696087205]
  Right shift: 39
  Maximum multiplier relative error: 1.885475109e-10

Benchmark
  Q → T46:
    Local vs TFLite: matches=4096/4096 (100.000%), max INT8-step difference=0, mean absolute difference=0.000000
    Chained vs TFLite: matches=3276/4096 (79.980%), max INT8-step difference=2, mean absolute difference=0.202148
    Chained vs local: matches=3276/4096 (79.980%), max INT8-step difference=2, mean absolute difference=0.202148
    Local saturation count: 1
    Chained saturation count: 1
  K → T47:
    Local vs TFLite: matches=4096/4096 (100.000%), max INT8-step difference=0, mean absolute difference=0.000000
    Chained vs TFLite: matches=3001/4096 (73.267%), max INT8-step difference=2, mean a

### Block-0 Q/K/V projections: integer validation, Op 15–17

Source block-0 SHA-256:
`06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9`

Test inputs: one 32-token benchmark sequence and seven calibration
sequences; hidden size 128.

All three projections receive **T45 (INT8 LayerNorm output)**:
shape `[1,32,128]`, scale `0.03362198546528816`, zero point `-17`.

Each projection has an INT8 weight matrix of shape `[128,128]`,
one weight scale, weight zero point 0, and no bias.
Each final output is INT8 with shape `[1,32,128]`.

| Projection | Weight tensor | Weight scale | Output tensor | Output scale | Output zero point |
|---|---|---:|---|---:|---:|
| Q | T27 | 0.0017367515247315168 | T46 | 0.025061121210455894 | -12 |
| K | T25 | 0.0017351132119074464 | T47 | 0.018255505710840225 | -30 |
| V | T26 | 0.0006367104360833764 | T48 | 0.006938841659575701 | 4 |

#### Accumulation and requantization

For token position $t$ and output feature $o$:

$$
A[t,o]=\sum_{k=0}^{127}(q_{T45}[t,k]+17)\,q_w[o,k]
$$

Accumulators use INT32 storage. Operands are promoted before arithmetic;
the Python validation calculates sums in INT64 and checks INT32 bounds.

The output conversion multiplier is:

$$
M=\frac{s_{\text{input}}s_{\text{weight}}}{s_{\text{output}}}
\approx M_q\,2^{-39}
$$

| Projection | Real multiplier | INT32 multiplier $M_q$ | Right shift |
|---|---:|---:|---:|
| Q | 0.0023300248233497876 | 1280944693 | 39 |
| K | 0.003195635996911343 | 1756819468 | 39 |
| V | 0.003085164653966409 | 1696087205 | 39 |

Integer output calculation:

    output_q = ((accumulator * multiplier_q + (1 << 38)) >> 39)
    output_q = clip(output_q + output_zero_point, -128, 127)

The multiplier product uses INT64 arithmetic.
Rounding is nearest, with ties toward positive infinity.

**The multiplier's scale $2^{-39}$ is separate from the output tensor
scale.** T46/T47/T48 retain the output scales and zero points above.

#### Validation results

- **Local:** use TFLite's T45 to validate the projection calculation.
- **Chained:** use our integer LayerNorm's T45 to measure propagated error.

| Projection | Local match: benchmark | Local match: calibration |
|---|---:|---:|
| Q | 4096/4096 | 28672/28672 |
| K | 4096/4096 | 28672/28672 |
| V | 4096/4096 | 28672/28672 |

All **98,304 local output values** matched the quantized TFLite reference.

| Projection | Chained match: benchmark | Chained match: calibration | Maximum difference: benchmark | Maximum difference: calibration |
|---|---:|---:|---:|---:|
| Q | 79.980% | 80.720% | 2 INT8 steps | 3 INT8 steps |
| K | 73.267% | 74.090% | 2 INT8 steps | 2 INT8 steps |
| V | 65.234% | 65.726% | 2 INT8 steps | 2 INT8 steps |

Chained differences originate from the changed LayerNorm input.
These percentages measure activation agreement, not model accuracy.

All accumulator range checks passed for the tested inputs.
The requantization product bounds were also checked for the full
signed INT32 accumulator range.

#### Workload and status

For one 32-token sequence:

- MACs per projection: **524,288**.
- Total Q/K/V MACs: **1,572,864**.
- Output elements per projection: **4,096**.

Q/K/V calculation is locally validated in Python on the tested cases.
C, Spike, RTL, and projection performance have not yet been validated
for these new exports. Attention is being analyzed separately.

### Attention


In [15]:
# Part 3A: Op 18–28
# Rearrange Q/K/V into heads and calculate integer attention scores.
# Masking and softmax will follow after inspecting these results.

ops = interpreter._get_ops_details()
assert ops[28]["op_name"] == "BATCH_MATMUL"

score_index = int(ops[28]["outputs"][0])
query_head_index = int(ops[28]["inputs"][0])
key_head_index = int(ops[28]["inputs"][1])
value_head_index = int(ops[22]["outputs"][0])

Q_ZERO_POINT = qkv_specs["Q"]["output_zero_point"]
K_ZERO_POINT = qkv_specs["K"]["output_zero_point"]
V_ZERO_POINT = qkv_specs["V"]["output_zero_point"]

ATTENTION_SCORE_SCALE = (
    qkv_specs["Q"]["output_scale"]
    * qkv_specs["K"]["output_scale"]
)
ATTENTION_SCORE_ZERO_POINT = 0

int32_limits = np.iinfo(np.int32)

# Conservative bound for all possible INT8 Q/K inputs.
q_centered_bound = max(
    abs(-128 - Q_ZERO_POINT), abs(127 - Q_ZERO_POINT)
)
k_centered_bound = max(
    abs(-128 - K_ZERO_POINT), abs(127 - K_ZERO_POINT)
)
score_bound = 8 * q_centered_bound * k_centered_bound
assert score_bound <= int32_limits.max


def prepare_attention_heads(projections, output_key):
    # Seed the graph traversal with our INT8 projection outputs.
    tensors = {
        spec["output_index"]: projections[name][output_key]
        for name, spec in qkv_specs.items()
    }

    # Follow the actual exported shape constants and permutations.
    for op_index in range(18, 28):
        op = ops[op_index]
        source_index = int(op["inputs"][0])
        destination_index = int(op["outputs"][0])
        source = tensors[source_index]

        if op["op_name"] == "RESHAPE":
            shape = tuple(int(v) for v in interpreter.get_tensor(
                int(op["inputs"][1])
            ).reshape(-1))
            tensors[destination_index] = source.reshape(shape)

        elif op["op_name"] == "TRANSPOSE":
            permutation = tuple(int(v) for v in interpreter.get_tensor(
                int(op["inputs"][1])
            ).reshape(-1))
            tensors[destination_index] = source.transpose(permutation)

        elif op["op_name"] == "DEQUANTIZE":
            # Integer candidate retains the quantized representation.
            # Its scale/zero point are applied through score metadata
            # and zero-point subtraction, rather than FP32 conversion.
            tensors[destination_index] = source

        else:
            raise ValueError(
                f"Unexpected Op {op_index}: {op['op_name']}"
            )

    query_q = tensors[query_head_index]
    key_q = tensors[key_head_index]
    value_q = tensors[value_head_index].reshape(16, 32, 8)

    assert query_q.shape == (16, 32, 8)
    assert key_q.shape == (16, 8, 32)
    assert value_q.shape == (16, 32, 8)

    return query_q, key_q, value_q, tensors


attention_score_trial = []
assert len(op0_cases) == len(qkv_integer_trial)

for case, projection_trial in zip(op0_cases, qkv_integer_trial):
    assert case["name"] == projection_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    score_ref = interpreter.get_tensor(score_index).copy()
    assert score_ref.shape == (16, 32, 32)

    paths = {}

    for path, output_key in [
        ("local", "output_local"),
        ("chained", "output_chained"),
    ]:
        query_q, key_q, value_q, layout_tensors = prepare_attention_heads(
            projection_trial["projections"], output_key
        )

        # Remove zero points after promotion.
        query_centered = query_q.astype(np.int64) - Q_ZERO_POINT
        key_centered = key_q.astype(np.int64) - K_ZERO_POINT

        # Integer dot products, using wide arithmetic for checking.
        score_wide = query_centered @ key_centered
        assert score_wide.min() >= int32_limits.min
        assert score_wide.max() <= int32_limits.max
        score_q = score_wide.astype(np.int32)

        # Diagnostic reconstruction only.
        score_approx = (
            score_q.astype(np.float64) * ATTENTION_SCORE_SCALE
        )

        # Local projection outputs already matched TFLite exactly.
        # Check all INT8 reshape/transpose outputs too.
        if path == "local":
            for op_index in range(18, 28):
                if ops[op_index]["op_name"] in ("RESHAPE", "TRANSPOSE"):
                    output_index = int(ops[op_index]["outputs"][0])
                    np.testing.assert_array_equal(
                        layout_tensors[output_index],
                        interpreter.get_tensor(output_index),
                    )

        paths[path] = {
            "query_head_q": query_q.copy(),
            "key_head_q": key_q.copy(),
            "value_head_q": value_q.copy(),
            "score_q": score_q,
            "score_approx": score_approx,
            "error": np.abs(score_approx - score_ref),
        }

    attention_score_trial.append({
        "name": case["name"],
        "score_ref": score_ref,
        "paths": paths,
    })


def report_attention_scores(label, trials):
    print(f"\n{label}")
    reference = np.concatenate([
        t["score_ref"].reshape(-1) for t in trials
    ])
    print(
        f"  T{score_index} (TFLite FP32 attention scores) range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )

    for path in ["local", "chained"]:
        integers = np.concatenate([
            t["paths"][path]["score_q"].reshape(-1) for t in trials
        ])
        errors = np.concatenate([
            t["paths"][path]["error"].reshape(-1) for t in trials
        ])
        print(f"  {path.capitalize()} integer score range:",
              f"{integers.min()} to {integers.max()}")
        print(f"    Max absolute error vs TFLite:  {errors.max():.9e}")
        print(f"    Mean absolute error vs TFLite: {errors.mean():.9e}")

    print("  Local INT8 layout checks: pass")
    print("  Score INT32 range checks: pass")


print("Q head shape: (16, 32, 8)")
print("K transposed head shape: (16, 8, 32)")
print("V head shape: (16, 32, 8)")
print(f"T{score_index} candidate scores: INT32, shape (16, 32, 32)")
print("Score scale:", ATTENTION_SCORE_SCALE)
print("Score zero point:", ATTENTION_SCORE_ZERO_POINT)
print("Full-range integer score bound:", score_bound)
print("Score output requantization: none")
print("Benchmark QK-transpose MACs:", 16 * 32 * 32 * 8)

report_attention_scores("Benchmark", attention_score_trial[:1])
report_attention_scores("Calibration", attention_score_trial[1:])

first = attention_score_trial[0]
print("\nHead 0, query token 0, first 8 key-token scores:")
print("  TFLite reference:", first["score_ref"][0, 0, :8])
print("  Local represented:",
      first["paths"]["local"]["score_approx"][0, 0, :8])
print("  Chained represented:",
      first["paths"]["chained"]["score_approx"][0, 0, :8])

Q head shape: (16, 32, 8)
K transposed head shape: (16, 8, 32)
V head shape: (16, 32, 8)
T59 candidate scores: INT32, shape (16, 32, 32)
Score scale: 0.0004575034413775367
Score zero point: 0
Full-range integer score bound: 174584
Score output requantization: none
Benchmark QK-transpose MACs: 131072

Benchmark
  T59 (TFLite FP32 attention scores) range: -6.657589912 to 5.850554466
  Local integer score range: -14552 to 12788
    Max absolute error vs TFLite:  5.669663921e-07
    Mean absolute error vs TFLite: 2.918397286e-08
  Chained integer score range: -14373 to 12614
    Max absolute error vs TFLite:  1.958116131e-01
    Mean absolute error vs TFLite: 1.579852991e-02
  Local INT8 layout checks: pass
  Score INT32 range checks: pass

Calibration
  T59 (TFLite FP32 attention scores) range: -8.119771004 to 5.561869621
  Local integer score range: -17748 to 12157
    Max absolute error vs TFLite:  9.951586715e-07
    Mean absolute error vs TFLite: 2.955508351e-08
  Chained integer scor

In [16]:
# Attention softmax preparation:
# Integer causal masking and maximum subtraction.
# FP64 exponential/normalization are diagnostics only.

ops = interpreter._get_ops_details()
assert ops[29]["op_name"] == "RESHAPE"
assert ops[30]["op_name"] == "SELECT_V2"
assert ops[31]["op_name"] == "SOFTMAX"
assert ops[32]["op_name"] == "RESHAPE"

mask_index = int(ops[30]["inputs"][0])
probability_index = int(ops[31]["outputs"][0])

attention_mask = interpreter.get_tensor(mask_index).copy()
assert attention_mask.dtype == np.bool_
assert attention_mask.shape == (1, 1, 32, 32)

# Confirm the saved mask is the expected causal mask.
np.testing.assert_array_equal(
    attention_mask[0, 0],
    np.tri(32, dtype=bool),
)

score_shape = (1, 16, 32, 32)
allowed = np.broadcast_to(attention_mask, score_shape)
assert np.all(np.any(allowed, axis=-1))

int32_limits = np.iinfo(np.int32)
attention_softmax_inputs = []

assert len(op0_cases) == len(attention_score_trial)

for case, score_trial in zip(op0_cases, attention_score_trial):
    assert case["name"] == score_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    probability_ref = interpreter.get_tensor(probability_index).copy()
    assert probability_ref.shape == score_shape
    assert np.all(probability_ref[~allowed] == 0)

    # Reference score offsets, for diagnostic comparison.
    reference_scores = score_trial["score_ref"].astype(
        np.float64
    ).reshape(score_shape)
    reference_maximum = np.max(
        np.where(allowed, reference_scores, -np.inf),
        axis=-1,
        keepdims=True,
    )
    reference_shifted = reference_scores - reference_maximum

    paths = {}

    for path in ["local", "chained"]:
        score_q = score_trial["paths"][path]["score_q"].reshape(
            score_shape
        )

        # Exclude masked entries when selecting each row's maximum.
        row_maximum_q = np.max(
            np.where(allowed, score_q, int32_limits.min),
            axis=-1,
            keepdims=True,
        )

        shifted_wide = (
            score_q.astype(np.int64)
            - row_maximum_q.astype(np.int64)
        )

        # Masked entries use a placeholder 0 and MUST be ignored.
        shifted_wide = np.where(allowed, shifted_wide, 0)

        assert shifted_wide.min() >= int32_limits.min
        assert shifted_wide.max() <= int32_limits.max
        shifted_score_q = shifted_wide.astype(np.int32)
        assert np.all(shifted_score_q[allowed] <= 0)

        # Every row has at least one allowed maximum, with offset 0.
        assert np.all(np.max(
            np.where(allowed, shifted_score_q, int32_limits.min),
            axis=-1,
        ) == 0)

        # Diagnostics only: reconstruct offsets and calculate ideal
        # softmax for the candidate scores.
        shifted_real = (
            shifted_score_q.astype(np.float64)
            * ATTENTION_SCORE_SCALE
        )
        diagnostic_exp = np.where(
            allowed, np.exp(shifted_real), 0.0
        )
        denominator = diagnostic_exp.sum(axis=-1, keepdims=True)
        assert np.all(denominator > 0)

        diagnostic_probabilities = diagnostic_exp / denominator
        np.testing.assert_allclose(
            diagnostic_probabilities.sum(axis=-1),
            1.0, rtol=0, atol=1e-12,
        )
        assert np.all(diagnostic_probabilities[~allowed] == 0)

        paths[path] = {
            "row_maximum_q": row_maximum_q,
            "shifted_score_q": shifted_score_q,
            "shifted_real": shifted_real,
            "shift_error": np.abs(
                shifted_real[allowed] - reference_shifted[allowed]
            ),
            "diagnostic_probabilities": diagnostic_probabilities,
            "probability_error": np.abs(
                diagnostic_probabilities - probability_ref
            ),
        }

    attention_softmax_inputs.append({
        "name": case["name"],
        "probability_ref": probability_ref,
        "paths": paths,
    })


def report_softmax_inputs(label, trials):
    print(f"\n{label}")

    for path in ["local", "chained"]:
        shifted_integers = np.concatenate([
            t["paths"][path]["shifted_score_q"][allowed]
            for t in trials
        ])
        shifted_values = np.concatenate([
            t["paths"][path]["shifted_real"][allowed]
            for t in trials
        ])
        probability_errors = np.concatenate([
            t["paths"][path]["probability_error"].reshape(-1)
            for t in trials
        ])

        print(f"  {path.capitalize()} path:")
        print(
            "    Allowed maximum-subtracted integer range:",
            f"{shifted_integers.min()} to {shifted_integers.max()}",
        )
        print(
            "    Allowed represented range:",
            f"{shifted_values.min():.9f} to {shifted_values.max():.9f}",
        )

        for cutoff in [-8, -12, -16]:
            count = int(np.count_nonzero(shifted_values < cutoff))
            print(f"    Allowed values below {cutoff}: {count}")

        print(
            "    Diagnostic softmax max error vs TFLite:",
            f"{probability_errors.max():.9e}",
        )
        print(
            "    Diagnostic softmax mean error vs TFLite:",
            f"{probability_errors.mean():.9e}",
        )

    print("  Integer maximum subtraction and mask checks: pass")


print(f"T{mask_index} (causal mask):", attention_mask.shape)
print("Allowed entries per head:", int(attention_mask.sum()))
print("Masked entries per head:", int((~attention_mask).sum()))
print("Maximum-subtracted score scale:", ATTENTION_SCORE_SCALE)
print("Maximum-subtracted score storage: INT32")
print("Integer exponential implementation: pending")

report_softmax_inputs("Benchmark", attention_softmax_inputs[:1])
report_softmax_inputs("Calibration", attention_softmax_inputs[1:])

first = attention_softmax_inputs[0]
print("\nHead 0, query token 0, first 8 probabilities:")
print("  TFLite reference:", first["probability_ref"][0, 0, 0, :8])
print("  Local diagnostic:",
      first["paths"]["local"]["diagnostic_probabilities"][0, 0, 0, :8])
print("  Chained diagnostic:",
      first["paths"]["chained"]["diagnostic_probabilities"][0, 0, 0, :8])

T21 (causal mask): (1, 1, 32, 32)
Allowed entries per head: 528
Masked entries per head: 496
Maximum-subtracted score scale: 0.0004575034413775367
Maximum-subtracted score storage: INT32
Integer exponential implementation: pending

Benchmark
  Local path:
    Allowed maximum-subtracted integer range: -20410 to 0
    Allowed represented range: -9.337645239 to 0.000000000
    Allowed values below -8: 3
    Allowed values below -12: 0
    Allowed values below -16: 0
    Diagnostic softmax max error vs TFLite: 7.899361815e-08
    Diagnostic softmax mean error vs TFLite: 1.958477854e-09
  Chained path:
    Allowed maximum-subtracted integer range: -20190 to 0
    Allowed represented range: -9.236994481 to 0.000000000
    Allowed values below -8: 2
    Allowed values below -12: 0
    Allowed values below -16: 0
    Diagnostic softmax max error vs TFLite: 2.590189826e-02
    Diagnostic softmax mean error vs TFLite: 3.359444158e-04
  Integer maximum subtraction and mask checks: pass

Calibrati

In [17]:
# Op 31–32: integer LUT softmax and probability reshape.

EXP_FRAC_BITS = 30
EXP_SCALE = 2.0 ** (-EXP_FRAC_BITS)

PROB_FRAC_BITS = 16
PROB_SCALE = 2.0 ** (-PROB_FRAC_BITS)
PROB_ONE_Q = 1 << PROB_FRAC_BITS

# Table covers exp(-u), where 0 <= u <= 16.
LUT_LIMIT = 16
LUT_INDEX_FRAC_BITS = 8          # Table spacing: 1/256.
LUT_COORD_FRAC_BITS = 16         # Coordinate spacing: 1/65536.
LUT_INTERP_BITS = (
    LUT_COORD_FRAC_BITS - LUT_INDEX_FRAC_BITS
)
LUT_INTERVALS = LUT_LIMIT * (1 << LUT_INDEX_FRAC_BITS)
LUT_CUTOFF_Q = LUT_LIMIT << LUT_COORD_FRAC_BITS

# OFFLINE: generate integer table values.
lut_coordinates = (
    np.arange(LUT_INTERVALS + 1, dtype=np.float64)
    / (1 << LUT_INDEX_FRAC_BITS)
)
exp_lut_q = np.rint(
    np.exp(-lut_coordinates) * (1 << EXP_FRAC_BITS)
).astype(np.int32)

assert exp_lut_q[0] == 1 << EXP_FRAC_BITS
assert np.all(np.diff(exp_lut_q.astype(np.int64)) <= 0)
assert np.all(exp_lut_q >= 0)

# OFFLINE: convert the score scale into an integer coordinate multiplier.
# u_coord_q ≈ (-shifted_score_q) * score_scale * 2^16
EXP_COORD_SHIFT = 24
EXP_COORD_MULTIPLIER_Q = int(np.rint(
    ATTENTION_SCORE_SCALE
    * (1 << LUT_COORD_FRAC_BITS)
    * (1 << EXP_COORD_SHIFT)
))

int32_limits = np.iinfo(np.int32)
int64_max = int(np.iinfo(np.int64).max)

assert 0 < EXP_COORD_MULTIPLIER_Q <= int32_limits.max

# This bound covers any nonpositive INT32 shifted score.
assert (
    (1 << 31) * EXP_COORD_MULTIPLIER_Q
    + (1 << (EXP_COORD_SHIFT - 1))
    <= int64_max
)

ops = interpreter._get_ops_details()
assert ops[31]["op_name"] == "SOFTMAX"
assert ops[32]["op_name"] == "RESHAPE"

probability_matrix_index = int(ops[32]["outputs"][0])
probability_matrix_shape = tuple(
    int(v) for v in interpreter.get_tensor(
        int(ops[32]["inputs"][1])
    ).reshape(-1)
)
assert probability_matrix_shape == (16, 32, 32)

softmax_allowed = np.broadcast_to(
    attention_mask, (1, 16, 32, 32)
)


def integer_lut_softmax(shifted_score_q, allowed_mask):
    assert shifted_score_q.dtype == np.int32
    assert shifted_score_q.shape == allowed_mask.shape
    assert np.all(shifted_score_q[allowed_mask] <= 0)
    assert np.all(np.any(allowed_mask, axis=-1))

    # Integer conversion to positive table coordinates.
    magnitude_q = -shifted_score_q.astype(np.int64)
    coordinate_q = (
        magnitude_q * EXP_COORD_MULTIPLIER_Q
        + (1 << (EXP_COORD_SHIFT - 1))
    ) >> EXP_COORD_SHIFT

    active = allowed_mask & (coordinate_q < LUT_CUTOFF_Q)
    cutoff_count = int(np.count_nonzero(
        allowed_mask & ~active
    ))

    # Safe indices, including entries that will subsequently become zero.
    safe_coordinate = np.clip(
        coordinate_q, 0, LUT_CUTOFF_Q - 1
    )
    index = safe_coordinate >> LUT_INTERP_BITS
    fraction = safe_coordinate & ((1 << LUT_INTERP_BITS) - 1)

    lower = exp_lut_q[index].astype(np.int64)
    upper = exp_lut_q[index + 1].astype(np.int64)

        # Linear interpolation; nearest rounding, ties toward +infinity.
    interpolated = lower + (
        (
            (upper - lower) * fraction
            + (1 << (LUT_INTERP_BITS - 1))
        ) >> LUT_INTERP_BITS
    )

    exp_wide = np.where(active, interpolated, 0)
    assert exp_wide.min() >= 0
    assert exp_wide.max() <= 1 << EXP_FRAC_BITS
    exp_q = exp_wide.astype(np.int32)

    sum_exp_q = np.sum(
        exp_q, axis=-1, keepdims=True, dtype=np.int64
    )
    assert np.all(sum_exp_q > 0)
    assert sum_exp_q.max() <= 32 * (1 << EXP_FRAC_BITS)

    # exp_q * 2^16 is at most 2^46; use INT64 arithmetic.
    numerator = exp_q.astype(np.int64) * PROB_ONE_Q
    assert int(numerator.max()) + int(sum_exp_q.max()) // 2 <= int64_max

    # Positive integer division with nearest rounding.
    probability_wide = (
        numerator + sum_exp_q // 2
    ) // sum_exp_q

    assert probability_wide.min() >= 0
    assert probability_wide.max() <= PROB_ONE_Q
    probability_q = probability_wide.astype(np.int32)
    assert np.all(probability_q[~allowed_mask] == 0)

    # Independent rounding may slightly change the row sum.
    row_sum_q = probability_q.sum(axis=-1, dtype=np.int64)
    assert np.abs(row_sum_q - PROB_ONE_Q).max() <= 16

    # First query token has exactly one allowed key.
    assert np.all(probability_q[0, :, 0, 0] == PROB_ONE_Q)
    assert np.all(probability_q[0, :, 0, 1:] == 0)

    return probability_q, exp_q, sum_exp_q, cutoff_count


softmax_integer_trial = []

for prepared in attention_softmax_inputs:
    paths = {}

    for path in ["local", "chained"]:
        source = prepared["paths"][path]

        probability_q, exp_q, sum_exp_q, cutoff_count = (
            integer_lut_softmax(
                source["shifted_score_q"], softmax_allowed
            )
        )

        # Op 32: reshape without changing values or scale.
        probability_matrix_q = probability_q.reshape(
            probability_matrix_shape
        )

        # Diagnostics only.
        probability_approx = (
            probability_q.astype(np.float64) * PROB_SCALE
        )
        own_error = np.abs(
            probability_approx - source["diagnostic_probabilities"]
        )
        total_error = np.abs(
            probability_approx - prepared["probability_ref"]
        )
        row_sum_error_q = (
            probability_q.sum(axis=-1, dtype=np.int64) - PROB_ONE_Q
        )

        paths[path] = {
            "probability_q": probability_q,
            "probability_matrix_q": probability_matrix_q,
            "probability_approx": probability_approx,
            "exp_q": exp_q,
            "sum_exp_q": sum_exp_q,
            "own_error": own_error,
            "total_error": total_error,
            "row_sum_error_q": row_sum_error_q,
            "cutoff_count": cutoff_count,
        }

    softmax_integer_trial.append({
        "name": prepared["name"],
        "probability_ref": prepared["probability_ref"],
        "paths": paths,
    })


def report_integer_softmax(label, trials):
    print(f"\n{label}")

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        for key, description in [
            ("own_error", "Integer softmax's own approximation error"),
            ("total_error", "Total probability error vs TFLite"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1) for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        row_errors = np.concatenate([
            t["paths"][path]["row_sum_error_q"].reshape(-1)
            for t in trials
        ])
        print("    Maximum row-sum error, integer steps:",
              int(np.abs(row_errors).max()))
        print("    Maximum represented row-sum error:",
              f"{np.abs(row_errors).max() * PROB_SCALE:.9e}")
        print("    Exponentials discarded by cutoff:",
              sum(t["paths"][path]["cutoff_count"] for t in trials))

    print("  Masked probabilities: exactly zero")
    print("  Width and probability-range checks: pass")


print("Exponential LUT entries:", exp_lut_q.size)
print("Exponential LUT bytes:", exp_lut_q.nbytes)
print("Exponential value scale:", EXP_SCALE)
print("Coordinate multiplier:", EXP_COORD_MULTIPLIER_Q)
print("Coordinate right shift:", EXP_COORD_SHIFT)
print(f"T{probability_index} (probabilities): INT32, scale =", PROB_SCALE)
print(f"T{probability_matrix_index} (probability matrices):",
      probability_matrix_shape)
print("Stored probability 1:", PROB_ONE_Q)

report_integer_softmax("Benchmark", softmax_integer_trial[:1])
report_integer_softmax("Calibration", softmax_integer_trial[1:])

first = softmax_integer_trial[0]
print("\nHead 0, query token 1, first 4 probabilities:")
print("  TFLite reference:", first["probability_ref"][0, 0, 1, :4])
for path in ["local", "chained"]:
    print(f"  {path} integers:",
          first["paths"][path]["probability_q"][0, 0, 1, :4])
    print(f"  {path} represented:",
          first["paths"][path]["probability_approx"][0, 0, 1, :4])

Exponential LUT entries: 4097
Exponential LUT bytes: 16388
Exponential value scale: 9.313225746154785e-10
Coordinate multiplier: 503030354
Coordinate right shift: 24
T62 (probabilities): INT32, scale = 1.52587890625e-05
T63 (probability matrices): (16, 32, 32)
Stored probability 1: 65536

Benchmark
  Local path:
    Integer softmax's own approximation error:
      Max:  8.565865151e-06
      Mean: 1.967145974e-06
    Total probability error vs TFLite:
      Max:  8.583068848e-06
      Mean: 1.967082174e-06
    Maximum row-sum error, integer steps: 4
    Maximum represented row-sum error: 6.103515625e-05
    Exponentials discarded by cutoff: 0
  Chained path:
    Integer softmax's own approximation error:
      Max:  8.631592864e-06
      Mean: 1.966028624e-06
    Total probability error vs TFLite:
      Max:  2.590093017e-02
      Mean: 3.359635556e-04
    Maximum row-sum error, integer steps: 5
    Maximum represented row-sum error: 7.629394531e-05
    Exponentials discarded by cutoff

In [18]:
# Part 3C, preparation: Op 33–35.
# Calculate integer probability × V, then inspect T67.
# Existing exports and earlier trial results are read only.

assert hashlib.sha256(BLOCK0_PATH.read_bytes()).hexdigest() == (
    "06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9"
)

context_ops = interpreter._get_ops_details()
context_details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}

for op_index, expected in [
    (33, "RESHAPE"),
    (34, "DEQUANTIZE"),
    (35, "BATCH_MATMUL"),
    (36, "QUANTIZE"),
]:
    assert context_ops[op_index]["op_name"] == expected

value_matrix_index = int(context_ops[33]["outputs"][0])  # T64
value_float_index = int(context_ops[34]["outputs"][0])   # T65
context_index = int(context_ops[35]["outputs"][0])       # T66
context_quant_index = int(context_ops[36]["outputs"][0]) # T67

assert value_matrix_index == 64
assert value_float_index == 65
assert context_index == 66
assert context_quant_index == 67

assert int(context_ops[33]["inputs"][0]) == value_head_index
assert int(context_ops[34]["inputs"][0]) == value_matrix_index
assert tuple(int(v) for v in context_ops[35]["inputs"]) == (
    probability_matrix_index, value_float_index
)
assert int(context_ops[36]["inputs"][0]) == context_index

value_matrix_shape = tuple(
    int(v) for v in interpreter.get_tensor(
        int(context_ops[33]["inputs"][1])
    ).reshape(-1)
)
assert value_matrix_shape == (16, 32, 8)

# Read V metadata from T64, the actual input to Op 34.
value_detail = context_details[value_matrix_index]
assert value_detail["dtype"] == np.int8

value_params = value_detail["quantization_parameters"]
assert len(value_params["scales"]) == 1
assert len(value_params["zero_points"]) == 1

CONTEXT_V_SCALE = float(value_params["scales"][0])
CONTEXT_V_ZERO_POINT = int(value_params["zero_points"][0])

assert CONTEXT_V_SCALE == qkv_specs["V"]["output_scale"]
assert CONTEXT_V_ZERO_POINT == qkv_specs["V"]["output_zero_point"]
assert CONTEXT_V_ZERO_POINT == 4

CONTEXT_SCALE = PROB_SCALE * CONTEXT_V_SCALE
CONTEXT_ZERO_POINT = 0

# Read T67 metadata from the fixed saved export.
context_quant_detail = context_details[context_quant_index]
assert context_quant_detail["dtype"] == np.int8
assert tuple(context_quant_detail["shape"]) == (16, 32, 8)

context_quant_params = context_quant_detail["quantization_parameters"]
assert len(context_quant_params["scales"]) == 1
assert len(context_quant_params["zero_points"]) == 1

T67_SCALE = float(context_quant_params["scales"][0])
T67_ZERO_POINT = int(context_quant_params["zero_points"][0])
assert np.isfinite(T67_SCALE) and T67_SCALE > 0

# Offline metadata calculation, for the next stage.
T67_REAL_MULTIPLIER = CONTEXT_SCALE / T67_SCALE

context_int32_limits = np.iinfo(np.int32)

# Conservative bound: 32 products, each probability <= 65536.
# This does not even rely on the near-unit probability row sums.
value_centered_bound = max(
    abs(-128 - CONTEXT_V_ZERO_POINT),
    abs(127 - CONTEXT_V_ZERO_POINT),
)
context_full_bound = (
    value_matrix_shape[1] * PROB_ONE_Q * value_centered_bound
)
assert context_full_bound <= context_int32_limits.max

attention_context_trial = []

assert (
    len(op0_cases)
    == len(softmax_integer_trial)
    == len(attention_score_trial)
)

for case, softmax_trial, score_trial in zip(
    op0_cases, softmax_integer_trial, attention_score_trial
):
    assert case["name"] == softmax_trial["name"] == score_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    context_ref = interpreter.get_tensor(context_index).copy()
    context_quant_ref = interpreter.get_tensor(context_quant_index).copy()

    assert context_ref.shape == (16, 32, 8)
    assert np.isfinite(context_ref).all()

    paths = {}

    for path in ["local", "chained"]:
        probability_q = softmax_trial["paths"][path][
            "probability_matrix_q"
        ]

        # The earlier head-layout helper already produced this shape.
        # Op 33 changes layout only; values and metadata stay unchanged.
        value_q = score_trial["paths"][path][
            "value_head_q"
        ].reshape(value_matrix_shape)

        assert probability_q.dtype == np.int32
        assert probability_q.shape == (16, 32, 32)
        assert value_q.dtype == np.int8
        assert np.all((probability_q >= 0) & (probability_q <= PROB_ONE_Q))

        if path == "local":
            np.testing.assert_array_equal(
                value_q,
                interpreter.get_tensor(value_matrix_index),
            )

        # Op 34 counterpart: remove V's zero point after promotion.
        # No floating-point dequantization in the candidate.
        value_centered_q = (
            value_q.astype(np.int64) - CONTEXT_V_ZERO_POINT
        )

        # Op 35: wide validation arithmetic, then INT32 storage.
        context_wide = (
            probability_q.astype(np.int64) @ value_centered_q
        )
        assert context_wide.shape == context_ref.shape
        assert context_wide.min() >= context_int32_limits.min
        assert context_wide.max() <= context_int32_limits.max

        context_q = context_wide.astype(np.int32)

        # Query token 0 attends only to key token 0.
        # Its context must therefore equal the first V vector exactly
        # in the candidate's integer representation.
        np.testing.assert_array_equal(
            context_q[:, 0, :].astype(np.int64),
            PROB_ONE_Q * value_centered_q[:, 0, :],
        )

        # Floating-point reconstruction: diagnostics only.
        context_approx = context_q.astype(np.float64) * CONTEXT_SCALE

        paths[path] = {
            "context_q": context_q,
            "context_approx": context_approx,
            "context_error": np.abs(context_approx - context_ref),
        }

    attention_context_trial.append({
        "name": case["name"],
        "context_ref": context_ref,
        "t67_ref": context_quant_ref,
        "paths": paths,
    })


def report_attention_context(label, trials):
    if not trials:
        return

    print(f"\n{label}")
    reference = np.concatenate([
        t["context_ref"].reshape(-1) for t in trials
    ])
    print(
        "  T66 (TFLite FP32 context) range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )

    for path in ["local", "chained"]:
        integers = np.concatenate([
            t["paths"][path]["context_q"].reshape(-1)
            for t in trials
        ])
        errors = np.concatenate([
            t["paths"][path]["context_error"].reshape(-1)
            for t in trials
        ])

        print(f"  {path.capitalize()} path:")
        print(
            "    T66 integer range:",
            f"{integers.min()} to {integers.max()}",
        )
        print(f"    Max absolute error vs TFLite:  {errors.max():.9e}")
        print(f"    Mean absolute error vs TFLite: {errors.mean():.9e}")

    print("  Local T64 (V matrix) layout check: pass")
    print("  First-query context identity: pass")
    print("  Context INT32 range checks: pass")


print("T63 (probabilities): INT32 [16,32,32]")
print("  Scale:", PROB_SCALE)
print("T64 (V matrices): INT8", value_matrix_shape)
print("  Scale:", CONTEXT_V_SCALE)
print("  Zero point:", CONTEXT_V_ZERO_POINT)
print("T66 candidate (context): INT32 [16,32,8]")
print("  Scale:", CONTEXT_SCALE)
print("  Zero point:", CONTEXT_ZERO_POINT)
print("  Conservative absolute accumulator bound:", context_full_bound)

print("\nT67 (INT8 context), read from the fixed export:")
print("  Scale:", T67_SCALE)
print("  Zero point:", T67_ZERO_POINT)
print("  T66→T67 real conversion multiplier:", T67_REAL_MULTIPLIER)

print("\nDense probability × V workload per sequence:")
print("  MACs:", 16 * 32 * 32 * 8)
print("  Context elements:", 16 * 32 * 8)

report_attention_context("Benchmark", attention_context_trial[:1])
report_attention_context("Calibration", attention_context_trial[1:])

first_context = attention_context_trial[0]
print("\nHead 0, query token 1, all 8 context features:")
print("  TFLite reference:", first_context["context_ref"][0, 1, :])
for path in ["local", "chained"]:
    print(
        f"  {path} represented:",
        first_context["paths"][path]["context_approx"][0, 1, :],
    )

T63 (probabilities): INT32 [16,32,32]
  Scale: 1.52587890625e-05
T64 (V matrices): INT8 (16, 32, 8)
  Scale: 0.006938841659575701
  Zero point: 4
T66 candidate (context): INT32 [16,32,8]
  Scale: 1.0587832122155305e-07
  Zero point: 0
  Conservative absolute accumulator bound: 276824064

T67 (INT8 context), read from the fixed export:
  Scale: 0.006692775525152683
  Zero point: 9
  T66→T67 real conversion multiplier: 1.581979267400241e-05

Dense probability × V workload per sequence:
  MACs: 131072
  Context elements: 4096

Benchmark
  T66 (TFLite FP32 context) range: -0.714700699 to 0.676839352
  Local path:
    T66 integer range: -6750208 to 6392616
    Max absolute error vs TFLite:  2.243796236e-05
    Mean absolute error vs TFLite: 2.441016188e-06
  Chained path:
    T66 integer range: -6684672 to 6392232
    Max absolute error vs TFLite:  1.025677345e-02
    Mean absolute error vs TFLite: 1.345797040e-03
  Local T64 (V matrix) layout check: pass
  First-query context identity: pas

In [19]:
# Part 3C, completion: Op 36–39.
# Integer context requantization and head merging.

T67_REQUANT_SHIFT = 46
T67_ROUNDING_OFFSET = 1 << (T67_REQUANT_SHIFT - 1)

context_int64_max = int(np.iinfo(np.int64).max)
context_int32_max = int(np.iinfo(np.int32).max)

# OFFLINE: encode the scale ratio as an integer multiplier.
T67_MULTIPLIER_Q = int(np.rint(
    T67_REAL_MULTIPLIER * (1 << T67_REQUANT_SHIFT)
))
assert 0 < T67_MULTIPLIER_Q <= context_int32_max

# Covers every signed INT32 context accumulator.
assert (
    (1 << 31) * T67_MULTIPLIER_Q + T67_ROUNDING_OFFSET
    <= context_int64_max
)

multiplier_approx = (
    T67_MULTIPLIER_Q * 2.0 ** (-T67_REQUANT_SHIFT)
)

merge_ops = interpreter._get_ops_details()
merge_details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}

merge_specs = []
previous_index = context_quant_index

for op_index, expected_name, expected_output in [
    (37, "RESHAPE", 68),
    (38, "TRANSPOSE", 69),
    (39, "RESHAPE", 70),
]:
    op = merge_ops[op_index]
    assert op["op_name"] == expected_name
    assert int(op["inputs"][0]) == previous_index

    output_index = int(op["outputs"][0])
    assert output_index == expected_output

    # Read the actual saved shape or permutation.
    parameter = tuple(
        int(v) for v in interpreter.get_tensor(
            int(op["inputs"][1])
        ).reshape(-1)
    )

    detail = merge_details[output_index]
    assert detail["dtype"] == np.int8

    # Shape operations preserve the quantized representation.
    params = detail["quantization_parameters"]
    assert len(params["scales"]) == 1
    assert len(params["zero_points"]) == 1
    assert float(params["scales"][0]) == T67_SCALE
    assert int(params["zero_points"][0]) == T67_ZERO_POINT

    merge_specs.append({
        "op_index": op_index,
        "name": expected_name,
        "output_index": output_index,
        "parameter": parameter,
        "output_shape": tuple(int(v) for v in detail["shape"]),
    })
    previous_index = output_index

t70_index = previous_index
assert t70_index == 70


def requantize_attention_context(context_q):
    assert context_q.dtype == np.int32

    # Integer-only candidate.
    rounded = (
        context_q.astype(np.int64) * T67_MULTIPLIER_Q
        + T67_ROUNDING_OFFSET
    ) >> T67_REQUANT_SHIFT

    before_saturation = rounded + T67_ZERO_POINT
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))

    output_q = np.clip(
        before_saturation, -128, 127
    ).astype(np.int8)

    return output_q, saturation_count


def merge_attention_heads(t67_q):
    assert t67_q.shape == (16, 32, 8)
    assert t67_q.dtype == np.int8

    current = t67_q
    outputs = {}

    for spec in merge_specs:
        if spec["name"] == "RESHAPE":
            current = current.reshape(spec["parameter"])
        else:
            current = current.transpose(spec["parameter"])

        assert current.shape == spec["output_shape"]
        outputs[spec["output_index"]] = current.copy()

    assert current.shape == (1, 32, 128)

    # Verify the meaning of the merged feature dimension:
    # features 8*h ... 8*h+7 belong to head h.
    for head in range(16):
        np.testing.assert_array_equal(
            current[0, :, 8 * head : 8 * (head + 1)],
            t67_q[head],
        )

    return current.copy(), outputs


attention_merge_integer_trial = []
assert len(op0_cases) == len(attention_context_trial)

for case, context_trial in zip(op0_cases, attention_context_trial):
    assert case["name"] == context_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    t67_ref = interpreter.get_tensor(context_quant_index).copy()
    t70_ref = interpreter.get_tensor(t70_index).copy()

    # Validate the head-merging helper using exact reference T67.
    reference_merged, reference_layout = merge_attention_heads(t67_ref)
    for index, values in reference_layout.items():
        np.testing.assert_array_equal(
            values, interpreter.get_tensor(index)
        )

    # DIAGNOSTIC: apply our rounding rule to original FP32 T66.
    # This helps identify differences caused by reference rounding
    # or FP32 conversion rather than the integer attention candidate.
    reference_with_candidate_rounding = np.clip(
        np.floor(
            context_trial["context_ref"].astype(np.float64)
            / T67_SCALE + 0.5
        ) + T67_ZERO_POINT,
        -128, 127,
    ).astype(np.int8)

    paths = {}

    for path in ["local", "chained"]:
        source = context_trial["paths"][path]

        t67_q, saturation_count = requantize_attention_context(
            source["context_q"]
        )
        t70_q, layout_tensors = merge_attention_heads(t67_q)

        # DIAGNOSTIC: ideal conversion of OUR represented T66,
        # using the same nearest/ties-toward-positive-infinity rule.
        ideal_t67_q = np.clip(
            np.floor(source["context_approx"] / T67_SCALE + 0.5)
            + T67_ZERO_POINT,
            -128, 127,
        ).astype(np.int8)

        paths[path] = {
            "t67_q": t67_q,
            "t70_q": t70_q,
            "layout_tensors": layout_tensors,
            "t67_difference": (
                t67_q.astype(np.int32) - t67_ref.astype(np.int32)
            ),
            "t70_difference": (
                t70_q.astype(np.int32) - t70_ref.astype(np.int32)
            ),
            "requant_mismatch_count": int(np.count_nonzero(
                t67_q != ideal_t67_q
            )),
            "saturation_count": saturation_count,
        }

    attention_merge_integer_trial.append({
        "name": case["name"],
        "t67_ref": t67_ref,
        "t70_ref": t70_ref,
        "reference_rounding_mismatch_count": int(np.count_nonzero(
            reference_with_candidate_rounding != t67_ref
        )),
        "paths": paths,
    })


def report_attention_merge(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    for path in ["local", "chained"]:
        differences = np.concatenate([
            t["paths"][path]["t70_difference"].reshape(-1)
            for t in trials
        ])
        values = np.concatenate([
            t["paths"][path]["t70_q"].reshape(-1)
            for t in trials
        ])
        matched = int(np.count_nonzero(differences == 0))

        print(f"  {path.capitalize()} path:")
        print(f"    T70 integer range: {values.min()} to {values.max()}")
        print(
            f"    T70 exact matches: {matched}/{differences.size} "
            f"({100 * matched / differences.size:.3f}%)"
        )
        print(
            "    Maximum INT8-step difference:",
            int(np.abs(differences).max()),
        )
        print(
            "    Mean absolute INT8-step difference:",
            f"{np.abs(differences).mean():.6f}",
        )

        unique, counts = np.unique(differences, return_counts=True)
        print("    Signed difference counts:", {
            int(d): int(c) for d, c in zip(unique, counts)
        })

        print("    Fixed multiplier vs ideal conversion mismatches:", sum(
            t["paths"][path]["requant_mismatch_count"]
            for t in trials
        ))
        print("    Values requiring saturation:", sum(
            t["paths"][path]["saturation_count"]
            for t in trials
        ))

    print("  Original T66 with candidate rounding vs T67 mismatches:", sum(
        t["reference_rounding_mismatch_count"] for t in trials
    ))
    print("  Reference head-layout checks: pass")
    print("  Candidate head-to-feature mapping checks: pass")


print("T66→T67 conversion:")
print("  Input context tensor scale:", CONTEXT_SCALE)
print("  Output T67 tensor scale:", T67_SCALE)
print("  Output zero point:", T67_ZERO_POINT)
print("  Real conversion multiplier:", T67_REAL_MULTIPLIER)
print("  Integer multiplier:", T67_MULTIPLIER_Q)
print("  Right shift:", T67_REQUANT_SHIFT)
print(
    "  Multiplier relative error:",
    f"{abs(multiplier_approx / T67_REAL_MULTIPLIER - 1):.9e}",
)
print("  Rounding: nearest, ties toward positive infinity")
print("  Full INT32-input multiplier-product bound: pass")

print("\nHead merging:")
for spec in merge_specs:
    print(
        f"  Op {spec['op_index']} {spec['name']}: "
        f"parameter={spec['parameter']} "
        f"→ T{spec['output_index']} {spec['output_shape']}"
    )

report_attention_merge("Benchmark", attention_merge_integer_trial[:1])
report_attention_merge("Calibration", attention_merge_integer_trial[1:])

first_merge = attention_merge_integer_trial[0]
print("\nT70: first benchmark token, first 16 merged features:")
print("  TFLite reference:", first_merge["t70_ref"][0, 0, :16])
for path in ["local", "chained"]:
    print(
        f"  {path}:",
        first_merge["paths"][path]["t70_q"][0, 0, :16],
    )

T66→T67 conversion:
  Input context tensor scale: 1.0587832122155305e-07
  Output T67 tensor scale: 0.006692775525152683
  Output zero point: 9
  Real conversion multiplier: 1.581979267400241e-05
  Integer multiplier: 1113218944
  Right shift: 46
  Multiplier relative error: 3.408504590e-10
  Rounding: nearest, ties toward positive infinity
  Full INT32-input multiplier-product bound: pass

Head merging:
  Op 37 RESHAPE: parameter=(1, 16, 32, 8) → T68 (1, 16, 32, 8)
  Op 38 TRANSPOSE: parameter=(0, 2, 1, 3) → T69 (1, 32, 16, 8)
  Op 39 RESHAPE: parameter=(1, 32, 128) → T70 (1, 32, 128)

Benchmark
  Local path:
    T70 integer range: -98 to 110
    T70 exact matches: 4093/4096 (99.927%)
    Maximum INT8-step difference: 1
    Mean absolute INT8-step difference: 0.000732
    Signed difference counts: {-1: 2, 0: 4093, 1: 1}
    Fixed multiplier vs ideal conversion mismatches: 0
    Values requiring saturation: 0
  Chained path:
    T70 integer range: -97 to 110
    T70 exact matches: 3282

### Block-0 attention: integer validation, Op 18–39

Source block-0 SHA-256:
`06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9`

Test inputs: one 32-token benchmark sequence and seven calibration
sequences; 16 attention heads, head dimension 8.

The original TFLite graph remains mixed FP32/INT8.
Integer counterparts are calculated separately in Python.

#### Tensor representations

| Tensor | Meaning | Integer candidate |
|---|---|---|
| T55 | Query matrices per head | INT8 `[16,32,8]`; Q scale, zero point -12 |
| T57 | Transposed key matrices per head | INT8 `[16,8,32]`; K scale, zero point -30 |
| T59 | Attention scores | INT32 `[16,32,32]`; scale `0.0004575034413775367`, zero point 0 |
| T21 | Causal mask | BOOL `[1,1,32,32]` |
| T62 | Attention probabilities | INT32 `[1,16,32,32]`; scale `2^-16`, zero point 0 |
| T63 | Probability matrices per head | INT32 `[16,32,32]`; same probability scale |
| T64 | Value matrices per head | INT8 `[16,32,8]`; scale `0.006938841659575701`, zero point 4 |
| T66 | Weighted value context | INT32 `[16,32,8]`; scale `1.0587832122155305e-07`, zero point 0 |
| T67 | Quantized context | INT8 `[16,32,8]`; scale `0.006692775525152683`, zero point 9 |
| T68 | Context with batch dimension | INT8 `[1,16,32,8]`; same scale/zero point as T67 |
| T69 | Context arranged by token | INT8 `[1,32,16,8]`; same scale/zero point as T67 |
| T70 | Merged attention output | INT8 `[1,32,128]`; same scale/zero point as T67 |

#### Op 18–28: head layouts and attention scores

Q/K/V are rearranged using the saved graph's shape and permutation
constants.

For head $h$, query token $t$, and key token $j$:

$$
q_{T59}[h,t,j]
=
\sum_{d=0}^{7}
(q_Q[h,t,d]+12)(q_K[h,d,j]+30)
$$

The score tensor scale is the product of Q and K scales:

$$
s_{T59}=s_Qs_K=0.0004575034413775367
$$

No score requantization or additional attention scaling is inserted.

Operands are promoted before arithmetic. Python calculates scores
in INT64, checks their range, then stores them as INT32.

#### Op 29–32: causal masking and integer softmax

T21 allows each query token to attend to itself and earlier tokens:
528 allowed and 496 masked entries per head.

Each row's maximum is selected from allowed scores only and
subtracted before exponential evaluation.

| Softmax component | Implementation |
|---|---|
| Exponential table | 4,097 INT32 entries; 16,388 bytes |
| Table function | `exp(-u)` over `[0,16]`, spacing `1/256` |
| Exponential tensor scale | `2^-30` |
| Table coordinate | 16 fractional bits |
| Coordinate multiplier | 503030354, right shift 24 |
| Interpolation | Linear, using 8 fractional coordinate bits |
| Exponential sum | INT64 |
| Normalization | Integer division with nearest rounding |
| Probability tensor scale | `2^-16`; probability 1 is integer 65536 |
| Masked probabilities | Exactly zero |
| Beyond-cutoff exponentials | Set to zero; none occurred in tested cases |

Probabilities are rounded independently, so their row sums may
deviate slightly from 65536. The largest observed deviation was
5 integer steps, representing `7.62939453125e-5`.

#### Op 33–35: probability × V

V's zero point is removed before multiplication:

$$
q_{T66}[h,t,d]
=
\sum_{j=0}^{31}
q_{T63}[h,t,j](q_{T64}[h,j,d]-4)
$$

The context tensor scale is:

$$
s_{T66}
=
s_{\text{probability}}s_V
=
1.0587832122155305\times10^{-7}
$$

Python accumulates in INT64, checks the range, then stores T66
as INT32. A conservative absolute accumulator bound is 276,824,064,
which fits INT32.

For the first query token, the integer context exactly equals
65536 times the first centered V vector, as required by the mask.

#### Op 36–39: context requantization and head merging

The T66→T67 conversion ratio is:

$$
M=\frac{s_{T66}}{s_{T67}}
=1.581979267400241\times10^{-5}
\approx1113218944\times2^{-46}
$$

Integer conversion:

    q67 = ((q66 * 1113218944 + (1 << 45)) >> 46) + 9
    q67 = clip(q67, -128, 127)

The product uses INT64 arithmetic.
Rounding is nearest, with ties toward positive infinity.

The multiplier's scale `2^-46` is separate from T67's tensor scale.
T67 retains scale `0.006692775525152683` and zero point 9.

Head merging follows the saved graph:

    Op 37: [16,32,8] → [1,16,32,8]
    Op 38: transpose (0,2,1,3) → [1,32,16,8]
    Op 39: reshape → [1,32,128]

These operations preserve values, scale, and zero point.
Features `8*h` through `8*h+7` in T70 belong to head `h`.

#### Validation measurements

- **Local:** attention starts from Q/K/V outputs that match TFLite.
  It includes the integer softmax approximation.
- **Chained:** attention starts from Q/K/V produced using the
  integer LayerNorm candidate.

| Intermediate measurement | Benchmark | Calibration |
|---|---:|---:|
| Local maximum score error vs TFLite | 5.6697e-7 | 9.9516e-7 |
| Chained maximum score error vs TFLite | 0.195812 | 0.231954 |
| Local maximum probability error vs TFLite | 8.5831e-6 | 9.2387e-6 |
| Chained maximum probability error vs TFLite | 0.025901 | 0.034797 |
| Local maximum context error vs TFLite | 2.2438e-5 | 3.2888e-5 |
| Local mean context error vs TFLite | 2.4410e-6 | 2.4953e-6 |
| Chained maximum context error vs TFLite | 0.010257 | 0.015926 |
| Chained mean context error vs TFLite | 0.001346 | 0.001308 |

| Final T70 measurement | Benchmark | Calibration |
|---|---:|---:|
| Local exact matches | 4093/4096 (99.927%) | 28661/28672 (99.962%) |
| Local maximum INT8-step difference | 1 | 1 |
| Local mean absolute INT8-step difference | 0.000732 | 0.000384 |
| Chained exact matches | 3282/4096 (80.127%) | 23065/28672 (80.444%) |
| Chained maximum INT8-step difference | 2 | 3 |
| Chained mean absolute INT8-step difference | 0.198975 | 0.197301 |
| Local values requiring saturation | 0 | 2 |
| Chained values requiring saturation | 0 | 1 |

Across all eight sequences, 14 of 32,768 local T70 values differed
from TFLite, each by one INT8 step.

Fixed-multiplier conversion matched ideal conversion of the
candidate's represented T66 for every tested value.
Applying the candidate rounding rule to original TFLite T66 also
matched reference T67 for every tested value.

All implemented width, mask, reference-layout, first-query context,
and head-to-feature mapping checks passed.

#### Analysis and status

Local attention closely matches the saved mixed-precision graph.
The small local T70 differences arise from differences already
present before context requantization; the fixed multiplier and
head merging introduced no additional observed output discrepancies.

Chained differences include propagated integer LayerNorm and
Q/K/V errors. These measurements describe activation agreement,
not complete-model accuracy.

For one 32-token sequence:

- QK-transpose MACs: **131,072**.
- Probability × V MACs: **131,072**.
- Total dense attention matrix-product MACs: **262,144**.
- Final T70 output elements: **4,096**.

Attention Op 18–39 is validated in Python on the tested cases.
C, Spike, RTL, complete-block accuracy, and attention performance
have not yet been validated.

Next: **Op 40–42, attention output projection and residual addition**.

In [20]:
# Part 4, preparation: inspect Op 40–42 and accumulate Op 40.
# Requantization and integer residual addition follow next.

part4_ops = interpreter._get_ops_details()
part4_details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}

assert part4_ops[40]["op_name"] == "FULLY_CONNECTED"
assert part4_ops[41]["op_name"] == "DEQUANTIZE"
assert part4_ops[42]["op_name"] == "ADD"

op40 = part4_ops[40]
out_input_index = int(op40["inputs"][0])
out_weight_index = int(op40["inputs"][1])
out_bias_index = int(op40["inputs"][2])
t71_index = int(op40["outputs"][0])
t72_index = int(part4_ops[41]["outputs"][0])
t73_index = int(part4_ops[42]["outputs"][0])

assert (out_input_index, out_weight_index, out_bias_index) == (70, 28, 24)
assert (t71_index, t72_index, t73_index) == (71, 72, 73)
assert int(part4_ops[41]["inputs"][0]) == t71_index
assert set(int(v) for v in part4_ops[42]["inputs"]) == {
    t72_index, x_index
}

input_detail = part4_details[out_input_index]
weight_detail = part4_details[out_weight_index]
output_detail = part4_details[t71_index]
bias_detail = part4_details[out_bias_index]

assert input_detail["dtype"] == np.int8
assert weight_detail["dtype"] == np.int8
assert output_detail["dtype"] == np.int8
assert bias_detail["dtype"] == np.int32

input_params = input_detail["quantization_parameters"]
weight_params = weight_detail["quantization_parameters"]
output_params = output_detail["quantization_parameters"]
bias_params = bias_detail["quantization_parameters"]

assert len(input_params["scales"]) == 1
assert len(input_params["zero_points"]) == 1
assert len(output_params["scales"]) == 1
assert len(output_params["zero_points"]) == 1

out_input_scale = float(input_params["scales"][0])
out_input_zero_point = int(input_params["zero_points"][0])
out_output_scale = float(output_params["scales"][0])
out_output_zero_point = int(output_params["zero_points"][0])

assert out_input_scale == T67_SCALE
assert out_input_zero_point == T67_ZERO_POINT
assert out_output_scale > 0

out_weights_q = interpreter.get_tensor(out_weight_index).copy()
out_bias_q = interpreter.get_tensor(out_bias_index).copy().reshape(-1)

assert out_weights_q.shape == (128, 128)
assert out_bias_q.shape == (128,)

out_weight_scales_raw = np.asarray(
    weight_params["scales"], dtype=np.float64
)
out_weight_zero_points_raw = np.asarray(
    weight_params["zero_points"], dtype=np.int64
)

assert out_weight_scales_raw.size in (1, 128)
assert out_weight_zero_points_raw.size in (1, 128)
assert np.all(out_weight_scales_raw > 0)

if out_weight_scales_raw.size > 1:
    assert int(weight_params["quantized_dimension"]) == 0

out_weight_scales = np.broadcast_to(
    out_weight_scales_raw, (128,)
).copy()
out_weight_zero_points = np.broadcast_to(
    out_weight_zero_points_raw, (128,)
).copy()

# Bias integers must represent values at accumulator scale.
assert len(bias_params["scales"]) in (1, 128)
assert np.all(bias_params["zero_points"] == 0)

np.testing.assert_allclose(
    np.broadcast_to(bias_params["scales"], (128,)),
    out_input_scale * out_weight_scales,
    rtol=1e-5,
    atol=0,
)

out_proj_spec = {
    "op_index": 40,
    "input_index": out_input_index,
    "weight_index": out_weight_index,
    "output_index": t71_index,
    "input_scale": out_input_scale,
    "input_zero_point": out_input_zero_point,
    "weight_scales": out_weight_scales,
    "weight_zero_points": out_weight_zero_points,
    "output_scale": out_output_scale,
    "output_zero_point": out_output_zero_point,
    "weights_q": out_weights_q,
    "bias_q": out_bias_q,
    "real_multipliers": (
        out_input_scale * out_weight_scales / out_output_scale
    ),
}

print("Op 40: attention output projection")
print("  T70 (input): scale =", out_input_scale)
print("    Zero point:", out_input_zero_point)
print("  T28 (weights):", out_weights_q.shape)
print("    Original scale count:", out_weight_scales_raw.size)
print("    Scale range:", out_weight_scales.min(), "to",
      out_weight_scales.max())
print("    Zero points:", np.unique(out_weight_zero_points))
print("  T24 (bias): INT32", out_bias_q.shape)
print("    Integer range:", out_bias_q.min(), "to", out_bias_q.max())
print("    Bias scale compatibility: pass")
print("  Accumulator scale range:",
      (out_input_scale * out_weight_scales).min(), "to",
      (out_input_scale * out_weight_scales).max())
print("  T71 (output): scale =", out_output_scale)
print("    Zero point:", out_output_zero_point)
print("  Conversion multiplier range:",
      out_proj_spec["real_multipliers"].min(), "to",
      out_proj_spec["real_multipliers"].max())

print("\nResidual branch:")
print("  T0 candidate: INT8, scale =", T0_SCALE, ", zero point = 0")
print("  T72/T73 in saved graph: FP32")
print("  Integer residual format: next stage")

out_proj_cases = []

assert (
    len(op0_cases)
    == len(attention_merge_integer_trial)
    == len(op0_integer_trial)
)

for case, attention_trial, input_trial in zip(
    op0_cases, attention_merge_integer_trial, op0_integer_trial
):
    assert case["name"] == attention_trial["name"] == input_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    input_ref = interpreter.get_tensor(out_input_index).copy()
    input_chained = attention_trial["paths"]["chained"]["t70_q"]

    # Reuse the verified 128→128 accumulator helper.
    # It promotes operands, removes zero points, adds bias,
    # and checks INT32 accumulator bounds.
    acc_local = qkv_accumulate(input_ref, out_proj_spec)
    acc_chained = qkv_accumulate(input_chained, out_proj_spec)

    out_proj_cases.append({
        "name": case["name"],
        "input_ref": input_ref,
        "input_chained": input_chained.copy(),
        "acc_local": acc_local,
        "acc_chained": acc_chained,
        "t71_ref": interpreter.get_tensor(t71_index).copy(),
        "t72_ref": interpreter.get_tensor(t72_index).copy(),
        "t73_ref": interpreter.get_tensor(t73_index).copy(),
        "t0_q": input_trial["t0_q"].copy(),
    })


def report_out_projection_inputs(label, cases):
    if not cases:
        return

    print(f"\n{label}")

    for key, description in [
        ("acc_local", "Local INT32 accumulator, including bias"),
        ("acc_chained", "Chained INT32 accumulator, including bias"),
        ("t71_ref", "T71 reference INT8 projection output"),
        ("t72_ref", "T72 reference FP32 projection output"),
        ("t73_ref", "T73 reference FP32 residual output"),
    ]:
        values = np.concatenate([c[key].reshape(-1) for c in cases])
        print(f"  {description}: {values.min()} to {values.max()}")

    print("  Accumulator INT32 range checks: pass")


print("\nOutput-projection workload per sequence:")
print("  MACs:", 32 * 128 * 128)
print("  Output elements:", 32 * 128)

report_out_projection_inputs("Benchmark", out_proj_cases[:1])
report_out_projection_inputs("Calibration", out_proj_cases[1:])

Op 40: attention output projection
  T70 (input): scale = 0.006692775525152683
    Zero point: 9
  T28 (weights): (128, 128)
    Original scale count: 1
    Scale range: 0.0006937290891073644 to 0.0006937290891073644
    Zero points: [0]
  T24 (bias): INT32 (128,)
    Integer range: -13036 to 8460
    Bias scale compatibility: pass
  Accumulator scale range: 4.6429730686642335e-06 to 4.6429730686642335e-06
  T71 (output): scale = 0.003416289109736681
    Zero point: -13
  Conversion multiplier range: 0.0013590691301363842 to 0.0013590691301363842

Residual branch:
  T0 candidate: INT8, scale = 0.00390625 , zero point = 0
  T72/T73 in saved graph: FP32
  Integer residual format: next stage

Output-projection workload per sequence:
  MACs: 524288
  Output elements: 4096

Benchmark
  Local INT32 accumulator, including bias: -74101 to 97146
  Chained INT32 accumulator, including bias: -73642 to 96475
  T71 reference INT8 projection output: -114 to 119
  T72 reference FP32 projection output

In [21]:
# Part 4, completion: Op 40–42.
# Projection requantization and scale-aligned residual addition.

part4_int32 = np.iinfo(np.int32)
part4_int64_max = int(np.iinfo(np.int64).max)

# ----- Op 40: accumulator -> T71 -----

OUT_PROJ_REQUANT_SHIFT = 39
OUT_PROJ_ROUNDING_OFFSET = 1 << (OUT_PROJ_REQUANT_SHIFT - 1)

out_multiplier_rounded = np.rint(
    out_proj_spec["real_multipliers"]
    * (1 << OUT_PROJ_REQUANT_SHIFT)
)
assert np.all(out_multiplier_rounded > 0)
assert out_multiplier_rounded.max() <= part4_int32.max

out_proj_multiplier_q = out_multiplier_rounded.astype(np.int32)

# Covers the full signed INT32 accumulator range.
assert (
    (1 << 31) * int(out_proj_multiplier_q.max())
    + OUT_PROJ_ROUNDING_OFFSET
    <= part4_int64_max
)

out_multiplier_approx = (
    out_proj_multiplier_q.astype(np.float64)
    * 2.0 ** (-OUT_PROJ_REQUANT_SHIFT)
)

# ----- Op 41–42: internal residual representation -----

T72_FRAC_BITS = 16
T72_SCALE = 2.0 ** (-T72_FRAC_BITS)
T72_ZERO_POINT = 0

T73_FRAC_BITS = T72_FRAC_BITS
T73_SCALE = T72_SCALE
T73_ZERO_POINT = 0

T0_RESIDUAL_ALIGN_SHIFT = T73_FRAC_BITS - T0_FRAC_BITS
assert T0_RESIDUAL_ALIGN_SHIFT >= 0
assert T0_SCALE == 2.0 ** (-T0_FRAC_BITS)

# OFFLINE: encode the T71 -> T72 scale ratio.
T72_REAL_MULTIPLIER = out_proj_spec["output_scale"] / T72_SCALE
T72_REQUANT_SHIFT = 22
T72_ROUNDING_OFFSET = 1 << (T72_REQUANT_SHIFT - 1)

T72_MULTIPLIER_Q = int(np.rint(
    T72_REAL_MULTIPLIER * (1 << T72_REQUANT_SHIFT)
))
assert 0 < T72_MULTIPLIER_Q <= part4_int32.max

t71_centered_bound = max(
    abs(-128 - out_proj_spec["output_zero_point"]),
    abs(127 - out_proj_spec["output_zero_point"]),
)
assert (
    t71_centered_bound * T72_MULTIPLIER_Q
    + T72_ROUNDING_OFFSET
    <= part4_int64_max
)


def out_projection_requantize(accumulator, output_shape):
    product = (
        accumulator.astype(np.int64)
        * out_proj_multiplier_q.astype(np.int64)[None, :]
    )
    rounded = (
        product + OUT_PROJ_ROUNDING_OFFSET
    ) >> OUT_PROJ_REQUANT_SHIFT

    before_saturation = (
        rounded + out_proj_spec["output_zero_point"]
    )
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))

    output_q = np.clip(
        before_saturation, -128, 127
    ).astype(np.int8).reshape(output_shape)

    return output_q, saturation_count


def integer_attention_residual(t71_q, t0_q):
    # Op 41 counterpart: retain integer arithmetic.
    centered_t71 = (
        t71_q.astype(np.int64)
        - out_proj_spec["output_zero_point"]
    )

    t72_wide = (
        centered_t71 * T72_MULTIPLIER_Q + T72_ROUNDING_OFFSET
    ) >> T72_REQUANT_SHIFT

    assert t72_wide.min() >= part4_int32.min
    assert t72_wide.max() <= part4_int32.max
    t72_q = t72_wide.astype(np.int32)

    # T0 already has zero point 0.
    # Alignment is exact because both scales are powers of two.
    aligned_t0_q = (
        t0_q.astype(np.int64) * (1 << T0_RESIDUAL_ALIGN_SHIFT)
    )
    assert aligned_t0_q.shape == t72_q.shape

    # Op 42: add at a shared scale.
    t73_wide = t72_q.astype(np.int64) + aligned_t0_q
    assert t73_wide.min() >= part4_int32.min
    assert t73_wide.max() <= part4_int32.max
    t73_q = t73_wide.astype(np.int32)

    return t72_q, t73_q


out_residual_integer_trial = []
assert len(op0_cases) == len(out_proj_cases)

for case, prepared in zip(op0_cases, out_proj_cases):
    assert case["name"] == prepared["name"]

    # Diagnostics for the unchanged INT8 residual input.
    t0_approx = prepared["t0_q"].astype(np.float64) * T0_SCALE
    t0_error = np.abs(t0_approx - case["x_fp32"])

    paths = {}

    for path in ["local", "chained"]:
        accumulator = prepared[f"acc_{path}"]

        t71_q, saturation_count = out_projection_requantize(
            accumulator, prepared["t71_ref"].shape
        )
        t72_q, t73_q = integer_attention_residual(
            t71_q, prepared["t0_q"]
        )

        # Floating-point calculations below are diagnostics only.
        ideal_t71_q = np.clip(
            np.floor(
                accumulator.astype(np.float64)
                * out_proj_spec["real_multipliers"][None, :]
                + 0.5
            ) + out_proj_spec["output_zero_point"],
            -128, 127,
        ).astype(np.int8).reshape(t71_q.shape)

        ideal_t72 = (
            t71_q.astype(np.float64)
            - out_proj_spec["output_zero_point"]
        ) * out_proj_spec["output_scale"]

        t72_approx = t72_q.astype(np.float64) * T72_SCALE
        t73_approx = t73_q.astype(np.float64) * T73_SCALE

        # Isolate conversion/addition error from upstream errors.
        ideal_residual_from_candidate = ideal_t72 + t0_approx
        residual_own_error = np.abs(
            t73_approx - ideal_residual_from_candidate
        )

        # A nearest conversion to scale 2^-16 should add at most
        # half a step here; allow tiny FP diagnostic tolerance.
        assert residual_own_error.max() <= T73_SCALE / 2 + 1e-12

        paths[path] = {
            "t71_q": t71_q,
            "t72_q": t72_q,
            "t73_q": t73_q,
            "t72_approx": t72_approx,
            "t73_approx": t73_approx,
            "t71_difference": (
                t71_q.astype(np.int32)
                - prepared["t71_ref"].astype(np.int32)
            ),
            "t72_error": np.abs(
                t72_approx - prepared["t72_ref"]
            ),
            "t73_error": np.abs(
                t73_approx - prepared["t73_ref"]
            ),
            "residual_own_error": residual_own_error,
            "requant_mismatch_count": int(np.count_nonzero(
                t71_q != ideal_t71_q
            )),
            "saturation_count": saturation_count,
        }

    out_residual_integer_trial.append({
        "name": prepared["name"],
        "t0_error": t0_error,
        "t73_ref": prepared["t73_ref"].copy(),
        "paths": paths,
    })


def report_out_residual(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    t0_errors = np.concatenate([
        t["t0_error"].reshape(-1) for t in trials
    ])
    print("  Shared T0 residual-input quantization error:")
    print(f"    Max:  {t0_errors.max():.9e}")
    print(f"    Mean: {t0_errors.mean():.9e}")

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        differences = np.concatenate([
            t["paths"][path]["t71_difference"].reshape(-1)
            for t in trials
        ])
        matched = int(np.count_nonzero(differences == 0))

        print(
            f"    T71 exact matches: {matched}/{differences.size} "
            f"({100 * matched / differences.size:.3f}%)"
        )
        print(
            "    T71 maximum INT8-step difference:",
            int(np.abs(differences).max()),
        )
        print(
            "    T71 mean absolute INT8-step difference:",
            f"{np.abs(differences).mean():.6f}",
        )
        print("    Fixed multiplier vs ideal conversion mismatches:", sum(
            t["paths"][path]["requant_mismatch_count"]
            for t in trials
        ))
        print("    Projection values requiring saturation:", sum(
            t["paths"][path]["saturation_count"]
            for t in trials
        ))

        for key, description in [
            ("t72_error", "T72 projection error vs TFLite"),
            ("residual_own_error", "Residual conversion/addition own error"),
            ("t73_error", "T73 total residual error vs TFLite"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1) for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        residual_q = np.concatenate([
            t["paths"][path]["t73_q"].reshape(-1) for t in trials
        ])
        print(
            "    T73 INT32 range:",
            f"{residual_q.min()} to {residual_q.max()}",
        )

    print("  T0 scale alignment: exact")
    print("  T72/T73 INT32 range checks: pass")


print("Op 40: accumulator -> T71")
print("  Integer multipliers:", np.unique(out_proj_multiplier_q))
print("  Right shift:", OUT_PROJ_REQUANT_SHIFT)
print(
    "  Maximum multiplier relative error:",
    f"{np.abs(out_multiplier_approx / out_proj_spec['real_multipliers'] - 1).max():.9e}",
)
print("  T71 tensor scale:", out_proj_spec["output_scale"])
print("  T71 zero point:", out_proj_spec["output_zero_point"])

print("\nOp 41–42: residual representation")
print("  T72/T73: INT32, shape [1,32,128]")
print("  Tensor scale:", T73_SCALE)
print("  Zero point:", T73_ZERO_POINT)
print("  T71->T72 integer multiplier:", T72_MULTIPLIER_Q)
print("  T71->T72 right shift:", T72_REQUANT_SHIFT)
print("  T0 alignment multiplier:", 1 << T0_RESIDUAL_ALIGN_SHIFT)
print("  Rounding: nearest, ties toward positive infinity")

report_out_residual("Benchmark", out_residual_integer_trial[:1])
report_out_residual("Calibration", out_residual_integer_trial[1:])

first_residual = out_residual_integer_trial[0]
print("\nFirst benchmark token, first 8 T73 features:")
print("  TFLite reference:", first_residual["t73_ref"][0, 0, :8])
for path in ["local", "chained"]:
    print(
        f"  {path} represented:",
        first_residual["paths"][path]["t73_approx"][0, 0, :8],
    )

Op 40: accumulator -> T71
  Integer multipliers: [747156156]
  Right shift: 39
  Maximum multiplier relative error: 3.102644808e-10
  T71 tensor scale: 0.003416289109736681
  T71 zero point: -13

Op 41–42: residual representation
  T72/T73: INT32, shape [1,32,128]
  Tensor scale: 1.52587890625e-05
  Zero point: 0
  T71->T72 integer multiplier: 939062400
  T71->T72 right shift: 22
  T0 alignment multiplier: 256
  Rounding: nearest, ties toward positive infinity

Benchmark
  Shared T0 residual-input quantization error:
    Max:  1.952394843e-03
    Mean: 9.877213568e-04
  Local path:
    T71 exact matches: 4096/4096 (100.000%)
    T71 maximum INT8-step difference: 0
    T71 mean absolute INT8-step difference: 0.000000
    Fixed multiplier vs ideal conversion mismatches: 0
    Projection values requiring saturation: 0
    T72 projection error vs TFLite:
      Max:  7.569789886e-06
      Mean: 3.758923299e-06
    Residual conversion/addition own error:
      Max:  7.570721209e-06
      Mea

In [22]:
out_residual_integer_trial[i]["paths"]["chained"]["t73_q"]

array([[[ 10719,  -3712,  -4062, ...,  -8254,  -3233,   3839],
        [ -7394,   1631, -12894, ...,  -7134,   1216,  -1089],
        [  8607,  -1825,  -3550, ..., -10110,   2912,   6495],
        ...,
        [  6848,   6431,   2145, ...,  -8351,   7840,  -5887],
        [ 10944,  -4033,   1921, ...,  -9631,  -3360,  11009],
        [ -3872,  -3489,  -4543, ..., -17311,   1024,  -2207]]],
      shape=(1, 32, 128), dtype=int32)

### Block-0 attention output projection and residual: Op 40–42

Source block-0 SHA-256:
`06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9`

Test inputs: one 32-token benchmark sequence and seven calibration
sequences; hidden size 128.

The saved TFLite graph remains unchanged. Integer counterparts for
its FP32 dequantization and residual addition are calculated in Python.

#### Tensor representations

| Tensor | Meaning | Integer candidate |
|---|---|---|
| T70 | Merged attention input | INT8 `[1,32,128]`; scale `0.006692775525152683`, zero point 9 |
| T28 | Output-projection weights | INT8 `[128,128]`; scale `0.0006937290891073644`, zero point 0 |
| T24 | Output-projection bias | INT32 `[128]`; scale `4.6429730686642335e-06`, zero point 0 |
| Accumulator | Projection dot products plus bias | INT32 `[32,128]`; scale `4.6429730686642335e-06`, zero point 0 |
| T71 | Quantized projection output | INT8 `[1,32,128]`; scale `0.003416289109736681`, zero point -13 |
| T72 | Projection output aligned for residual addition | INT32 `[1,32,128]`; scale `2^-16`, zero point 0 |
| T0 | Original block input, residual branch | Existing INT8 candidate `[1,32,128]`; scale `2^-8`, zero point 0 |
| T73 | First residual output / second LayerNorm input | INT32 `[1,32,128]`; scale `2^-16`, zero point 0 |

#### Op 40: projection accumulation and requantization

For token $t$ and output feature $o$:

$$
A[t,o]
=
\sum_{k=0}^{127}(q_{T70}[t,k]-9)q_{T28}[o,k]
+
q_{T24}[o]
$$

The bias is added directly because its scale equals the accumulator
scale:

$$
s_A=s_{T70}s_{T28}
=4.6429730686642335\times10^{-6}
$$

Operands are promoted before arithmetic. Python accumulates in
INT64, checks the range, then stores the result as INT32.

The accumulator-to-T71 conversion ratio is:

$$
M=\frac{s_A}{s_{T71}}
=0.0013590691301363842
\approx747156156\times2^{-39}
$$

Integer requantization:

    q71 = ((accumulator * 747156156 + (1 << 38)) >> 39) - 13
    q71 = clip(q71, -128, 127)

The multiplier product uses INT64 arithmetic.
Rounding is nearest, with ties toward positive infinity.

T71 retains its output tensor scale and zero point.
The multiplier's scale `2^-39` describes the conversion ratio,
not T71's stored representation.

#### Op 41–42: integer residual addition

The saved graph dequantizes T71 into FP32 T72, then adds FP32 T0.
The integer candidate instead aligns both branches to scale `2^-16`.

T71→T72 conversion:

$$
q_{T72}
\approx
\operatorname{round}\left(
(q_{T71}+13)\frac{s_{T71}}{2^{-16}}
\right)
$$

Integer implementation:

    q72 = (((q71 + 13) * 939062400 + (1 << 21)) >> 22)

T0 alignment is exact:

    aligned_q0 = q0 * 256

Residual addition:

    q73 = q72 + aligned_q0

T72 and T73 use INT32 storage at scale `2^-16`, zero point 0.
INT64 temporaries are used before checking and storing the results.

No additional INT8 quantization is introduced before the second
LayerNorm. The finer residual format preserves the sum but does not
recover information already lost in T0's INT8 quantization.

#### Validation measurements

- **Local projection:** uses TFLite's T70 to isolate Op 40.
- **Chained projection:** uses the chained integer attention T70.
- **Both residual paths:** use the existing INT8 T0 candidate.

Therefore, local T73 error includes T0's input quantization.
The residual operation's own error is measured separately against
the ideal sum of its represented candidate inputs.

| Projection measurement | Benchmark | Calibration |
|---|---:|---:|
| Local T71 exact matches | 4096/4096 (100%) | 28672/28672 (100%) |
| Chained T71 exact matches | 3428/4096 (83.691%) | 24479/28672 (85.376%) |
| Chained maximum INT8-step difference | 2 | 2 |
| Chained mean absolute INT8-step difference | 0.163330 | 0.146275 |
| Fixed multiplier vs ideal conversion mismatches | 0 | 0 |
| Local values requiring saturation | 0 | 2 |
| Chained values requiring saturation | 0 | 2 |

| Residual measurement | Benchmark | Calibration |
|---|---:|---:|
| Shared T0 maximum quantization error | 1.9524e-3 | 1.9530e-3 |
| Shared T0 mean quantization error | 9.8772e-4 | 9.7425e-4 |
| Local T72 maximum error vs TFLite | 7.5698e-6 | 7.5698e-6 |
| Chained T72 maximum error vs TFLite | 0.006829 | 0.006830 |
| Maximum residual conversion/addition own error | 7.5707e-6 | 7.5707e-6 |
| Local T73 maximum error vs TFLite | 0.001959 | 0.001959 |
| Local T73 mean error vs TFLite | 0.000988 | 0.000974 |
| Chained T73 maximum error vs TFLite | 0.005364 | 0.007440 |
| Chained T73 mean error vs TFLite | 0.001368 | 0.001333 |

| Stored T73 integer range | Benchmark | Calibration |
|---|---|---|
| Local | -33337 to 28342 | -33337 to 33201 |
| Chained | -33113 to 28118 | -33113 to 32977 |

Bias scale compatibility, projection accumulator ranges,
requantization product bounds, and T72/T73 INT32 range checks passed.
T0 scale alignment was exact.

#### Analysis and status

The local output projection matched every reference T71 value.
Its fixed-point requantization introduced no observed differences
relative to ideal conversion.

Local residual error is dominated by the existing INT8 T0
quantization. The residual conversion/addition itself adds less
than half a `2^-16` step.

Chained residual error also includes differences propagated through
the integer LayerNorm, Q/K/V, attention, and output projection.
These measurements describe activation error, not complete-model
accuracy.

For one 32-token sequence:

- Output-projection MACs: **524,288**.
- Projection output elements: **4,096**.
- Residual elementwise additions: **4,096**.

Op 40–42 is validated in Python on the tested cases.
C, Spike, RTL, complete-block accuracy, and performance have not
yet been validated for this implementation.

Next: **Op 43–57, second LayerNorm**, consuming chained T73 at
scale `2^-16`.

In [23]:
# Part 5A: second LayerNorm, Op 43–44.
# Integer means and broadcast layout.

ln2_ops = interpreter._get_ops_details()
assert ln2_ops[43]["op_name"] == "MEAN"
assert ln2_ops[44]["op_name"] == "RESHAPE"

ln2_input_index = int(ln2_ops[43]["inputs"][0])
t74_index = int(ln2_ops[43]["outputs"][0])
t75_index = int(ln2_ops[44]["outputs"][0])

assert (ln2_input_index, t74_index, t75_index) == (73, 74, 75)
assert int(ln2_ops[44]["inputs"][0]) == t74_index

ln2_mean_axes = tuple(
    int(v) for v in interpreter.get_tensor(
        int(ln2_ops[43]["inputs"][1])
    ).reshape(-1)
)
assert ln2_mean_axes == (0, 2)

ln2_broadcast_shape = tuple(
    int(v) for v in interpreter.get_tensor(
        int(ln2_ops[44]["inputs"][1])
    ).reshape(-1)
)
assert ln2_broadcast_shape == (1, 32, 1)

LN2_FEATURE_COUNT = 128
LN2_INPUT_FRAC_BITS = T73_FRAC_BITS
LN2_INPUT_SCALE = T73_SCALE

LN2_MEAN_FRAC_BITS = LN2_INPUT_FRAC_BITS + 7
LN2_MEAN_SCALE = LN2_INPUT_SCALE / LN2_FEATURE_COUNT

assert LN2_INPUT_SCALE == 2.0 ** (-LN2_INPUT_FRAC_BITS)
assert LN2_MEAN_SCALE == 2.0 ** (-LN2_MEAN_FRAC_BITS)

ln2_int32_limits = np.iinfo(np.int32)
ln2_mean_integer_trial = []

assert len(op0_cases) == len(out_residual_integer_trial)

for case, residual_trial in zip(
    op0_cases, out_residual_integer_trial
):
    assert case["name"] == residual_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    input_ref = interpreter.get_tensor(ln2_input_index).copy()
    mean_ref = interpreter.get_tensor(t74_index).copy()
    broadcast_mean_ref = interpreter.get_tensor(t75_index).copy()

    assert input_ref.shape == (1, 32, 128)
    assert mean_ref.shape == (32,)
    assert np.isfinite(input_ref).all()

    # Local input: quantize reference T73 at the chosen input scale.
    # Nearest rounding, ties to even; no clipping.
    local_rounded = np.rint(
        input_ref.astype(np.float64) / LN2_INPUT_SCALE
    )
    assert local_rounded.min() >= ln2_int32_limits.min
    assert local_rounded.max() <= ln2_int32_limits.max

    local_input_q = local_rounded.astype(np.int32)

    # Chained input: retain the existing residual integers unchanged.
    chained_input_q = residual_trial["paths"]["chained"]["t73_q"]
    assert chained_input_q.dtype == np.int32
    assert chained_input_q.shape == input_ref.shape

    paths = {}

    for path, input_q in [
        ("local", local_input_q),
        ("chained", chained_input_q),
    ]:
        # Op 43: store the sum; division is represented by mean scale.
        mean_wide = np.sum(
            input_q.astype(np.int64),
            axis=ln2_mean_axes,
            dtype=np.int64,
        )
        assert mean_wide.min() >= ln2_int32_limits.min
        assert mean_wide.max() <= ln2_int32_limits.max

        t74_q = mean_wide.astype(np.int32)
        assert t74_q.shape == mean_ref.shape

        # Op 44: reshape only.
        t75_q = t74_q.reshape(ln2_broadcast_shape)
        assert t75_q.shape == broadcast_mean_ref.shape

        # Floating-point reconstruction: diagnostics only.
        input_approx = input_q.astype(np.float64) * LN2_INPUT_SCALE
        mean_approx = t74_q.astype(np.float64) * LN2_MEAN_SCALE

        input_error = np.abs(input_approx - input_ref)

        if path == "local":
            assert input_error.max() <= LN2_INPUT_SCALE / 2 + 1e-12

        # Verify that the stored sum represents the mean exactly.
        np.testing.assert_allclose(
            mean_approx,
            input_approx.mean(axis=ln2_mean_axes),
            rtol=0,
            atol=1e-12,
        )

        paths[path] = {
            "t73_q": input_q.copy(),
            "t74_q": t74_q,
            "t75_q": t75_q.copy(),
            "input_approx": input_approx,
            "mean_approx": mean_approx,
            "input_error": input_error,
            "mean_error": np.abs(mean_approx - mean_ref),
        }

    ln2_mean_integer_trial.append({
        "name": case["name"],
        "input_ref": input_ref,
        "mean_ref": mean_ref,
        "broadcast_mean_ref": broadcast_mean_ref,
        "paths": paths,
    })


def report_ln2_means(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    reference = np.concatenate([
        t["mean_ref"].reshape(-1) for t in trials
    ])
    print(
        "  T74 reference mean range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        for key, description in [
            ("input_error", "T73 input error vs TFLite"),
            ("mean_error", "T74 mean error vs TFLite"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1)
                for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        means_q = np.concatenate([
            t["paths"][path]["t74_q"].reshape(-1)
            for t in trials
        ])
        print(
            "    T74 stored integer range:",
            f"{means_q.min()} to {means_q.max()}",
        )

    print("  Mean representation identity: pass")
    print("  Mean INT32 range and broadcast-shape checks: pass")


print("Second LayerNorm input T73: INT32 [1,32,128]")
print("  Scale:", LN2_INPUT_SCALE)
print("  Zero point: 0")
print("T74 (means): INT32 [32]")
print("  Scale:", LN2_MEAN_SCALE)
print("  Fractional bits:", LN2_MEAN_FRAC_BITS)
print("T75 (broadcast means):", ln2_broadcast_shape)
print("  Same integers and scale as T74")
print("Reduction axes:", ln2_mean_axes)

report_ln2_means("Benchmark", ln2_mean_integer_trial[:1])
report_ln2_means("Calibration", ln2_mean_integer_trial[1:])

Second LayerNorm input T73: INT32 [1,32,128]
  Scale: 1.52587890625e-05
  Zero point: 0
T74 (means): INT32 [32]
  Scale: 1.1920928955078125e-07
  Fractional bits: 23
T75 (broadcast means): (1, 32, 1)
  Same integers and scale as T74
Reduction axes: (0, 2)

Benchmark
  T74 reference mean range: -0.011050868 to 0.001243779
  Local path:
    T73 input error vs TFLite:
      Max:  7.629394531e-06
      Mean: 3.815025650e-06
    T74 mean error vs TFLite:
      Max:  8.587376215e-07
      Mean: 3.339318937e-07
    T74 stored integer range: -92696 to 10434
  Chained path:
    T73 input error vs TFLite:
      Max:  5.363769829e-03
      Mean: 1.367635223e-03
    T74 mean error vs TFLite:
      Max:  3.511926625e-04
      Mean: 1.163299876e-04
    T74 stored integer range: -91321 to 8881
  Mean representation identity: pass
  Mean INT32 range and broadcast-shape checks: pass

Calibration
  T74 reference mean range: -0.011224828 to 0.006237695
  Local path:
    T73 input error vs TFLite:
      M

In [24]:
# Part 5B: second LayerNorm, Op 45–46.
# Squared differences and variance.

assert ln2_ops[45]["op_name"] == "SQUARED_DIFFERENCE"
assert ln2_ops[46]["op_name"] == "MEAN"

t76_index = int(ln2_ops[45]["outputs"][0])
t77_index = int(ln2_ops[46]["outputs"][0])

assert (t76_index, t77_index) == (76, 77)
assert set(int(v) for v in ln2_ops[45]["inputs"]) == {
    ln2_input_index, t75_index
}
assert int(ln2_ops[46]["inputs"][0]) == t76_index

ln2_variance_axes = tuple(
    int(v) for v in interpreter.get_tensor(
        int(ln2_ops[46]["inputs"][1])
    ).reshape(-1)
)
assert ln2_variance_axes == (0, 2)

LN2_CENTER_ALIGN_SHIFT = (
    LN2_MEAN_FRAC_BITS - LN2_INPUT_FRAC_BITS
)
assert LN2_CENTER_ALIGN_SHIFT == 7

LN2_SQUARE_FINE_FRAC_BITS = 2 * LN2_MEAN_FRAC_BITS
LN2_SQUARE_FINE_SCALE = 2.0 ** (-LN2_SQUARE_FINE_FRAC_BITS)

LN2_SQUARE_FRAC_BITS = 23
LN2_SQUARE_SCALE = 2.0 ** (-LN2_SQUARE_FRAC_BITS)

LN2_SQUARE_REQUANT_SHIFT = (
    LN2_SQUARE_FINE_FRAC_BITS - LN2_SQUARE_FRAC_BITS
)
LN2_SQUARE_ROUNDING_OFFSET = (
    1 << (LN2_SQUARE_REQUANT_SHIFT - 1)
)

LN2_VARIANCE_FRAC_BITS = LN2_SQUARE_FRAC_BITS + 7
LN2_VARIANCE_SCALE = LN2_SQUARE_SCALE / LN2_FEATURE_COUNT
LN2_VARIANCE_FINE_SCALE = (
    LN2_SQUARE_FINE_SCALE / LN2_FEATURE_COUNT
)

assert LN2_SQUARE_FINE_FRAC_BITS == 46
assert LN2_SQUARE_REQUANT_SHIFT == 23
assert LN2_VARIANCE_FRAC_BITS == 30
assert LN2_VARIANCE_SCALE == 2.0 ** (-LN2_VARIANCE_FRAC_BITS)

ln2_int64_max = int(np.iinfo(np.int64).max)
ln2_variance_integer_trial = []

assert len(op0_cases) == len(ln2_mean_integer_trial)

for case, mean_trial in zip(op0_cases, ln2_mean_integer_trial):
    assert case["name"] == mean_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    square_ref = interpreter.get_tensor(t76_index).copy()
    variance_ref = interpreter.get_tensor(t77_index).copy()

    assert square_ref.shape == (1, 32, 128)
    assert variance_ref.shape == (32,)
    assert np.all(square_ref >= 0)
    assert np.all(variance_ref >= 0)

    paths = {}

    for path in ["local", "chained"]:
        source = mean_trial["paths"][path]

        # Align input to the broadcast mean's scale, then subtract.
        input_aligned_q = (
            source["t73_q"].astype(np.int64)
            * (1 << LN2_CENTER_ALIGN_SHIFT)
        )
        difference_q = (
            input_aligned_q - source["t75_q"].astype(np.int64)
        )

        # Check the square BEFORE multiplying.
        difference_bound = max(
            abs(int(difference_q.min())),
            abs(int(difference_q.max())),
        )
        square_bound = difference_bound * difference_bound
        assert (
            square_bound + LN2_SQUARE_ROUNDING_OFFSET
            <= ln2_int64_max
        )

        # Op 45: fine squared differences at scale 2^-46.
        t76_fine_q = difference_q * difference_q

        # Convert nonnegative squares to scale 2^-23.
        # Nearest rounding, ties upward.
        t76_wide = (
            t76_fine_q + LN2_SQUARE_ROUNDING_OFFSET
        ) >> LN2_SQUARE_REQUANT_SHIFT

        assert t76_wide.min() >= 0
        assert t76_wide.max() <= ln2_int32_limits.max
        t76_q = t76_wide.astype(np.int32)

        # Fine variance: diagnostic alternative without square conversion.
        assert (
            int(t76_fine_q.max()) * LN2_FEATURE_COUNT
            <= ln2_int64_max
        )
        t77_fine_q = np.sum(
            t76_fine_q,
            axis=ln2_variance_axes,
            dtype=np.int64,
        )

        # Op 46: store the sum at scale 2^-30.
        # Check the wide sum before INT32 storage.
        assert (
            int(t76_q.max()) * LN2_FEATURE_COUNT
            <= ln2_int64_max
        )
        variance_wide = np.sum(
            t76_q,
            axis=ln2_variance_axes,
            dtype=np.int64,
        )
        assert variance_wide.min() >= 0
        assert variance_wide.max() <= ln2_int32_limits.max

        t77_q = variance_wide.astype(np.int32)
        assert t77_q.shape == variance_ref.shape

        # Floating-point reconstruction: diagnostics only.
        square_fine_approx = (
            t76_fine_q.astype(np.float64) * LN2_SQUARE_FINE_SCALE
        )
        square_approx = (
            t76_q.astype(np.float64) * LN2_SQUARE_SCALE
        )
        variance_fine_approx = (
            t77_fine_q.astype(np.float64) * LN2_VARIANCE_FINE_SCALE
        )
        variance_approx = (
            t77_q.astype(np.float64) * LN2_VARIANCE_SCALE
        )

        square_requant_error = np.abs(
            square_approx - square_fine_approx
        )
        variance_requant_error = np.abs(
            variance_approx - variance_fine_approx
        )

        # Averaging cannot exceed the per-square conversion-error bound.
        assert (
            square_requant_error.max()
            <= LN2_SQUARE_SCALE / 2 + 1e-12
        )
        assert (
            variance_requant_error.max()
            <= LN2_SQUARE_SCALE / 2 + 1e-12
        )

        paths[path] = {
            "difference_q": difference_q,
            "t76_fine_q": t76_fine_q,
            "t76_q": t76_q,
            "t77_fine_q": t77_fine_q,
            "t77_q": t77_q,
            "variance_approx": variance_approx,
            "square_error": np.abs(square_approx - square_ref),
            "square_requant_error": square_requant_error,
            "variance_fine_error": np.abs(
                variance_fine_approx - variance_ref
            ),
            "variance_error": np.abs(
                variance_approx - variance_ref
            ),
            "variance_requant_error": variance_requant_error,
        }

    ln2_variance_integer_trial.append({
        "name": case["name"],
        "square_ref": square_ref,
        "variance_ref": variance_ref,
        "paths": paths,
    })


def report_ln2_variance(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    reference = np.concatenate([
        t["variance_ref"].reshape(-1) for t in trials
    ])
    print(
        "  T77 reference variance range:",
        f"{reference.min():.9e} to {reference.max():.9e}",
    )

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        for key, description in [
            ("square_error", "T76 squared-difference error vs TFLite"),
            ("square_requant_error", "Additional square conversion error"),
            ("variance_fine_error", "Fine variance error vs TFLite"),
            ("variance_error", "Candidate T77 variance error vs TFLite"),
            ("variance_requant_error", "Additional variance conversion error"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1)
                for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        for key, description in [
            ("t76_q", "T76 INT32 squared differences"),
            ("t77_q", "T77 INT32 variance"),
        ]:
            values = np.concatenate([
                t["paths"][path][key].reshape(-1)
                for t in trials
            ])
            print(
                f"    {description}: {values.min()} to {values.max()}"
            )

    print("  Pre-multiplication and sum-width checks: pass")
    print("  T76/T77 INT32 range checks: pass")


print("T76 fine squared differences: INT64")
print("  Scale:", LN2_SQUARE_FINE_SCALE)
print("T76 candidate squared differences: INT32")
print("  Scale:", LN2_SQUARE_SCALE)
print("  Conversion right shift:", LN2_SQUARE_REQUANT_SHIFT)
print("T77 candidate variance: INT32")
print("  Scale:", LN2_VARIANCE_SCALE)
print("  Fractional bits:", LN2_VARIANCE_FRAC_BITS)
print("  Zero point: 0")

report_ln2_variance("Benchmark", ln2_variance_integer_trial[:1])
report_ln2_variance("Calibration", ln2_variance_integer_trial[1:])

T76 fine squared differences: INT64
  Scale: 1.4210854715202004e-14
T76 candidate squared differences: INT32
  Scale: 1.1920928955078125e-07
  Conversion right shift: 23
T77 candidate variance: INT32
  Scale: 9.313225746154785e-10
  Fractional bits: 30
  Zero point: 0

Benchmark
  T77 reference variance range: 1.144323684e-02 to 2.093932964e-02
  Local path:
    T76 squared-difference error vs TFLite:
      Max:  5.334615707e-06
      Mean: 7.081201752e-07
    Additional square conversion error:
      Max:  5.960464478e-08
      Mean: 2.921643780e-08
    Fine variance error vs TFLite:
      Max:  2.037688773e-07
      Mean: 6.850933376e-08
    Candidate T77 variance error vs TFLite:
      Max:  2.002343535e-07
      Mean: 6.859772839e-08
    Additional variance conversion error:
      Max:  6.024492905e-09
      Mean: 1.873719402e-09
    T76 INT32 squared differences: 0 to 2137139
    T77 INT32 variance: 12287105 to 22483578
  Chained path:
    T76 squared-difference error vs TFLite:
 

In [25]:
# Part 5C: second LayerNorm, Op 47–48.
# Epsilon addition and integer reciprocal square root.

from math import isqrt

assert ln2_ops[47]["op_name"] == "ADD"
assert ln2_ops[48]["op_name"] == "RSQRT"

assert t77_index in ln2_ops[47]["inputs"]

ln2_epsilon_index = next(
    int(v) for v in ln2_ops[47]["inputs"]
    if int(v) != t77_index
)
t78_index = int(ln2_ops[47]["outputs"][0])
t79_index = int(ln2_ops[48]["outputs"][0])

assert (ln2_epsilon_index, t78_index, t79_index) == (2, 78, 79)
assert int(ln2_ops[48]["inputs"][0]) == t78_index

# Read epsilon from the fixed export.
ln2_epsilon_tensor = interpreter.get_tensor(ln2_epsilon_index)
assert ln2_epsilon_tensor.size == 1

ln2_epsilon_fp32 = float(ln2_epsilon_tensor.reshape(-1)[0])
assert np.isfinite(ln2_epsilon_fp32)
assert ln2_epsilon_fp32 > 0

LN2_EPSILON_VARIANCE_FRAC_BITS = LN2_VARIANCE_FRAC_BITS
LN2_EPSILON_VARIANCE_SCALE = LN2_VARIANCE_SCALE

# OFFLINE: convert epsilon at the current variance scale.
LN2_EPSILON_Q = int(np.rint(
    ln2_epsilon_fp32 / LN2_EPSILON_VARIANCE_SCALE
))
assert 0 < LN2_EPSILON_Q <= ln2_int32_limits.max

ln2_epsilon_approx = (
    LN2_EPSILON_Q * LN2_EPSILON_VARIANCE_SCALE
)

LN2_RSQRT_FRAC_BITS = 16
LN2_RSQRT_SCALE = 2.0 ** (-LN2_RSQRT_FRAC_BITS)

LN2_RSQRT_NUMERATOR_BITS = (
    LN2_EPSILON_VARIANCE_FRAC_BITS + 2 * LN2_RSQRT_FRAC_BITS
)
LN2_RSQRT_NUMERATOR = 1 << LN2_RSQRT_NUMERATOR_BITS

assert LN2_RSQRT_NUMERATOR_BITS == 62
assert LN2_RSQRT_NUMERATOR <= ln2_int64_max

# Variance is nonnegative, so epsilon bounds the smallest input.
# Check the largest possible output under that bound.
assert (
    isqrt(LN2_RSQRT_NUMERATOR // LN2_EPSILON_Q)
    <= ln2_int32_limits.max
)

ln2_rsqrt_integer_trial = []
assert len(op0_cases) == len(ln2_variance_integer_trial)

for case, variance_trial in zip(
    op0_cases, ln2_variance_integer_trial
):
    assert case["name"] == variance_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    t78_ref = interpreter.get_tensor(t78_index).copy()
    t79_ref = interpreter.get_tensor(t79_index).copy()

    assert t78_ref.shape == (32,)
    assert t79_ref.shape == (32,)
    assert np.all(np.isfinite(t79_ref))
    assert np.all(t79_ref > 0)

    paths = {}

    for path in ["local", "chained"]:
        variance_q = variance_trial["paths"][path]["t77_q"]
        assert np.all(variance_q >= 0)

        # Op 47: same scale, so add epsilon directly.
        t78_wide = variance_q.astype(np.int64) + LN2_EPSILON_Q
        assert t78_wide.min() > 0
        assert t78_wide.max() <= ln2_int32_limits.max
        t78_q = t78_wide.astype(np.int32)

        # Op 48: integer division followed by integer square root.
        division_values = []
        inverse_std_values = []

        for stored_input in t78_q:
            q = int(stored_input)
            quotient = LN2_RSQRT_NUMERATOR // q
            inverse_std_q = isqrt(quotient)

            assert 0 <= quotient <= ln2_int64_max
            assert 0 <= inverse_std_q <= ln2_int32_limits.max

            # Exact integer checks of floor rounding.
            assert (
                inverse_std_q * inverse_std_q
                <= quotient
                < (inverse_std_q + 1) * (inverse_std_q + 1)
            )
            assert (
                inverse_std_q * inverse_std_q * q
                <= LN2_RSQRT_NUMERATOR
                < (inverse_std_q + 1) * (inverse_std_q + 1) * q
            )

            division_values.append(quotient)
            inverse_std_values.append(inverse_std_q)

        division_q = np.asarray(division_values, dtype=np.int64)
        t79_q = np.asarray(inverse_std_values, dtype=np.int32)

        # Floating-point calculations below are diagnostics only.
        t78_approx = (
            t78_q.astype(np.float64) * LN2_EPSILON_VARIANCE_SCALE
        )
        t79_approx = t79_q.astype(np.float64) * LN2_RSQRT_SCALE

        ideal_rsqrt_from_candidate = 1.0 / np.sqrt(t78_approx)

        rsqrt_own_error = np.abs(
            t79_approx - ideal_rsqrt_from_candidate
        )
        rsqrt_upstream_error = np.abs(
            ideal_rsqrt_from_candidate - t79_ref
        )
        rsqrt_error = np.abs(t79_approx - t79_ref)

        assert np.all(
            t79_approx <= ideal_rsqrt_from_candidate + 1e-12
        )
        assert rsqrt_own_error.max() < LN2_RSQRT_SCALE + 1e-12

        paths[path] = {
            "t78_q": t78_q,
            "division_q": division_q,
            "t79_q": t79_q,
            "t78_approx": t78_approx,
            "t79_approx": t79_approx,
            "t78_error": np.abs(t78_approx - t78_ref),
            "rsqrt_own_error": rsqrt_own_error,
            "rsqrt_upstream_error": rsqrt_upstream_error,
            "rsqrt_error": rsqrt_error,
            "rsqrt_relative_error": (
                rsqrt_error / np.abs(t79_ref)
            ),
        }

    ln2_rsqrt_integer_trial.append({
        "name": case["name"],
        "t78_ref": t78_ref,
        "t79_ref": t79_ref,
        "paths": paths,
    })


def report_ln2_rsqrt(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    reference = np.concatenate([
        t["t79_ref"].reshape(-1) for t in trials
    ])
    print(
        "  T79 reference inverse-standard-deviation range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        for key, description in [
            ("t78_error", "T78 variance-plus-epsilon error vs TFLite"),
            ("rsqrt_own_error", "Integer RSQRT's own approximation error"),
            ("rsqrt_upstream_error", "Effect of earlier input/variance errors"),
            ("rsqrt_error", "T79 total error vs TFLite"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1)
                for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        relative_errors = np.concatenate([
            t["paths"][path]["rsqrt_relative_error"].reshape(-1)
            for t in trials
        ])
        print(
            "    Maximum T79 relative error:",
            f"{100 * relative_errors.max():.6f}%",
        )

        for key, description in [
            ("t78_q", "T78 INT32 variance plus epsilon"),
            ("t79_q", "T79 INT32 inverse standard deviation"),
        ]:
            values = np.concatenate([
                t["paths"][path][key].reshape(-1)
                for t in trials
            ])
            print(
                f"    {description}: {values.min()} to {values.max()}"
            )

    print("  Positive-input and width checks: pass")
    print("  Exact integer floor-square-root checks: pass")


print("T2 (epsilon):")
print("  Original FP32 value:", ln2_epsilon_fp32)
print("  Integer at variance scale:", LN2_EPSILON_Q)
print("  Represented value:", ln2_epsilon_approx)
print(
    "  Absolute conversion error:",
    f"{abs(ln2_epsilon_approx - ln2_epsilon_fp32):.9e}",
)

print("\nT78 (variance plus epsilon): INT32")
print("  Scale:", LN2_EPSILON_VARIANCE_SCALE)
print("T79 (inverse standard deviation): INT32")
print("  Scale:", LN2_RSQRT_SCALE)
print("  Fractional bits:", LN2_RSQRT_FRAC_BITS)
print("Integer numerator: 2^", LN2_RSQRT_NUMERATOR_BITS, sep="")
print("RSQRT rounding: floor")
print("RSQRT own absolute-error bound: less than", LN2_RSQRT_SCALE)

report_ln2_rsqrt("Benchmark", ln2_rsqrt_integer_trial[:1])
report_ln2_rsqrt("Calibration", ln2_rsqrt_integer_trial[1:])

T2 (epsilon):
  Original FP32 value: 9.999999747378752e-06
  Integer at variance scale: 10737
  Represented value: 9.999610483646393e-06
  Absolute conversion error: 3.892637324e-10

T78 (variance plus epsilon): INT32
  Scale: 9.313225746154785e-10
T79 (inverse standard deviation): INT32
  Scale: 1.52587890625e-05
  Fractional bits: 16
Integer numerator: 2^62
RSQRT rounding: floor
RSQRT own absolute-error bound: less than 1.52587890625e-05

Benchmark
  T79 reference inverse-standard-deviation range: 6.908995628 to 9.344065666
  Local path:
    T78 variance-plus-epsilon error vs TFLite:
      Max:  2.002343535e-07
      Mean: 6.859772839e-08
    Integer RSQRT's own approximation error:
      Max:  1.416295770e-05
      Mean: 7.797349619e-06
    Effect of earlier input/variance errors:
      Max:  5.952279971e-05
      Mean: 2.096739759e-05
    T79 total error vs TFLite:
      Max:  6.103515625e-05
      Mean: 2.127885818e-05
    Maximum T79 relative error: 0.000692%
    T78 INT32 varian

In [26]:
# Part 5D: second LayerNorm, Op 49–54.
# Normalization, followed by conversion to INT32 at scale 2^-16.

ln2_normalization_names = {
    49: "MUL",
    50: "SUB",
    51: "RESHAPE",
    52: "MUL",
    53: "RESHAPE",
    54: "ADD",
}

for op_index, expected in ln2_normalization_names.items():
    assert ln2_ops[op_index]["op_name"] == expected

ln2_normalization_indices = {
    op_index: int(ln2_ops[op_index]["outputs"][0])
    for op_index in ln2_normalization_names
}
assert tuple(ln2_normalization_indices.values()) == (
    80, 81, 82, 83, 84, 85
)

assert set(int(v) for v in ln2_ops[49]["inputs"]) == {
    t79_index, t74_index
}
assert int(ln2_ops[50]["inputs"][1]) == 80
assert int(ln2_ops[51]["inputs"][0]) == t79_index
assert set(int(v) for v in ln2_ops[52]["inputs"]) == {
    ln2_input_index, 82
}
assert int(ln2_ops[53]["inputs"][0]) == 81
assert set(int(v) for v in ln2_ops[54]["inputs"]) == {83, 84}

ln2_zero = interpreter.get_tensor(int(ln2_ops[50]["inputs"][0]))
assert np.all(ln2_zero == 0)

ln2_rsqrt_broadcast_shape = tuple(
    int(v) for v in interpreter.get_tensor(
        int(ln2_ops[51]["inputs"][1])
    ).reshape(-1)
)
ln2_correction_broadcast_shape = tuple(
    int(v) for v in interpreter.get_tensor(
        int(ln2_ops[53]["inputs"][1])
    ).reshape(-1)
)
assert ln2_rsqrt_broadcast_shape == (1, 32, 1)
assert ln2_correction_broadcast_shape == (1, 32, 1)

LN2_CORRECTION_FRAC_BITS = (
    LN2_MEAN_FRAC_BITS + LN2_RSQRT_FRAC_BITS
)
LN2_SCALED_INPUT_FRAC_BITS = (
    LN2_INPUT_FRAC_BITS + LN2_RSQRT_FRAC_BITS
)
LN2_NORMALIZED_FINE_FRAC_BITS = LN2_CORRECTION_FRAC_BITS
LN2_NORMALIZED_FINE_SCALE = (
    2.0 ** (-LN2_NORMALIZED_FINE_FRAC_BITS)
)

LN2_NORMALIZED_FRAC_BITS = 16
LN2_NORMALIZED_SCALE = 2.0 ** (-LN2_NORMALIZED_FRAC_BITS)

LN2_NORMALIZATION_ALIGN_SHIFT = (
    LN2_NORMALIZED_FINE_FRAC_BITS - LN2_SCALED_INPUT_FRAC_BITS
)
LN2_NORMALIZATION_REQUANT_SHIFT = (
    LN2_NORMALIZED_FINE_FRAC_BITS - LN2_NORMALIZED_FRAC_BITS
)
LN2_NORMALIZATION_ROUNDING_OFFSET = (
    1 << (LN2_NORMALIZATION_REQUANT_SHIFT - 1)
)

assert LN2_NORMALIZED_FINE_FRAC_BITS == 39
assert LN2_NORMALIZATION_ALIGN_SHIFT == 7
assert LN2_NORMALIZATION_REQUANT_SHIFT == 23

ln2_stage_scales = {
    49: 2.0 ** (-LN2_CORRECTION_FRAC_BITS),
    50: 2.0 ** (-LN2_CORRECTION_FRAC_BITS),
    51: LN2_RSQRT_SCALE,
    52: 2.0 ** (-LN2_SCALED_INPUT_FRAC_BITS),
    53: 2.0 ** (-LN2_CORRECTION_FRAC_BITS),
    54: LN2_NORMALIZED_FINE_SCALE,
}

ln2_normalization_integer_trial = []

assert (
    len(op0_cases)
    == len(ln2_mean_integer_trial)
    == len(ln2_rsqrt_integer_trial)
)

for case, mean_trial, rsqrt_trial in zip(
    op0_cases, ln2_mean_integer_trial, ln2_rsqrt_integer_trial
):
    assert case["name"] == mean_trial["name"] == rsqrt_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    references = {
        op_index: interpreter.get_tensor(tensor_index).copy()
        for op_index, tensor_index in ln2_normalization_indices.items()
    }

    paths = {}

    for path in ["local", "chained"]:
        mean_source = mean_trial["paths"][path]
        rsqrt_source = rsqrt_trial["paths"][path]

        input_q = mean_source["t73_q"].astype(np.int64)
        mean_q = mean_source["t74_q"].astype(np.int64)
        inverse_std_q = rsqrt_source["t79_q"].astype(np.int64)

        # Bounds are evaluated as Python integers before multiplication.
        input_bound = max(abs(int(input_q.min())), abs(int(input_q.max())))
        mean_bound = max(abs(int(mean_q.min())), abs(int(mean_q.max())))
        rsqrt_bound = int(inverse_std_q.max())

        assert mean_bound * rsqrt_bound <= ln2_int64_max
        assert input_bound * rsqrt_bound <= ln2_int64_max

        # Op 49: mean * inverse standard deviation, scale 2^-39.
        correction_q = mean_q * inverse_std_q

        # Op 50: zero - correction.
        negative_correction_q = -correction_q

        # Op 51: reshape inverse standard deviation.
        broadcast_rsqrt_q = inverse_std_q.astype(np.int32).reshape(
            ln2_rsqrt_broadcast_shape
        )

        # Op 52: input * inverse standard deviation, scale 2^-32.
        scaled_input_q = (
            input_q * broadcast_rsqrt_q.astype(np.int64)
        )

        # Op 53: reshape the negative correction.
        broadcast_negative_q = negative_correction_q.reshape(
            ln2_correction_broadcast_shape
        )

        # Op 54: align scaled input to scale 2^-39, then add.
        assert (
            input_bound * rsqrt_bound
            * (1 << LN2_NORMALIZATION_ALIGN_SHIFT)
            + mean_bound * rsqrt_bound
            <= ln2_int64_max
        )

        t85_fine_q = (
            scaled_input_q * (1 << LN2_NORMALIZATION_ALIGN_SHIFT)
            + broadcast_negative_q
        )

        # Verify equivalence to (input - mean) * inverse_std.
        centered_input_q = (
            input_q * (1 << LN2_CENTER_ALIGN_SHIFT)
            - mean_q.reshape(ln2_correction_broadcast_shape)
        )
        centered_bound = max(
            abs(int(centered_input_q.min())),
            abs(int(centered_input_q.max())),
        )
        assert centered_bound * rsqrt_bound <= ln2_int64_max

        fused_q = centered_input_q * broadcast_rsqrt_q.astype(np.int64)
        np.testing.assert_array_equal(t85_fine_q, fused_q)

        stage_q = {
            49: correction_q,
            50: negative_correction_q,
            51: broadcast_rsqrt_q,
            52: scaled_input_q,
            53: broadcast_negative_q,
            54: t85_fine_q,
        }

        stage_errors = {}
        for op_index, integers in stage_q.items():
            reference = references[op_index]
            assert integers.shape == reference.shape

            represented = (
                integers.astype(np.float64) * ln2_stage_scales[op_index]
            )
            stage_errors[op_index] = np.abs(represented - reference)

        # Convert normalized features from scale 2^-39 to 2^-16.
        fine_bound = max(
            abs(int(t85_fine_q.min())),
            abs(int(t85_fine_q.max())),
        )
        assert (
            fine_bound + LN2_NORMALIZATION_ROUNDING_OFFSET
            <= ln2_int64_max
        )

        t85_wide = (
            t85_fine_q + LN2_NORMALIZATION_ROUNDING_OFFSET
        ) >> LN2_NORMALIZATION_REQUANT_SHIFT

        assert t85_wide.min() >= ln2_int32_limits.min
        assert t85_wide.max() <= ln2_int32_limits.max
        t85_q = t85_wide.astype(np.int32)

        # Floating-point reconstruction: diagnostics only.
        fine_approx = (
            t85_fine_q.astype(np.float64) * LN2_NORMALIZED_FINE_SCALE
        )
        t85_approx = (
            t85_q.astype(np.float64) * LN2_NORMALIZED_SCALE
        )
        requant_error = np.abs(t85_approx - fine_approx)

        assert requant_error.max() <= LN2_NORMALIZED_SCALE / 2 + 1e-12

        paths[path] = {
            "stage_q": stage_q,
            "stage_errors": stage_errors,
            "t85_fine_q": t85_fine_q,
            "t85_q": t85_q,
            "t85_approx": t85_approx,
            "t85_error": np.abs(t85_approx - references[54]),
            "requant_error": requant_error,
        }

    ln2_normalization_integer_trial.append({
        "name": case["name"],
        "t85_ref": references[54],
        "paths": paths,
    })


def report_ln2_normalization(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    reference = np.concatenate([
        t["t85_ref"].reshape(-1) for t in trials
    ])
    print(
        "  T85 reference normalized-feature range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        for op_index, tensor_index in ln2_normalization_indices.items():
            errors = np.concatenate([
                t["paths"][path]["stage_errors"][op_index].reshape(-1)
                for t in trials
            ])
            print(
                f"    Op {op_index}, T{tensor_index} error vs TFLite: "
                f"max={errors.max():.9e}, mean={errors.mean():.9e}"
            )

        for key, description in [
            ("t85_error", "Candidate T85 total error vs TFLite"),
            ("requant_error", "Additional T85 conversion error"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1) for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        values = np.concatenate([
            t["paths"][path]["t85_q"].reshape(-1) for t in trials
        ])
        print(
            "    T85 INT32 range:",
            f"{values.min()} to {values.max()}",
        )

    print("  Graph decomposition vs fused expression: exact")
    print("  Intermediate-width and T85 INT32 checks: pass")


print("T85 fine normalized features: INT64")
print("  Scale:", LN2_NORMALIZED_FINE_SCALE)
print("T85 candidate normalized features: INT32")
print("  Scale:", LN2_NORMALIZED_SCALE)
print("  Zero point: 0")
print("Input-product alignment multiplier:",
      1 << LN2_NORMALIZATION_ALIGN_SHIFT)
print("T85 conversion right shift:", LN2_NORMALIZATION_REQUANT_SHIFT)
print("Conversion rounding: nearest, ties toward positive infinity")

report_ln2_normalization("Benchmark", ln2_normalization_integer_trial[:1])
report_ln2_normalization("Calibration", ln2_normalization_integer_trial[1:])

T85 fine normalized features: INT64
  Scale: 1.8189894035458565e-12
T85 candidate normalized features: INT32
  Scale: 1.52587890625e-05
  Zero point: 0
Input-product alignment multiplier: 128
T85 conversion right shift: 23
Conversion rounding: nearest, ties toward positive infinity

Benchmark
  T85 reference normalized-feature range: -3.487281322 to 3.548740387
  Local path:
    Op 49, T80 error vs TFLite: max=6.596876119e-06, mean=2.882935860e-06
    Op 50, T81 error vs TFLite: max=6.596876119e-06, mean=2.882935860e-06
    Op 51, T82 error vs TFLite: max=6.103515625e-05, mean=2.127885818e-05
    Op 52, T83 error vs TFLite: max=7.565319538e-05, mean=3.252646408e-05
    Op 53, T84 error vs TFLite: max=6.596876119e-06, mean=2.882935860e-06
    Op 54, T85 error vs TFLite: max=7.570062553e-05, mean=3.229209259e-05
    Candidate T85 total error vs TFLite:
      Max:  8.189678192e-05
      Mean: 3.239601568e-05
    Additional T85 conversion error:
      Max:  7.627717423e-06
      Mean: 3.84

In [27]:
# Part 5E: second LayerNorm, Op 55–57.
# Learned affine transform and conversion to T88 (INT8 MLP input).

assert ln2_ops[55]["op_name"] == "MUL"
assert ln2_ops[56]["op_name"] == "ADD"
assert ln2_ops[57]["op_name"] == "QUANTIZE"

t85_index = ln2_normalization_indices[54]
t86_index = int(ln2_ops[55]["outputs"][0])
t87_index = int(ln2_ops[56]["outputs"][0])
t88_index = int(ln2_ops[57]["outputs"][0])

ln2_gamma_index = next(
    int(v) for v in ln2_ops[55]["inputs"] if int(v) != t85_index
)
ln2_beta_index = next(
    int(v) for v in ln2_ops[56]["inputs"] if int(v) != t86_index
)

assert (t85_index, t86_index, t87_index, t88_index) == (85, 86, 87, 88)
assert (ln2_gamma_index, ln2_beta_index) == (8, 9)
assert t85_index in ln2_ops[55]["inputs"]
assert t86_index in ln2_ops[56]["inputs"]
assert int(ln2_ops[57]["inputs"][0]) == t87_index

ln2_gamma_fp32 = interpreter.get_tensor(ln2_gamma_index).copy()
ln2_beta_fp32 = interpreter.get_tensor(ln2_beta_index).copy()

assert ln2_gamma_fp32.shape == (128,)
assert ln2_beta_fp32.shape == (128,)
assert np.isfinite(ln2_gamma_fp32).all()
assert np.isfinite(ln2_beta_fp32).all()

LN2_GAMMA_FRAC_BITS = 16
LN2_BETA_FRAC_BITS = 16
LN2_GAMMA_SCALE = 2.0 ** (-LN2_GAMMA_FRAC_BITS)
LN2_BETA_SCALE = 2.0 ** (-LN2_BETA_FRAC_BITS)


def ln2_quantize_parameter(values, scale):
    # OFFLINE: nearest rounding, ties to even.
    rounded = np.rint(values.astype(np.float64) / scale)
    assert rounded.min() >= ln2_int32_limits.min
    assert rounded.max() <= ln2_int32_limits.max
    return rounded.astype(np.int32)


ln2_gamma_q = ln2_quantize_parameter(
    ln2_gamma_fp32, LN2_GAMMA_SCALE
)
ln2_beta_q = ln2_quantize_parameter(
    ln2_beta_fp32, LN2_BETA_SCALE
)

LN2_AFFINE_FRAC_BITS = (
    LN2_NORMALIZED_FRAC_BITS + LN2_GAMMA_FRAC_BITS
)
LN2_AFFINE_SCALE = 2.0 ** (-LN2_AFFINE_FRAC_BITS)
LN2_BETA_ALIGN_SHIFT = LN2_AFFINE_FRAC_BITS - LN2_BETA_FRAC_BITS

assert LN2_AFFINE_FRAC_BITS == 32
assert LN2_BETA_ALIGN_SHIFT == 16

# Read T88's actual INT8 representation.
ln2_details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}
t88_detail = ln2_details[t88_index]
assert t88_detail["dtype"] == np.int8
assert tuple(t88_detail["shape"]) == (1, 32, 128)

t88_params = t88_detail["quantization_parameters"]
assert len(t88_params["scales"]) == 1
assert len(t88_params["zero_points"]) == 1

T88_SCALE = float(t88_params["scales"][0])
T88_ZERO_POINT = int(t88_params["zero_points"][0])
assert np.isfinite(T88_SCALE) and T88_SCALE > 0

# OFFLINE: encode the affine-output-to-T88 scale ratio.
T88_REAL_MULTIPLIER = LN2_AFFINE_SCALE / T88_SCALE
T88_REQUANT_SHIFT = 47
T88_ROUNDING_OFFSET = 1 << (T88_REQUANT_SHIFT - 1)

T88_MULTIPLIER_Q = int(np.rint(
    T88_REAL_MULTIPLIER * (1 << T88_REQUANT_SHIFT)
))
assert 0 < T88_MULTIPLIER_Q <= ln2_int32_limits.max

t88_multiplier_approx = (
    T88_MULTIPLIER_Q * 2.0 ** (-T88_REQUANT_SHIFT)
)

ln2_affine_integer_trial = []
assert len(op0_cases) == len(ln2_normalization_integer_trial)

for case, normalization_trial in zip(
    op0_cases, ln2_normalization_integer_trial
):
    assert case["name"] == normalization_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    t86_ref = interpreter.get_tensor(t86_index).copy()
    t87_ref = interpreter.get_tensor(t87_index).copy()
    t88_ref = interpreter.get_tensor(t88_index).copy()

    # Diagnostic control: original T87 with our output rounding rule.
    reference_with_candidate_rounding = np.clip(
        np.floor(t87_ref.astype(np.float64) / T88_SCALE + 0.5)
        + T88_ZERO_POINT,
        -128, 127,
    ).astype(np.int8)

    paths = {}

    for path in ["local", "chained"]:
        source = normalization_trial["paths"][path]

        # Op 55: normalized features * quantized gamma.
        normalized_bound = max(
            abs(int(source["t85_q"].min())),
            abs(int(source["t85_q"].max())),
        )
        gamma_bound = max(
            abs(int(ln2_gamma_q.min())),
            abs(int(ln2_gamma_q.max())),
        )
        assert normalized_bound * gamma_bound <= ln2_int64_max

        t86_q = (
            source["t85_q"].astype(np.int64)
            * ln2_gamma_q.astype(np.int64)
        )

        # Op 56: align beta to scale 2^-32, then add.
        aligned_beta_q = (
            ln2_beta_q.astype(np.int64) * (1 << LN2_BETA_ALIGN_SHIFT)
        )
        product_bound = max(
            abs(int(t86_q.min())), abs(int(t86_q.max()))
        )
        beta_bound = max(
            abs(int(aligned_beta_q.min())),
            abs(int(aligned_beta_q.max())),
        )
        assert product_bound + beta_bound <= ln2_int64_max

        t87_q = t86_q + aligned_beta_q

        # Op 57: integer-only conversion to T88.
        affine_bound = max(
            abs(int(t87_q.min())), abs(int(t87_q.max()))
        )
        assert (
            affine_bound * T88_MULTIPLIER_Q + T88_ROUNDING_OFFSET
            <= ln2_int64_max
        )

        rounded_q = (
            t87_q * T88_MULTIPLIER_Q + T88_ROUNDING_OFFSET
        ) >> T88_REQUANT_SHIFT

        before_saturation = rounded_q + T88_ZERO_POINT
        saturation_count = int(np.count_nonzero(
            (before_saturation < -128) | (before_saturation > 127)
        ))
        t88_q = np.clip(
            before_saturation, -128, 127
        ).astype(np.int8)

        # Floating-point calculations below are diagnostics only.
        t86_approx = t86_q.astype(np.float64) * LN2_AFFINE_SCALE
        t87_approx = t87_q.astype(np.float64) * LN2_AFFINE_SCALE

        ideal_affine_from_candidate = (
            source["t85_approx"] * ln2_gamma_fp32.astype(np.float64)
            + ln2_beta_fp32.astype(np.float64)
        )

        ideal_t88_q = np.clip(
            np.floor(t87_approx / T88_SCALE + 0.5) + T88_ZERO_POINT,
            -128, 127,
        ).astype(np.int8)

        paths[path] = {
            "t86_q": t86_q,
            "t87_q": t87_q,
            "t88_q": t88_q,
            "t87_approx": t87_approx,
            "t86_error": np.abs(t86_approx - t86_ref),
            "t87_error": np.abs(t87_approx - t87_ref),
            "parameter_error": np.abs(
                t87_approx - ideal_affine_from_candidate
            ),
            "t88_difference": (
                t88_q.astype(np.int32) - t88_ref.astype(np.int32)
            ),
            "requant_mismatch_count": int(np.count_nonzero(
                t88_q != ideal_t88_q
            )),
            "saturation_count": saturation_count,
        }

    ln2_affine_integer_trial.append({
        "name": case["name"],
        "t87_ref": t87_ref,
        "t88_ref": t88_ref,
        "reference_rounding_mismatch_count": int(np.count_nonzero(
            reference_with_candidate_rounding != t88_ref
        )),
        "paths": paths,
    })


def report_ln2_affine(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        for key, description in [
            ("t86_error", "T86 features-times-gamma error vs TFLite"),
            ("t87_error", "T87 affine-output error vs TFLite"),
            ("parameter_error", "Additional gamma/beta conversion error"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1) for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        differences = np.concatenate([
            t["paths"][path]["t88_difference"].reshape(-1)
            for t in trials
        ])
        values = np.concatenate([
            t["paths"][path]["t88_q"].reshape(-1) for t in trials
        ])
        matched = int(np.count_nonzero(differences == 0))

        print(
            f"    T88 exact matches: {matched}/{differences.size} "
            f"({100 * matched / differences.size:.3f}%)"
        )
        print("    Maximum INT8-step difference:",
              int(np.abs(differences).max()))
        print("    Mean absolute INT8-step difference:",
              f"{np.abs(differences).mean():.6f}")
        print("    T88 integer range:", values.min(), "to", values.max())

        unique, counts = np.unique(differences, return_counts=True)
        print("    Signed difference counts:", {
            int(d): int(c) for d, c in zip(unique, counts)
        })

        print("    Fixed multiplier vs ideal conversion mismatches:", sum(
            t["paths"][path]["requant_mismatch_count"] for t in trials
        ))
        print("    Values requiring saturation:", sum(
            t["paths"][path]["saturation_count"] for t in trials
        ))

    print("  Original T87 with candidate rounding vs T88 mismatches:", sum(
        t["reference_rounding_mismatch_count"] for t in trials
    ))
    print("  Affine arithmetic and requantization width checks: pass")


for index, meaning, original, quantized, scale in [
    (ln2_gamma_index, "gamma", ln2_gamma_fp32, ln2_gamma_q,
     LN2_GAMMA_SCALE),
    (ln2_beta_index, "beta", ln2_beta_fp32, ln2_beta_q,
     LN2_BETA_SCALE),
]:
    print(f"T{index} ({meaning}): INT32, scale = {scale}")
    print("  Original parameter range:", original.min(), "to", original.max())
    print(
        "  Maximum parameter conversion error:",
        f"{np.abs(quantized.astype(np.float64) * scale - original).max():.9e}",
    )

print("\nT86/T87 affine intermediates: INT64")
print("  Scale:", LN2_AFFINE_SCALE)
print("  Beta alignment multiplier:", 1 << LN2_BETA_ALIGN_SHIFT)

print("\nT88 (INT8 MLP input):")
print("  Scale:", T88_SCALE)
print("  Zero point:", T88_ZERO_POINT)
print("  Integer conversion multiplier:", T88_MULTIPLIER_Q)
print("  Right shift:", T88_REQUANT_SHIFT)
print(
    "  Multiplier relative error:",
    f"{abs(t88_multiplier_approx / T88_REAL_MULTIPLIER - 1):.9e}",
)
print("  Rounding: nearest, ties toward positive infinity")

report_ln2_affine("Benchmark", ln2_affine_integer_trial[:1])
report_ln2_affine("Calibration", ln2_affine_integer_trial[1:])

first_ln2_affine = ln2_affine_integer_trial[0]
print("\nFirst benchmark token, first 8 T88 features:")
print("  TFLite reference:", first_ln2_affine["t88_ref"][0, 0, :8])
for path in ["local", "chained"]:
    print(
        f"  {path}:",
        first_ln2_affine["paths"][path]["t88_q"][0, 0, :8],
    )

T8 (gamma): INT32, scale = 1.52587890625e-05
  Original parameter range: 1.017637 to 1.1421613
  Maximum parameter conversion error: 7.510185242e-06
T9 (beta): INT32, scale = 1.52587890625e-05
  Original parameter range: -0.035599805 to 0.05044351
  Maximum parameter conversion error: 7.528811693e-06

T86/T87 affine intermediates: INT64
  Scale: 2.3283064365386963e-10
  Beta alignment multiplier: 65536

T88 (INT8 MLP input):
  Scale: 0.03247051313519478
  Zero point: 2
  Integer conversion multiplier: 1009162
  Right shift: 47
  Multiplier relative error: 2.434246653e-07
  Rounding: nearest, ties toward positive infinity

Benchmark
  Local path:
    T86 features-times-gamma error vs TFLite:
      Max:  9.222654626e-05
      Mean: 3.456281539e-05
    T87 affine-output error vs TFLite:
      Max:  9.890226647e-05
      Mean: 3.470644828e-05
    Additional gamma/beta conversion error:
      Max:  2.745721758e-05
      Mean: 4.770707216e-06
    T88 exact matches: 4094/4096 (99.951%)
    Ma

### Block-0 second LayerNorm: integer validation, Op 43–57

Source block-0 SHA-256:
`06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9`

Test inputs: one 32-token benchmark sequence and seven calibration
sequences; hidden size 128.

The saved TFLite graph remains unchanged. Integer counterparts are
calculated separately in Python.

#### Tensor representations

| Tensor | Meaning | Integer candidate |
|---|---|---|
| T73 | Second LayerNorm input | INT32 `[1,32,128]`; scale `2^-16`, zero point 0 |
| T74/T75 | Per-token means / broadcast means | INT32; scale `2^-23`, zero point 0 |
| T76 | Squared differences | INT64 at `2^-46`; converted to INT32 at `2^-23` |
| T77 | Per-token variance | INT32; scale `2^-30`, zero point 0 |
| T78 | Variance plus epsilon | INT32; scale `2^-30`, zero point 0 |
| T79 | Inverse standard deviation | INT32; scale `2^-16`, zero point 0 |
| T80/T81 | Mean correction / negative correction | INT64; scale `2^-39` |
| T82 | Broadcast inverse standard deviation | INT32; scale `2^-16` |
| T83 | Input × inverse standard deviation | INT64; scale `2^-32` |
| T84 | Broadcast negative correction | INT64; scale `2^-39` |
| T85 | Normalized features | INT64 at `2^-39`; converted to INT32 at `2^-16` |
| T8/T9 | Learned gamma / beta | INT32; scale `2^-16` |
| T86/T87 | Affine intermediates / LayerNorm output | INT64; scale `2^-32` |
| T88 | INT8 MLP input | INT8 `[1,32,128]`; scale `0.03247051313519478`, zero point 2 |

#### Op 43–46: mean and variance

The mean stores the sum of 128 input features:

$$
q_{T74}[t]=\sum_{k=0}^{127}q_{T73}[0,t,k]
$$

Division by 128 is represented through the mean scale:
`2^-16 / 128 = 2^-23`.

T75 reshapes the means to `[1,32,1]` without changing their values
or scale.

Input and mean are aligned before calculating squared differences:

$$
d_q=128q_{T73}-q_{T75}
$$

$$
q_{T76,\mathrm{fine}}=d_q^2
$$

Fine squared differences use INT64 at scale `2^-46`.
They are converted to INT32 at scale `2^-23`:

    q76 = (q76_fine + (1 << 22)) >> 23

Variance stores the sum of the converted squared differences,
with division by 128 represented by output scale `2^-30`.

#### Op 47–48: epsilon and inverse standard deviation

Epsilon is read from T2 in the fixed export.

| Epsilon measurement | Value |
|---|---:|
| Original FP32 value | `9.999999747378752e-06` |
| Integer at scale `2^-30` | 10737 |
| Represented value | `9.999610483646393e-06` |
| Absolute conversion error | `3.892637324e-10` |

Epsilon is added directly to the variance:

    q78 = q77 + 10737

Integer reciprocal square root:

    q79 = isqrt((1 << 62) // q78)

T79 uses scale `2^-16`. The numerator `2^62` fits signed INT64.

Rounding is floor. The algorithm's own absolute approximation
error is less than one T79 step: `1.52587890625e-5`.

#### Op 49–54: normalization

The candidate follows the saved graph's separate multiplication,
subtraction, reshape, and addition operations.

Their result exactly matches the fused expression:

$$
q_{T85,\mathrm{fine}}
=
(128q_{T73}-q_{T75})q_{T82}
$$

The fine result has scale `2^-39`.
It is converted to INT32 at scale `2^-16`:

    q85 = (q85_fine + (1 << 22)) >> 23

Conversion uses nearest rounding, with ties toward positive infinity.

#### Op 55–57: learned affine transform and INT8 conversion

Gamma and beta are converted offline to INT32 at scale `2^-16`.

| Parameter | Original range | Maximum conversion error |
|---|---|---:|
| T8 gamma | 1.017637 to 1.1421613 | 7.5102e-6 |
| T9 beta | -0.035599805 to 0.05044351 | 7.5288e-6 |

Affine arithmetic:

    q86 = q85 * gamma_q
    q87 = q86 + beta_q * 65536

T86 and T87 use INT64 at scale `2^-32`.

The T87→T88 conversion ratio is:

$$
M=\frac{2^{-32}}{0.03247051313519478}
\approx1009162\times2^{-47}
$$

Integer output conversion:

    q88 = ((q87 * 1009162 + (1 << 46)) >> 47) + 2
    q88 = clip(q88, -128, 127)

The product uses INT64 arithmetic.
Rounding is nearest, with ties toward positive infinity.

The multiplier's scale `2^-47` describes the conversion ratio.
T88 retains tensor scale `0.03247051313519478` and zero point 2.

#### Validation measurements

- **Local:** starts from TFLite's T73, quantized directly to INT32
  at scale `2^-16`, to isolate the second LayerNorm.
- **Chained:** starts from the existing chained integer residual T73.

The local path therefore differs from Part 4's local residual path:
it starts from the reference residual rather than the INT8 T0 branch.

| Intermediate measurement | Benchmark | Calibration |
|---|---:|---:|
| Local maximum T74 mean error | 8.5874e-7 | 1.0780e-6 |
| Chained maximum T74 mean error | 0.000351 | 0.000412 |
| Local maximum T77 variance error | 2.0023e-7 | 2.6822e-7 |
| Chained maximum T77 variance error | 2.4885e-4 | 2.4885e-4 |
| Local maximum T79 inverse-standard-deviation error | 6.1035e-5 | 1.1635e-4 |
| Chained maximum T79 inverse-standard-deviation error | 0.042701 | 0.065284 |
| Local maximum T85 normalized-feature error | 8.1897e-5 | 9.1314e-5 |
| Chained maximum T85 normalized-feature error | 0.052997 | 0.053643 |
| Local maximum T87 affine-output error | 9.8902e-5 | 1.0516e-4 |
| Local mean T87 affine-output error | 3.4706e-5 | 3.5469e-5 |
| Chained maximum T87 affine-output error | 0.057087 | 0.057442 |
| Chained mean T87 affine-output error | 0.012255 | 0.012244 |

| Final T88 measurement | Benchmark | Calibration |
|---|---:|---:|
| Local exact matches | 4094/4096 (99.951%) | 28644/28672 (99.902%) |
| Local maximum INT8-step difference | 1 | 1 |
| Local mean absolute INT8-step difference | 0.000488 | 0.000977 |
| Chained exact matches | 2587/4096 (63.159%) | 18257/28672 (63.675%) |
| Chained maximum INT8-step difference | 2 | 2 |
| Chained mean absolute INT8-step difference | 0.380127 | 0.380755 |
| Fixed multiplier vs ideal conversion mismatches | 0 | 0 |
| Local values requiring saturation | 0 | 2 |
| Chained values requiring saturation | 0 | 2 |

Across all eight sequences, 30 of 32,768 local T88 values differed
from TFLite, each by one INT8 step.

Applying the candidate rounding rule to original TFLite T87 matched
reference T88 for every tested value.

#### Analysis and status

The local second LayerNorm closely matches the saved graph.
Square conversion, integer RSQRT, normalized-output conversion,
and gamma/beta conversion introduce comparatively small errors.

The fixed output multiplier produced no observed differences
relative to ideal conversion of the candidate's represented T87.

Chained errors include differences propagated from the first
LayerNorm, attention, and residual branches. Normalization can
magnify those input differences because the inverse standard
deviation is approximately 6.5–9.8 on the tested cases.

The T88 match percentages describe exact activation agreement,
not complete-model accuracy.

All implemented mean identities, broadcast-shape checks,
pre-multiplication bounds, sum-width checks, integer square-root
checks, fused-normalization checks, and storage-range checks passed.

Op 43–57 is validated in Python on the tested cases.
C, Spike, RTL, complete-block accuracy, and performance have not
yet been validated for this implementation.

Next: **Op 58–70, MLP expansion from 128 to 512 features and GELU**.

In [28]:
# Part 6A, preparation: Op 58.
# Inspect the 128→512 projection and calculate integer accumulators.

mlp_ops = interpreter._get_ops_details()
mlp_details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}

op58 = mlp_ops[58]
assert op58["op_name"] == "FULLY_CONNECTED"

mlp_input_index = int(op58["inputs"][0])
mlp_weight_index = int(op58["inputs"][1])
mlp_bias_index = int(op58["inputs"][2])
t89_index = int(op58["outputs"][0])

assert (
    mlp_input_index, mlp_weight_index, mlp_bias_index, t89_index
) == (88, 29, 23, 89)

mlp_input_detail = mlp_details[mlp_input_index]
mlp_weight_detail = mlp_details[mlp_weight_index]
mlp_bias_detail = mlp_details[mlp_bias_index]
mlp_output_detail = mlp_details[t89_index]

assert mlp_input_detail["dtype"] == np.int8
assert mlp_weight_detail["dtype"] == np.int8
assert mlp_bias_detail["dtype"] == np.int32
assert mlp_output_detail["dtype"] == np.int8

assert tuple(mlp_input_detail["shape"]) == (1, 32, 128)
assert tuple(mlp_output_detail["shape"]) == (1, 32, 512)

mlp_input_params = mlp_input_detail["quantization_parameters"]
mlp_weight_params = mlp_weight_detail["quantization_parameters"]
mlp_bias_params = mlp_bias_detail["quantization_parameters"]
mlp_output_params = mlp_output_detail["quantization_parameters"]

assert len(mlp_input_params["scales"]) == 1
assert len(mlp_input_params["zero_points"]) == 1
assert len(mlp_output_params["scales"]) == 1
assert len(mlp_output_params["zero_points"]) == 1

mlp_input_scale = float(mlp_input_params["scales"][0])
mlp_input_zero_point = int(mlp_input_params["zero_points"][0])
mlp_output_scale = float(mlp_output_params["scales"][0])
mlp_output_zero_point = int(mlp_output_params["zero_points"][0])

assert mlp_input_scale == T88_SCALE
assert mlp_input_zero_point == T88_ZERO_POINT
assert mlp_input_scale > 0
assert mlp_output_scale > 0

mlp_weights_q = interpreter.get_tensor(mlp_weight_index).copy()
mlp_bias_q = interpreter.get_tensor(mlp_bias_index).copy().reshape(-1)

assert mlp_weights_q.shape == (512, 128)
assert mlp_bias_q.shape == (512,)

mlp_weight_scales_raw = np.asarray(
    mlp_weight_params["scales"], dtype=np.float64
)
mlp_weight_zero_points_raw = np.asarray(
    mlp_weight_params["zero_points"], dtype=np.int64
)

assert mlp_weight_scales_raw.size in (1, 512)
assert mlp_weight_zero_points_raw.size in (1, 512)
assert np.all(mlp_weight_scales_raw > 0)

if mlp_weight_scales_raw.size > 1:
    assert int(mlp_weight_params["quantized_dimension"]) == 0

mlp_weight_scales = np.broadcast_to(
    mlp_weight_scales_raw, (512,)
).copy()
mlp_weight_zero_points = np.broadcast_to(
    mlp_weight_zero_points_raw, (512,)
).copy()

# Bias must use each output channel's accumulator scale.
assert len(mlp_bias_params["scales"]) in (1, 512)
assert np.all(mlp_bias_params["zero_points"] == 0)

np.testing.assert_allclose(
    np.broadcast_to(mlp_bias_params["scales"], (512,)),
    mlp_input_scale * mlp_weight_scales,
    rtol=1e-5,
    atol=0,
)

mlp_expand_spec = {
    "op_index": 58,
    "input_index": mlp_input_index,
    "weight_index": mlp_weight_index,
    "output_index": t89_index,
    "input_scale": mlp_input_scale,
    "input_zero_point": mlp_input_zero_point,
    "weight_scales": mlp_weight_scales,
    "weight_zero_points": mlp_weight_zero_points,
    "output_scale": mlp_output_scale,
    "output_zero_point": mlp_output_zero_point,
    "weights_q": mlp_weights_q,
    "bias_q": mlp_bias_q,
    "real_multipliers": (
        mlp_input_scale * mlp_weight_scales / mlp_output_scale
    ),
}

print("Op 58: MLP expansion, 128→512")
print("  T88 (input): scale =", mlp_input_scale)
print("    Zero point:", mlp_input_zero_point)
print("  T29 (weights):", mlp_weights_q.shape)
print("    Original scale count:", mlp_weight_scales_raw.size)
print("    Scale range:", mlp_weight_scales.min(), "to",
      mlp_weight_scales.max())
print("    Zero points:", np.unique(mlp_weight_zero_points))
print("  T23 (bias): INT32", mlp_bias_q.shape)
print("    Integer range:", mlp_bias_q.min(), "to", mlp_bias_q.max())
print("    Bias scale compatibility: pass")
print("  Accumulator scale range:",
      (mlp_input_scale * mlp_weight_scales).min(), "to",
      (mlp_input_scale * mlp_weight_scales).max())
print("  T89 (output): scale =", mlp_output_scale)
print("    Zero point:", mlp_output_zero_point)
print("  Conversion multiplier range:",
      mlp_expand_spec["real_multipliers"].min(), "to",
      mlp_expand_spec["real_multipliers"].max())

mlp_expand_cases = []

assert len(op0_cases) == len(ln2_affine_integer_trial)

for case, layernorm_trial in zip(
    op0_cases, ln2_affine_integer_trial
):
    assert case["name"] == layernorm_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    input_ref = interpreter.get_tensor(mlp_input_index).copy()
    input_chained = layernorm_trial["paths"]["chained"]["t88_q"]

    assert input_chained.shape == (1, 32, 128)
    assert input_chained.dtype == np.int8

    # The existing helper accepts 128 input features and any number
    # of weight rows, so it also supports this 512-output projection.
    # It removes zero points, adds bias, and checks INT32 bounds.
    acc_local = qkv_accumulate(input_ref, mlp_expand_spec)
    acc_chained = qkv_accumulate(input_chained, mlp_expand_spec)

    assert acc_local.shape == (32, 512)
    assert acc_chained.shape == (32, 512)

    mlp_expand_cases.append({
        "name": case["name"],
        "input_ref": input_ref,
        "input_chained": input_chained.copy(),
        "acc_local": acc_local,
        "acc_chained": acc_chained,
        "t89_ref": interpreter.get_tensor(t89_index).copy(),
    })


def report_mlp_expand_accumulators(label, cases):
    if not cases:
        return

    print(f"\n{label}")

    for key, description in [
        ("acc_local", "Local INT32 accumulator, including bias"),
        ("acc_chained", "Chained INT32 accumulator, including bias"),
        ("t89_ref", "T89 reference INT8 expansion output"),
    ]:
        values = np.concatenate([c[key].reshape(-1) for c in cases])
        print(f"  {description}: {values.min()} to {values.max()}")

    print("  Accumulator INT32 range checks: pass")


print("\nMLP-expansion workload per sequence:")
print("  MACs:", 32 * 128 * 512)
print("  Output elements:", 32 * 512)

report_mlp_expand_accumulators("Benchmark", mlp_expand_cases[:1])
report_mlp_expand_accumulators("Calibration", mlp_expand_cases[1:])

print("\nFirst benchmark token, first 8 outputs:")
print("  Local accumulators:", mlp_expand_cases[0]["acc_local"][0, :8])
print("  Chained accumulators:", mlp_expand_cases[0]["acc_chained"][0, :8])
print("  T89 reference INT8:", mlp_expand_cases[0]["t89_ref"][0, 0, :8])

Op 58: MLP expansion, 128→512
  T88 (input): scale = 0.03247051313519478
    Zero point: 2
  T29 (weights): (512, 128)
    Original scale count: 1
    Scale range: 0.000803247734438628 to 0.000803247734438628
    Zero points: [0]
  T23 (bias): INT32 (512,)
    Integer range: -3909 to 1823
    Bias scale compatibility: pass
  Accumulator scale range: 2.6081866111904916e-05 to 2.6081866111904916e-05
  T89 (output): scale = 0.011252585798501968
    Zero point: 37
  Conversion multiplier range: 0.0023178553426695166 to 0.0023178553426695166

MLP-expansion workload per sequence:
  MACs: 2097152
  Output elements: 16384

Benchmark
  Local INT32 accumulator, including bias: -67434 to 41610
  Chained INT32 accumulator, including bias: -67376 to 41963
  T89 reference INT8 expansion output: -119 to 127
  Accumulator INT32 range checks: pass

Calibration
  Local INT32 accumulator, including bias: -71271 to 42213
  Chained INT32 accumulator, including bias: -71223 to 42015
  T89 reference INT8 exp

In [29]:
# Part 6A, completion: Op 58 requantization.
# Then inspect the saved GELU graph and tensor formats.

MLP_EXPAND_REQUANT_SHIFT = 39
MLP_EXPAND_ROUNDING_OFFSET = (
    1 << (MLP_EXPAND_REQUANT_SHIFT - 1)
)

mlp_int32_limits = np.iinfo(np.int32)
mlp_int64_max = int(np.iinfo(np.int64).max)

# OFFLINE: encode accumulator-to-T89 scale ratios.
mlp_multiplier_rounded = np.rint(
    mlp_expand_spec["real_multipliers"]
    * (1 << MLP_EXPAND_REQUANT_SHIFT)
)
assert np.all(mlp_multiplier_rounded > 0)
assert mlp_multiplier_rounded.max() <= mlp_int32_limits.max

mlp_expand_multiplier_q = mlp_multiplier_rounded.astype(np.int32)

# Covers every signed INT32 accumulator.
assert (
    (1 << 31) * int(mlp_expand_multiplier_q.max())
    + MLP_EXPAND_ROUNDING_OFFSET
    <= mlp_int64_max
)

mlp_multiplier_approx = (
    mlp_expand_multiplier_q.astype(np.float64)
    * 2.0 ** (-MLP_EXPAND_REQUANT_SHIFT)
)


def mlp_expand_requantize(accumulator, output_shape):
    product = (
        accumulator.astype(np.int64)
        * mlp_expand_multiplier_q.astype(np.int64)[None, :]
    )
    rounded_q = (
        product + MLP_EXPAND_ROUNDING_OFFSET
    ) >> MLP_EXPAND_REQUANT_SHIFT

    before_saturation = (
        rounded_q + mlp_expand_spec["output_zero_point"]
    )
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))

    output_q = np.clip(
        before_saturation, -128, 127
    ).astype(np.int8).reshape(output_shape)

    return output_q, saturation_count


mlp_expand_integer_trial = []

for prepared in mlp_expand_cases:
    reference = prepared["t89_ref"]
    paths = {}

    for path in ["local", "chained"]:
        accumulator = prepared[f"acc_{path}"]

        t89_q, saturation_count = mlp_expand_requantize(
            accumulator, reference.shape
        )

        # DIAGNOSTIC: ideal conversion of our accumulator,
        # using the same output rounding rule.
        ideal_t89_q = np.clip(
            np.floor(
                accumulator.astype(np.float64)
                * mlp_expand_spec["real_multipliers"][None, :]
                + 0.5
            ) + mlp_expand_spec["output_zero_point"],
            -128, 127,
        ).astype(np.int8).reshape(reference.shape)

        paths[path] = {
            "t89_q": t89_q,
            "difference": (
                t89_q.astype(np.int32) - reference.astype(np.int32)
            ),
            "requant_mismatch_count": int(np.count_nonzero(
                t89_q != ideal_t89_q
            )),
            "saturation_count": saturation_count,
        }

    mlp_expand_integer_trial.append({
        "name": prepared["name"],
        "t89_ref": reference.copy(),
        "paths": paths,
    })


def report_mlp_expand_outputs(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    for path in ["local", "chained"]:
        differences = np.concatenate([
            t["paths"][path]["difference"].reshape(-1)
            for t in trials
        ])
        values = np.concatenate([
            t["paths"][path]["t89_q"].reshape(-1)
            for t in trials
        ])
        matched = int(np.count_nonzero(differences == 0))

        print(f"  {path.capitalize()} path:")
        print(
            f"    T89 exact matches: {matched}/{differences.size} "
            f"({100 * matched / differences.size:.3f}%)"
        )
        print("    Maximum INT8-step difference:",
              int(np.abs(differences).max()))
        print("    Mean absolute INT8-step difference:",
              f"{np.abs(differences).mean():.6f}")
        print("    T89 integer range:", values.min(), "to", values.max())

        unique, counts = np.unique(differences, return_counts=True)
        print("    Signed difference counts:", {
            int(d): int(c) for d, c in zip(unique, counts)
        })

        print("    Fixed multiplier vs ideal conversion mismatches:", sum(
            t["paths"][path]["requant_mismatch_count"]
            for t in trials
        ))
        print("    Values requiring saturation:", sum(
            t["paths"][path]["saturation_count"]
            for t in trials
        ))


print("Op 58: accumulator -> T89")
print("  Integer multipliers:", np.unique(mlp_expand_multiplier_q))
print("  Right shift:", MLP_EXPAND_REQUANT_SHIFT)
print(
    "  Maximum multiplier relative error:",
    f"{np.abs(mlp_multiplier_approx / mlp_expand_spec['real_multipliers'] - 1).max():.9e}",
)
print("  T89 tensor scale:", mlp_expand_spec["output_scale"])
print("  T89 zero point:", mlp_expand_spec["output_zero_point"])
print("  Full INT32-input product bound: pass")
print("  Rounding: nearest, ties toward positive infinity")

report_mlp_expand_outputs("Benchmark", mlp_expand_integer_trial[:1])
report_mlp_expand_outputs("Calibration", mlp_expand_integer_trial[1:])

first_expansion = mlp_expand_integer_trial[0]
print("\nFirst benchmark token, first 8 T89 features:")
print("  TFLite reference:", first_expansion["t89_ref"][0, 0, :8])
for path in ["local", "chained"]:
    print(
        f"  {path}:",
        first_expansion["paths"][path]["t89_q"][0, 0, :8],
    )

# ----- Inspection only: GELU Op 59–70 -----

gelu_tensor_meanings = {
    89: "expansion output x",
    90: "dequantized x",
    91: "half-factor constant",
    92: "half-input branch",
    93: "cubed input",
    94: "quantized cubed input",
    95: "cubic-coefficient constant",
    96: "weighted cubic term",
    97: "input plus weighted cubic term",
    98: "tanh-argument coefficient",
    99: "quantized tanh argument",
    100: "dequantized tanh argument",
    101: "tanh output",
    102: "one plus tanh output",
    103: "quantized one-plus-tanh factor",
    104: "final GELU output",
}

print("\nSaved GELU operations:")
for op_index in range(59, 71):
    op = mlp_ops[op_index]
    inputs = ", ".join(f"T{int(v)}" for v in op["inputs"] if int(v) >= 0)
    outputs = ", ".join(f"T{int(v)}" for v in op["outputs"])
    print(f"  Op {op_index} {op['op_name']}: {inputs} -> {outputs}")

print("\nSaved GELU tensor formats:")
for index, meaning in gelu_tensor_meanings.items():
    detail = mlp_details[index]
    params = detail["quantization_parameters"]

    print(
        f"  T{index} ({meaning}): "
        f"{np.dtype(detail['dtype']).name} "
        f"{tuple(int(v) for v in detail['shape'])}"
    )
    if len(params["scales"]) > 0:
        print("    Scales:", params["scales"])
        print("    Zero points:", params["zero_points"])

print("\nSaved GELU constants:")
for index in [3, 4, 91, 95, 98]:
    values = interpreter.get_tensor(index).copy()
    params = mlp_details[index]["quantization_parameters"]

    print(f"  T{index}: shape={values.shape}, first values={values.reshape(-1)[:8]}")

    if len(params["scales"]) == 1:
        represented = (
            values.astype(np.float64) - int(params["zero_points"][0])
        ) * float(params["scales"][0])
        print("    Represented first values:", represented.reshape(-1)[:8])

Op 58: accumulator -> T89
  Integer multipliers: [1274254450]
  Right shift: 39
  Maximum multiplier relative error: 3.012972094e-10
  T89 tensor scale: 0.011252585798501968
  T89 zero point: 37
  Full INT32-input product bound: pass
  Rounding: nearest, ties toward positive infinity

Benchmark
  Local path:
    T89 exact matches: 16384/16384 (100.000%)
    Maximum INT8-step difference: 0
    Mean absolute INT8-step difference: 0.000000
    T89 integer range: -119 to 127
    Signed difference counts: {0: 16384}
    Fixed multiplier vs ideal conversion mismatches: 0
    Values requiring saturation: 1
  Chained path:
    T89 exact matches: 11021/16384 (67.267%)
    Maximum INT8-step difference: 2
    Mean absolute INT8-step difference: 0.331909
    T89 integer range: -119 to 127
    Signed difference counts: {-2: 41, -1: 2478, 0: 11021, 1: 2810, 2: 34}
    Fixed multiplier vs ideal conversion mismatches: 0
    Values requiring saturation: 1

Calibration
  Local path:
    T89 exact matche

In [30]:
# Part 6B: GELU Op 59–63.
# Half-input branch, integer cube, and weighted cubic term.

for op_index, expected in [
    (59, "DEQUANTIZE"),
    (60, "MUL"),
    (61, "POW"),
    (62, "QUANTIZE"),
    (63, "MUL"),
]:
    assert mlp_ops[op_index]["op_name"] == expected

assert tuple(int(v) for v in mlp_ops[60]["inputs"]) == (89, 91)
assert tuple(int(v) for v in mlp_ops[61]["inputs"]) == (90, 3)
assert int(mlp_ops[62]["inputs"][0]) == 93
assert tuple(int(v) for v in mlp_ops[63]["inputs"]) == (94, 95)

assert float(interpreter.get_tensor(3).reshape(-1)[0]) == 3.0

gelu_int32_limits = np.iinfo(np.int32)
gelu_int64_max = int(np.iinfo(np.int64).max)


def read_gelu_quant_spec(index):
    detail = mlp_details[index]
    assert detail["dtype"] == np.int8

    params = detail["quantization_parameters"]
    assert len(params["scales"]) == 1
    assert len(params["zero_points"]) == 1

    scale = float(params["scales"][0])
    assert scale > 0

    return {
        "index": index,
        "scale": scale,
        "zero_point": int(params["zero_points"][0]),
    }


gelu_quant_specs = {
    index: read_gelu_quant_spec(index)
    for index in [89, 91, 92, 94, 95, 96]
}

gelu_half_constant_q = interpreter.get_tensor(91).copy()
gelu_cubic_constant_q = interpreter.get_tensor(95).copy()

assert gelu_half_constant_q.size == 1
assert gelu_cubic_constant_q.size == 1

GELU_HALF_CENTERED_Q = (
    int(gelu_half_constant_q.reshape(-1)[0])
    - gelu_quant_specs[91]["zero_point"]
)
GELU_CUBIC_COEFF_CENTERED_Q = (
    int(gelu_cubic_constant_q.reshape(-1)[0])
    - gelu_quant_specs[95]["zero_point"]
)

GELU_CUBE_SCALE = gelu_quant_specs[89]["scale"] ** 3

# Full INT8 input-domain cube bound.
gelu_input_centered_bound = max(
    abs(-128 - gelu_quant_specs[89]["zero_point"]),
    abs(127 - gelu_quant_specs[89]["zero_point"]),
)
GELU_CUBE_BOUND = gelu_input_centered_bound ** 3
assert GELU_CUBE_BOUND <= gelu_int32_limits.max


def make_gelu_conversion(source_scale, output_index, shift):
    # OFFLINE: encode the scale ratio.
    output_spec = gelu_quant_specs[output_index]
    real_multiplier = source_scale / output_spec["scale"]
    multiplier_q = int(np.rint(real_multiplier * (1 << shift)))
    offset = 1 << (shift - 1)

    assert 0 < multiplier_q <= gelu_int32_limits.max
    assert (1 << 31) * multiplier_q + offset <= gelu_int64_max

    return {
        "output_index": output_index,
        "real_multiplier": real_multiplier,
        "multiplier_q": multiplier_q,
        "shift": shift,
        "offset": offset,
        "output_zero_point": output_spec["zero_point"],
    }


# Each conversion gets a shift that keeps its multiplier in INT32.
gelu_half_conversion = make_gelu_conversion(
    gelu_quant_specs[89]["scale"] * gelu_quant_specs[91]["scale"],
    92,
    38,
)
gelu_cube_conversion = make_gelu_conversion(
    GELU_CUBE_SCALE,
    94,
    45,
)
gelu_weighted_cube_conversion = make_gelu_conversion(
    gelu_quant_specs[94]["scale"] * gelu_quant_specs[95]["scale"],
    96,
    38,
)


def gelu_integer_convert(accumulator, spec):
    assert accumulator.min() >= gelu_int32_limits.min
    assert accumulator.max() <= gelu_int32_limits.max

    rounded = (
        accumulator.astype(np.int64) * spec["multiplier_q"]
        + spec["offset"]
    ) >> spec["shift"]

    before_saturation = rounded + spec["output_zero_point"]
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))
    output_q = np.clip(
        before_saturation, -128, 127
    ).astype(np.int8)

    # Diagnostic only: ideal conversion with the same rounding rule.
    ideal_q = np.clip(
        np.floor(
            accumulator.astype(np.float64) * spec["real_multiplier"]
            + 0.5
        ) + spec["output_zero_point"],
        -128, 127,
    ).astype(np.int8)

    mismatch_count = int(np.count_nonzero(output_q != ideal_q))
    return output_q, saturation_count, mismatch_count


gelu_cubic_integer_trial = []
assert len(op0_cases) == len(mlp_expand_integer_trial)

for case, expansion_trial in zip(op0_cases, mlp_expand_integer_trial):
    assert case["name"] == expansion_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    references = {
        index: interpreter.get_tensor(index).copy()
        for index in [89, 92, 93, 94, 96]
    }

    # Control for Op 63 alone: use exact reference T94.
    control_product = (
        references[94].astype(np.int64)
        - gelu_quant_specs[94]["zero_point"]
    ) * GELU_CUBIC_COEFF_CENTERED_Q

    control_t96_q, _, _ = gelu_integer_convert(
        control_product, gelu_weighted_cube_conversion
    )

    paths = {}

    for path in ["local", "chained"]:
        t89_q = expansion_trial["paths"][path]["t89_q"]

        if path == "local":
            np.testing.assert_array_equal(t89_q, references[89])

        # Op 59 counterpart: centered integers at T89's scale.
        centered_input_q = (
            t89_q.astype(np.int64)
            - gelu_quant_specs[89]["zero_point"]
        )

        # Op 60: input * saved half-factor constant.
        half_product = centered_input_q * GELU_HALF_CENTERED_Q
        t92_q, sat92, mismatch92 = gelu_integer_convert(
            half_product, gelu_half_conversion
        )

        # Op 61: signed integer cube at scale s_T89^3.
        # The full input-domain bound was checked before multiplication.
        cube_wide = (
            centered_input_q * centered_input_q * centered_input_q
        )
        assert cube_wide.min() >= gelu_int32_limits.min
        assert cube_wide.max() <= gelu_int32_limits.max
        t93_q = cube_wide.astype(np.int32)

        # Op 62: cube -> quantized cube.
        t94_q, sat94, mismatch94 = gelu_integer_convert(
            t93_q, gelu_cube_conversion
        )

        # Op 63: quantized cube * saved cubic coefficient.
        weighted_cube_product = (
            t94_q.astype(np.int64)
            - gelu_quant_specs[94]["zero_point"]
        ) * GELU_CUBIC_COEFF_CENTERED_Q

        t96_q, sat96, mismatch96 = gelu_integer_convert(
            weighted_cube_product, gelu_weighted_cube_conversion
        )

        # Floating-point reconstruction: diagnostic only.
        cube_approx = t93_q.astype(np.float64) * GELU_CUBE_SCALE

        paths[path] = {
            "t92_q": t92_q,
            "t93_q": t93_q,
            "t94_q": t94_q,
            "t96_q": t96_q,
            "cube_error": np.abs(cube_approx - references[93]),
            "differences": {
                index: candidate.astype(np.int32)
                - references[index].astype(np.int32)
                for index, candidate in [
                    (92, t92_q), (94, t94_q), (96, t96_q)
                ]
            },
            "saturation_counts": {92: sat92, 94: sat94, 96: sat96},
            "requant_mismatch_counts": {
                92: mismatch92, 94: mismatch94, 96: mismatch96
            },
        }

    gelu_cubic_integer_trial.append({
        "name": case["name"],
        "paths": paths,
        "op63_control_mismatch_count": int(np.count_nonzero(
            control_t96_q != references[96]
        )),
    })


def report_gelu_cubic(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        cube_errors = np.concatenate([
            t["paths"][path]["cube_error"].reshape(-1)
            for t in trials
        ])
        print("    T93 represented cube error vs TFLite:")
        print(f"      Max:  {cube_errors.max():.9e}")
        print(f"      Mean: {cube_errors.mean():.9e}")

        for index, meaning in [
            (92, "half-input branch"),
            (94, "quantized cube"),
            (96, "weighted cubic term"),
        ]:
            differences = np.concatenate([
                t["paths"][path]["differences"][index].reshape(-1)
                for t in trials
            ])
            matched = int(np.count_nonzero(differences == 0))

            print(f"    T{index} ({meaning}):")
            print(
                f"      Exact matches: {matched}/{differences.size} "
                f"({100 * matched / differences.size:.3f}%)"
            )
            print("      Maximum INT8-step difference:",
                  int(np.abs(differences).max()))
            print("      Mean absolute INT8-step difference:",
                  f"{np.abs(differences).mean():.6f}")
            print("      Fixed multiplier vs ideal conversion mismatches:",
                  sum(t["paths"][path]["requant_mismatch_counts"][index]
                      for t in trials))
            print("      Values requiring saturation:",
                  sum(t["paths"][path]["saturation_counts"][index]
                      for t in trials))

    print("  Op 63 reference-T94 control vs T96 mismatches:", sum(
        t["op63_control_mismatch_count"] for t in trials
    ))
    print("  Cube and conversion width checks: pass")


print("T93 candidate cube: INT32")
print("  Scale:", GELU_CUBE_SCALE)
print("  Zero point: 0")
print("  Full input-domain absolute cube bound:", GELU_CUBE_BOUND)

for meaning, spec in [
    ("Op 60 -> T92", gelu_half_conversion),
    ("Op 62 -> T94", gelu_cube_conversion),
    ("Op 63 -> T96", gelu_weighted_cube_conversion),
]:
    represented_multiplier = spec["multiplier_q"] * 2.0 ** (-spec["shift"])
    print(f"\n{meaning}:")
    print("  Integer multiplier:", spec["multiplier_q"])
    print("  Right shift:", spec["shift"])
    print(
        "  Multiplier relative error:",
        f"{abs(represented_multiplier / spec['real_multiplier'] - 1):.9e}",
    )

report_gelu_cubic("Benchmark", gelu_cubic_integer_trial[:1])
report_gelu_cubic("Calibration", gelu_cubic_integer_trial[1:])

T93 candidate cube: INT32
  Scale: 1.4248101460504496e-06
  Zero point: 0
  Full input-domain absolute cube bound: 4492125

Op 60 -> T92:
  Integer multiplier: 1077952640
  Right shift: 38
  Multiplier relative error: 0.000000000e+00

Op 62 -> T94:
  Integer multiplier: 1670962322
  Right shift: 45
  Multiplier relative error: 8.507217153e-11

Op 63 -> T96:
  Integer multiplier: 1500813243
  Right shift: 38
  Multiplier relative error: 2.848115077e-10

Benchmark
  Local path:
    T93 represented cube error vs TFLite:
      Max:  6.164187836e-07
      Mean: 1.852212068e-09
    T92 (half-input branch):
      Exact matches: 16384/16384 (100.000%)
      Maximum INT8-step difference: 0
      Mean absolute INT8-step difference: 0.000000
      Fixed multiplier vs ideal conversion mismatches: 0
      Values requiring saturation: 0
    T94 (quantized cube):
      Exact matches: 16384/16384 (100.000%)
      Maximum INT8-step difference: 0
      Mean absolute INT8-step difference: 0.000000
      

In [31]:
# Part 6C: GELU Op 64–65.
# Add the linear/cubic branches and form the tanh argument.

assert mlp_ops[64]["op_name"] == "ADD"
assert mlp_ops[65]["op_name"] == "MUL"
assert tuple(int(v) for v in mlp_ops[64]["inputs"]) == (89, 96)
assert tuple(int(v) for v in mlp_ops[65]["inputs"]) == (97, 98)
assert int(mlp_ops[64]["outputs"][0]) == 97
assert int(mlp_ops[65]["outputs"][0]) == 99

# Extend metadata without changing earlier candidate arrays.
gelu_quant_specs = {
    **gelu_quant_specs,
    **{
        index: read_gelu_quant_spec(index)
        for index in [97, 98, 99]
    },
}

GELU_ADD_SHIFT = 31
GELU_ADD_ROUNDING_OFFSET = 1 << (GELU_ADD_SHIFT - 1)

gelu_add_real_multipliers = np.asarray([
    gelu_quant_specs[89]["scale"] / gelu_quant_specs[97]["scale"],
    gelu_quant_specs[96]["scale"] / gelu_quant_specs[97]["scale"],
], dtype=np.float64)

gelu_add_multiplier_rounded = np.rint(
    gelu_add_real_multipliers * (1 << GELU_ADD_SHIFT)
)
assert np.all(gelu_add_multiplier_rounded > 0)
assert gelu_add_multiplier_rounded.max() <= gelu_int32_limits.max

gelu_add_multiplier_q = gelu_add_multiplier_rounded.astype(np.int32)

# Bound the combined product over both full INT8 input domains.
gelu_add_bounds = [
    max(abs(-128 - gelu_quant_specs[index]["zero_point"]),
        abs(127 - gelu_quant_specs[index]["zero_point"]))
    for index in [89, 96]
]
assert (
    sum(
        bound * int(multiplier)
        for bound, multiplier in zip(
            gelu_add_bounds, gelu_add_multiplier_q
        )
    ) + GELU_ADD_ROUNDING_OFFSET
    <= gelu_int64_max
)

gelu_argument_constant_q = interpreter.get_tensor(98).copy()
assert gelu_argument_constant_q.size == 1

GELU_ARGUMENT_COEFF_CENTERED_Q = (
    int(gelu_argument_constant_q.reshape(-1)[0])
    - gelu_quant_specs[98]["zero_point"]
)

gelu_argument_conversion = make_gelu_conversion(
    gelu_quant_specs[97]["scale"] * gelu_quant_specs[98]["scale"],
    99,
    38,
)


def gelu_add_branches(t89_q, t96_q):
    assert t89_q.dtype == np.int8
    assert t96_q.dtype == np.int8
    assert t89_q.shape == t96_q.shape

    linear_q = (
        t89_q.astype(np.int64) - gelu_quant_specs[89]["zero_point"]
    )
    cubic_q = (
        t96_q.astype(np.int64) - gelu_quant_specs[96]["zero_point"]
    )

    combined_product = (
        linear_q * int(gelu_add_multiplier_q[0])
        + cubic_q * int(gelu_add_multiplier_q[1])
    )
    rounded = (
        combined_product + GELU_ADD_ROUNDING_OFFSET
    ) >> GELU_ADD_SHIFT

    before_saturation = rounded + gelu_quant_specs[97]["zero_point"]
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))
    t97_q = np.clip(
        before_saturation, -128, 127
    ).astype(np.int8)

    # Diagnostic only: ideal single-rounded conversion.
    ideal_t97_q = np.clip(
        np.floor(
            (
                linear_q.astype(np.float64) * gelu_quant_specs[89]["scale"]
                + cubic_q.astype(np.float64) * gelu_quant_specs[96]["scale"]
            ) / gelu_quant_specs[97]["scale"] + 0.5
        ) + gelu_quant_specs[97]["zero_point"],
        -128, 127,
    ).astype(np.int8)

    mismatch_count = int(np.count_nonzero(t97_q != ideal_t97_q))
    return t97_q, saturation_count, mismatch_count


def gelu_form_argument(t97_q):
    product = (
        t97_q.astype(np.int64) - gelu_quant_specs[97]["zero_point"]
    ) * GELU_ARGUMENT_COEFF_CENTERED_Q

    return gelu_integer_convert(product, gelu_argument_conversion)


gelu_argument_integer_trial = []

assert (
    len(op0_cases)
    == len(mlp_expand_integer_trial)
    == len(gelu_cubic_integer_trial)
)

for case, expansion_trial, cubic_trial in zip(
    op0_cases, mlp_expand_integer_trial, gelu_cubic_integer_trial
):
    assert case["name"] == expansion_trial["name"] == cubic_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    references = {
        index: interpreter.get_tensor(index).copy()
        for index in [89, 96, 97, 99]
    }

    # Controls isolate the conversion rules from upstream differences.
    control_t97_q, _, _ = gelu_add_branches(
        references[89], references[96]
    )
    control_t99_q, _, _ = gelu_form_argument(references[97])

    paths = {}

    for path in ["local", "chained"]:
        t89_q = expansion_trial["paths"][path]["t89_q"]
        t96_q = cubic_trial["paths"][path]["t96_q"]

        t97_q, sat97, mismatch97 = gelu_add_branches(t89_q, t96_q)
        t99_q, sat99, mismatch99 = gelu_form_argument(t97_q)

        # Diagnostic reconstruction of the stored tanh argument.
        argument_approx = (
            t99_q.astype(np.float64) - gelu_quant_specs[99]["zero_point"]
        ) * gelu_quant_specs[99]["scale"]

        paths[path] = {
            "t97_q": t97_q,
            "t99_q": t99_q,
            "argument_approx": argument_approx,
            "differences": {
                97: t97_q.astype(np.int32) - references[97].astype(np.int32),
                99: t99_q.astype(np.int32) - references[99].astype(np.int32),
            },
            "saturation_counts": {97: sat97, 99: sat99},
            "requant_mismatch_counts": {
                97: mismatch97, 99: mismatch99
            },
        }

    gelu_argument_integer_trial.append({
        "name": case["name"],
        "paths": paths,
        "op64_control_mismatch_count": int(np.count_nonzero(
            control_t97_q != references[97]
        )),
        "op65_control_mismatch_count": int(np.count_nonzero(
            control_t99_q != references[99]
        )),
    })


def report_gelu_argument(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        for index, meaning in [
            (97, "linear plus cubic term"),
            (99, "tanh argument"),
        ]:
            differences = np.concatenate([
                t["paths"][path]["differences"][index].reshape(-1)
                for t in trials
            ])
            matched = int(np.count_nonzero(differences == 0))

            print(f"    T{index} ({meaning}):")
            print(
                f"      Exact matches: {matched}/{differences.size} "
                f"({100 * matched / differences.size:.3f}%)"
            )
            print("      Maximum INT8-step difference:",
                  int(np.abs(differences).max()))
            print("      Mean absolute INT8-step difference:",
                  f"{np.abs(differences).mean():.6f}")
            print("      Fixed arithmetic vs ideal conversion mismatches:",
                  sum(t["paths"][path]["requant_mismatch_counts"][index]
                      for t in trials))
            print("      Values requiring saturation:",
                  sum(t["paths"][path]["saturation_counts"][index]
                      for t in trials))

        arguments = np.concatenate([
            t["paths"][path]["argument_approx"].reshape(-1)
            for t in trials
        ])
        print(
            "    Represented tanh-argument range:",
            f"{arguments.min():.9f} to {arguments.max():.9f}",
        )

    print("  Op 64 reference-input control vs T97 mismatches:", sum(
        t["op64_control_mismatch_count"] for t in trials
    ))
    print("  Op 65 reference-input control vs T99 mismatches:", sum(
        t["op65_control_mismatch_count"] for t in trials
    ))


for index, meaning in [(97, "branch sum"), (99, "tanh argument")]:
    print(f"T{index} ({meaning}): INT8")
    print("  Tensor scale:", gelu_quant_specs[index]["scale"])
    print("  Zero point:", gelu_quant_specs[index]["zero_point"])

print("\nOp 64 branch conversion:")
print("  Integer multipliers:", gelu_add_multiplier_q)
print("  Common right shift:", GELU_ADD_SHIFT)
print("  Full input-domain product-sum bound: pass")
print(
    "  Maximum multiplier relative error:",
    f"{np.abs(gelu_add_multiplier_q.astype(np.float64) * 2.0**(-GELU_ADD_SHIFT) / gelu_add_real_multipliers - 1).max():.9e}",
)

print("\nOp 65 argument conversion:")
print("  Integer multiplier:", gelu_argument_conversion["multiplier_q"])
print("  Right shift:", gelu_argument_conversion["shift"])
print("  Rounding: nearest, ties toward positive infinity")

report_gelu_argument("Benchmark", gelu_argument_integer_trial[:1])
report_gelu_argument("Calibration", gelu_argument_integer_trial[1:])

T97 (branch sum): INT8
  Tensor scale: 0.012609509751200676
  Zero point: 37
T99 (tanh argument): INT8
  Tensor scale: 0.010060933418571949
  Zero point: 37

Op 64 branch conversion:
  Integer multipliers: [1916390445  228467824]
  Common right shift: 31
  Full input-domain product-sum bound: pass
  Maximum multiplier relative error: 1.584106402e-09

Op 65 argument conversion:
  Integer multiplier: 1077952560
  Right shift: 38
  Rounding: nearest, ties toward positive infinity

Benchmark
  Local path:
    T97 (linear plus cubic term):
      Exact matches: 16384/16384 (100.000%)
      Maximum INT8-step difference: 0
      Mean absolute INT8-step difference: 0.000000
      Fixed arithmetic vs ideal conversion mismatches: 0
      Values requiring saturation: 0
    T99 (tanh argument):
      Exact matches: 16384/16384 (100.000%)
      Maximum INT8-step difference: 0
      Mean absolute INT8-step difference: 0.000000
      Fixed arithmetic vs ideal conversion mismatches: 0
      Values requ

In [32]:
# Part 6D: GELU Op 66–69.
# Integer tanh lookup, addition of one, and conversion to T103.

for op_index, expected in [
    (66, "DEQUANTIZE"),
    (67, "TANH"),
    (68, "ADD"),
    (69, "QUANTIZE"),
]:
    assert mlp_ops[op_index]["op_name"] == expected

assert int(mlp_ops[66]["inputs"][0]) == 99
assert int(mlp_ops[67]["inputs"][0]) == 100
assert set(int(v) for v in mlp_ops[68]["inputs"]) == {101, 4}
assert int(mlp_ops[69]["inputs"][0]) == 102
assert int(mlp_ops[69]["outputs"][0]) == 103

gelu_one_tensor = interpreter.get_tensor(4).copy()
assert gelu_one_tensor.size == 1
assert float(gelu_one_tensor.reshape(-1)[0]) == 1.0

gelu_quant_specs = {
    **gelu_quant_specs,
    103: read_gelu_quant_spec(103),
}

GELU_TANH_FRAC_BITS = 24
GELU_TANH_SCALE = 2.0 ** (-GELU_TANH_FRAC_BITS)
GELU_ONE_Q = 1 << GELU_TANH_FRAC_BITS

# OFFLINE: one table entry for every possible T99 INT8 code.
gelu_tanh_input_codes = np.arange(-128, 128, dtype=np.int32)
gelu_tanh_lut_arguments = (
    gelu_tanh_input_codes.astype(np.float64)
    - gelu_quant_specs[99]["zero_point"]
) * gelu_quant_specs[99]["scale"]

gelu_tanh_lut_real = np.tanh(gelu_tanh_lut_arguments)
gelu_tanh_lut_q = np.rint(
    gelu_tanh_lut_real * GELU_ONE_Q
).astype(np.int32)

assert gelu_tanh_lut_q.shape == (256,)
assert np.all(np.diff(gelu_tanh_lut_q.astype(np.int64)) >= 0)
assert gelu_tanh_lut_q.min() >= -GELU_ONE_Q
assert gelu_tanh_lut_q.max() <= GELU_ONE_Q

# T99's zero-point code represents argument zero.
gelu_tanh_zero_index = gelu_quant_specs[99]["zero_point"] + 128
assert gelu_tanh_lut_q[gelu_tanh_zero_index] == 0

gelu_tanh_lut_error = np.abs(
    gelu_tanh_lut_q.astype(np.float64) * GELU_TANH_SCALE
    - gelu_tanh_lut_real
)
assert gelu_tanh_lut_error.max() <= GELU_TANH_SCALE / 2 + 1e-12

gelu_factor_conversion = make_gelu_conversion(
    GELU_TANH_SCALE,
    103,
    47,
)


def integer_gelu_tanh_factor(t99_q):
    assert t99_q.dtype == np.int8

    # Op 66–67 counterpart: direct integer lookup.
    table_index = t99_q.astype(np.int16) + 128
    t101_q = gelu_tanh_lut_q[table_index]

    # Op 68: add exact integer one at the same scale.
    t102_wide = t101_q.astype(np.int64) + GELU_ONE_Q
    assert t102_wide.min() >= 0
    assert t102_wide.max() <= gelu_int32_limits.max
    t102_q = t102_wide.astype(np.int32)

    # Op 69: convert the factor to its saved INT8 format.
    t103_q, saturation_count, mismatch_count = gelu_integer_convert(
        t102_q, gelu_factor_conversion
    )

    return t101_q, t102_q, t103_q, saturation_count, mismatch_count


gelu_tanh_integer_trial = []
assert len(op0_cases) == len(gelu_argument_integer_trial)

for case, argument_trial in zip(
    op0_cases, gelu_argument_integer_trial
):
    assert case["name"] == argument_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    references = {
        index: interpreter.get_tensor(index).copy()
        for index in [99, 101, 102, 103]
    }

    # Control: isolate tanh/factor conversion using reference T99.
    _, _, control_t103_q, _, _ = integer_gelu_tanh_factor(
        references[99]
    )

    # Diagnostic control for reference output quantization.
    reference_with_candidate_rounding = np.clip(
        np.floor(
            references[102].astype(np.float64)
            / gelu_quant_specs[103]["scale"] + 0.5
        ) + gelu_quant_specs[103]["zero_point"],
        -128, 127,
    ).astype(np.int8)

    paths = {}

    for path in ["local", "chained"]:
        source = argument_trial["paths"][path]
        t99_q = source["t99_q"]

        t101_q, t102_q, t103_q, saturation_count, mismatch_count = (
            integer_gelu_tanh_factor(t99_q)
        )

        # Floating-point reconstruction: diagnostics only.
        tanh_approx = t101_q.astype(np.float64) * GELU_TANH_SCALE
        factor_approx = t102_q.astype(np.float64) * GELU_TANH_SCALE
        ideal_tanh_from_candidate = np.tanh(source["argument_approx"])

        tanh_own_error = np.abs(
            tanh_approx - ideal_tanh_from_candidate
        )
        assert tanh_own_error.max() <= GELU_TANH_SCALE / 2 + 1e-12

        paths[path] = {
            "t101_q": t101_q,
            "t102_q": t102_q,
            "t103_q": t103_q,
            "tanh_own_error": tanh_own_error,
            "tanh_error": np.abs(tanh_approx - references[101]),
            "factor_error": np.abs(factor_approx - references[102]),
            "t103_difference": (
                t103_q.astype(np.int32)
                - references[103].astype(np.int32)
            ),
            "saturation_count": saturation_count,
            "requant_mismatch_count": mismatch_count,
        }

    gelu_tanh_integer_trial.append({
        "name": case["name"],
        "paths": paths,
        "reference_input_control_mismatch_count": int(np.count_nonzero(
            control_t103_q != references[103]
        )),
        "reference_rounding_mismatch_count": int(np.count_nonzero(
            reference_with_candidate_rounding != references[103]
        )),
    })


def report_gelu_tanh(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        for key, description in [
            ("tanh_own_error", "Tanh table's own approximation error"),
            ("tanh_error", "T101 total tanh error vs TFLite"),
            ("factor_error", "T102 one-plus-tanh error vs TFLite"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1)
                for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        differences = np.concatenate([
            t["paths"][path]["t103_difference"].reshape(-1)
            for t in trials
        ])
        matched = int(np.count_nonzero(differences == 0))

        print(
            f"    T103 exact matches: {matched}/{differences.size} "
            f"({100 * matched / differences.size:.3f}%)"
        )
        print("    Maximum INT8-step difference:",
              int(np.abs(differences).max()))
        print("    Mean absolute INT8-step difference:",
              f"{np.abs(differences).mean():.6f}")
        print("    Fixed multiplier vs ideal conversion mismatches:", sum(
            t["paths"][path]["requant_mismatch_count"] for t in trials
        ))
        print("    Values requiring saturation:", sum(
            t["paths"][path]["saturation_count"] for t in trials
        ))

    print("  Reference-T99 control vs T103 mismatches:", sum(
        t["reference_input_control_mismatch_count"] for t in trials
    ))
    print("  Original T102 with candidate rounding vs T103 mismatches:", sum(
        t["reference_rounding_mismatch_count"] for t in trials
    ))
    print("  Table monotonicity, zero, and width checks: pass")


print("Integer tanh table:")
print("  Entries:", gelu_tanh_lut_q.size)
print("  Bytes:", gelu_tanh_lut_q.nbytes)
print(
    "  Full-domain represented argument range:",
    gelu_tanh_lut_arguments.min(), "to", gelu_tanh_lut_arguments.max(),
)
print("  T101/T102 candidate tensor scale:", GELU_TANH_SCALE)
print("  T101/T102 zero point: 0")
print("  Integer one:", GELU_ONE_Q)
print("  Full-table maximum approximation error:",
      f"{gelu_tanh_lut_error.max():.9e}")

print("\nT103 (INT8 one-plus-tanh factor):")
print("  Tensor scale:", gelu_quant_specs[103]["scale"])
print("  Zero point:", gelu_quant_specs[103]["zero_point"])
print("  Conversion multiplier:", gelu_factor_conversion["multiplier_q"])
print("  Right shift:", gelu_factor_conversion["shift"])
print("  Conversion rounding: nearest, ties toward positive infinity")

report_gelu_tanh("Benchmark", gelu_tanh_integer_trial[:1])
report_gelu_tanh("Calibration", gelu_tanh_integer_trial[1:])

Integer tanh table:
  Entries: 256
  Bytes: 1024
  Full-domain represented argument range: -1.6600540140643716 to 0.9054840076714754
  T101/T102 candidate tensor scale: 5.960464477539063e-08
  T101/T102 zero point: 0
  Integer one: 16777216
  Full-table maximum approximation error: 2.979223762e-08

T103 (INT8 one-plus-tanh factor):
  Tensor scale: 0.0066078659147024155
  Zero point: -128
  Conversion multiplier: 1269488229
  Right shift: 47
  Conversion rounding: nearest, ties toward positive infinity

Benchmark
  Local path:
    Tanh table's own approximation error:
      Max:  2.979223762e-08
      Mean: 1.697999224e-08
    T101 total tanh error vs TFLite:
      Max:  5.960464478e-08
      Mean: 1.825168283e-08
    T102 one-plus-tanh error vs TFLite:
      Max:  1.192092896e-07
      Mean: 1.593798515e-08
    T103 exact matches: 16384/16384 (100.000%)
    Maximum INT8-step difference: 0
    Mean absolute INT8-step difference: 0.000000
    Fixed multiplier vs ideal conversion mismatch

In [33]:
# Part 6E: Op 70, final GELU multiplication.
# T92 (half-input branch) * T103 (one-plus-tanh factor) -> T104.

assert mlp_ops[70]["op_name"] == "MUL"
assert tuple(int(v) for v in mlp_ops[70]["inputs"]) == (92, 103)
assert int(mlp_ops[70]["outputs"][0]) == 104

gelu_quant_specs = {
    **gelu_quant_specs,
    104: read_gelu_quant_spec(104),
}
assert tuple(mlp_details[104]["shape"]) == (1, 32, 512)

GELU_OUTPUT_PRODUCT_SCALE = (
    gelu_quant_specs[92]["scale"] * gelu_quant_specs[103]["scale"]
)

gelu_output_conversion = make_gelu_conversion(
    GELU_OUTPUT_PRODUCT_SCALE,
    104,
    37,
)

# Full INT8 input-domain product bound.
gelu_half_bound = max(
    abs(-128 - gelu_quant_specs[92]["zero_point"]),
    abs(127 - gelu_quant_specs[92]["zero_point"]),
)
gelu_factor_bound = max(
    abs(-128 - gelu_quant_specs[103]["zero_point"]),
    abs(127 - gelu_quant_specs[103]["zero_point"]),
)
GELU_OUTPUT_PRODUCT_BOUND = gelu_half_bound * gelu_factor_bound
assert GELU_OUTPUT_PRODUCT_BOUND <= gelu_int32_limits.max


def integer_gelu_output(t92_q, t103_q):
    assert t92_q.dtype == np.int8
    assert t103_q.dtype == np.int8
    assert t92_q.shape == t103_q.shape

    # Promote before subtracting zero points and multiplying.
    half_centered_q = (
        t92_q.astype(np.int64) - gelu_quant_specs[92]["zero_point"]
    )
    factor_centered_q = (
        t103_q.astype(np.int64) - gelu_quant_specs[103]["zero_point"]
    )

    product_q = half_centered_q * factor_centered_q
    assert product_q.min() >= gelu_int32_limits.min
    assert product_q.max() <= gelu_int32_limits.max

    return gelu_integer_convert(product_q, gelu_output_conversion)


gelu_output_integer_trial = []

assert (
    len(op0_cases)
    == len(gelu_cubic_integer_trial)
    == len(gelu_tanh_integer_trial)
)

for case, cubic_trial, tanh_trial in zip(
    op0_cases, gelu_cubic_integer_trial, gelu_tanh_integer_trial
):
    assert case["name"] == cubic_trial["name"] == tanh_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    t92_ref = interpreter.get_tensor(92).copy()
    t103_ref = interpreter.get_tensor(103).copy()
    t104_ref = interpreter.get_tensor(104).copy()

    # Control isolates Op 70 from all earlier candidate differences.
    control_t104_q, _, _ = integer_gelu_output(t92_ref, t103_ref)

    paths = {}

    for path in ["local", "chained"]:
        t92_q = cubic_trial["paths"][path]["t92_q"]
        t103_q = tanh_trial["paths"][path]["t103_q"]

        t104_q, saturation_count, mismatch_count = integer_gelu_output(
            t92_q, t103_q
        )

        difference = (
            t104_q.astype(np.int32) - t104_ref.astype(np.int32)
        )

        paths[path] = {
            "t104_q": t104_q,
            "difference": difference,
            # Diagnostic error between represented quantized outputs.
            "represented_error": (
                np.abs(difference.astype(np.float64))
                * gelu_quant_specs[104]["scale"]
            ),
            "saturation_count": saturation_count,
            "requant_mismatch_count": mismatch_count,
        }

    gelu_output_integer_trial.append({
        "name": case["name"],
        "t104_ref": t104_ref,
        "paths": paths,
        "reference_input_control_mismatch_count": int(np.count_nonzero(
            control_t104_q != t104_ref
        )),
    })


def report_gelu_output(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    for path in ["local", "chained"]:
        differences = np.concatenate([
            t["paths"][path]["difference"].reshape(-1)
            for t in trials
        ])
        values = np.concatenate([
            t["paths"][path]["t104_q"].reshape(-1)
            for t in trials
        ])
        errors = np.concatenate([
            t["paths"][path]["represented_error"].reshape(-1)
            for t in trials
        ])
        matched = int(np.count_nonzero(differences == 0))

        print(f"  {path.capitalize()} path:")
        print(
            f"    T104 exact matches: {matched}/{differences.size} "
            f"({100 * matched / differences.size:.3f}%)"
        )
        print("    Maximum INT8-step difference:",
              int(np.abs(differences).max()))
        print("    Mean absolute INT8-step difference:",
              f"{np.abs(differences).mean():.6f}")
        print("    T104 integer range:", values.min(), "to", values.max())

        unique, counts = np.unique(differences, return_counts=True)
        print("    Signed difference counts:", {
            int(d): int(c) for d, c in zip(unique, counts)
        })

        print("    Represented error vs quantized TFLite output:")
        print(f"      Max:  {errors.max():.9e}")
        print(f"      Mean: {errors.mean():.9e}")

        print("    Fixed multiplier vs ideal conversion mismatches:", sum(
            t["paths"][path]["requant_mismatch_count"] for t in trials
        ))
        print("    Values requiring saturation:", sum(
            t["paths"][path]["saturation_count"] for t in trials
        ))

    print("  Op 70 reference-input control vs T104 mismatches:", sum(
        t["reference_input_control_mismatch_count"] for t in trials
    ))
    print("  Product and conversion width checks: pass")


print("Op 70: final GELU multiplication")
print("  Product scale:", GELU_OUTPUT_PRODUCT_SCALE)
print("  Full input-domain absolute product bound:",
      GELU_OUTPUT_PRODUCT_BOUND)

print("\nT104 (INT8 GELU output):")
print("  Shape: [1,32,512]")
print("  Tensor scale:", gelu_quant_specs[104]["scale"])
print("  Zero point:", gelu_quant_specs[104]["zero_point"])
print("  Integer conversion multiplier:",
      gelu_output_conversion["multiplier_q"])
print("  Right shift:", gelu_output_conversion["shift"])

represented_multiplier = (
    gelu_output_conversion["multiplier_q"]
    * 2.0 ** (-gelu_output_conversion["shift"])
)
print(
    "  Multiplier relative error:",
    f"{abs(represented_multiplier / gelu_output_conversion['real_multiplier'] - 1):.9e}",
)
print("  Rounding: nearest, ties toward positive infinity")

report_gelu_output("Benchmark", gelu_output_integer_trial[:1])
report_gelu_output("Calibration", gelu_output_integer_trial[1:])

first_gelu_output = gelu_output_integer_trial[0]
print("\nFirst benchmark token, first 8 T104 features:")
print("  TFLite reference:", first_gelu_output["t104_ref"][0, 0, :8])
for path in ["local", "chained"]:
    print(
        f"  {path}:",
        first_gelu_output["paths"][path]["t104_q"][0, 0, :8],
    )

Op 70: final GELU multiplication
  Product scale: 3.717778907509281e-05
  Full input-domain absolute product bound: 42075

T104 (INT8 GELU output):
  Shape: [1,32,512]
  Tensor scale: 0.004273270256817341
  Zero point: -88
  Integer conversion multiplier: 1195729761
  Right shift: 37
  Multiplier relative error: 2.133848653e-12
  Rounding: nearest, ties toward positive infinity

Benchmark
  Local path:
    T104 exact matches: 16377/16384 (99.957%)
    Maximum INT8-step difference: 1
    Mean absolute INT8-step difference: 0.000427
    T104 integer range: -128 to 112
    Signed difference counts: {-1: 7, 0: 16377}
    Represented error vs quantized TFLite output:
      Max:  4.273270257e-03
      Mean: 1.825738025e-06
    Fixed multiplier vs ideal conversion mismatches: 0
    Values requiring saturation: 0
  Chained path:
    T104 exact matches: 11514/16384 (70.276%)
    Maximum INT8-step difference: 4
    Mean absolute INT8-step difference: 0.406616
    T104 integer range: -128 to 112


In [34]:
# Op 70 rounding check.
# Preserve gelu_output_integer_trial as the single-rounding baseline.
# Test Q31 high multiplication followed by rounded power-of-two division.

from math import frexp, floor

gelu_op70_real_multiplier = (
    GELU_OUTPUT_PRODUCT_SCALE / gelu_quant_specs[104]["scale"]
)

# OFFLINE: normalized Q31 multiplier and binary exponent.
gelu_op70_fraction, GELU_OP70_BINARY_EXPONENT = frexp(
    gelu_op70_real_multiplier
)
GELU_OP70_Q31_MULTIPLIER = int(floor(
    gelu_op70_fraction * (1 << 31) + 0.5
))

if GELU_OP70_Q31_MULTIPLIER == (1 << 31):
    GELU_OP70_Q31_MULTIPLIER //= 2
    GELU_OP70_BINARY_EXPONENT += 1

assert 0 < GELU_OP70_Q31_MULTIPLIER <= gelu_int32_limits.max
assert GELU_OP70_BINARY_EXPONENT <= 0

GELU_OP70_SECOND_RIGHT_SHIFT = -GELU_OP70_BINARY_EXPONENT

assert GELU_OP70_Q31_MULTIPLIER == 1195729761
assert GELU_OP70_SECOND_RIGHT_SHIFT == 6

assert (
    GELU_OUTPUT_PRODUCT_BOUND * GELU_OP70_Q31_MULTIPLIER
    + (1 << 30)
    <= gelu_int64_max
)


def integer_gelu_output_two_round(t92_q, t103_q):
    assert t92_q.dtype == np.int8
    assert t103_q.dtype == np.int8
    assert t92_q.shape == t103_q.shape

    half_q = (
        t92_q.astype(np.int64) - gelu_quant_specs[92]["zero_point"]
    )
    factor_q = (
        t103_q.astype(np.int64) - gelu_quant_specs[103]["zero_point"]
    )
    product_q = half_q * factor_q

    assert product_q.min() >= gelu_int32_limits.min
    assert product_q.max() <= gelu_int32_limits.max

    # Stage 1: rounded Q31 high multiplication.
    # With this positive multiplier, this shift expression implements
    # the signed high-multiply rounding rule without an overflow case.
    high_mul_q = (
        product_q * GELU_OP70_Q31_MULTIPLIER + (1 << 30)
    ) >> 31

    assert high_mul_q.min() >= gelu_int32_limits.min
    assert high_mul_q.max() <= gelu_int32_limits.max

    # Stage 2: nearest power-of-two division, ties away from zero.
    shift = GELU_OP70_SECOND_RIGHT_SHIFT

    if shift == 0:
        rounded_q = high_mul_q
    else:
        mask = (1 << shift) - 1
        remainder = high_mul_q & mask
        threshold = (
            (mask >> 1) + (high_mul_q < 0).astype(np.int64)
        )
        rounded_q = (
            (high_mul_q >> shift)
            + (remainder > threshold).astype(np.int64)
        )

    before_saturation = (
        rounded_q + gelu_quant_specs[104]["zero_point"]
    )
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))

    t104_q = np.clip(
        before_saturation, -128, 127
    ).astype(np.int8)

    return t104_q, saturation_count


gelu_output_two_round_trial = []

assert (
    len(op0_cases)
    == len(gelu_cubic_integer_trial)
    == len(gelu_tanh_integer_trial)
    == len(gelu_output_integer_trial)
)

for case, cubic_trial, tanh_trial, single_trial in zip(
    op0_cases,
    gelu_cubic_integer_trial,
    gelu_tanh_integer_trial,
    gelu_output_integer_trial,
):
    assert (
        case["name"] == cubic_trial["name"]
        == tanh_trial["name"] == single_trial["name"]
    )

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    t92_ref = interpreter.get_tensor(92).copy()
    t103_ref = interpreter.get_tensor(103).copy()
    t104_ref = interpreter.get_tensor(104).copy()

    control_q, _ = integer_gelu_output_two_round(t92_ref, t103_ref)

    paths = {}

    for path in ["local", "chained"]:
        t104_q, saturation_count = integer_gelu_output_two_round(
            cubic_trial["paths"][path]["t92_q"],
            tanh_trial["paths"][path]["t103_q"],
        )

        difference = (
            t104_q.astype(np.int32) - t104_ref.astype(np.int32)
        )

        paths[path] = {
            "t104_q": t104_q,
            "difference": difference,
            "represented_error": (
                np.abs(difference.astype(np.float64))
                * gelu_quant_specs[104]["scale"]
            ),
            "saturation_count": saturation_count,
            "changed_vs_single_count": int(np.count_nonzero(
                t104_q != single_trial["paths"][path]["t104_q"]
            )),
        }

    gelu_output_two_round_trial.append({
        "name": case["name"],
        "t92_ref": t92_ref,
        "t103_ref": t103_ref,
        "t104_ref": t104_ref,
        "paths": paths,
        "reference_input_control_mismatch_count": int(np.count_nonzero(
            control_q != t104_ref
        )),
    })


def report_gelu_two_round(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    for path in ["local", "chained"]:
        differences = np.concatenate([
            t["paths"][path]["difference"].reshape(-1)
            for t in trials
        ])
        errors = np.concatenate([
            t["paths"][path]["represented_error"].reshape(-1)
            for t in trials
        ])
        matched = int(np.count_nonzero(differences == 0))

        print(f"  {path.capitalize()} path:")
        print(
            f"    T104 exact matches: {matched}/{differences.size} "
            f"({100 * matched / differences.size:.3f}%)"
        )
        print("    Maximum INT8-step difference:",
              int(np.abs(differences).max()))
        print("    Mean absolute INT8-step difference:",
              f"{np.abs(differences).mean():.6f}")
        print(f"    Maximum represented error: {errors.max():.9e}")
        print(f"    Mean represented error: {errors.mean():.9e}")
        print("    Outputs changed from single-rounding baseline:", sum(
            t["paths"][path]["changed_vs_single_count"] for t in trials
        ))
        print("    Values requiring saturation:", sum(
            t["paths"][path]["saturation_count"] for t in trials
        ))

    print("  Reference-input control mismatches:", sum(
        t["reference_input_control_mismatch_count"] for t in trials
    ))


print("Op 70 two-stage rounding trial:")
print("  Q31 multiplier:", GELU_OP70_Q31_MULTIPLIER)
print("  Binary exponent:", GELU_OP70_BINARY_EXPONENT)
print("  First rounding shift: 31")
print("  Second rounding shift:", GELU_OP70_SECOND_RIGHT_SHIFT)
print("  T104 tensor scale:", gelu_quant_specs[104]["scale"])
print("  T104 zero point:", gelu_quant_specs[104]["zero_point"])

report_gelu_two_round("Benchmark", gelu_output_two_round_trial[:1])
report_gelu_two_round("Calibration", gelu_output_two_round_trial[1:])

# Show how the alternative treats previously mismatched examples.
first_two = gelu_output_two_round_trial[0]
first_single = gelu_output_integer_trial[0]

old_mismatch_positions = np.argwhere(
    first_single["paths"]["local"]["t104_q"]
    != first_two["t104_ref"]
)

print("\nPreviously mismatched benchmark examples:")
for position in old_mismatch_positions[:5]:
    index = tuple(int(v) for v in position)
    print(
        f"  {index}: "
        f"single={int(first_single['paths']['local']['t104_q'][index])}, "
        f"two-stage={int(first_two['paths']['local']['t104_q'][index])}, "
        f"reference={int(first_two['t104_ref'][index])}"
    )

Op 70 two-stage rounding trial:
  Q31 multiplier: 1195729761
  Binary exponent: -6
  First rounding shift: 31
  Second rounding shift: 6
  T104 tensor scale: 0.004273270256817341
  T104 zero point: -88

Benchmark
  Local path:
    T104 exact matches: 16384/16384 (100.000%)
    Maximum INT8-step difference: 0
    Mean absolute INT8-step difference: 0.000000
    Maximum represented error: 0.000000000e+00
    Mean represented error: 0.000000000e+00
    Outputs changed from single-rounding baseline: 7
    Values requiring saturation: 0
  Chained path:
    T104 exact matches: 11519/16384 (70.306%)
    Maximum INT8-step difference: 4
    Mean absolute INT8-step difference: 0.406433
    Maximum represented error: 1.709308103e-02
    Mean represented error: 1.736798501e-03
    Outputs changed from single-rounding baseline: 7
    Values requiring saturation: 0
  Reference-input control mismatches: 0

Calibration
  Local path:
    T104 exact matches: 114688/114688 (100.000%)
    Maximum INT8-ste

### Block-0 MLP expansion and GELU: integer validation, Op 58–70

Source block-0 SHA-256:
`06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9`

Test inputs: one 32-token benchmark sequence and seven calibration
sequences.

The expansion maps 128 input features to 512 features per token.
The saved mixed FP32/INT8 graph remains unchanged; integer
counterparts are calculated separately in Python.

#### Tensor representations

Approximate scales below reflect the rounded metadata display;
the implementation reads full-precision scales from the export.

| Tensor | Meaning | Integer candidate |
|---|---|---|
| T88 | Expansion input | INT8 `[1,32,128]`; scale `0.03247051313519478`, zero point 2 |
| T29 | Expansion weights | INT8 `[512,128]`; scale `0.000803247734438628`, zero point 0 |
| T23 | Expansion bias | INT32 `[512]`; scale `2.6081866111904916e-05`, zero point 0 |
| T89 | Expansion output | INT8 `[1,32,512]`; scale `0.011252585798501968`, zero point 37 |
| T90 | Dequantized input in the saved graph | Candidate retains centered T89 integers at T89's scale |
| T92 | Half-input branch | INT8; scale approximately `0.00562629`, zero point 37 |
| T93 | Cubed input | INT32; scale `1.4248101460504496e-06`, zero point 0 |
| T94 | Quantized cube | INT8; scale approximately `0.03000131`, zero point 85 |
| T96 | Weighted cubic term | INT8; scale approximately `0.00134151`, zero point 85 |
| T97 | Input plus weighted cubic term | INT8; scale `0.012609509751200676`, zero point 37 |
| T99 | Quantized tanh argument | INT8; scale `0.010060933418571949`, zero point 37 |
| T100 | Dequantized argument in the saved graph | Candidate uses T99 directly as a table index |
| T101/T102 | Tanh / one-plus-tanh | INT32; scale `2^-24`, zero point 0 |
| T103 | Quantized one-plus-tanh factor | INT8; scale `0.0066078659147024155`, zero point -128 |
| T104 | Final GELU output | INT8 `[1,32,512]`; scale `0.004273270256817341`, zero point -88 |

GELU tensors T92–T104 contain 16,384 values per sequence,
except scalar constants.

#### Op 58: expansion projection

For token $t$ and output feature $o$:

$$
A[t,o]
=
\sum_{k=0}^{127}(q_{T88}[t,k]-2)q_{T29}[o,k]
+
q_{T23}[o]
$$

Bias scale equals the accumulator scale:

$$
s_A=s_{T88}s_{T29}
=2.6081866111904916\times10^{-5}
$$

Integer requantization:

    q89 = ((accumulator * 1274254450 + (1 << 38)) >> 39) + 37
    q89 = clip(q89, -128, 127)

Python accumulates in INT64, checks the range, and stores the
accumulators as INT32. Requantization products use INT64.

#### Op 59–65: half-input and cubic branches

The implementation uses the saved quantized constants:

| Constant | Meaning | Stored INT8 | Zero point | Represented value |
|---|---|---:|---:|---:|
| T91 | Half factor | 127 | -128 | approximately 0.50000003 |
| T95 | Cubic coefficient | 55 | -128 | approximately 0.04467773 |
| T98 | Tanh-argument coefficient | 127 | -128 | approximately 0.79788457 |

The integer cube is:

$$
q_{T93}=(q_{T89}-37)^3
$$

Its scale is $s_{T89}^3$.
The full input-domain absolute cube bound is 4,492,125, fitting INT32.

T93 is converted to T94, then multiplied by the centered T95
constant and converted to T96.

Op 64 aligns the linear and cubic branches before adding:

$$
q_{T97}\approx
\operatorname{round}\left(
(q_{T89}-37)\frac{s_{89}}{s_{97}}
+
(q_{T96}-85)\frac{s_{96}}{s_{97}}
\right)+37
$$

Both scale ratios use a shared fixed-point precision; their
weighted sum is rounded once.

Op 65 multiplies centered T97 by the saved T98 constant,
then converts the result to T99.

| Conversion | Integer multiplier | Right shift |
|---|---:|---:|
| Op 60 → T92 | 1077952640 | 38 |
| Op 62 → T94 | 1670962322 | 45 |
| Op 63 → T96 | 1500813243 | 38 |
| Op 64 linear branch | 1916390445 | 31 |
| Op 64 cubic branch | 228467824 | 31 |
| Op 65 → T99 | 1077952560 | 38 |

These multipliers encode scale ratios; they are separate from
the destination tensors' scales.

#### Op 66–69: integer tanh and factor conversion

A table covers all 256 possible T99 INT8 codes.

| Table property | Value |
|---|---|
| Entries | 256 INT32 values |
| Storage | 1,024 bytes |
| Represented argument range | -1.660054014 to 0.905484008 |
| Tanh output scale | `2^-24` |
| Maximum full-table approximation error | `2.979223762e-08` |
| Integer representation of one | 16,777,216 |

The table is generated offline. Candidate evaluation uses integer
lookup without interpolation.

Op 68 adds one exactly:

    q102 = q101 + 16777216

Op 69 converts the factor to T103:

    q103 = ((q102 * 1269488229 + (1 << 46)) >> 47) - 128
    q103 = clip(q103, -128, 127)

#### Op 70: final multiplication and reference rounding

The final product is:

$$
P=(q_{T92}-37)(q_{T103}+128)
$$

Its scale is `3.717778907509281e-05`.
The full input-domain absolute product bound is 42,075.

The confirmed candidate uses two rounding stages:

    high = (product * 1195729761 + (1 << 30)) >> 31

    remainder = high & 63
    threshold = 31 + int(high < 0)
    scaled = (high >> 6) + int(remainder > threshold)

    q104 = clip(scaled - 88, -128, 127)

The second stage performs nearest division by 64, with ties
away from zero.

Although the effective multiplier ratio is approximately
`1195729761 * 2^-37`, combining both stages into one rounding
operation does not reproduce every reference output.

The original single-rounding trial differed at 7 benchmark and
27 calibration values, each by one INT8 step.
The two-stage trial removed all 34 local differences.

The single-rounding trial remains preserved for comparison.
The selected result for subsequent stages is:

    gelu_output_two_round_trial[i]["paths"][path]["t104_q"]

#### Validation measurements

- **Local:** Op 58 starts from reference T88; its integer output
  then continues through the integer GELU stages.
- **Chained:** Op 58 starts from the chained integer LayerNorm T88.

All tested local quantized outputs T89, T92, T94, T96, T97,
T99, T103, and final two-stage T104 matched their references.

| Chained intermediate agreement | Benchmark | Calibration |
|---|---:|---:|
| T89 expansion output | 67.267% | 66.073% |
| T94/T96 cubic branch | 97.607% | 97.675% |
| T97/T99 tanh-argument branch | 70.483% | 69.440% |
| T103 one-plus-tanh factor | 70.557% | 69.503% |

| Final T104 measurement, two-stage rounding | Benchmark | Calibration |
|---|---:|---:|
| Local exact matches | 16384/16384 (100%) | 114688/114688 (100%) |
| Local maximum INT8-step difference | 0 | 0 |
| Chained exact matches | 11519/16384 (70.306%) | 79110/114688 (68.978%) |
| Chained maximum INT8-step difference | 4 | 6 |
| Chained mean absolute INT8-step difference | 0.406433 | 0.423322 |
| Chained maximum represented error | 0.017093 | 0.025640 |
| Chained mean represented error | 0.001737 | 0.001809 |
| Op 70 reference-input control mismatches | 0 | 0 |
| Final T104 values requiring saturation | 0 | 0 |

Represented errors compare against the quantized TFLite output,
not an ideal floating-point GELU function.

Local tanh error versus TFLite was at most `5.960464478e-08`.
These small floating-point differences did not change tested T103
INT8 outputs.

#### Analysis and status

The expansion and GELU now reproduce every tested local T104 value.

The initial Op 70 discrepancies were caused by the rounding
sequence. Using reference inputs reproduced the discrepancies;
changing to two-stage rounding removed them.

Chained differences originate upstream and propagate through the
GELU calculation. Cubing can amplify differences at larger input
magnitudes. Intermediate match percentages also depend on tensor
scales and should not be interpreted as model accuracy.

All implemented bias-scale, accumulator, cube, product,
conversion-width, table-monotonicity, table-zero, and
reference-input control checks passed.

For one 32-token sequence:

- Expansion MACs: **2,097,152**.
- GELU input/output elements: **16,384**.
- Integer tanh table: **1,024 bytes**.

Op 58–70 is validated in Python on the tested cases.
C, Spike, RTL, complete-block accuracy, and performance have not
yet been validated for this implementation.

Next: **Op 71–73, MLP output projection from 512 to 128 features
and final residual addition**.

In [35]:
# Part 7, preparation: Op 71–73.
# Inspect the 512→128 projection and calculate accumulators.

final_ops = interpreter._get_ops_details()
final_details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}

assert final_ops[71]["op_name"] == "FULLY_CONNECTED"
assert final_ops[72]["op_name"] == "DEQUANTIZE"
assert final_ops[73]["op_name"] == "ADD"

op71 = final_ops[71]
mlp_out_input_index = int(op71["inputs"][0])
mlp_out_weight_index = int(op71["inputs"][1])
mlp_out_bias_index = int(op71["inputs"][2])
t105_index = int(op71["outputs"][0])
t106_index = int(final_ops[72]["outputs"][0])
t107_index = int(final_ops[73]["outputs"][0])

assert (
    mlp_out_input_index, mlp_out_weight_index, mlp_out_bias_index
) == (104, 30, 22)
assert (t105_index, t106_index, t107_index) == (105, 106, 107)
assert int(final_ops[72]["inputs"][0]) == t105_index
assert set(int(v) for v in final_ops[73]["inputs"]) == {t106_index, t73_index}

input_detail = final_details[mlp_out_input_index]
weight_detail = final_details[mlp_out_weight_index]
bias_detail = final_details[mlp_out_bias_index]
output_detail = final_details[t105_index]

assert input_detail["dtype"] == np.int8
assert weight_detail["dtype"] == np.int8
assert bias_detail["dtype"] == np.int32
assert output_detail["dtype"] == np.int8
assert tuple(input_detail["shape"]) == (1, 32, 512)
assert tuple(output_detail["shape"]) == (1, 32, 128)

input_params = input_detail["quantization_parameters"]
weight_params = weight_detail["quantization_parameters"]
bias_params = bias_detail["quantization_parameters"]
output_params = output_detail["quantization_parameters"]

assert len(input_params["scales"]) == 1
assert len(input_params["zero_points"]) == 1
assert len(output_params["scales"]) == 1
assert len(output_params["zero_points"]) == 1

mlp_out_input_scale = float(input_params["scales"][0])
mlp_out_input_zero_point = int(input_params["zero_points"][0])
mlp_out_output_scale = float(output_params["scales"][0])
mlp_out_output_zero_point = int(output_params["zero_points"][0])

assert mlp_out_input_scale == gelu_quant_specs[104]["scale"]
assert mlp_out_input_zero_point == gelu_quant_specs[104]["zero_point"]
assert mlp_out_output_scale > 0

mlp_out_weights_q = interpreter.get_tensor(mlp_out_weight_index).copy()
mlp_out_bias_q = interpreter.get_tensor(mlp_out_bias_index).copy().reshape(-1)

assert mlp_out_weights_q.shape == (128, 512)
assert mlp_out_bias_q.shape == (128,)

weight_scales_raw = np.asarray(
    weight_params["scales"], dtype=np.float64
)
weight_zero_points_raw = np.asarray(
    weight_params["zero_points"], dtype=np.int64
)

assert weight_scales_raw.size in (1, 128)
assert weight_zero_points_raw.size in (1, 128)
assert np.all(weight_scales_raw > 0)
assert np.all(
    (weight_zero_points_raw >= -128) & (weight_zero_points_raw <= 127)
)

if weight_scales_raw.size > 1:
    assert int(weight_params["quantized_dimension"]) == 0

mlp_out_weight_scales = np.broadcast_to(
    weight_scales_raw, (128,)
).copy()
mlp_out_weight_zero_points = np.broadcast_to(
    weight_zero_points_raw, (128,)
).copy()

assert len(bias_params["scales"]) in (1, 128)
assert np.all(bias_params["zero_points"] == 0)

np.testing.assert_allclose(
    np.broadcast_to(bias_params["scales"], (128,)),
    mlp_out_input_scale * mlp_out_weight_scales,
    rtol=1e-5,
    atol=0,
)

mlp_output_spec = {
    "input_scale": mlp_out_input_scale,
    "input_zero_point": mlp_out_input_zero_point,
    "weight_scales": mlp_out_weight_scales,
    "weight_zero_points": mlp_out_weight_zero_points,
    "weights_q": mlp_out_weights_q,
    "bias_q": mlp_out_bias_q,
    "output_scale": mlp_out_output_scale,
    "output_zero_point": mlp_out_output_zero_point,
    "real_multipliers": (
        mlp_out_input_scale * mlp_out_weight_scales
        / mlp_out_output_scale
    ),
}

final_int32_limits = np.iinfo(np.int32)

# Conservative bound over full INT8 input/weight ranges,
# including the saved bias for each output channel.
final_input_bound = max(
    abs(-128 - mlp_out_input_zero_point),
    abs(127 - mlp_out_input_zero_point),
)
final_weight_bounds = np.maximum(
    np.abs(-128 - mlp_out_weight_zero_points),
    np.abs(127 - mlp_out_weight_zero_points),
)
final_accumulator_bounds = (
    512 * final_input_bound * final_weight_bounds
    + np.abs(mlp_out_bias_q.astype(np.int64))
)
assert final_accumulator_bounds.max() <= final_int32_limits.max


def mlp_output_accumulate(input_q):
    assert input_q.dtype == np.int8
    assert input_q.shape == (1, 32, 512)

    x = input_q.reshape(-1, 512).astype(np.int64)
    x = x - mlp_output_spec["input_zero_point"]

    w = mlp_output_spec["weights_q"].astype(np.int64)
    w = w - mlp_output_spec["weight_zero_points"][:, None]

    accumulator = (
        x @ w.T + mlp_output_spec["bias_q"].astype(np.int64)[None, :]
    )

    assert accumulator.shape == (32, 128)
    assert accumulator.min() >= final_int32_limits.min
    assert accumulator.max() <= final_int32_limits.max

    return accumulator.astype(np.int32)


mlp_output_cases = []

assert (
    len(op0_cases)
    == len(gelu_output_two_round_trial)
    == len(out_residual_integer_trial)
)

for case, gelu_trial, residual_trial in zip(
    op0_cases,
    gelu_output_two_round_trial,
    out_residual_integer_trial,
):
    assert case["name"] == gelu_trial["name"] == residual_trial["name"]

    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    input_ref = interpreter.get_tensor(mlp_out_input_index).copy()
    input_chained = gelu_trial["paths"]["chained"]["t104_q"]

    t73_ref = interpreter.get_tensor(t73_index).copy()
    t106_ref = interpreter.get_tensor(t106_index).copy()
    t107_ref = interpreter.get_tensor(t107_index).copy()

    assert t73_ref.shape == (1, 32, 128)
    assert t107_ref.shape == (1, 32, 128)

    # Verify the reference final residual relationship.
    np.testing.assert_allclose(
        t107_ref, t73_ref + t106_ref,
        rtol=0,
        atol=1e-7,
    )

    mlp_output_cases.append({
        "name": case["name"],
        "acc_local": mlp_output_accumulate(input_ref),
        "acc_chained": mlp_output_accumulate(input_chained),
        "t105_ref": interpreter.get_tensor(t105_index).copy(),
        "t106_ref": t106_ref,
        "t107_ref": t107_ref,
        "t73_ref": t73_ref,
        "t73_chained_q": (
            residual_trial["paths"]["chained"]["t73_q"].copy()
        ),
    })


def report_final_projection_inputs(label, cases):
    if not cases:
        return

    print(f"\n{label}")

    for key, description in [
        ("acc_local", "Local INT32 accumulator, including bias"),
        ("acc_chained", "Chained INT32 accumulator, including bias"),
        ("t105_ref", "T105 reference INT8 projection output"),
        ("t106_ref", "T106 reference FP32 projection output"),
        ("t107_ref", "T107 reference FP32 block output"),
    ]:
        values = np.concatenate([c[key].reshape(-1) for c in cases])
        print(f"  {description}: {values.min()} to {values.max()}")

    print("  Accumulator INT32 checks: pass")
    print("  Reference final-residual relationship: pass")


print("Op 71: MLP output projection, 512→128")
print("  T104 input scale:", mlp_out_input_scale)
print("  T104 zero point:", mlp_out_input_zero_point)
print("  T30 weight shape:", mlp_out_weights_q.shape)
print("  Original weight scale count:", weight_scales_raw.size)
print("  Weight scale range:",
      mlp_out_weight_scales.min(), "to", mlp_out_weight_scales.max())
print("  Weight zero points:", np.unique(mlp_out_weight_zero_points))
print("  T22 bias integer range:",
      mlp_out_bias_q.min(), "to", mlp_out_bias_q.max())
print("  Bias scale compatibility: pass")
print("  Accumulator scale range:",
      (mlp_out_input_scale * mlp_out_weight_scales).min(), "to",
      (mlp_out_input_scale * mlp_out_weight_scales).max())
print("  Full input/weight-domain accumulator bound:",
      int(final_accumulator_bounds.max()))
print("  T105 output scale:", mlp_out_output_scale)
print("  T105 zero point:", mlp_out_output_zero_point)
print("  Conversion multiplier range:",
      mlp_output_spec["real_multipliers"].min(), "to",
      mlp_output_spec["real_multipliers"].max())

print("\nFinal residual:")
print("  Chained T73: INT32, scale =", T73_SCALE, ", zero point = 0")
print("  Saved T106/T107: FP32")
print("  Candidate final output format: next cell")

print("\nWorkload per sequence:")
print("  Projection MACs:", 32 * 512 * 128)
print("  Projection outputs:", 32 * 128)
print("  Final residual additions:", 32 * 128)

report_final_projection_inputs("Benchmark", mlp_output_cases[:1])
report_final_projection_inputs("Calibration", mlp_output_cases[1:])

Op 71: MLP output projection, 512→128
  T104 input scale: 0.004273270256817341
  T104 zero point: -88
  T30 weight shape: (128, 512)
  Original weight scale count: 1
  Weight scale range: 0.0007148781442083418 to 0.0007148781442083418
  Weight zero points: [0]
  T22 bias integer range: -16701 to 16823
  Bias scale compatibility: pass
  Accumulator scale range: 3.054867510894285e-06 to 3.054867510894285e-06
  Full input/weight-domain accumulator bound: 14107063
  T105 output scale: 0.0026197750121355057
  T105 zero point: 10
  Conversion multiplier range: 0.0011660801010557446 to 0.0011660801010557446

Final residual:
  Chained T73: INT32, scale = 1.52587890625e-05 , zero point = 0
  Saved T106/T107: FP32
  Candidate final output format: next cell

Workload per sequence:
  Projection MACs: 2097152
  Projection outputs: 4096
  Final residual additions: 4096

Benchmark
  Local INT32 accumulator, including bias: -95953 to 114507
  Chained INT32 accumulator, including bias: -96833 to 114088

In [36]:
# Part 7, completion: Op 71–73.
# Final projection requantization and block-output residual addition.

final_int64_max = int(np.iinfo(np.int64).max)

# ----- Op 71: accumulator -> T105 -----

MLP_OUT_REQUANT_SHIFT = 39
MLP_OUT_ROUNDING_OFFSET = 1 << (MLP_OUT_REQUANT_SHIFT - 1)

mlp_out_multiplier_rounded = np.rint(
    mlp_output_spec["real_multipliers"]
    * (1 << MLP_OUT_REQUANT_SHIFT)
)
assert np.all(mlp_out_multiplier_rounded > 0)
assert mlp_out_multiplier_rounded.max() <= final_int32_limits.max

mlp_out_multiplier_q = mlp_out_multiplier_rounded.astype(np.int32)

assert (
    (1 << 31) * int(mlp_out_multiplier_q.max())
    + MLP_OUT_ROUNDING_OFFSET
    <= final_int64_max
)

# ----- Op 72–73: final residual format -----

T106_FRAC_BITS = 16
T106_SCALE = 2.0 ** (-T106_FRAC_BITS)
T107_FRAC_BITS = T106_FRAC_BITS
T107_SCALE = T106_SCALE
T107_ZERO_POINT = 0

assert T107_SCALE == T73_SCALE
assert T107_FRAC_BITS == T73_FRAC_BITS

# OFFLINE: encode centered T105 -> T106 scale ratio.
T106_REAL_MULTIPLIER = mlp_output_spec["output_scale"] / T106_SCALE
T106_REQUANT_SHIFT = 22
T106_ROUNDING_OFFSET = 1 << (T106_REQUANT_SHIFT - 1)

T106_MULTIPLIER_Q = int(np.rint(
    T106_REAL_MULTIPLIER * (1 << T106_REQUANT_SHIFT)
))
assert 0 < T106_MULTIPLIER_Q <= final_int32_limits.max

final_projection_centered_bound = max(
    abs(-128 - mlp_output_spec["output_zero_point"]),
    abs(127 - mlp_output_spec["output_zero_point"]),
)
assert (
    final_projection_centered_bound * T106_MULTIPLIER_Q
    + T106_ROUNDING_OFFSET
    <= final_int64_max
)


def final_projection_requantize(accumulator, output_shape):
    product = (
        accumulator.astype(np.int64)
        * mlp_out_multiplier_q.astype(np.int64)[None, :]
    )
    rounded = (
        product + MLP_OUT_ROUNDING_OFFSET
    ) >> MLP_OUT_REQUANT_SHIFT

    before_saturation = rounded + mlp_output_spec["output_zero_point"]
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))

    t105_q = np.clip(
        before_saturation, -128, 127
    ).astype(np.int8).reshape(output_shape)

    return t105_q, saturation_count


def final_integer_residual(t105_q, t73_q):
    assert t105_q.dtype == np.int8
    assert t73_q.dtype == np.int32
    assert t105_q.shape == t73_q.shape

    centered_projection = (
        t105_q.astype(np.int64) - mlp_output_spec["output_zero_point"]
    )

    # Op 72 counterpart: convert to the residual's integer scale.
    t106_wide = (
        centered_projection * T106_MULTIPLIER_Q
        + T106_ROUNDING_OFFSET
    ) >> T106_REQUANT_SHIFT

    assert t106_wide.min() >= final_int32_limits.min
    assert t106_wide.max() <= final_int32_limits.max
    t106_q = t106_wide.astype(np.int32)

    # Op 73: both branches already have scale 2^-16.
    t107_wide = t106_q.astype(np.int64) + t73_q.astype(np.int64)

    assert t107_wide.min() >= final_int32_limits.min
    assert t107_wide.max() <= final_int32_limits.max
    t107_q = t107_wide.astype(np.int32)

    return t106_q, t107_q


block0_integer_trial = []

for prepared in mlp_output_cases:
    # Local residual input: encode reference T73 at scale 2^-16.
    local_residual_rounded = np.rint(
        prepared["t73_ref"].astype(np.float64) / T73_SCALE
    )
    assert local_residual_rounded.min() >= final_int32_limits.min
    assert local_residual_rounded.max() <= final_int32_limits.max
    local_residual_q = local_residual_rounded.astype(np.int32)

    paths = {}

    for path in ["local", "chained"]:
        accumulator = prepared[f"acc_{path}"]
        residual_q = (
            local_residual_q if path == "local"
            else prepared["t73_chained_q"]
        )

        t105_q, saturation_count = final_projection_requantize(
            accumulator, prepared["t105_ref"].shape
        )
        t106_q, t107_q = final_integer_residual(t105_q, residual_q)

        # Floating-point calculations below are diagnostics only.
        ideal_t105_q = np.clip(
            np.floor(
                accumulator.astype(np.float64)
                * mlp_output_spec["real_multipliers"][None, :]
                + 0.5
            ) + mlp_output_spec["output_zero_point"],
            -128, 127,
        ).astype(np.int8).reshape(t105_q.shape)

        ideal_projection = (
            t105_q.astype(np.float64)
            - mlp_output_spec["output_zero_point"]
        ) * mlp_output_spec["output_scale"]

        residual_approx = residual_q.astype(np.float64) * T73_SCALE
        t106_approx = t106_q.astype(np.float64) * T106_SCALE
        t107_approx = t107_q.astype(np.float64) * T107_SCALE

        ideal_sum_from_candidate = ideal_projection + residual_approx
        residual_own_error = np.abs(
            t107_approx - ideal_sum_from_candidate
        )
        assert residual_own_error.max() <= T107_SCALE / 2 + 1e-12

        paths[path] = {
            "t105_q": t105_q,
            "t106_q": t106_q,
            "t107_q": t107_q,
            "t107_approx": t107_approx,
            "t105_difference": (
                t105_q.astype(np.int32)
                - prepared["t105_ref"].astype(np.int32)
            ),
            "t73_input_error": np.abs(
                residual_approx - prepared["t73_ref"]
            ),
            "t106_error": np.abs(
                t106_approx - prepared["t106_ref"]
            ),
            "t107_error": np.abs(
                t107_approx - prepared["t107_ref"]
            ),
            "residual_own_error": residual_own_error,
            "requant_mismatch_count": int(np.count_nonzero(
                t105_q != ideal_t105_q
            )),
            "saturation_count": saturation_count,
        }

    block0_integer_trial.append({
        "name": prepared["name"],
        "t107_ref": prepared["t107_ref"].copy(),
        "paths": paths,
    })


def report_block0_output(label, trials):
    if not trials:
        return

    print(f"\n{label}")

    reference = np.concatenate([
        t["t107_ref"].astype(np.float64).reshape(-1)
        for t in trials
    ])
    reference_energy = float(np.dot(reference, reference))
    assert reference_energy > 0

    print(
        "  T107 reference block-output range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )

    for path in ["local", "chained"]:
        print(f"  {path.capitalize()} path:")

        projection_differences = np.concatenate([
            t["paths"][path]["t105_difference"].reshape(-1)
            for t in trials
        ])
        matched = int(np.count_nonzero(projection_differences == 0))

        print(
            f"    T105 exact matches: {matched}/{projection_differences.size} "
            f"({100 * matched / projection_differences.size:.3f}%)"
        )
        print("    T105 maximum INT8-step difference:",
              int(np.abs(projection_differences).max()))
        print("    T105 mean absolute INT8-step difference:",
              f"{np.abs(projection_differences).mean():.6f}")
        print("    Fixed multiplier vs ideal conversion mismatches:", sum(
            t["paths"][path]["requant_mismatch_count"] for t in trials
        ))
        print("    Projection values requiring saturation:", sum(
            t["paths"][path]["saturation_count"] for t in trials
        ))

        for key, description in [
            ("t73_input_error", "T73 residual-input error vs TFLite"),
            ("t106_error", "T106 projection error vs TFLite"),
            ("residual_own_error", "Final conversion/addition own error"),
            ("t107_error", "T107 final block-output error vs TFLite"),
        ]:
            errors = np.concatenate([
                t["paths"][path][key].reshape(-1) for t in trials
            ])
            print(f"    {description}:")
            print(f"      Max:  {errors.max():.9e}")
            print(f"      Mean: {errors.mean():.9e}")

        output = np.concatenate([
            t["paths"][path]["t107_approx"].reshape(-1)
            for t in trials
        ])
        signed_error = output - reference
        error_energy = float(np.dot(signed_error, signed_error))
        rmse = np.sqrt(error_energy / reference.size)
        relative_l2 = np.sqrt(error_energy / reference_energy)

        print(f"    T107 RMSE: {rmse:.9e}")
        print(f"    T107 relative L2 error: {100 * relative_l2:.6f}%")

        output_q = np.concatenate([
            t["paths"][path]["t107_q"].reshape(-1) for t in trials
        ])
        print("    T107 INT32 range:", output_q.min(), "to", output_q.max())

    print("  Final residual addition uses a shared scale")
    print("  T106/T107 INT32 checks: pass")


print("Op 71: accumulator -> T105")
print("  Integer multipliers:", np.unique(mlp_out_multiplier_q))
print("  Right shift:", MLP_OUT_REQUANT_SHIFT)
print(
    "  Maximum multiplier relative error:",
    f"{np.abs(mlp_out_multiplier_q.astype(np.float64) * 2.0**(-MLP_OUT_REQUANT_SHIFT) / mlp_output_spec['real_multipliers'] - 1).max():.9e}",
)

print("\nOp 72–73: final block-output format")
print("  T106/T107: INT32 [1,32,128]")
print("  Tensor scale:", T107_SCALE)
print("  Zero point:", T107_ZERO_POINT)
print("  T105->T106 integer multiplier:", T106_MULTIPLIER_Q)
print("  T105->T106 right shift:", T106_REQUANT_SHIFT)
print("  Requantization rounding: nearest, ties toward positive infinity")

report_block0_output("Benchmark", block0_integer_trial[:1])
report_block0_output("Calibration", block0_integer_trial[1:])

first_block_output = block0_integer_trial[0]
print("\nFirst benchmark token, first 8 T107 features:")
print("  TFLite reference:", first_block_output["t107_ref"][0, 0, :8])
for path in ["local", "chained"]:
    print(
        f"  {path} represented:",
        first_block_output["paths"][path]["t107_approx"][0, 0, :8],
    )

Op 71: accumulator -> T105
  Integer multipliers: [641059315]
  Right shift: 39
  Maximum multiplier relative error: 2.262223742e-11

Op 72–73: final block-output format
  T106/T107: INT32 [1,32,128]
  Tensor scale: 1.52587890625e-05
  Zero point: 0
  T105->T106 integer multiplier: 720118272
  T105->T106 right shift: 22
  Requantization rounding: nearest, ties toward positive infinity

Benchmark
  T107 reference block-output range: -0.653274477 to 0.533836246
  Local path:
    T105 exact matches: 4096/4096 (100.000%)
    T105 maximum INT8-step difference: 0
    T105 mean absolute INT8-step difference: 0.000000
    Fixed multiplier vs ideal conversion mismatches: 0
    Projection values requiring saturation: 2
    T73 residual-input error vs TFLite:
      Max:  7.629394531e-06
      Mean: 3.815025650e-06
    T106 projection error vs TFLite:
      Max:  7.480382919e-06
      Mean: 3.865807685e-06
    Final conversion/addition own error:
      Max:  7.482245564e-06
      Mean: 3.865613053

  T107 reference block-output range: -0.653274477 to 0.605844736
  Local path:
    T105 exact matches: 28672/28672 (100.000%)
    T105 maximum INT8-step difference: 0
    T105 mean absolute INT8-step difference: 0.000000
    Fixed multiplier vs ideal conversion mismatches: 0
    Projection values requiring saturation: 2
    T73 residual-input error vs TFLite:
      Max:  7.629394531e-06
      Mean: 3.800485266e-06
    T106 projection error vs TFLite:
      Max:  7.510185242e-06
      Mean: 3.813139691e-06
    Final conversion/addition own error:
      Max:  7.517635822e-06
      Mean: 3.812946423e-06
    T107 final block-output error vs TFLite:
      Max:  1.498311758e-05
      Mean: 5.080643145e-06
    T107 RMSE: 6.219611354e-06
    T107 relative L2 error: 0.005238%
    T107 INT32 range: -42813 to 39704
  Chained path:
    T105 exact matches: 13952/28672 (48.661%)
    T105 maximum INT8-step difference: 4
    T105 mean absolute INT8-step difference: 0.568394
    Fixed multiplier vs ide

In [37]:
block0_integer_trial[i]["paths"]["chained"]["t107_q"]

array([[[ 10719,     65,  -4062, ..., -13061,   -658,  -8179],
        [  3079,   -429,  -5168, ..., -10911,  -2733, -11390],
        [ 13758,  -1825,  -9903, ...,  -8736,   8578,  -5180],
        ...,
        [ 13029,   1109,  -7470, ...,  -9381,   9729,  -8634],
        [  9399,    -84,  -6492, ...,  -5510,   1276,  -6332],
        [ 10722,  -3661,  -4886, ..., -13019,  -2581,  -7186]]],
      shape=(1, 32, 128), dtype=int32)

### Block-0 MLP output projection and final residual: Op 71–73

Source block-0 SHA-256:
`06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9`

Test inputs: one 32-token benchmark sequence and seven calibration
sequences.

The saved mixed FP32/INT8 graph remains unchanged.
Integer counterparts are calculated separately in Python.

#### Tensor representations

| Tensor | Meaning | Integer candidate |
|---|---|---|
| T104 | GELU output | INT8 `[1,32,512]`; scale `0.004273270256817341`, zero point -88 |
| T30 | Final projection weights | INT8 `[128,512]`; scale `0.0007148781442083418`, zero point 0 |
| T22 | Final projection bias | INT32 `[128]`; scale `3.054867510894285e-06`, zero point 0 |
| Accumulator | Projection dot products plus bias | INT32 `[32,128]`; scale `3.054867510894285e-06`, zero point 0 |
| T105 | Quantized projection output | INT8 `[1,32,128]`; scale `0.0026197750121355057`, zero point 10 |
| T106 | Projection aligned for residual addition | INT32 `[1,32,128]`; scale `2^-16`, zero point 0 |
| T73 | Residual branch | INT32 `[1,32,128]`; scale `2^-16`, zero point 0 |
| T107 | Final block-0 output | INT32 `[1,32,128]`; scale `2^-16`, zero point 0 |

#### Op 71: final projection

For token $t$ and output feature $o$:

$$
A[t,o]
=
\sum_{k=0}^{511}(q_{T104}[t,k]+88)q_{T30}[o,k]
+
q_{T22}[o]
$$

Bias scale equals the accumulator scale:

$$
s_A=s_{T104}s_{T30}
=3.054867510894285\times10^{-6}
$$

The full input/weight-domain absolute accumulator bound,
including the saved bias, is 14,107,063 and fits INT32.

Python accumulates in INT64, checks the range, and stores INT32.

Accumulator-to-T105 conversion:

    q105 = ((accumulator * 641059315 + (1 << 38)) >> 39) + 10
    q105 = clip(q105, -128, 127)

The multiplier's relative error is `2.262223742e-11`.
Its product uses INT64 arithmetic.

#### Op 72–73: final residual addition

Centered T105 is converted to the residual scale `2^-16`:

    q106 = (((q105 - 10) * 720118272 + (1 << 21)) >> 22)

T73 already uses that scale, so the final addition is direct:

    q107 = q106 + q73

T106 and T107 use INT32 storage at scale `2^-16`, zero point 0.
No additional INT8 quantization is introduced at the block output.

Requantization uses nearest rounding, with ties toward positive
infinity. Wide temporaries are checked before INT32 storage.

#### Validation measurements

- **Local:** starts from reference T104 and reference T73.
  Reference T73 is encoded at scale `2^-16` using nearest rounding,
  ties to even. This isolates Op 71–73.
- **Chained:** uses the confirmed two-stage integer GELU T104
  and the existing chained integer residual T73.
  This represents the complete block-0 integer path.

| Projection measurement | Benchmark | Calibration |
|---|---:|---:|
| Local T105 exact matches | 4096/4096 (100%) | 28672/28672 (100%) |
| Chained T105 exact matches | 2022/4096 (49.365%) | 13952/28672 (48.661%) |
| Chained maximum INT8-step difference | 3 | 4 |
| Chained mean absolute INT8-step difference | 0.557617 | 0.568394 |
| Fixed multiplier vs ideal conversion mismatches | 0 | 0 |
| Local projection values requiring saturation | 2 | 2 |
| Chained projection values requiring saturation | 2 | 3 |

| Final T107 error vs saved TFLite block | Benchmark | Calibration |
|---|---:|---:|
| Local maximum absolute error | 1.5005e-5 | 1.4983e-5 |
| Local mean absolute error | 5.0792e-6 | 5.0806e-6 |
| Local RMSE | 6.2099e-6 | 6.2196e-6 |
| Local relative L2 error | 0.005232% | 0.005238% |
| Chained maximum absolute error | 0.0101973 | 0.0084476 |
| Chained mean absolute error | 0.0018416 | 0.0018122 |
| Chained RMSE | 0.0022950 | 0.0022524 |
| Chained relative L2 error | 1.933702% | 1.896908% |

Relative L2 error is:

$$
\frac{\lVert y_{\mathrm{candidate}}-y_{\mathrm{reference}}\rVert_2}
{\lVert y_{\mathrm{reference}}\rVert_2}
$$

It measures error relative to the reference output's overall magnitude.
It is not a model-accuracy percentage.

| Stored T107 integer range | Benchmark | Calibration |
|---|---|---|
| Local | -42813 to 34986 | -42813 to 39704 |
| Chained | -42556 to 34935 | -42556 to 39330 |

The final conversion/addition's own maximum error was below
half a `2^-16` step.

Bias-scale compatibility, accumulator bounds, requantization bounds,
reference residual relationships, and T106/T107 INT32 checks passed.

#### Analysis

The local projection matched every tested T105 value.
Final-part local T107 error is approximately one output-format step
or less, including reference T73 encoding and projection conversion.

Chained T105 exact agreement is approximately 49%, while complete
block-output relative L2 error is approximately 1.9%.
Exact activation matching and overall numerical error therefore
describe different properties.

The projection and residual errors combine at T107 and can partly
cancel. Final output error must be measured directly rather than
estimated by adding intermediate maximum errors.

#### Block-0 completion checkpoint

All seven parts, Op 0–73, have Python integer candidates:

| Part | Operations | Final tensor |
|---|---|---|
| First LayerNorm | 0–14 | T45 |
| Q/K/V projections | 15–17 | T46/T47/T48 |
| Attention | 18–39 | T70 |
| Attention output projection and residual | 40–42 | T73 |
| Second LayerNorm | 43–57 | T88 |
| MLP expansion and GELU | 58–70 | T104 |
| MLP output projection and final residual | 71–73 | T107 |

For one 32-token sequence:

| Dense matrix-product workload | MACs |
|---|---:|
| Q/K/V projections | 1,572,864 |
| QK-transpose | 131,072 |
| Probability × V | 131,072 |
| Attention output projection | 524,288 |
| MLP expansion | 2,097,152 |
| MLP output projection | 2,097,152 |
| **Total** | **6,553,600** |

These are algorithmic dense MAC counts. Instruction counts,
cycle counts, and nonlinear-operation costs remain to be measured.

The complete chained result is available as:

    block0_integer_trial[i]["paths"]["chained"]["t107_q"]

Its represented output is:

    t107_q.astype(np.float64) * 2**-16

The current baseline retains INT8 T0 at scale `2^-8`.
An INT16 input alternative has not yet been tested.

Reference comparisons here use the fixed mixed-precision TFLite
block, whose final output is stored as FP32.

Complete-model accuracy, remaining blocks, C implementation,
new Spike validation, RTL validation, and performance evaluation
remain outstanding.

## EXPORT T0 and Golden Model T31

In [38]:
# from pathlib import Path
# import hashlib
# import numpy as np

# # Export the existing selected candidate; do not quantize again.
# expected_sha256 = (
#     "06481f6c5151a83e3c01896d314d902ee"
#     "9608773d94834848777ccc7282d87a9"
# )
# assert hashlib.sha256(BLOCK0_PATH.read_bytes()).hexdigest() == expected_sha256
# assert T0_SCALE == 2.0 ** -8
# assert T31_SCALE == 2.0 ** -15
# assert len(op0_integer_trial) == len(op0_cases) == 8

# case_names = []
# inputs = []
# goldens = []

# for case, trial in zip(op0_cases, op0_integer_trial):
#     assert case["name"] == trial["name"]

#     t0 = np.asarray(trial["t0_q"])
#     t31 = np.asarray(trial["t31_mean_q"])

#     assert t0.dtype == np.int8
#     assert t0.shape == (1, 32, 128)
#     assert t31.dtype == np.int32
#     assert t31.shape == (32,)

#     # Check the stored golden against its corresponding integer input.
#     np.testing.assert_array_equal(
#         t31,
#         t0.sum(axis=-1, dtype=np.int32).reshape(32),
#     )

#     case_names.append(case["name"])
#     inputs.append(t0.reshape(32 * 128).copy())
#     goldens.append(t31.copy())

# inputs = np.stack(inputs)
# goldens = np.stack(goldens)

# fixture_dir = (
#     PROJECT_DIR / "experiments/integer_inference/fixtures/block0"
# )
# fixture_dir.mkdir(parents=True, exist_ok=True)

# # C storage omits the singleton batch dimension.
# # Each input row contains 32 tokens × 128 features, in token-major order.
# case_comment = "\n".join(
#     f" * Case {i}: {name}"
#     for i, name in enumerate(case_names)
# )

# header_text = f"""\
# #ifndef BLOCK0_LN1_SUM_DATA_H
# #define BLOCK0_LN1_SUM_DATA_H

# #include <stdint.h>

# /*
#  * Source block SHA-256: {expected_sha256}
#  * T0: INT8, scale 2^-8, zero point 0.
#  * T31: INT32 feature sums, scale 2^-15, zero point 0.
#  * Input indexing: token * 128 + feature.
# {case_comment}
#  */

# #define BLOCK0_LN1_CASE_COUNT 8
# #define BLOCK0_LN1_TOKEN_COUNT 32
# #define BLOCK0_LN1_FEATURE_COUNT 128

# extern const int8_t block0_ln1_t0[
#     BLOCK0_LN1_CASE_COUNT
# ][BLOCK0_LN1_TOKEN_COUNT * BLOCK0_LN1_FEATURE_COUNT];

# extern const int32_t block0_ln1_t31_golden[
#     BLOCK0_LN1_CASE_COUNT
# ][BLOCK0_LN1_TOKEN_COUNT];

# #endif
# """

# def c_array_2d(name, c_type, values):
#     rows, columns = values.shape
#     lines = [
#         f"const {c_type} {name}[{rows}][{columns}] = {{"
#     ]

#     for i, row in enumerate(values):
#         lines.append(f"    {{ /* Case {i}: {case_names[i]} */")
#         for start in range(0, columns, 16):
#             numbers = ", ".join(
#                 str(int(v)) for v in row[start:start + 16]
#             )
#             lines.append(f"        {numbers},")
#         lines.append("    },")

#     lines.append("};")
#     return "\n".join(lines)

# source_text = (
#     '#include "ln1_sum_data.h"\n\n'
#     + c_array_2d("block0_ln1_t0", "int8_t", inputs)
#     + "\n\n"
#     + c_array_2d(
#         "block0_ln1_t31_golden", "int32_t", goldens
#     )
#     + "\n"
# )

# outputs = {
#     fixture_dir / "ln1_sum_data.h": header_text,
#     fixture_dir / "ln1_sum_data.c": source_text,
# }

# # Allow an identical rerun; refuse to overwrite different existing data.
# for path, content in outputs.items():
#     if path.exists() and path.read_text(encoding="utf-8") != content:
#         raise FileExistsError(f"Existing file differs: {path}")

# for path, content in outputs.items():
#     if not path.exists():
#         path.write_text(content, encoding="utf-8")

# print("Exported cases:", len(case_names))
# print("T0:", inputs.shape, inputs.dtype, inputs.nbytes, "bytes")
# print("Golden T31:", goldens.shape, goldens.dtype, goldens.nbytes, "bytes")
# print("First benchmark T0 values:", inputs[0, :8])
# print("First benchmark T31:", int(goldens[0, 0]))
# for path in outputs:
#     print("File:", path)

### Beta Alignment for op12-13

In [39]:
# S_beta / (S_normalized * S_gamma)
# For power-of-two scales, derive the ratio using their exponents.
beta_alignment_exponent = (
    T42_FRAC_BITS + GAMMA_FRAC_BITS - BETA_FRAC_BITS
)

# This helper supports exact integer multiplication for alignment.
assert 0 <= beta_alignment_exponent <= 62

ln1_beta_alignment = 1 << beta_alignment_exponent

assert BETA_SCALE == (
    ln1_beta_alignment * T42_SCALE * GAMMA_SCALE
)

print("Beta alignment:", ln1_beta_alignment)
print(
    "#define BLOCK0_LN1_BETA_ALIGNMENT "
    f"INT64_C({ln1_beta_alignment})"
)

Beta alignment: 65536
#define BLOCK0_LN1_BETA_ALIGNMENT INT64_C(65536)


In [40]:
# from pathlib import Path
# import hashlib
# import numpy as np

# source_sha = (
#     "06481f6c5151a83e3c01896d314d902ee"
#     "9608773d94834848777ccc7282d87a9"
# )
# assert hashlib.sha256(BLOCK0_PATH.read_bytes()).hexdigest() == source_sha

# base = PROJECT_DIR / "experiments/integer_inference"
# params_dir = base / "params"
# fixtures_dir = base / "fixtures/block0"

# # Confirm the selected formats and conversion parameters.
# assert FEATURE_COUNT == 128
# assert T42_FRAC_BITS == GAMMA_FRAC_BITS == BETA_FRAC_BITS == 16
# assert REQUANT_SHIFT == NORMALIZATION_REQUANT_SHIFT == 15
# assert epsilon_q == 42
# assert RSQRT_NUMERATOR == 1 << 54
# assert T45_MULTIPLIER_Q == 974600
# assert T45_REQUANT_SHIFT == 47
# assert T45_ZERO_POINT == -17

# beta_alignment = 1 << (
#     T42_FRAC_BITS + GAMMA_FRAC_BITS - BETA_FRAC_BITS
# )
# assert beta_alignment == 65536

# case_names = [case["name"] for case in op0_cases]
# assert len(case_names) == 8

# gamma = np.asarray(ln1_gamma_q).reshape(-1)
# beta = np.asarray(ln1_beta_q).reshape(-1)
# assert gamma.dtype == beta.dtype == np.dtype("int32")
# assert gamma.shape == beta.shape == (128,)

# # Each entry is an existing candidate output at a C function boundary.
# specs = [
#     ("t33_fine", op2_integer_trial, "t33_fine_q", "int64_t", np.int64, 4096),
#     ("t33", op2_integer_trial, "t33_coarse_q", "int32_t", np.int32, 4096),
#     ("t34", op3_integer_trial, "t34_coarse_q", "int32_t", np.int32, 32),
#     ("t35", op4_integer_trial, "t35_q", "int32_t", np.int32, 32),
#     ("t36", op5_integer_trial, "t36_q", "int32_t", np.int32, 32),
#     ("t42_fine", normalization_integer_trial, "t42_fine_q",
#      "int64_t", np.int64, 4096),
#     ("t42", normalization_integer_trial, "t42_q",
#      "int32_t", np.int32, 4096),
#     ("t44", ln1_affine_integer_trial, "t44_q", "int64_t", np.int64, 4096),
#     ("t45", ln1_affine_integer_trial, "t45_q", "int8_t", np.int8, 4096),
# ]

# goldens = {}
# for label, trials, key, c_type, dtype, columns in specs:
#     assert [trial["name"] for trial in trials] == case_names

#     rows = []
#     for trial in trials:
#         values = np.asarray(trial[key])
#         assert values.dtype == np.dtype(dtype)
#         assert values.size == columns
#         rows.append(values.reshape(columns).copy())

#     goldens[label] = (c_type, np.stack(rows))

# # Check that the earlier exported input/sum fixture still corresponds
# # to these exact candidates. Files are compared below before writing.
# assert [trial["name"] for trial in op0_integer_trial] == case_names
# for mean_trial, squared_trial in zip(op0_integer_trial, op2_integer_trial):
#     np.testing.assert_array_equal(
#         mean_trial["t31_mean_q"],
#         squared_trial["t32_q"].reshape(32),
#     )

# def c_values(values, indent="    "):
#     values = np.asarray(values).reshape(-1)
#     return "\n".join(
#         indent + ", ".join(str(int(v)) for v in values[i:i + 16]) + ","
#         for i in range(0, values.size, 16)
#     )

# def c_array_1d(name, values):
#     return (
#         f"const int32_t {name}[128] = {{\n"
#         + c_values(values)
#         + "\n};\n"
#     )

# def c_array_2d(name, c_type, values):
#     rows, columns = values.shape
#     lines = [f"const {c_type} {name}[{rows}][{columns}] = {{"]
#     for i, row in enumerate(values):
#         lines.append(f"    {{ /* Case {i}: {case_names[i]} */")
#         lines.append(c_values(row, indent="        "))
#         lines.append("    },")
#     lines.append("};")
#     return "\n".join(lines) + "\n"

# # Preserve existing parameter-file content using a marked section.
# marker_start = "/* BEGIN GENERATED BLOCK0 LN1 ARRAYS */"
# marker_end = "/* END GENERATED BLOCK0 LN1 ARRAYS */"

# def add_section(existing, body, symbols, header=False):
#     section = marker_start + "\n" + body + "\n" + marker_end

#     if marker_start in existing:
#         start = existing.index(marker_start)
#         end = existing.index(marker_end, start) + len(marker_end)
#         assert existing[start:end] == section, (
#             "Existing generated parameters differ; no files written."
#         )
#         return existing

#     assert marker_end not in existing
#     assert all(symbol not in existing for symbol in symbols), (
#         "Array names already exist outside the generated section."
#     )

#     if header:
#         end_guard = existing.rfind("#endif")
#         assert end_guard >= 0
#         return existing[:end_guard] + section + "\n\n" + existing[end_guard:]

#     return existing.rstrip() + "\n\n" + section + "\n"

# symbols = ["block0_ln1_gamma", "block0_ln1_beta"]
# header_path = params_dir / "block0_params.h"
# source_path = params_dir / "block0_params.c"

# header_existing = header_path.read_text(encoding="utf-8")
# source_existing = (
#     source_path.read_text(encoding="utf-8") if source_path.exists() else ""
# )

# parameter_declarations = """\
# #include <stdint.h>

# /* Gamma and beta: INT32 [128], scale 2^-16, zero point 0. */
# extern const int32_t block0_ln1_gamma[128];
# extern const int32_t block0_ln1_beta[128];"""

# parameter_definitions = (
#     '#include "block0_params.h"\n\n'
#     + c_array_1d(symbols[0], gamma)
#     + "\n"
#     + c_array_1d(symbols[1], beta)
# )

# planned = {
#     header_path: add_section(
#         header_existing, parameter_declarations, symbols, header=True
#     ),
#     source_path: add_section(
#         source_existing, parameter_definitions, symbols
#     ),
# }

# golden_header = [
#     "#ifndef BLOCK0_LN1_GOLDEN_H",
#     "#define BLOCK0_LN1_GOLDEN_H",
#     "",
#     "#include <stdint.h>",
#     "",
#     f"/* Source SHA-256: {source_sha}",
#     " * Selected integer candidate starting from encoded T0.",
#     " * Feature tensors are flattened in token-major order.",
# ]
# golden_header += [
#     f" * Case {i}: {name}" for i, name in enumerate(case_names)
# ]
# golden_header += [" */", ""]

# golden_source = ['#include "ln1_golden.h"', ""]

# for label, (c_type, values) in goldens.items():
#     name = f"block0_ln1_{label}_golden"
#     golden_header.append(
#         f"extern const {c_type} {name}[8][{values.shape[1]}];"
#     )
#     golden_source.append(c_array_2d(name, c_type, values))

# golden_header += ["", "#endif", ""]

# new_fixture_files = {
#     fixtures_dir / "ln1_golden.h": "\n".join(golden_header),
#     fixtures_dir / "ln1_golden.c": "\n".join(golden_source),
# }

# # Refuse to replace different existing golden fixtures.
# # Complete every check before writing any file.
# for path, content in new_fixture_files.items():
#     if path.exists():
#         assert path.read_text(encoding="utf-8") == content, (
#             f"Existing fixture differs: {path}; no files written."
#         )
# planned.update(new_fixture_files)

# for path, content in planned.items():
#     path.parent.mkdir(parents=True, exist_ok=True)
#     if not path.exists() or path.read_text(encoding="utf-8") != content:
#         path.write_text(content, encoding="utf-8")

# print("Exported LN1 cases:", len(case_names))
# print("Gamma + beta payload:", gamma.nbytes + beta.nbytes, "bytes")
# print("Beta alignment:", beta_alignment)
# for label, (c_type, values) in goldens.items():
#     print(f"{label}: {values.shape}, {values.dtype}")

# print("First benchmark T45 values:", goldens["t45"][1][0, :8])
# for path in planned:
#     print("File:", path)

### QKV EXPORT

In [41]:
# from pathlib import Path
# import hashlib
# import numpy as np

# expected_sha = (
#     "06481f6c5151a83e3c01896d314d902ee"
#     "9608773d94834848777ccc7282d87a9"
# )
# assert hashlib.sha256(BLOCK0_PATH.read_bytes()).hexdigest() == expected_sha
# assert QKV_REQUANT_SHIFT == 39

# qkv_case_names = [case["name"] for case in op0_cases]
# assert len(qkv_case_names) == 8
# for trials in [qkv_cases, qkv_integer_trial, ln1_affine_integer_trial]:
#     assert [trial["name"] for trial in trials] == qkv_case_names

# # Preserve the exact selected integer inputs.
# for case, ln1 in zip(qkv_cases, ln1_affine_integer_trial):
#     np.testing.assert_array_equal(
#         case["input_candidate"], ln1["t45_q"]
#     )

# parameter_arrays = {}
# fixture_arrays = {}
# parameter_lines = [
#     "#include <stdint.h>",
#     "",
#     f"/* Fixed export SHA-256: {expected_sha} */",
#     "#define BLOCK0_QKV_INPUT_ZERO_POINT (-17)",
#     "#define BLOCK0_QKV_REQUANT_SHIFT 39u",
#     "",
# ]

# def collect_fixture(name, arrays, dtype, c_type):
#     rows = []
#     for array in arrays:
#         values = np.asarray(array)
#         assert values.dtype == np.dtype(dtype)
#         assert values.shape == (1, 32, 128) or values.shape == (32, 128)
#         rows.append(values.reshape(4096).copy())
#     fixture_arrays[name] = (c_type, np.stack(rows))

# collect_fixture(
#     "block0_qkv_input_local",
#     [case["input_ref"] for case in qkv_cases],
#     np.int8, "int8_t",
# )
# collect_fixture(
#     "block0_qkv_input_chained",
#     [case["input_candidate"] for case in qkv_cases],
#     np.int8, "int8_t",
# )

# expected_specs = {
#     "Q": (15, 27, 46, 1280944693, -12),
#     "K": (16, 25, 47, 1756819468, -30),
#     "V": (17, 26, 48, 1696087205, 4),
# }

# for name, expected in expected_specs.items():
#     spec = qkv_specs[name]
#     op, weight_id, output_id, multiplier, zero_point = expected

#     assert spec["op_index"] == op
#     assert spec["input_index"] == 45
#     assert spec["weight_index"] == weight_id
#     assert spec["output_index"] == output_id
#     assert spec["input_zero_point"] == -17
#     assert spec["input_scale"] == T45_SCALE
#     assert spec["output_zero_point"] == zero_point
#     assert spec["bias_q"] is None
#     assert np.all(spec["weight_zero_points"] == 0)

#     multipliers = np.unique(spec["multiplier_q"])
#     weight_scales = np.unique(spec["weight_scales"])
#     assert multipliers.size == weight_scales.size == 1
#     assert int(multipliers[0]) == multiplier

#     weights = np.asarray(spec["weights_q"])
#     assert weights.dtype == np.int8
#     assert weights.shape == (128, 128)
#     np.testing.assert_array_equal(
#         weights, interpreter.get_tensor(weight_id)
#     )

#     lower = name.lower()
#     symbol = f"block0_{lower}_weights"
#     parameter_arrays[symbol] = ("int8_t", weights.reshape(16384).copy())

#     parameter_lines.extend([
#         f"/* {name}: weights T{weight_id}, output T{output_id}.",
#         " * Weights flattened from [output_feature, input_feature].",
#         f" * Weight scale: {float(weight_scales[0])!r}, zero point 0.",
#         f" * Output scale: {spec['output_scale']!r}. No bias. */",
#         f"#define BLOCK0_{name}_MULTIPLIER INT64_C({multiplier})",
#         f"#define BLOCK0_{name}_OUTPUT_ZERO_POINT ({zero_point})",
#         f"extern const int8_t {symbol}[16384];",
#         "",
#     ])

#     for path in ["local", "chained"]:
#         collect_fixture(
#             f"block0_{lower}_acc_{path}_golden",
#             [case["projections"][name][f"acc_{path}"]
#              for case in qkv_cases],
#             np.int32, "int32_t",
#         )
#         collect_fixture(
#             f"block0_{lower}_output_{path}_golden",
#             [trial["projections"][name][f"output_{path}"]
#              for trial in qkv_integer_trial],
#             np.int8, "int8_t",
#         )

# def qkv_values(values, indent):
#     flat = np.asarray(values).reshape(-1)
#     return "\n".join(
#         indent + ", ".join(str(int(v)) for v in flat[i:i + 16]) + ","
#         for i in range(0, flat.size, 16)
#     )

# def qkv_c_array(name, c_type, values):
#     dimensions = "".join(f"[{n}]" for n in values.shape)
#     lines = [f"const {c_type} {name}{dimensions} = {{"]

#     if values.ndim == 1:
#         lines.append(qkv_values(values, "    "))
#     else:
#         for i, row in enumerate(values):
#             lines.append(f"    {{ /* Case {i}: {qkv_case_names[i]} */")
#             lines.append(qkv_values(row, "        "))
#             lines.append("    },")

#     lines.append("};")
#     return "\n".join(lines) + "\n"

# # Append a separately marked Q/K/V section, preserving existing LN1 data.
# begin_marker = "/* BEGIN GENERATED BLOCK0 QKV PARAMETERS */"
# end_marker = "/* END GENERATED BLOCK0 QKV PARAMETERS */"

# def qkv_add_section(existing, body, header=False):
#     section = begin_marker + "\n" + body + "\n" + end_marker

#     if begin_marker in existing:
#         start = existing.index(begin_marker)
#         end = existing.index(end_marker, start) + len(end_marker)
#         assert existing[start:end] == section, (
#             "Existing Q/K/V section differs; no files written."
#         )
#         return existing

#     assert end_marker not in existing
#     assert all(symbol not in existing for symbol in parameter_arrays), (
#         "Q/K/V weight declarations already exist outside the marked section."
#     )
#     assert "BLOCK0_QKV_INPUT_ZERO_POINT" not in existing

#     if header:
#         position = existing.rfind("#endif")
#         assert position >= 0
#         return existing[:position] + section + "\n\n" + existing[position:]

#     return existing.rstrip() + "\n\n" + section + "\n"

# base = PROJECT_DIR / "experiments/integer_inference"
# parameter_header = base / "params/block0_params.h"
# parameter_source = base / "params/block0_params.c"
# fixture_dir = base / "fixtures/block0"

# parameter_source_body = (
#     '#include "block0_params.h"\n\n'
#     + "\n".join(
#         qkv_c_array(name, c_type, values)
#         for name, (c_type, values) in parameter_arrays.items()
#     )
# )

# planned_files = {
#     parameter_header: qkv_add_section(
#         parameter_header.read_text(encoding="utf-8"),
#         "\n".join(parameter_lines), header=True,
#     ),
#     parameter_source: qkv_add_section(
#         parameter_source.read_text(encoding="utf-8"),
#         parameter_source_body,
#     ),
# }

# fixture_header_lines = [
#     "#ifndef BLOCK0_QKV_GOLDEN_H",
#     "#define BLOCK0_QKV_GOLDEN_H",
#     "",
#     "#include <stdint.h>",
#     "",
#     "#define BLOCK0_QKV_CASE_COUNT 8",
#     "#define BLOCK0_QKV_ELEMENTS 4096",
#     "",
#     f"/* Source SHA-256: {expected_sha}",
#     " * local: reference T45 input.",
#     " * chained: selected integer LN1 T45 input.",
#     " * Feature arrays are flattened in token-major order.",
# ]
# fixture_header_lines.extend(
#     f" * Case {i}: {name}" for i, name in enumerate(qkv_case_names)
# )
# fixture_header_lines.extend([" */", ""])

# for name, (c_type, values) in fixture_arrays.items():
#     fixture_header_lines.append(
#         f"extern const {c_type} {name}[8][4096];"
#     )
# fixture_header_lines.extend(["", "#endif", ""])

# fixture_files = {
#     fixture_dir / "qkv_golden.h": "\n".join(fixture_header_lines),
#     fixture_dir / "qkv_golden.c": (
#         '#include "qkv_golden.h"\n\n'
#         + "\n".join(
#             qkv_c_array(name, c_type, values)
#             for name, (c_type, values) in fixture_arrays.items()
#         )
#     ),
# }

# # Check all existing fixtures before writing anything.
# for path, content in fixture_files.items():
#     if path.exists():
#         assert path.read_text(encoding="utf-8") == content, (
#             f"Existing fixture differs: {path}; no files written."
#         )
# planned_files.update(fixture_files)

# for path, content in planned_files.items():
#     path.parent.mkdir(parents=True, exist_ok=True)
#     if not path.exists() or path.read_text(encoding="utf-8") != content:
#         path.write_text(content, encoding="utf-8")

# print("Exported Q/K/V cases:", len(qkv_case_names))
# print("Weight payload:",
#       sum(values.nbytes for _, values in parameter_arrays.values()), "bytes")
# print("Fixture payload:",
#       sum(values.nbytes for _, values in fixture_arrays.values()), "bytes")

# for name in ["Q", "K", "V"]:
#     symbol = f"block0_{name.lower()}_output_chained_golden"
#     print(f"First benchmark {name}, chained:",
#           fixture_arrays[symbol][1][0, :8])

# for path in planned_files:
#     print("File:", path)

In [42]:
from pathlib import Path
import hashlib
import numpy as np

expected_sha = (
    "06481f6c5151a83e3c01896d314d902ee"
    "9608773d94834848777ccc7282d87a9"
)
assert hashlib.sha256(BLOCK0_PATH.read_bytes()).hexdigest() == expected_sha
assert int(interpreter._get_ops_details()[28]["outputs"][0]) == 59
assert qkv_specs["Q"]["output_zero_point"] == -12
assert qkv_specs["K"]["output_zero_point"] == -30

score_scale = (
    qkv_specs["Q"]["output_scale"]
    * qkv_specs["K"]["output_scale"]
)
assert score_scale == 0.0004575034413775367

case_names = [case["name"] for case in op0_cases]
assert len(case_names) == 8
assert [trial["name"] for trial in attention_score_trial] == case_names
assert [trial["name"] for trial in qkv_integer_trial] == case_names

score_arrays = {}

for path in ["local", "chained"]:
    rows = []
    output_key = f"output_{path}"

    for trial, projection in zip(
        attention_score_trial, qkv_integer_trial
    ):
        selected = trial["paths"][path]

        # Confirm these scores belong to our exported Q/K candidates.
        q = projection["projections"]["Q"][output_key]
        k = projection["projections"]["K"][output_key]

        np.testing.assert_array_equal(
            selected["query_head_q"],
            q.reshape(32, 16, 8).transpose(1, 0, 2),
        )
        np.testing.assert_array_equal(
            selected["key_head_q"],
            k.reshape(32, 16, 8).transpose(1, 2, 0),
        )

        scores = np.asarray(selected["score_q"])
        assert scores.dtype == np.int32
        assert scores.shape == (16, 32, 32)
        assert np.all(scores >= -174584)
        assert np.all(scores <= 174584)

        # Flatten in [head, query_token, key_token] order.
        rows.append(scores.reshape(16384).copy())

    score_arrays[path] = np.stack(rows)

fixture_dir = (
    PROJECT_DIR / "experiments/integer_inference/fixtures/block0"
)

header_lines = [
    "#ifndef BLOCK0_ATTENTION_SCORES_GOLDEN_H",
    "#define BLOCK0_ATTENTION_SCORES_GOLDEN_H",
    "",
    "#include <stdint.h>",
    "",
    "#define BLOCK0_ATTN_SCORE_CASE_COUNT 8",
    "#define BLOCK0_ATTN_SCORE_ELEMENTS 16384",
    "",
    f"/* Source SHA-256: {expected_sha}",
    " * T59: INT32 [16,32,32], zero point 0.",
    f" * Scale: {score_scale!r}.",
    " * Flattened index: (head * 32 + query_token) * 32 + key_token.",
    " * Includes all scores before masking.",
    " * local: Q/K from reference T45.",
    " * chained: Q/K from selected integer LN1 T45.",
]
header_lines.extend(
    f" * Case {i}: {name}" for i, name in enumerate(case_names)
)
header_lines.extend([" */", ""])

source_lines = ['#include "attention_scores_golden.h"', ""]

for path, values in score_arrays.items():
    name = f"block0_attention_scores_{path}_golden"

    header_lines.append(
        f"extern const int32_t {name}[8][16384];"
    )
    source_lines.append(f"const int32_t {name}[8][16384] = {{")

    for i, row in enumerate(values):
        source_lines.append(f"    {{ /* Case {i}: {case_names[i]} */")

        for start in range(0, 16384, 16):
            numbers = ", ".join(
                str(int(v)) for v in row[start:start + 16]
            )
            source_lines.append(f"        {numbers},")

        source_lines.append("    },")

    source_lines.extend(["};", ""])

header_lines.extend(["", "#endif", ""])

planned_files = {
    fixture_dir / "attention_scores_golden.h": "\n".join(header_lines),
    fixture_dir / "attention_scores_golden.c": "\n".join(source_lines),
}

# Allow identical reruns; preserve different existing files.
for file_path, content in planned_files.items():
    if file_path.exists():
        assert file_path.read_text(encoding="utf-8") == content, (
            f"Existing file differs: {file_path}; no files written."
        )

fixture_dir.mkdir(parents=True, exist_ok=True)

for file_path, content in planned_files.items():
    if not file_path.exists():
        file_path.write_text(content, encoding="utf-8")

print("Exported attention-score cases:", len(case_names))
print("Score scale:", score_scale)
print("Score zero point: 0")

for path, values in score_arrays.items():
    print(f"{path}: {values.shape}, {values.dtype}")
    print("  First benchmark head 0, query 0, first 8 scores:",
          values[0, :8])

print("Golden payload:",
      sum(values.nbytes for values in score_arrays.values()), "bytes")

for file_path in planned_files:
    print("File:", file_path)

Exported attention-score cases: 8
Score scale: 0.0004575034413775367
Score zero point: 0
local: (8, 16384), int32
  First benchmark head 0, query 0, first 8 scores: [-522 -219  144 -546 -270  277 -432  -83]
chained: (8, 16384), int32
  First benchmark head 0, query 0, first 8 scores: [-510 -223  144 -530 -281  289 -432  -68]
Golden payload: 1048576 bytes
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/attention_scores_golden.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/attention_scores_golden.c


In [44]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
params_dir = project_root / "experiments/integer_inference/params"
header_path = params_dir / "block0_params.h"
source_path = params_dir / "block0_params.c"

assert header_path.is_file(), header_path.resolve()
assert source_path.is_file(), source_path.resolve()

lut = np.asarray(exp_lut_q)
assert lut.dtype == np.int32
assert lut.shape == (4097,)
assert int(lut[0]) == 1 << 30
assert np.all(lut >= 0)
assert np.all(np.diff(lut.astype(np.int64)) <= 0)

assert EXP_COORD_MULTIPLIER_Q == 503030354
assert EXP_COORD_SHIFT == 24

begin = "/* BEGIN GENERATED BLOCK0 SOFTMAX LUT */"
end = "/* END GENERATED BLOCK0 SOFTMAX LUT */"

header_body = f"""#include <stdint.h>

#define BLOCK0_SOFTMAX_EXP_LUT_SIZE 4097u
#define BLOCK0_SOFTMAX_COORD_MULTIPLIER INT64_C({EXP_COORD_MULTIPLIER_Q})
#define BLOCK0_SOFTMAX_COORD_SHIFT {EXP_COORD_SHIFT}u

extern const int32_t block0_softmax_exp_lut[
    BLOCK0_SOFTMAX_EXP_LUT_SIZE
];"""

rows = [
    "    " + ", ".join(str(int(v)) for v in lut[i:i + 8]) + ","
    for i in range(0, lut.size, 8)
]

source_body = (
    "const int32_t block0_softmax_exp_lut[\n"
    "    BLOCK0_SOFTMAX_EXP_LUT_SIZE\n"
    "] = {\n"
    + "\n".join(rows)
    + "\n};"
)

def update_section(text, body, *, header=False):
    section = f"{begin}\n{body}\n{end}"

    if begin in text or end in text:
        assert text.count(begin) == text.count(end) == 1
        start = text.index(begin)
        finish = text.index(end) + len(end)
        assert start < finish - len(end)
        return text[:start] + section + text[finish:]

    if header:
        position = text.rfind("#endif")
        assert position >= 0
        return text[:position] + section + "\n\n" + text[position:]

    return text.rstrip() + "\n\n" + section + "\n"

# Prepare both updates before writing.
new_header = update_section(
    header_path.read_text(), header_body, header=True
)
new_source = update_section(
    source_path.read_text(), source_body
)

header_path.write_text(new_header)
source_path.write_text(new_source)

print("Exponential LUT entries:", lut.size)
print("Exponential LUT payload:", lut.nbytes, "bytes")
print("Coordinate multiplier:", EXP_COORD_MULTIPLIER_Q)
print("Coordinate shift:", EXP_COORD_SHIFT)
print("File:", header_path.resolve())
print("File:", source_path.resolve())

Exponential LUT entries: 4097
Exponential LUT payload: 16388 bytes
Coordinate multiplier: 503030354
Coordinate shift: 24
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.c


In [45]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
fixture_dir = (
    project_root
    / "experiments/integer_inference/fixtures/block0"
)
assert fixture_dir.is_dir(), fixture_dir

trials = [
    op0_integer_trial,
    qkv_integer_trial,
    attention_score_trial,
    attention_softmax_inputs,
    softmax_integer_trial,
    attention_context_trial,
    attention_merge_integer_trial,
]

names = [case["name"] for case in op0_integer_trial]
assert len(names) == 8

for trial in trials:
    assert [case["name"] for case in trial] == names

assert PROB_FRAC_BITS == 16
assert EXP_COORD_MULTIPLIER_Q == 503030354
assert EXP_COORD_SHIFT == 24
assert T67_MULTIPLIER_Q == 1113218944
assert T67_REQUANT_SHIFT == 46
assert T67_ZERO_POINT == 9

arrays = {}

def add_array(name, values, dtype, elements):
    array = np.stack([
        np.asarray(value).reshape(-1) for value in values
    ])
    assert array.dtype == np.dtype(dtype), (name, array.dtype)
    assert array.shape == (len(names), elements), (
        name, array.shape
    )
    arrays[name] = array

# Include T0 for the complete LN1 -> QKV -> attention test.
add_array(
    "block0_attention_t0",
    [case["t0_q"] for case in op0_integer_trial],
    np.int8,
    4096,
)

for path in ("local", "chained"):
    add_array(
        f"block0_attention_shifted_{path}_golden",
        [
            case["paths"][path]["shifted_score_q"]
            for case in attention_softmax_inputs
        ],
        np.int32,
        16384,
    )

    add_array(
        f"block0_attention_exp_{path}_golden",
        [
            case["paths"][path]["exp_q"]
            for case in softmax_integer_trial
        ],
        np.int32,
        16384,
    )

    add_array(
        f"block0_attention_probabilities_{path}_golden",
        [
            case["paths"][path]["probability_matrix_q"]
            for case in softmax_integer_trial
        ],
        np.int32,
        16384,
    )

    # Notebook: [head, token, feature].
    # C output: [token, head, feature] -> [token, hidden].
    contexts = []
    for case in attention_context_trial:
        context = case["paths"][path]["context_q"]
        assert context.shape == (16, 32, 8)
        contexts.append(
            context.transpose(1, 0, 2).reshape(32, 128)
        )

    add_array(
        f"block0_attention_context_{path}_golden",
        contexts,
        np.int32,
        4096,
    )

    add_array(
        f"block0_attention_output_{path}_golden",
        [
            case["paths"][path]["t70_q"]
            for case in attention_merge_integer_trial
        ],
        np.int8,
        4096,
    )

header_path = fixture_dir / "attention_golden.h"
source_path = fixture_dir / "attention_golden.c"

c_types = {
    np.dtype(np.int8): "int8_t",
    np.dtype(np.int32): "int32_t",
}

header_lines = [
    "#ifndef BLOCK0_ATTENTION_GOLDEN_H",
    "#define BLOCK0_ATTENTION_GOLDEN_H",
    "",
    "#include <stdint.h>",
    "",
    f"#define BLOCK0_ATTENTION_CASE_COUNT {len(names)}u",
    "#define BLOCK0_ATTENTION_SCORE_ELEMENTS 16384u",
    "#define BLOCK0_ATTENTION_FEATURE_ELEMENTS 4096u",
    "",
]

source_sections = ['#include "attention_golden.h"\n']

for name, array in arrays.items():
    c_type = c_types[array.dtype]
    dimensions = f"[{array.shape[0]}][{array.shape[1]}]"

    header_lines.append(
        f"extern const {c_type} {name}{dimensions};"
    )

    lines = [f"const {c_type} {name}{dimensions} = {{"]

    for row in array:
        lines.append("    {")
        for offset in range(0, row.size, 16):
            values = ", ".join(
                str(int(value))
                for value in row[offset:offset + 16]
            )
            lines.append(f"        {values},")
        lines.append("    },")

    lines.append("};")
    source_sections.append("\n".join(lines))

header_lines.extend(["", "#endif", ""])

header_path.write_text("\n".join(header_lines))
source_path.write_text("\n\n".join(source_sections) + "\n")

print("Exported attention cases:", len(names))
print("Golden payload:", sum(a.nbytes for a in arrays.values()), "bytes")
for name, array in arrays.items():
    print(f"{name}: {array.shape}, {array.dtype}")

print("File:", header_path)
print("File:", source_path)

Exported attention cases: 8
Golden payload: 3506176 bytes
block0_attention_t0: (8, 4096), int8
block0_attention_shifted_local_golden: (8, 16384), int32
block0_attention_exp_local_golden: (8, 16384), int32
block0_attention_probabilities_local_golden: (8, 16384), int32
block0_attention_context_local_golden: (8, 4096), int32
block0_attention_output_local_golden: (8, 4096), int8
block0_attention_shifted_chained_golden: (8, 16384), int32
block0_attention_exp_chained_golden: (8, 16384), int32
block0_attention_probabilities_chained_golden: (8, 16384), int32
block0_attention_context_chained_golden: (8, 4096), int32
block0_attention_output_chained_golden: (8, 4096), int8
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/attention_golden.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/attention_golden.c


In [46]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
base = project_root / "experiments/integer_inference"

params_h = base / "params/block0_params.h"
params_c = base / "params/block0_params.c"
golden_h = base / "fixtures/block0/out_residual_golden.h"
golden_c = base / "fixtures/block0/out_residual_golden.c"

assert params_h.is_file()
assert params_c.is_file()
assert golden_h.parent.is_dir()

names = [case["name"] for case in out_proj_cases]
assert len(names) == 8

for trial in (
    op0_integer_trial,
    attention_merge_integer_trial,
    out_residual_integer_trial,
):
    assert [case["name"] for case in trial] == names

weights = np.asarray(out_proj_spec["weights_q"])
bias = np.asarray(out_proj_spec["bias_q"])

assert weights.dtype == np.int8 and weights.shape == (128, 128)
assert bias.dtype == np.int32 and bias.shape == (128,)
assert np.all(out_proj_spec["weight_zero_points"] == 0)
assert np.all(out_proj_multiplier_q == 747156156)
assert OUT_PROJ_REQUANT_SHIFT == 39
assert out_proj_spec["input_zero_point"] == 9
assert out_proj_spec["output_zero_point"] == -13
assert T72_MULTIPLIER_Q == 939062400
assert T72_REQUANT_SHIFT == 22
assert T73_FRAC_BITS == 16
assert T0_RESIDUAL_ALIGN_SHIFT == 8

for prepared, original, attention in zip(
    out_proj_cases,
    op0_integer_trial,
    attention_merge_integer_trial,
):
    np.testing.assert_array_equal(
        prepared["t0_q"], original["t0_q"]
    )
    np.testing.assert_array_equal(
        prepared["input_chained"],
        attention["paths"]["chained"]["t70_q"],
    )

def values_initializer(values, indent=""):
    flat = np.asarray(values).reshape(-1)
    lines = [
        indent + "    " + ", ".join(
            str(int(value)) for value in flat[i:i + 16]
        ) + ","
        for i in range(0, flat.size, 16)
    ]
    return "{\n" + "\n".join(lines) + "\n" + indent + "}"

def patch_section(text, tag, body, *, header=False):
    begin = f"/* BEGIN GENERATED {tag} */"
    end = f"/* END GENERATED {tag} */"
    section = f"{begin}\n{body}\n{end}"

    if begin in text or end in text:
        assert text.count(begin) == text.count(end) == 1
        start = text.index(begin)
        finish = text.index(end) + len(end)
        assert text.index(end) > start
        return text[:start] + section + text[finish:]

    if header:
        position = text.rfind("#endif")
        assert position >= 0
        return text[:position] + section + "\n\n" + text[position:]

    return text.rstrip() + "\n\n" + section + "\n"

parameter_header = """#include <stdint.h>

#define BLOCK0_OUT_PROJ_INPUT_ZERO_POINT 9
#define BLOCK0_OUT_PROJ_MULTIPLIER INT64_C(747156156)
#define BLOCK0_OUT_PROJ_SHIFT 39u
#define BLOCK0_OUT_PROJ_OUTPUT_ZERO_POINT (-13)

#define BLOCK0_RESIDUAL1_BRANCH_MULTIPLIER INT64_C(939062400)
#define BLOCK0_RESIDUAL1_BRANCH_SHIFT 22u
#define BLOCK0_RESIDUAL1_INPUT_ALIGNMENT INT64_C(256)
#define BLOCK0_RESIDUAL1_INPUT_ZERO_POINT 0
#define BLOCK0_RESIDUAL1_FRAC_BITS 16u

extern const int8_t block0_out_weights[16384];
extern const int32_t block0_out_bias[128];"""

parameter_source = (
    "const int8_t block0_out_weights[16384] = "
    + values_initializer(weights)
    + ";\n\n"
    + "const int32_t block0_out_bias[128] = "
    + values_initializer(bias)
    + ";"
)

golden_header = """#ifndef BLOCK0_OUT_RESIDUAL_GOLDEN_H
#define BLOCK0_OUT_RESIDUAL_GOLDEN_H

#include <stdint.h>

#define BLOCK0_OUT_RESIDUAL_CASE_COUNT 8u
#define BLOCK0_OUT_RESIDUAL_ELEMENTS 4096u

typedef struct {
    int8_t t70[BLOCK0_OUT_RESIDUAL_ELEMENTS];
    int32_t accumulator[BLOCK0_OUT_RESIDUAL_ELEMENTS];
    int8_t t71[BLOCK0_OUT_RESIDUAL_ELEMENTS];
    int32_t t72[BLOCK0_OUT_RESIDUAL_ELEMENTS];
    int32_t t73[BLOCK0_OUT_RESIDUAL_ELEMENTS];
} Block0OutResidualGolden;

extern const int8_t block0_out_residual_t0[8][4096];
extern const Block0OutResidualGolden block0_out_residual_local[8];
extern const Block0OutResidualGolden block0_out_residual_chained[8];

#endif
"""

t0 = np.stack([
    case["t0_q"].reshape(-1) for case in out_proj_cases
])
assert t0.dtype == np.int8 and t0.shape == (8, 4096)

sections = [
    '#include "out_residual_golden.h"',
    "const int8_t block0_out_residual_t0[8][4096] = {\n"
    + ",\n".join(
        "    " + values_initializer(row, "    ") for row in t0
    )
    + "\n};",
]

payload = t0.nbytes

for path in ("local", "chained"):
    records = []

    for prepared, result in zip(
        out_proj_cases, out_residual_integer_trial
    ):
        fields = {
            "t70": prepared[
                "input_ref" if path == "local" else "input_chained"
            ],
            "accumulator": prepared[f"acc_{path}"],
            "t71": result["paths"][path]["t71_q"],
            "t72": result["paths"][path]["t72_q"],
            "t73": result["paths"][path]["t73_q"],
        }

        lines = ["    {"]
        for field, array in fields.items():
            array = np.asarray(array).reshape(-1)
            expected_dtype = (
                np.int8 if field in ("t70", "t71") else np.int32
            )
            assert array.dtype == np.dtype(expected_dtype)
            assert array.size == 4096

            payload += array.nbytes
            lines.append(
                f"        .{field} = "
                + values_initializer(array, "        ")
                + ","
            )
        lines.append("    }")
        records.append("\n".join(lines))

    sections.append(
        "const Block0OutResidualGolden "
        f"block0_out_residual_{path}[8] = {{\n"
        + ",\n".join(records)
        + "\n};"
    )

updates = {
    params_h: patch_section(
        params_h.read_text(),
        "BLOCK0 OUT RESIDUAL PARAMETERS",
        parameter_header,
        header=True,
    ),
    params_c: patch_section(
        params_c.read_text(),
        "BLOCK0 OUT RESIDUAL PARAMETERS",
        parameter_source,
    ),
    golden_h: golden_header,
    golden_c: "\n\n".join(sections) + "\n",
}

for path, text in updates.items():
    path.write_text(text)
    print("File:", path)

print("Exported Part 4 cases:", len(names))
print("Weight + bias payload:", weights.nbytes + bias.nbytes, "bytes")
print("Golden payload:", payload, "bytes")
print(
    "First chained T73 values:",
    out_residual_integer_trial[0]["paths"]["chained"][
        "t73_q"
    ].reshape(-1)[:8],
)

File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.c
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/out_residual_golden.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/out_residual_golden.c
Exported Part 4 cases: 8
Weight + bias payload: 16896 bytes
Golden payload: 950272 bytes
First chained T73 values: [ -4960   2398   9024  -1215    417  -4415  -1347 -11937]


In [47]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
params_dir = (
    project_root / "experiments/integer_inference/params"
)

header_path = params_dir / "block0_params.h"
source_path = params_dir / "block0_params.c"

assert header_path.is_file(), header_path
assert source_path.is_file(), source_path

gamma = np.asarray(ln2_gamma_q)
beta = np.asarray(ln2_beta_q)

assert gamma.dtype == np.int32 and gamma.shape == (128,)
assert beta.dtype == np.int32 and beta.shape == (128,)

assert LN2_SQUARE_REQUANT_SHIFT == 23
assert LN2_EPSILON_Q == 10737
assert LN2_RSQRT_NUMERATOR == 1 << 62
assert LN2_NORMALIZATION_REQUANT_SHIFT == 23
assert LN2_BETA_ALIGN_SHIFT == 16
assert T88_MULTIPLIER_Q == 1009162
assert T88_REQUANT_SHIFT == 47
assert T88_ZERO_POINT == 2

header_body = f"""#include "../src/layernorm.h"

#define BLOCK0_LN2_SQUARE_SHIFT {LN2_SQUARE_REQUANT_SHIFT}u
#define BLOCK0_LN2_EPSILON_Q INT32_C({LN2_EPSILON_Q})
#define BLOCK0_LN2_RSQRT_NUMERATOR (UINT64_C(1) << 62)
#define BLOCK0_LN2_NORMALIZE_SHIFT {LN2_NORMALIZATION_REQUANT_SHIFT}u
#define BLOCK0_LN2_BETA_ALIGNMENT INT64_C({1 << LN2_BETA_ALIGN_SHIFT})

#define BLOCK0_LN2_OUTPUT_MULTIPLIER INT64_C({T88_MULTIPLIER_Q})
#define BLOCK0_LN2_OUTPUT_SHIFT {T88_REQUANT_SHIFT}u
#define BLOCK0_LN2_OUTPUT_ZERO_POINT ({T88_ZERO_POINT})

extern const int32_t block0_ln2_gamma[128];
extern const int32_t block0_ln2_beta[128];

extern const LayerNormI8Params block0_ln2_params;"""

def array_definition(name, values):
    lines = [f"const int32_t {name}[128] = {{"]
    for i in range(0, 128, 8):
        row = ", ".join(str(int(v)) for v in values[i:i + 8])
        lines.append(f"    {row},")
    lines.append("};")
    return "\n".join(lines)

config_definition = """const LayerNormI8Params block0_ln2_params = {
    .gamma = block0_ln2_gamma,
    .beta = block0_ln2_beta,

    .square_shift = BLOCK0_LN2_SQUARE_SHIFT,
    .epsilon_q = BLOCK0_LN2_EPSILON_Q,
    .rsqrt_numerator = BLOCK0_LN2_RSQRT_NUMERATOR,
    .normalize_shift = BLOCK0_LN2_NORMALIZE_SHIFT,
    .beta_alignment = BLOCK0_LN2_BETA_ALIGNMENT,

    .output_multiplier = BLOCK0_LN2_OUTPUT_MULTIPLIER,
    .output_shift = BLOCK0_LN2_OUTPUT_SHIFT,
    .output_zero_point = BLOCK0_LN2_OUTPUT_ZERO_POINT
};"""

source_body = "\n\n".join([
    array_definition("block0_ln2_gamma", gamma),
    array_definition("block0_ln2_beta", beta),
    config_definition,
])

begin = "/* BEGIN GENERATED BLOCK0 LN2 PARAMETERS */"
end = "/* END GENERATED BLOCK0 LN2 PARAMETERS */"

def update_section(text, body, *, header=False):
    section = f"{begin}\n{body}\n{end}"

    if begin in text or end in text:
        assert text.count(begin) == text.count(end) == 1
        start = text.index(begin)
        finish = text.index(end) + len(end)
        assert text.index(end) > start
        return text[:start] + section + text[finish:]

    if header:
        position = text.rfind("#endif")
        assert position >= 0
        return text[:position] + section + "\n\n" + text[position:]

    return text.rstrip() + "\n\n" + section + "\n"

# Prepare both updates before writing.
new_header = update_section(
    header_path.read_text(), header_body, header=True
)
new_source = update_section(
    source_path.read_text(), source_body
)

header_path.write_text(new_header)
source_path.write_text(new_source)

print("LN2 gamma + beta payload:", gamma.nbytes + beta.nbytes, "bytes")
print("Square requantization shift:", LN2_SQUARE_REQUANT_SHIFT)
print("Epsilon integer:", LN2_EPSILON_Q)
print("RSQRT numerator: 2^62")
print("Normalization requantization shift:", LN2_NORMALIZATION_REQUANT_SHIFT)
print("Output multiplier:", T88_MULTIPLIER_Q)
print("Output shift:", T88_REQUANT_SHIFT)
print("Output zero point:", T88_ZERO_POINT)
print("File:", header_path)
print("File:", source_path)

LN2 gamma + beta payload: 1024 bytes
Square requantization shift: 23
Epsilon integer: 10737
RSQRT numerator: 2^62
Normalization requantization shift: 23
Output multiplier: 1009162
Output shift: 47
Output zero point: 2
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.c


In [48]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
fixture_dir = (
    project_root
    / "experiments/integer_inference/fixtures/block0"
)
assert fixture_dir.is_dir(), fixture_dir

names = [case["name"] for case in op0_integer_trial]
assert len(names) == 8
assert LN2_FEATURE_COUNT == 128
assert LN2_INPUT_FRAC_BITS == 16
assert LN2_SQUARE_REQUANT_SHIFT == 23
assert LN2_NORMALIZATION_REQUANT_SHIFT == 23
assert T88_MULTIPLIER_Q == 1009162
assert T88_REQUANT_SHIFT == 47
assert T88_ZERO_POINT == 2

trials = [
    ln2_mean_integer_trial,
    ln2_variance_integer_trial,
    ln2_rsqrt_integer_trial,
    ln2_normalization_integer_trial,
    ln2_affine_integer_trial,
    out_residual_integer_trial,
]

for trial in trials:
    assert [case["name"] for case in trial] == names

# Each entry describes one C-visible checkpoint.
fields = [
    ("t73", "int32_t", np.int32, 4096,
     ln2_mean_integer_trial, "t73_q"),

    ("t74", "int32_t", np.int32, 32,
     ln2_mean_integer_trial, "t74_q"),

    ("t76_fine", "int64_t", np.int64, 4096,
     ln2_variance_integer_trial, "t76_fine_q"),

    ("t76", "int32_t", np.int32, 4096,
     ln2_variance_integer_trial, "t76_q"),

    ("t77", "int32_t", np.int32, 32,
     ln2_variance_integer_trial, "t77_q"),

    ("t78", "int32_t", np.int32, 32,
     ln2_rsqrt_integer_trial, "t78_q"),

    ("t79", "int32_t", np.int32, 32,
     ln2_rsqrt_integer_trial, "t79_q"),

    ("t85_fine", "int64_t", np.int64, 4096,
     ln2_normalization_integer_trial, "t85_fine_q"),

    ("t85", "int32_t", np.int32, 4096,
     ln2_normalization_integer_trial, "t85_q"),

    ("t87", "int64_t", np.int64, 4096,
     ln2_affine_integer_trial, "t87_q"),

    ("t88", "int8_t", np.int8, 4096,
     ln2_affine_integer_trial, "t88_q"),
]

for mean_case, residual_case in zip(
    ln2_mean_integer_trial, out_residual_integer_trial
):
    np.testing.assert_array_equal(
        mean_case["paths"]["chained"]["t73_q"],
        residual_case["paths"]["chained"]["t73_q"],
    )

def initializer(values, indent=""):
    values = np.asarray(values).reshape(-1)
    rows = [
        indent + "    " + ", ".join(
            str(int(v)) for v in values[i:i + 16]
        ) + ","
        for i in range(0, values.size, 16)
    ]
    return "{\n" + "\n".join(rows) + "\n" + indent + "}"

header_lines = [
    "#ifndef BLOCK0_LN2_GOLDEN_H",
    "#define BLOCK0_LN2_GOLDEN_H",
    "",
    "#include <stdint.h>",
    "",
    "#define BLOCK0_LN2_GOLDEN_CASE_COUNT 8u",
    "#define BLOCK0_LN2_GOLDEN_TOKEN_COUNT 32u",
    "#define BLOCK0_LN2_GOLDEN_ELEMENTS 4096u",
    "",
    "typedef struct {",
]

for field, c_type, _, count, _, _ in fields:
    header_lines.append(f"    {c_type} {field}[{count}];")

header_lines.extend([
    "} Block0Ln2Golden;",
    "",
    "extern const int8_t block0_ln2_t0[8][4096];",
    "extern const Block0Ln2Golden block0_ln2_local[8];",
    "extern const Block0Ln2Golden block0_ln2_chained[8];",
    "",
    "#endif",
    "",
])

t0 = np.stack([
    case["t0_q"].reshape(-1) for case in op0_integer_trial
])
assert t0.dtype == np.int8 and t0.shape == (8, 4096)

source_sections = [
    '#include "ln2_golden.h"',
    "const int8_t block0_ln2_t0[8][4096] = {\n"
    + ",\n".join(
        "    " + initializer(row, "    ") for row in t0
    )
    + "\n};",
]

payload = t0.nbytes

for path in ("local", "chained"):
    records = []

    for c in range(len(names)):
        lines = ["    {"]

        for field, _, dtype, count, trial, key in fields:
            values = np.asarray(
                trial[c]["paths"][path][key]
            ).reshape(-1)

            assert values.dtype == np.dtype(dtype), (
                path, field, values.dtype
            )
            assert values.size == count, (path, field, values.shape)

            payload += values.nbytes

            lines.append(
                f"        .{field} = "
                + initializer(values, "        ")
                + ","
            )

        lines.append("    }")
        records.append("\n".join(lines))

    source_sections.append(
        f"const Block0Ln2Golden block0_ln2_{path}[8] = {{\n"
        + ",\n".join(records)
        + "\n};"
    )

header_path = fixture_dir / "ln2_golden.h"
source_path = fixture_dir / "ln2_golden.c"

header_path.write_text("\n".join(header_lines))
source_path.write_text("\n\n".join(source_sections) + "\n")

print("Exported LN2 cases:", len(names))
print("Golden payload:", payload, "bytes")
print(
    "First chained T88 values:",
    ln2_affine_integer_trial[0]["paths"]["chained"][
        "t88_q"
    ].reshape(-1)[:8],
)
print("File:", header_path)
print("File:", source_path)

Exported LN2 cases: 8
Golden payload: 2465792 bytes
First chained T88 values: [-14  11  34  -2   5 -14  -2 -38]
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/ln2_golden.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/ln2_golden.c


In [49]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
params_dir = (
    project_root / "experiments/integer_inference/params"
)

header_path = params_dir / "block0_params.h"
source_path = params_dir / "block0_params.c"

assert header_path.is_file(), header_path
assert source_path.is_file(), source_path

spec = mlp_expand_spec

assert spec["op_index"] == 58
assert spec["input_index"] == 88
assert spec["weight_index"] == 29
assert spec["output_index"] == 89

weights = np.asarray(spec["weights_q"])
bias = np.asarray(spec["bias_q"])

assert weights.dtype == np.int8
assert weights.shape == (512, 128)
assert bias.dtype == np.int32
assert bias.shape == (512,)

# Our shared projection kernel assumes weight zero point 0.
assert np.all(np.asarray(spec["weight_zero_points"]) == 0)

multipliers = np.asarray(mlp_expand_multiplier_q)
assert multipliers.shape == (512,)

unique_multipliers = np.unique(multipliers)
assert unique_multipliers.size == 1

multiplier = int(unique_multipliers[0])
shift = int(MLP_EXPAND_REQUANT_SHIFT)
input_zero_point = int(spec["input_zero_point"])
output_zero_point = int(spec["output_zero_point"])

assert multiplier == 1274254450
assert shift == 39
assert input_zero_point == 2
assert output_zero_point == 37

# Confirm the conversion product fits INT64 for every INT32 accumulator.
assert (
    (1 << 31) * multiplier + (1 << (shift - 1))
    <= np.iinfo(np.int64).max
)

header_body = f"""#include <stdint.h>

/*
 * Op 58: MLP expansion, 128 -> 512.
 * Weights are flattened in [output_feature, input_feature] order.
 *
 * Input scale:  {float(spec["input_scale"]):.17g}
 * Output scale: {float(spec["output_scale"]):.17g}
 * Bias uses the corresponding accumulator scale.
 */
#define BLOCK0_MLP_EXPAND_INPUT_FEATURES 128u
#define BLOCK0_MLP_EXPAND_OUTPUT_FEATURES 512u
#define BLOCK0_MLP_EXPAND_WEIGHT_COUNT 65536u

#define BLOCK0_MLP_EXPAND_INPUT_ZERO_POINT ({input_zero_point})
#define BLOCK0_MLP_EXPAND_MULTIPLIER INT64_C({multiplier})
#define BLOCK0_MLP_EXPAND_SHIFT {shift}u
#define BLOCK0_MLP_EXPAND_OUTPUT_ZERO_POINT ({output_zero_point})

extern const int8_t block0_mlp_expand_weights[
    BLOCK0_MLP_EXPAND_WEIGHT_COUNT
];

extern const int32_t block0_mlp_expand_bias[
    BLOCK0_MLP_EXPAND_OUTPUT_FEATURES
];"""

def array_definition(c_type, name, dimension, values):
    flat = np.asarray(values).reshape(-1)
    lines = [f"const {c_type} {name}[{dimension}] = {{"]

    for i in range(0, flat.size, 16):
        row = ", ".join(str(int(v)) for v in flat[i:i + 16])
        lines.append(f"    {row},")

    lines.append("};")
    return "\n".join(lines)

source_body = "\n\n".join([
    array_definition(
        "int8_t",
        "block0_mlp_expand_weights",
        "BLOCK0_MLP_EXPAND_WEIGHT_COUNT",
        weights,
    ),
    array_definition(
        "int32_t",
        "block0_mlp_expand_bias",
        "BLOCK0_MLP_EXPAND_OUTPUT_FEATURES",
        bias,
    ),
])

begin = "/* BEGIN GENERATED BLOCK0 MLP EXPAND PARAMETERS */"
end = "/* END GENERATED BLOCK0 MLP EXPAND PARAMETERS */"

def update_section(text, body, *, header=False):
    section = f"{begin}\n{body}\n{end}"

    if begin in text or end in text:
        assert text.count(begin) == text.count(end) == 1
        start = text.index(begin)
        finish = text.index(end) + len(end)
        assert text.index(end) > start
        return text[:start] + section + text[finish:]

    if header:
        position = text.rfind("#endif")
        assert position >= 0
        return text[:position] + section + "\n\n" + text[position:]

    return text.rstrip() + "\n\n" + section + "\n"

# Prepare both updates before writing.
new_header = update_section(
    header_path.read_text(), header_body, header=True
)
new_source = update_section(
    source_path.read_text(), source_body
)

header_path.write_text(new_header)
source_path.write_text(new_source)

print("Exported Op 58 parameters")
print("Weights:", weights.shape, weights.dtype, weights.nbytes, "bytes")
print("Bias:", bias.shape, bias.dtype, bias.nbytes, "bytes")
print("Total payload:", weights.nbytes + bias.nbytes, "bytes")
print("Input zero point:", input_zero_point)
print("Output zero point:", output_zero_point)
print("Shared multiplier:", multiplier)
print("Right shift:", shift)
print("File:", header_path)
print("File:", source_path)

Exported Op 58 parameters
Weights: (512, 128) int8 65536 bytes
Bias: (512,) int32 2048 bytes
Total payload: 67584 bytes
Input zero point: 2
Output zero point: 37
Shared multiplier: 1274254450
Right shift: 39
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.c


In [50]:
from pathlib import Path

project_root = Path("/home/randy/Documents/ACompArc/model/project")
params_dir = (
    project_root / "experiments/integer_inference/params"
)

header_path = params_dir / "block0_params.h"

assert header_path.is_file(), header_path

input_zero_point = int(gelu_quant_specs[89]["zero_point"])
centered_constant = int(GELU_HALF_CENTERED_Q)

conversion = gelu_half_conversion
multiplier = int(conversion["multiplier_q"])
shift = int(conversion["shift"])
output_zero_point = int(conversion["output_zero_point"])

assert conversion["output_index"] == 92
assert input_zero_point == 37
assert centered_constant == 255
assert multiplier == 1077952640
assert shift == 38
assert output_zero_point == 37

body = f"""#include <stdint.h>

/* Op 60: centered T89 * centered T91, then requantize to T92. */
#define BLOCK0_GELU_HALF_INPUT_ZERO_POINT ({input_zero_point})
#define BLOCK0_GELU_HALF_CENTERED_CONSTANT INT32_C({centered_constant})
#define BLOCK0_GELU_HALF_MULTIPLIER INT64_C({multiplier})
#define BLOCK0_GELU_HALF_SHIFT {shift}u
#define BLOCK0_GELU_HALF_OUTPUT_ZERO_POINT ({output_zero_point})"""

begin = "/* BEGIN GENERATED BLOCK0 GELU HALF PARAMETERS */"
end = "/* END GENERATED BLOCK0 GELU HALF PARAMETERS */"
section = f"{begin}\n{body}\n{end}"

text = header_path.read_text()

if begin in text or end in text:
    assert text.count(begin) == text.count(end) == 1
    start = text.index(begin)
    finish = text.index(end) + len(end)
    assert text.index(end) > start

    updated = text[:start] + section + text[finish:]
else:
    position = text.rfind("#endif")
    assert position >= 0

    updated = (
        text[:position]
        + section
        + "\n\n"
        + text[position:]
    )

header_path.write_text(updated)

print("Exported GELU half-branch parameters")
print("Input zero point:", input_zero_point)
print("Centered scalar:", centered_constant)
print("Multiplier:", multiplier)
print("Right shift:", shift)
print("Output zero point:", output_zero_point)
print("File:", header_path)

Exported GELU half-branch parameters
Input zero point: 37
Centered scalar: 255
Multiplier: 1077952640
Right shift: 38
Output zero point: 37
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h


In [51]:
from pathlib import Path

project_root = Path("/home/randy/Documents/ACompArc/model/project")
header_path = (
    project_root
    / "experiments/integer_inference/params/block0_params.h"
)

assert header_path.is_file(), header_path

input_zero_point = int(gelu_quant_specs[89]["zero_point"])

conversion = gelu_cube_conversion
multiplier = int(conversion["multiplier_q"])
shift = int(conversion["shift"])
output_zero_point = int(conversion["output_zero_point"])

assert conversion["output_index"] == 94
assert input_zero_point == 37
assert multiplier == 1670962322
assert shift == 45
assert output_zero_point == 85

body = f"""#include <stdint.h>

/*
 * Op 61: centered T89 cube, INT32, zero point 0.
 * Cube scale: {float(GELU_CUBE_SCALE):.17g}
 * Op 62: requantize the cube to INT8 T94.
 */
#define BLOCK0_GELU_CUBE_INPUT_ZERO_POINT ({input_zero_point})
#define BLOCK0_GELU_CUBE_MULTIPLIER INT64_C({multiplier})
#define BLOCK0_GELU_CUBE_SHIFT {shift}u
#define BLOCK0_GELU_CUBE_OUTPUT_ZERO_POINT ({output_zero_point})"""

begin = "/* BEGIN GENERATED BLOCK0 GELU CUBE PARAMETERS */"
end = "/* END GENERATED BLOCK0 GELU CUBE PARAMETERS */"
section = f"{begin}\n{body}\n{end}"

text = header_path.read_text()

if begin in text or end in text:
    assert text.count(begin) == text.count(end) == 1
    start = text.index(begin)
    finish = text.index(end) + len(end)
    assert text.index(end) > start

    updated = text[:start] + section + text[finish:]
else:
    position = text.rfind("#endif")
    assert position >= 0

    updated = (
        text[:position]
        + section
        + "\n\n"
        + text[position:]
    )

header_path.write_text(updated)

print("Exported GELU cube parameters")
print("Input zero point:", input_zero_point)
print("Cube scale:", GELU_CUBE_SCALE)
print("Multiplier:", multiplier)
print("Right shift:", shift)
print("Output zero point:", output_zero_point)
print("File:", header_path)

Exported GELU cube parameters
Input zero point: 37
Cube scale: 1.4248101460504496e-06
Multiplier: 1670962322
Right shift: 45
Output zero point: 85
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h


In [52]:
from pathlib import Path

project_root = Path("/home/randy/Documents/ACompArc/model/project")
header_path = (
    project_root
    / "experiments/integer_inference/params/block0_params.h"
)

assert header_path.is_file(), header_path

input_zero_point = int(gelu_quant_specs[94]["zero_point"])
centered_constant = int(GELU_CUBIC_COEFF_CENTERED_Q)

conversion = gelu_weighted_cube_conversion
multiplier = int(conversion["multiplier_q"])
shift = int(conversion["shift"])
output_zero_point = int(conversion["output_zero_point"])

assert conversion["output_index"] == 96
assert input_zero_point == 85
assert centered_constant == 183
assert multiplier == 1500813243
assert shift == 38
assert output_zero_point == 85

body = f"""#include <stdint.h>

/* Op 63: centered T94 * centered T95, then requantize to T96. */
#define BLOCK0_GELU_WEIGHTED_CUBE_INPUT_ZERO_POINT ({input_zero_point})
#define BLOCK0_GELU_WEIGHTED_CUBE_CENTERED_CONSTANT INT32_C({centered_constant})
#define BLOCK0_GELU_WEIGHTED_CUBE_MULTIPLIER INT64_C({multiplier})
#define BLOCK0_GELU_WEIGHTED_CUBE_SHIFT {shift}u
#define BLOCK0_GELU_WEIGHTED_CUBE_OUTPUT_ZERO_POINT ({output_zero_point})"""

begin = "/* BEGIN GENERATED BLOCK0 GELU WEIGHTED CUBE PARAMETERS */"
end = "/* END GENERATED BLOCK0 GELU WEIGHTED CUBE PARAMETERS */"
section = f"{begin}\n{body}\n{end}"

text = header_path.read_text()

if begin in text or end in text:
    assert text.count(begin) == text.count(end) == 1
    start = text.index(begin)
    finish = text.index(end) + len(end)
    assert text.index(end) > start

    updated = text[:start] + section + text[finish:]
else:
    position = text.rfind("#endif")
    assert position >= 0

    updated = (
        text[:position]
        + section
        + "\n\n"
        + text[position:]
    )

header_path.write_text(updated)

print("Exported Op 63 parameters")
print("Input zero point:", input_zero_point)
print("Centered cubic coefficient:", centered_constant)
print("Multiplier:", multiplier)
print("Right shift:", shift)
print("Output zero point:", output_zero_point)
print("File:", header_path)

Exported Op 63 parameters
Input zero point: 85
Centered cubic coefficient: 183
Multiplier: 1500813243
Right shift: 38
Output zero point: 85
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h


In [53]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
header_path = (
    project_root
    / "experiments/integer_inference/params/block0_params.h"
)

assert header_path.is_file(), header_path

multipliers = np.asarray(gelu_add_multiplier_q)
assert multipliers.shape == (2,)

linear_multiplier = int(multipliers[0])
cubic_multiplier = int(multipliers[1])

linear_zero_point = int(gelu_quant_specs[89]["zero_point"])
cubic_zero_point = int(gelu_quant_specs[96]["zero_point"])
output_zero_point = int(gelu_quant_specs[97]["zero_point"])
shift = int(GELU_ADD_SHIFT)

assert linear_multiplier == 1916390445
assert cubic_multiplier == 228467824
assert linear_zero_point == 37
assert cubic_zero_point == 85
assert output_zero_point == 37
assert shift == 31

linear_bound = max(
    abs(-128 - linear_zero_point),
    abs(127 - linear_zero_point),
)
cubic_bound = max(
    abs(-128 - cubic_zero_point),
    abs(127 - cubic_zero_point),
)

combined_bound = (
    linear_bound * linear_multiplier
    + cubic_bound * cubic_multiplier
)

assert (
    combined_bound + (1 << (shift - 1))
    <= np.iinfo(np.int64).max
)

body = f"""#include <stdint.h>

/* Op 64: combine scaled T89 and T96, then round once to T97. */
#define BLOCK0_GELU_ADD_LINEAR_ZERO_POINT ({linear_zero_point})
#define BLOCK0_GELU_ADD_CUBIC_ZERO_POINT ({cubic_zero_point})
#define BLOCK0_GELU_ADD_LINEAR_MULTIPLIER INT64_C({linear_multiplier})
#define BLOCK0_GELU_ADD_CUBIC_MULTIPLIER INT64_C({cubic_multiplier})
#define BLOCK0_GELU_ADD_SHIFT {shift}u
#define BLOCK0_GELU_ADD_OUTPUT_ZERO_POINT ({output_zero_point})"""

begin = "/* BEGIN GENERATED BLOCK0 GELU ADD PARAMETERS */"
end = "/* END GENERATED BLOCK0 GELU ADD PARAMETERS */"
section = f"{begin}\n{body}\n{end}"

text = header_path.read_text()

if begin in text or end in text:
    assert text.count(begin) == text.count(end) == 1
    start = text.index(begin)
    finish = text.index(end) + len(end)
    assert text.index(end) > start

    updated = text[:start] + section + text[finish:]
else:
    position = text.rfind("#endif")
    assert position >= 0

    updated = (
        text[:position]
        + section
        + "\n\n"
        + text[position:]
    )

header_path.write_text(updated)

print("Exported Op 64 parameters")
print("Linear multiplier:", linear_multiplier)
print("Cubic multiplier:", cubic_multiplier)
print("Common right shift:", shift)
print("Output zero point:", output_zero_point)
print("Full-domain absolute combined-product bound:", combined_bound)
print("File:", header_path)

Exported Op 64 parameters
Linear multiplier: 1916390445
Cubic multiplier: 228467824
Common right shift: 31
Output zero point: 37
Full-domain absolute combined-product bound: 364868069937
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h


In [54]:
from pathlib import Path

project_root = Path("/home/randy/Documents/ACompArc/model/project")
header_path = (
    project_root
    / "experiments/integer_inference/params/block0_params.h"
)

assert header_path.is_file(), header_path

input_zero_point = int(gelu_quant_specs[97]["zero_point"])
centered_constant = int(GELU_ARGUMENT_COEFF_CENTERED_Q)

conversion = gelu_argument_conversion
multiplier = int(conversion["multiplier_q"])
shift = int(conversion["shift"])
output_zero_point = int(conversion["output_zero_point"])

assert conversion["output_index"] == 99
assert input_zero_point == 37
assert centered_constant == 255
assert multiplier == 1077952560
assert shift == 38
assert output_zero_point == 37

body = f"""#include <stdint.h>

/* Op 65: centered T97 * centered T98, then requantize to T99. */
#define BLOCK0_GELU_ARGUMENT_INPUT_ZERO_POINT ({input_zero_point})
#define BLOCK0_GELU_ARGUMENT_CENTERED_CONSTANT INT32_C({centered_constant})
#define BLOCK0_GELU_ARGUMENT_MULTIPLIER INT64_C({multiplier})
#define BLOCK0_GELU_ARGUMENT_SHIFT {shift}u
#define BLOCK0_GELU_ARGUMENT_OUTPUT_ZERO_POINT ({output_zero_point})"""

begin = "/* BEGIN GENERATED BLOCK0 GELU ARGUMENT PARAMETERS */"
end = "/* END GENERATED BLOCK0 GELU ARGUMENT PARAMETERS */"
section = f"{begin}\n{body}\n{end}"

text = header_path.read_text()

if begin in text or end in text:
    assert text.count(begin) == text.count(end) == 1
    start = text.index(begin)
    finish = text.index(end) + len(end)
    assert text.index(end) > start

    updated = text[:start] + section + text[finish:]
else:
    position = text.rfind("#endif")
    assert position >= 0

    updated = (
        text[:position]
        + section
        + "\n\n"
        + text[position:]
    )

header_path.write_text(updated)

print("Exported Op 65 parameters")
print("Input zero point:", input_zero_point)
print("Centered argument coefficient:", centered_constant)
print("Multiplier:", multiplier)
print("Right shift:", shift)
print("Output zero point:", output_zero_point)
print("File:", header_path)

Exported Op 65 parameters
Input zero point: 37
Centered argument coefficient: 255
Multiplier: 1077952560
Right shift: 38
Output zero point: 37
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h


In [55]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
params_dir = (
    project_root / "experiments/integer_inference/params"
)

header_path = params_dir / "block0_params.h"
source_path = params_dir / "block0_params.c"

assert header_path.is_file(), header_path
assert source_path.is_file(), source_path

lut = np.asarray(gelu_tanh_lut_q)

assert lut.dtype == np.int32
assert lut.shape == (256,)
assert GELU_TANH_FRAC_BITS == 24
assert GELU_ONE_Q == 16777216
assert np.all(np.diff(lut.astype(np.int64)) >= 0)
assert int(lut.min()) >= -GELU_ONE_Q
assert int(lut.max()) <= GELU_ONE_Q

input_zero_point = int(gelu_quant_specs[99]["zero_point"])
assert input_zero_point == 37
assert int(lut[input_zero_point + 128]) == 0

conversion = gelu_factor_conversion
multiplier = int(conversion["multiplier_q"])
shift = int(conversion["shift"])
output_zero_point = int(conversion["output_zero_point"])

assert conversion["output_index"] == 103
assert multiplier == 1269488229
assert shift == 47
assert output_zero_point == -128

header_body = f"""#include <stdint.h>

/*
 * Op 66-67: table indexed by T99 code + 128.
 * T99 scale: {float(gelu_quant_specs[99]["scale"]):.17g}
 * T101/T102 scale: 2^-24, zero point 0.
 */
#define BLOCK0_GELU_TANH_LUT_SIZE 256u
#define BLOCK0_GELU_TANH_INPUT_ZERO_POINT ({input_zero_point})
#define BLOCK0_GELU_TANH_FRAC_BITS {GELU_TANH_FRAC_BITS}u
#define BLOCK0_GELU_TANH_ONE INT32_C({GELU_ONE_Q})

#define BLOCK0_GELU_FACTOR_MULTIPLIER INT64_C({multiplier})
#define BLOCK0_GELU_FACTOR_SHIFT {shift}u
#define BLOCK0_GELU_FACTOR_OUTPUT_ZERO_POINT ({output_zero_point})

extern const int32_t block0_gelu_tanh_lut[
    BLOCK0_GELU_TANH_LUT_SIZE
];"""

lines = [
    "const int32_t block0_gelu_tanh_lut[",
    "    BLOCK0_GELU_TANH_LUT_SIZE",
    "] = {",
]

for i in range(0, lut.size, 8):
    row = ", ".join(str(int(v)) for v in lut[i:i + 8])
    lines.append(f"    {row},")

lines.append("};")
source_body = "\n".join(lines)

begin = "/* BEGIN GENERATED BLOCK0 GELU TANH PARAMETERS */"
end = "/* END GENERATED BLOCK0 GELU TANH PARAMETERS */"

def update_section(text, body, *, header=False):
    section = f"{begin}\n{body}\n{end}"

    if begin in text or end in text:
        assert text.count(begin) == text.count(end) == 1
        start = text.index(begin)
        finish = text.index(end) + len(end)
        assert text.index(end) > start
        return text[:start] + section + text[finish:]

    if header:
        position = text.rfind("#endif")
        assert position >= 0
        return text[:position] + section + "\n\n" + text[position:]

    return text.rstrip() + "\n\n" + section + "\n"

new_header = update_section(
    header_path.read_text(), header_body, header=True
)
new_source = update_section(
    source_path.read_text(), source_body
)

header_path.write_text(new_header)
source_path.write_text(new_source)

print("Exported GELU tanh/factor parameters")
print("LUT entries:", lut.size)
print("LUT payload:", lut.nbytes, "bytes")
print("Integer one:", GELU_ONE_Q)
print("Factor multiplier:", multiplier)
print("Factor shift:", shift)
print("Factor output zero point:", output_zero_point)
print("File:", header_path)
print("File:", source_path)

Exported GELU tanh/factor parameters
LUT entries: 256
LUT payload: 1024 bytes
Integer one: 16777216
Factor multiplier: 1269488229
Factor shift: 47
Factor output zero point: -128
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.c


In [56]:
from pathlib import Path

project_root = Path("/home/randy/Documents/ACompArc/model/project")
header_path = (
    project_root
    / "experiments/integer_inference/params/block0_params.h"
)

assert header_path.is_file(), header_path

half_zero_point = int(gelu_quant_specs[92]["zero_point"])
factor_zero_point = int(gelu_quant_specs[103]["zero_point"])
output_zero_point = int(gelu_quant_specs[104]["zero_point"])

multiplier = int(GELU_OP70_Q31_MULTIPLIER)
second_shift = int(GELU_OP70_SECOND_RIGHT_SHIFT)

assert half_zero_point == 37
assert factor_zero_point == -128
assert output_zero_point == -88
assert multiplier == 1195729761
assert second_shift == 6

body = f"""#include <stdint.h>

/* Op 70: centered T92 * centered T103, then two-stage requantization. */
#define BLOCK0_GELU_OUTPUT_HALF_ZERO_POINT ({half_zero_point})
#define BLOCK0_GELU_OUTPUT_FACTOR_ZERO_POINT ({factor_zero_point})
#define BLOCK0_GELU_OUTPUT_Q31_MULTIPLIER INT32_C({multiplier})
#define BLOCK0_GELU_OUTPUT_SECOND_SHIFT {second_shift}u
#define BLOCK0_GELU_OUTPUT_ZERO_POINT ({output_zero_point})"""

begin = "/* BEGIN GENERATED BLOCK0 GELU OUTPUT PARAMETERS */"
end = "/* END GENERATED BLOCK0 GELU OUTPUT PARAMETERS */"
section = f"{begin}\n{body}\n{end}"

text = header_path.read_text()

if begin in text or end in text:
    assert text.count(begin) == text.count(end) == 1
    start = text.index(begin)
    finish = text.index(end) + len(end)
    assert text.index(end) > start

    updated = text[:start] + section + text[finish:]
else:
    position = text.rfind("#endif")
    assert position >= 0

    updated = (
        text[:position]
        + section
        + "\n\n"
        + text[position:]
    )

header_path.write_text(updated)

print("Exported Op 70 parameters")
print("Half-input zero point:", half_zero_point)
print("Factor zero point:", factor_zero_point)
print("Q31 multiplier:", multiplier)
print("First rounding shift: 31")
print("Second rounding shift:", second_shift)
print("Output zero point:", output_zero_point)
print("File:", header_path)

Exported Op 70 parameters
Half-input zero point: 37
Factor zero point: -128
Q31 multiplier: 1195729761
First rounding shift: 31
Second rounding shift: 6
Output zero point: -88
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h


In [57]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
fixture_dir = (
    project_root / "experiments/integer_inference/fixtures/block0"
)
assert fixture_dir.is_dir(), fixture_dir

names = [case["name"] for case in op0_integer_trial]
assert len(names) == 8

trials = [
    mlp_expand_cases,
    mlp_expand_integer_trial,
    gelu_cubic_integer_trial,
    gelu_argument_integer_trial,
    gelu_tanh_integer_trial,
    gelu_output_two_round_trial,
    ln2_affine_integer_trial,
]

for trial in trials:
    assert [case["name"] for case in trial] == names

assert GELU_OP70_Q31_MULTIPLIER == 1195729761
assert GELU_OP70_SECOND_RIGHT_SHIFT == 6

for prepared, ln2_case in zip(
    mlp_expand_cases, ln2_affine_integer_trial
):
    np.testing.assert_array_equal(
        prepared["input_chained"],
        ln2_case["paths"]["chained"]["t88_q"],
    )

# C-visible tensor boundaries and the projection accumulator.
fields = [
    ("t88", "int8_t", np.int8, 4096),
    ("accumulator", "int32_t", np.int32, 16384),
    ("t89", "int8_t", np.int8, 16384),
    ("t92", "int8_t", np.int8, 16384),
    ("t93", "int32_t", np.int32, 16384),
    ("t94", "int8_t", np.int8, 16384),
    ("t96", "int8_t", np.int8, 16384),
    ("t97", "int8_t", np.int8, 16384),
    ("t99", "int8_t", np.int8, 16384),
    ("t101", "int32_t", np.int32, 16384),
    ("t102", "int32_t", np.int32, 16384),
    ("t103", "int8_t", np.int8, 16384),
    ("t104", "int8_t", np.int8, 16384),
]

def get_record(c, path):
    prepared = mlp_expand_cases[c]
    cubic = gelu_cubic_integer_trial[c]["paths"][path]
    argument = gelu_argument_integer_trial[c]["paths"][path]
    tanh = gelu_tanh_integer_trial[c]["paths"][path]

    return {
        "t88": prepared[
            "input_ref" if path == "local" else "input_chained"
        ],
        "accumulator": prepared[f"acc_{path}"],
        "t89": mlp_expand_integer_trial[c]["paths"][path]["t89_q"],
        "t92": cubic["t92_q"],
        "t93": cubic["t93_q"],
        "t94": cubic["t94_q"],
        "t96": cubic["t96_q"],
        "t97": argument["t97_q"],
        "t99": argument["t99_q"],
        "t101": tanh["t101_q"],
        "t102": tanh["t102_q"],
        "t103": tanh["t103_q"],
        # Deliberately select the two-stage rounding candidate.
        "t104": gelu_output_two_round_trial[c]["paths"][path]["t104_q"],
    }

def initializer(values, indent=""):
    flat = np.asarray(values).reshape(-1)
    rows = [
        indent + "    " + ", ".join(
            str(int(v)) for v in flat[i:i + 16]
        ) + ","
        for i in range(0, flat.size, 16)
    ]
    return "{\n" + "\n".join(rows) + "\n" + indent + "}"

header_lines = [
    "#ifndef BLOCK0_MLP_GELU_GOLDEN_H",
    "#define BLOCK0_MLP_GELU_GOLDEN_H",
    "",
    "#include <stdint.h>",
    "",
    "#define BLOCK0_MLP_GELU_CASE_COUNT 8u",
    "#define BLOCK0_MLP_GELU_INPUT_ELEMENTS 4096u",
    "#define BLOCK0_MLP_GELU_ELEMENTS 16384u",
    "",
    "typedef struct {",
]

for field, c_type, _, count in fields:
    header_lines.append(f"    {c_type} {field}[{count}];")

header_lines.extend([
    "} Block0MlpGeluGolden;",
    "",
    "extern const int8_t block0_mlp_gelu_t0[8][4096];",
    "extern const Block0MlpGeluGolden block0_mlp_gelu_local[8];",
    "extern const Block0MlpGeluGolden block0_mlp_gelu_chained[8];",
    "",
    "#endif",
    "",
])

t0 = np.stack([
    case["t0_q"].reshape(-1) for case in op0_integer_trial
])
assert t0.dtype == np.int8 and t0.shape == (8, 4096)

sections = [
    '#include "mlp_gelu_golden.h"',
    "const int8_t block0_mlp_gelu_t0[8][4096] = {\n"
    + ",\n".join(
        "    " + initializer(row, "    ") for row in t0
    )
    + "\n};",
]

payload = t0.nbytes

for path in ("local", "chained"):
    records = []

    for c in range(len(names)):
        record = get_record(c, path)
        lines = ["    {"]

        for field, _, dtype, count in fields:
            values = np.asarray(record[field]).reshape(-1)

            assert values.dtype == np.dtype(dtype), (
                path, field, values.dtype
            )
            assert values.size == count, (
                path, field, values.shape
            )

            payload += values.nbytes
            lines.append(
                f"        .{field} = "
                + initializer(values, "        ")
                + ","
            )

        lines.append("    }")
        records.append("\n".join(lines))

    sections.append(
        "const Block0MlpGeluGolden "
        f"block0_mlp_gelu_{path}[8] = {{\n"
        + ",\n".join(records)
        + "\n};"
    )

header_path = fixture_dir / "mlp_gelu_golden.h"
source_path = fixture_dir / "mlp_gelu_golden.c"

header_path.write_text("\n".join(header_lines))
source_path.write_text("\n\n".join(sections) + "\n")

print("Exported Part 6 cases:", len(names))
print("Golden payload:", payload, "bytes")
print("T104 source: gelu_output_two_round_trial")
print(
    "First chained T104 values:",
    get_record(0, "chained")["t104"].reshape(-1)[:8],
)
print("File:", header_path)
print("File:", source_path)

Exported Part 6 cases: 8
Golden payload: 6389760 bytes
T104 source: gelu_output_two_round_trial
First chained T104 values: [ -24  -42  -50 -108  -74  -42  -81  -42]
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/mlp_gelu_golden.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/mlp_gelu_golden.c


In [58]:
from pathlib import Path
import numpy as np

project_root = Path("/home/randy/Documents/ACompArc/model/project")
base = project_root / "experiments/integer_inference"

params_h = base / "params/block0_params.h"
params_c = base / "params/block0_params.c"
golden_h = base / "fixtures/block0/mlp_out_residual_golden.h"
golden_c = base / "fixtures/block0/mlp_out_residual_golden.c"

assert params_h.is_file()
assert params_c.is_file()
assert golden_h.parent.is_dir()

names = [case["name"] for case in op0_integer_trial]
assert len(names) == 8

for trial in (
    mlp_output_cases,
    block0_integer_trial,
    gelu_output_two_round_trial,
    out_residual_integer_trial,
):
    assert [case["name"] for case in trial] == names

spec = mlp_output_spec
weights = np.asarray(spec["weights_q"])
bias = np.asarray(spec["bias_q"])

assert weights.dtype == np.int8
assert weights.shape == (128, 512)
assert bias.dtype == np.int32
assert bias.shape == (128,)
assert np.all(np.asarray(spec["weight_zero_points"]) == 0)

multipliers = np.asarray(mlp_out_multiplier_q)
assert multipliers.shape == (128,)
assert np.all(multipliers == 641059315)

assert spec["input_zero_point"] == -88
assert spec["output_zero_point"] == 10
assert MLP_OUT_REQUANT_SHIFT == 39
assert T106_MULTIPLIER_Q == 720118272
assert T106_REQUANT_SHIFT == 22
assert T73_SCALE == T106_SCALE == T107_SCALE == 2.0 ** -16

def initializer(values, indent=""):
    flat = np.asarray(values).reshape(-1)
    rows = [
        indent + "    " + ", ".join(
            str(int(v)) for v in flat[i:i + 16]
        ) + ","
        for i in range(0, flat.size, 16)
    ]
    return "{\n" + "\n".join(rows) + "\n" + indent + "}"

def patch_section(text, body, *, header=False):
    begin = "/* BEGIN GENERATED BLOCK0 MLP OUT PARAMETERS */"
    end = "/* END GENERATED BLOCK0 MLP OUT PARAMETERS */"
    section = f"{begin}\n{body}\n{end}"

    if begin in text or end in text:
        assert text.count(begin) == text.count(end) == 1
        start = text.index(begin)
        finish = text.index(end) + len(end)
        assert text.index(end) > start
        return text[:start] + section + text[finish:]

    if header:
        position = text.rfind("#endif")
        assert position >= 0
        return text[:position] + section + "\n\n" + text[position:]

    return text.rstrip() + "\n\n" + section + "\n"

parameter_header = """#include <stdint.h>

#define BLOCK0_MLP_OUT_INPUT_FEATURES 512u
#define BLOCK0_MLP_OUT_OUTPUT_FEATURES 128u
#define BLOCK0_MLP_OUT_WEIGHT_COUNT 65536u

#define BLOCK0_MLP_OUT_INPUT_ZERO_POINT (-88)
#define BLOCK0_MLP_OUT_MULTIPLIER INT64_C(641059315)
#define BLOCK0_MLP_OUT_SHIFT 39u
#define BLOCK0_MLP_OUT_OUTPUT_ZERO_POINT 10

#define BLOCK0_RESIDUAL2_BRANCH_MULTIPLIER INT64_C(720118272)
#define BLOCK0_RESIDUAL2_BRANCH_SHIFT 22u
#define BLOCK0_RESIDUAL2_FRAC_BITS 16u

extern const int8_t block0_mlp_out_weights[
    BLOCK0_MLP_OUT_WEIGHT_COUNT
];

extern const int32_t block0_mlp_out_bias[
    BLOCK0_MLP_OUT_OUTPUT_FEATURES
];"""

parameter_source = (
    "const int8_t block0_mlp_out_weights[\n"
    "    BLOCK0_MLP_OUT_WEIGHT_COUNT\n"
    "] = "
    + initializer(weights)
    + ";\n\n"
    + "const int32_t block0_mlp_out_bias[\n"
    "    BLOCK0_MLP_OUT_OUTPUT_FEATURES\n"
    "] = "
    + initializer(bias)
    + ";"
)

fields = [
    ("t104", "int8_t", np.int8, 16384),
    ("t73", "int32_t", np.int32, 4096),
    ("accumulator", "int32_t", np.int32, 4096),
    ("t105", "int8_t", np.int8, 4096),
    ("t106", "int32_t", np.int32, 4096),
    ("t107", "int32_t", np.int32, 4096),
]

header_lines = [
    "#ifndef BLOCK0_MLP_OUT_RESIDUAL_GOLDEN_H",
    "#define BLOCK0_MLP_OUT_RESIDUAL_GOLDEN_H",
    "",
    "#include <stdint.h>",
    "",
    "#define BLOCK0_MLP_OUT_RESIDUAL_CASE_COUNT 8u",
    "#define BLOCK0_MLP_OUT_RESIDUAL_INPUT_ELEMENTS 16384u",
    "#define BLOCK0_MLP_OUT_RESIDUAL_ELEMENTS 4096u",
    "",
    "typedef struct {",
]

for field, c_type, _, count in fields:
    header_lines.append(f"    {c_type} {field}[{count}];")

header_lines.extend([
    "} Block0MlpOutResidualGolden;",
    "",
    "extern const int8_t block0_mlp_out_residual_t0[8][4096];",
    "extern const Block0MlpOutResidualGolden",
    "    block0_mlp_out_residual_local[8];",
    "extern const Block0MlpOutResidualGolden",
    "    block0_mlp_out_residual_chained[8];",
    "",
    "#endif",
    "",
])

t0 = np.stack([
    case["t0_q"].reshape(-1) for case in op0_integer_trial
])
assert t0.dtype == np.int8 and t0.shape == (8, 4096)

sections = [
    '#include "mlp_out_residual_golden.h"',
    "const int8_t block0_mlp_out_residual_t0[8][4096] = {\n"
    + ",\n".join(
        "    " + initializer(row, "    ") for row in t0
    )
    + "\n};",
]

payload = t0.nbytes

for path in ("local", "chained"):
    records = []

    for c in range(len(names)):
        prepared = mlp_output_cases[c]
        result = block0_integer_trial[c]["paths"][path]
        gelu_case = gelu_output_two_round_trial[c]

        if path == "local":
            # Same encoding used by the Python Part 7 candidate.
            rounded = np.rint(
                prepared["t73_ref"].astype(np.float64) / T73_SCALE
            )
            assert rounded.min() >= np.iinfo(np.int32).min
            assert rounded.max() <= np.iinfo(np.int32).max
            residual = rounded.astype(np.int32)

            input_q = gelu_case["t104_ref"]
        else:
            residual = prepared["t73_chained_q"]
            input_q = gelu_case["paths"]["chained"]["t104_q"]

            np.testing.assert_array_equal(
                residual,
                out_residual_integer_trial[c]["paths"]["chained"]["t73_q"],
            )

        record = {
            "t104": input_q,
            "t73": residual,
            "accumulator": prepared[f"acc_{path}"],
            "t105": result["t105_q"],
            "t106": result["t106_q"],
            "t107": result["t107_q"],
        }

        lines = ["    {"]

        for field, _, dtype, count in fields:
            values = np.asarray(record[field]).reshape(-1)

            assert values.dtype == np.dtype(dtype), (
                path, field, values.dtype
            )
            assert values.size == count, (
                path, field, values.shape
            )

            payload += values.nbytes
            lines.append(
                f"        .{field} = "
                + initializer(values, "        ")
                + ","
            )

        lines.append("    }")
        records.append("\n".join(lines))

    sections.append(
        "const Block0MlpOutResidualGolden "
        f"block0_mlp_out_residual_{path}[8] = {{\n"
        + ",\n".join(records)
        + "\n};"
    )

updates = {
    params_h: patch_section(
        params_h.read_text(), parameter_header, header=True
    ),
    params_c: patch_section(
        params_c.read_text(), parameter_source
    ),
    golden_h: "\n".join(header_lines),
    golden_c: "\n\n".join(sections) + "\n",
}

for path, contents in updates.items():
    path.write_text(contents)
    print("File:", path)

print("Exported Part 7 cases:", len(names))
print("Weight + bias payload:", weights.nbytes + bias.nbytes, "bytes")
print("Golden payload:", payload, "bytes")
print(
    "First chained T107 values:",
    block0_integer_trial[0]["paths"]["chained"][
        "t107_q"
    ].reshape(-1)[:8],
)

File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/params/block0_params.c
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/mlp_out_residual_golden.h
File: /home/randy/Documents/ACompArc/model/project/experiments/integer_inference/fixtures/block0/mlp_out_residual_golden.c
Exported Part 7 cases: 8
Weight + bias payload: 66048 bytes
Golden payload: 1409024 bytes
First chained T107 values: [ 8088  3085 -2651  5653  1447  4513  -317 -1807]
